In [1]:
import os
import re
import copy
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.multiprocessing as mp
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

mp.set_sharing_strategy("file_system")

# =========================
# Configuration
# =========================
BASE_DIR = "./"

DATA_PATHS = [
    os.path.join(BASE_DIR, "data_physics_with_variances_total_v2.csv"),
]

NROWS_PER_DATASET = 5_000_000

BEST_MODEL_SAVE_PATH = os.path.join(
    BASE_DIR,
    "random_fulltrain_highber_best_v1_target020_040_transformer_multitask.pth",
)
LAST_MODEL_SAVE_PATH = os.path.join(
    BASE_DIR,
    "random_fulltrain_highber_last_v1_target020_040_transformer_multitask.pth",
)
SCALER_SAVE_PATH = os.path.join(
    BASE_DIR,
    "random_fulltrain_highber_scalers_v1_target020_040_transformer_multitask.pkl",
)

EPS = 1e-12
LOG10_HALF = float(np.log10(0.5))

ORDINAL_THRESHOLDS = [
    1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1,
    0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45,
]
NUM_ORDINAL_THRESHOLDS = len(ORDINAL_THRESHOLDS)
NUM_REGION_CLASSES = NUM_ORDINAL_THRESHOLDS + 1


# =========================
# Utilities
# =========================
def get_sorted_seq_cols(columns, prefix):
    pattern = re.compile(rf"^{re.escape(prefix)}_(\d+)$")
    matched = []
    for col in columns:
        m = pattern.match(col)
        if m:
            matched.append((int(m.group(1)), col))
    matched.sort(key=lambda x: x[0])
    return [col for _, col in matched]


def make_strat_bins(y_log, n_bins=10):
    y_flat = y_log.reshape(-1)
    quantiles = np.linspace(0, 1, n_bins + 1)
    edges = np.quantile(y_flat, quantiles)
    edges = np.unique(edges)

    if len(edges) < 3:
        return None

    bins = np.digitize(y_flat, edges[1:-1], right=True)
    counts = np.bincount(bins)
    if np.any(counts < 2):
        return None
    return bins


def has_nonfinite_tensor(x):
    return not torch.isfinite(x).all().item()


def validate_required_columns(df, file_path):
    if "mem_len" not in df.columns:
        raise ValueError(f"Required column 'mem_len' not found in {file_path}")
    if "N" not in df.columns:
        raise ValueError(f"Required column 'N' not found in {file_path}")

    tap_cols = get_sorted_seq_cols(df.columns, "tap")
    var_cols = get_sorted_seq_cols(df.columns, "var")

    if not tap_cols:
        raise ValueError(f"No tap_* columns found in {file_path}")
    if not var_cols:
        raise ValueError(f"No var_* columns found in {file_path}")
    if len(tap_cols) != len(var_cols):
        raise ValueError(
            f"tap/var length mismatch in {file_path}: "
            f"{len(tap_cols)} tap cols vs {len(var_cols)} var cols"
        )

    required_cols = tap_cols + var_cols + ["threshold", "BER", "N"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in {file_path}: {missing}")

    return tap_cols, var_cols


def load_and_merge_data(csv_paths, nrows_per_dataset):
    dfs = []
    reference_tap_cols = None
    reference_var_cols = None

    for path in csv_paths:
        if not os.path.exists(path):
            raise FileNotFoundError(f"Dataset not found: {path}")

        print(f"Loading up to {nrows_per_dataset:,} rows from: {path}")
        df = pd.read_csv(path, nrows=nrows_per_dataset)

        tap_cols, var_cols = validate_required_columns(df, path)

        if reference_tap_cols is None:
            reference_tap_cols = tap_cols
            reference_var_cols = var_cols
        else:
            if tap_cols != reference_tap_cols:
                raise ValueError("tap columns do not match across files.")
            if var_cols != reference_var_cols:
                raise ValueError("var columns do not match across files.")

        df["source_dataset"] = os.path.basename(path)
        dfs.append(df)

    merged = pd.concat(dfs, ignore_index=True)
    print(f"Combined rows before filtering: {len(merged):,}")

    return merged, reference_tap_cols, reference_var_cols


def y_log_to_raw_np(y_log):
    return np.clip(10 ** y_log, EPS, 0.5)


def raw_to_region_labels_np(y_raw):
    y = np.asarray(y_raw).reshape(-1)
    labels = np.searchsorted(np.asarray(ORDINAL_THRESHOLDS), y, side="right")
    return labels.astype(np.int64)


def region_labels_to_ordinal_targets_np(labels, num_thresholds=NUM_ORDINAL_THRESHOLDS):
    labels = np.asarray(labels).reshape(-1)
    thresholds = np.arange(1, num_thresholds + 1, dtype=np.int64)
    ordinal = (labels[:, None] >= thresholds[None, :]).astype(np.float32)
    return ordinal


def ordinal_logits_to_region_labels_torch(logits):
    probs = torch.sigmoid(logits)
    passed = (probs >= 0.5).sum(dim=1)
    return passed.long()


def compute_region_class_weights(labels_np, num_classes=NUM_REGION_CLASSES, max_weight=8.0):
    counts = np.bincount(labels_np.reshape(-1), minlength=num_classes).astype(np.float64)
    counts = np.maximum(counts, 1.0)
    weights = counts.sum() / (num_classes * counts)
    weights = np.sqrt(weights)
    weights = np.clip(weights, 1.0, max_weight)
    return weights.astype(np.float32)


def compute_ordinal_pos_weights_from_region_labels(
    region_labels_np,
    num_thresholds=NUM_ORDINAL_THRESHOLDS,
    max_weight=20.0,
):
    ordinal_targets = region_labels_to_ordinal_targets_np(region_labels_np, num_thresholds)
    pos_counts = ordinal_targets.sum(axis=0)
    neg_counts = ordinal_targets.shape[0] - pos_counts
    pos_counts = np.maximum(pos_counts, 1.0)
    pos_weight = neg_counts / pos_counts
    pos_weight = np.clip(pos_weight, 1.0, max_weight)
    return pos_weight.astype(np.float32)


# =========================
# Position-Independent Scaling
# =========================
class SharedFeatureScaler:
    """A scaler that stores a single (mean, std) per feature channel,
    shared across all sequence positions.

    For the first token (always position 0) we keep a separate scaler,
    since the current-symbol tap has a genuinely different distribution
    than the ISI taps at positions 1+.

    For all past positions we pool every valid entry into one distribution
    and fit a single mean/std.  This makes the scaler independent of
    sequence length — at inference time, any number of past taps can be
    scaled with the same parameters.
    """

    def __init__(self):
        self.first_mean = None  # shape [n_features]
        self.first_std = None
        self.past_mean = None   # shape [n_features]
        self.past_std = None

    def fit(self, first_data_list, past_data_list, past_valid_list):
        """
        Args:
            first_data_list: list of arrays, each [N, n_features] for the
                             first-token features (taps[:,0], vars[:,0], ...).
                             Concatenated across splits if desired, or just train.
            past_data_list:  list of arrays, each [N, L_past, n_features] or
                             [N, L_past] for a single feature channel.
            past_valid_list: list of bool arrays, each [N, L_past], True = valid.
        """
        # --- First token ---
        first_all = np.concatenate(first_data_list, axis=0)  # [N_total, F]
        self.first_mean = first_all.mean(axis=0).astype(np.float64)
        self.first_std = first_all.std(axis=0).astype(np.float64)
        self.first_std = np.maximum(self.first_std, 1e-12)

        # --- Past tokens (pool all valid entries per feature) ---
        valid_entries = []
        for data, valid in zip(past_data_list, past_valid_list):
            if data.ndim == 2:
                # single feature: [N, L] -> expand to [N, L, 1]
                data = data[:, :, None]
            # data: [N, L, F], valid: [N, L]
            valid_expanded = valid[:, :, None]  # [N, L, 1]
            # Gather valid entries: [?, F]
            valid_entries.append(data[np.broadcast_to(valid_expanded, data.shape)].reshape(-1, data.shape[-1]))

        pooled = np.concatenate(valid_entries, axis=0)  # [total_valid, F]
        self.past_mean = pooled.mean(axis=0).astype(np.float64)
        self.past_std = pooled.std(axis=0).astype(np.float64)
        self.past_std = np.maximum(self.past_std, 1e-12)

    def transform_first(self, first_data):
        """first_data: [N, F] -> scaled [N, F]"""
        return ((first_data - self.first_mean) / self.first_std).astype(np.float32)

    def transform_past(self, past_data, past_lens):
        """past_data: [N, L, F] or [N, L] -> scaled + re-zeroed [N, L, ...].
        past_lens: [N] int, number of valid past positions per row."""
        shape = past_data.shape
        if past_data.ndim == 2:
            scaled = ((past_data - self.past_mean[0]) / self.past_std[0]).astype(np.float32)
            L = shape[1]
        else:
            scaled = ((past_data - self.past_mean) / self.past_std).astype(np.float32)
            L = shape[1]

        # Re-zero padding positions
        valid = (np.arange(L)[None, :] < past_lens[:, None])  # [N, L]
        if scaled.ndim == 3:
            valid = valid[:, :, None]
        scaled = scaled * valid.astype(np.float32)
        return scaled


def fit_shared_scalar_scaler(train_data, train_valid_mask):
    """Fit a single-feature StandardScaler on pooled valid entries.
    train_data: [N, L], train_valid_mask: [N, L] bool.
    Returns (mean, std) as floats."""
    entries = train_data[train_valid_mask].reshape(-1)
    if len(entries) == 0:
        entries = train_data.reshape(-1)
    mean = float(entries.mean())
    std = float(max(entries.std(), 1e-12))
    return mean, std


def apply_shared_scale(data, mean, std, valid_mask=None):
    """Scale data with a single (mean, std), optionally re-zero invalid positions.
    data: [N, L] or [N, 1], valid_mask: [N, L] bool or None."""
    scaled = ((data - mean) / std).astype(np.float32)
    if valid_mask is not None and scaled.ndim == 2 and valid_mask.shape == scaled.shape:
        scaled = scaled * valid_mask.astype(np.float32)
    return scaled


# =========================
# Targeted Regression Loss
# =========================
class StableMultiObjectiveBERBoundedLogLoss(nn.Module):
    def __init__(
        self,
        log_delta=0.35,
        raw_delta=0.006,
        rel_delta=0.03,
        alpha_log=0.45,
        beta_raw=0.35,
        gamma_rel=0.20,
        use_regime_weights=True,
    ):
        super().__init__()
        self.log_delta = log_delta
        self.raw_delta = raw_delta
        self.rel_delta = rel_delta
        self.alpha_log = alpha_log
        self.beta_raw = beta_raw
        self.gamma_rel = gamma_rel
        self.use_regime_weights = use_regime_weights

    @staticmethod
    def huber_elementwise(pred, target, delta):
        err = pred - target
        abs_err = err.abs()
        return torch.where(
            abs_err < delta,
            0.5 * err * err,
            delta * (abs_err - 0.5 * delta),
        )

    @staticmethod
    def raw_to_pred_log(pred_raw_unconstrained):
        log10_half = torch.log10(
            torch.tensor(
                0.5,
                device=pred_raw_unconstrained.device,
                dtype=pred_raw_unconstrained.dtype,
            )
        )
        return log10_half - F.softplus(pred_raw_unconstrained)

    @staticmethod
    def raw_to_pred_ber(pred_raw_unconstrained):
        pred_log = StableMultiObjectiveBERBoundedLogLoss.raw_to_pred_log(
            pred_raw_unconstrained
        )
        return torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)

    def forward(self, pred_raw_unconstrained, target_log):
        pred_log = self.raw_to_pred_log(pred_raw_unconstrained)
        pred_raw = torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)

        target_raw = torch.pow(10.0, target_log).clamp(min=EPS, max=0.5)
        target_log_for_loss = torch.log10(target_raw)

        log_loss = self.huber_elementwise(pred_log, target_log_for_loss, self.log_delta)
        raw_loss = self.huber_elementwise(pred_raw, target_raw, self.raw_delta)
        rel_err = (pred_raw - target_raw) / torch.clamp(target_raw, min=1e-6)
        rel_loss = self.huber_elementwise(rel_err, torch.zeros_like(rel_err), self.rel_delta)

        total = (
            self.alpha_log * log_loss
            + self.beta_raw * raw_loss
            + self.gamma_rel * rel_loss
        )

        if self.use_regime_weights:
            weights = torch.ones_like(target_raw)
            weights = torch.where(
                (target_raw >= 1e-2) & (target_raw < 0.1),
                torch.full_like(weights, 1.5), weights,
            )
            weights = torch.where(
                (target_raw >= 0.1) & (target_raw < 0.15),
                torch.full_like(weights, 2.5), weights,
            )
            weights = torch.where(
                (target_raw >= 0.15) & (target_raw < 0.2),
                torch.full_like(weights, 3.0), weights,
            )
            weights = torch.where(
                (target_raw >= 0.2) & (target_raw < 0.3),
                torch.full_like(weights, 4.0), weights,
            )
            weights = torch.where(
                (target_raw >= 0.3) & (target_raw < 0.4),
                torch.full_like(weights, 4.5), weights,
            )
            weights = torch.where(
                (target_raw >= 0.4) & (target_raw < 0.45),
                torch.full_like(weights, 3.0), weights,
            )
            weights = torch.where(
                target_raw >= 0.45,
                torch.full_like(weights, 2.5), weights,
            )
            total = total * weights

        return total.mean()


class OrdinalBCELoss(nn.Module):
    def __init__(self, pos_weight=None, reduction="mean"):
        super().__init__()
        if pos_weight is not None and not isinstance(pos_weight, torch.Tensor):
            pos_weight = torch.tensor(pos_weight, dtype=torch.float32)
        self.register_buffer(
            "pos_weight", pos_weight if pos_weight is not None else None
        )
        self.reduction = reduction

    def forward(self, logits, ordinal_targets):
        return F.binary_cross_entropy_with_logits(
            logits,
            ordinal_targets,
            pos_weight=self.pos_weight,
            reduction=self.reduction,
        )


class MultiTaskBERLoss(nn.Module):
    def __init__(self, reg_loss, ord_loss, lambda_ord=0.25):
        super().__init__()
        self.reg_loss = reg_loss
        self.ord_loss = ord_loss
        self.lambda_ord = lambda_ord

    def forward(self, pred_raw_unconstrained, ord_logits, target_log, target_ord):
        reg = self.reg_loss(pred_raw_unconstrained, target_log)
        ordl = self.ord_loss(ord_logits, target_ord)
        total = reg + self.lambda_ord * ordl
        return total, reg.detach(), ordl.detach()


# =========================
# Set Transformer Blocks
# =========================
class SetSelfAttentionBlock(nn.Module):
    def __init__(self, d_model, num_heads, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norm2 = nn.LayerNorm(d_model)

        hidden = int(d_model * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x, key_padding_mask=None):
        """
        Args:
            x: [B, L, D]
            key_padding_mask: [B, L] bool, True = padding (ignore)
        """
        y = self.norm1(x)
        attn_out, _ = self.attn(
            y, y, y,
            need_weights=False,
            key_padding_mask=key_padding_mask,
        )
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x


class ConditionalFeatureModulation(nn.Module):
    def __init__(self, cond_dim, feat_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, 2 * feat_dim),
        )
        self.scale = nn.Parameter(torch.tensor(0.25, dtype=torch.float32))

    def forward(self, x, cond):
        gamma_beta = self.net(cond)
        gamma, beta = torch.chunk(gamma_beta, 2, dim=-1)

        if x.dim() == 3:
            gamma = gamma.unsqueeze(1)
            beta = beta.unsqueeze(1)

        return x * (1.0 + self.scale * gamma) + self.scale * beta


class SetAttentionPooling(nn.Module):
    def __init__(self, d_model, hidden_dim=128):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x, key_padding_mask=None):
        """
        Args:
            x: [B, L, D]
            key_padding_mask: [B, L] bool, True = padding
        Returns:
            pooled: [B, D]
        """
        logits = self.score(x)  # [B, L, 1]

        if key_padding_mask is not None:
            logits = logits.masked_fill(
                key_padding_mask.unsqueeze(-1), float("-inf")
            )

        weights = torch.softmax(logits, dim=1)  # [B, L, 1]
        # Safety: all-masked rows produce NaN from softmax(-inf); replace with 0
        weights = torch.nan_to_num(weights, nan=0.0)

        pooled = (weights * x).sum(dim=1)  # [B, D]
        return pooled


# =========================
# Model
# =========================
class BERFirstTokenSetTransformerBoundedLogHalfMultiTask(nn.Module):
    def __init__(
        self,
        max_past_seq_len,
        token_dim=4,
        first_token_dim=4,
        threshold_dim=1,
        global_dim=7,
        d_model=128,
        num_set_layers=4,
        num_heads=4,
        mlp_ratio=4.0,
        dropout=0.10,
        num_ordinal_thresholds=NUM_ORDINAL_THRESHOLDS,
    ):
        super().__init__()

        self.max_past_seq_len = max_past_seq_len
        self.token_dim = token_dim
        self.first_token_dim = first_token_dim
        self.d_model = d_model
        self.num_ordinal_thresholds = num_ordinal_thresholds

        self.first_proj = nn.Sequential(
            nn.Linear(first_token_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.token_proj = nn.Sequential(
            nn.Linear(token_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.thr_embed = nn.Sequential(
            nn.Linear(threshold_dim, 32),
            nn.GELU(),
            nn.Linear(32, 32),
            nn.GELU(),
        )

        self.global_embed = nn.Sequential(
            nn.Linear(global_dim, 96),
            nn.GELU(),
            nn.Linear(96, 96),
            nn.GELU(),
        )

        cond_dim = 32 + 96

        self.first_cond_mod = ConditionalFeatureModulation(
            cond_dim=cond_dim, feat_dim=d_model, hidden_dim=256,
        )
        self.set_cond_mod = ConditionalFeatureModulation(
            cond_dim=cond_dim, feat_dim=d_model, hidden_dim=256,
        )

        self.set_blocks = nn.ModuleList(
            [
                SetSelfAttentionBlock(
                    d_model=d_model,
                    num_heads=num_heads,
                    mlp_ratio=mlp_ratio,
                    dropout=dropout,
                )
                for _ in range(num_set_layers)
            ]
        )

        self.first_refine = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
        )

        self.final_set_norm = nn.LayerNorm(d_model)
        self.set_attn_pool = SetAttentionPooling(d_model=d_model, hidden_dim=128)

        set_summary_dim = 3 * d_model
        head_in = d_model + set_summary_dim + cond_dim

        self.reg_head = nn.Sequential(
            nn.Linear(head_in, 256),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(128, 1),
        )

        self.ord_head = nn.Sequential(
            nn.Linear(head_in, 256),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(128, num_ordinal_thresholds),
        )

    @staticmethod
    def raw_to_log10ber(pred_raw_unconstrained):
        log10_half = torch.log10(
            torch.tensor(
                0.5,
                device=pred_raw_unconstrained.device,
                dtype=pred_raw_unconstrained.dtype,
            )
        )
        return log10_half - F.softplus(pred_raw_unconstrained)

    @staticmethod
    def raw_to_ber(pred_raw_unconstrained):
        pred_log = BERFirstTokenSetTransformerBoundedLogHalfMultiTask.raw_to_log10ber(
            pred_raw_unconstrained
        )
        return torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)

    def forward(self, first_token, past_tokens, global_feats, threshold, key_padding_mask=None):
        """
        Args:
            first_token:       [B, first_token_dim]
            past_tokens:       [B, L, token_dim]  (L can vary between training and inference)
            global_feats:      [B, global_dim]
            threshold:         [B, 1]
            key_padding_mask:  [B, L] bool, True = padding position
        """
        thr = self.thr_embed(threshold)
        g = self.global_embed(global_feats)
        cond = torch.cat([thr, g], dim=-1)

        # First token path
        first_x = self.first_proj(first_token)
        first_x = self.first_cond_mod(first_x, cond)
        first_x = first_x + self.first_refine(first_x)

        # Past tokens path with mask
        set_x = self.token_proj(past_tokens)
        set_x = self.set_cond_mod(set_x, cond)

        for block in self.set_blocks:
            set_x = block(set_x, key_padding_mask=key_padding_mask)

        set_x = self.final_set_norm(set_x)

        # --- Masked pooling ---
        if key_padding_mask is not None:
            valid_mask = (~key_padding_mask).unsqueeze(-1).float()  # [B, L, 1]
        else:
            valid_mask = torch.ones(
                set_x.shape[0], set_x.shape[1], 1,
                device=set_x.device, dtype=set_x.dtype,
            )

        # Attention pooling
        pooled_attn = self.set_attn_pool(set_x, key_padding_mask=key_padding_mask)

        # Masked mean pooling
        valid_counts = valid_mask.sum(dim=1).clamp(min=1.0)  # [B, 1]
        pooled_mean = (set_x * valid_mask).sum(dim=1) / valid_counts

        # Masked max pooling
        if key_padding_mask is not None:
            x_for_max = set_x.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
        else:
            x_for_max = set_x
        pooled_max = x_for_max.amax(dim=1)
        pooled_max = torch.nan_to_num(pooled_max, nan=0.0, posinf=0.0, neginf=0.0)

        set_summary = torch.cat([pooled_attn, pooled_mean, pooled_max], dim=-1)
        fused = torch.cat([first_x, set_summary, cond], dim=-1)

        pred_raw_unconstrained = self.reg_head(fused)
        ord_logits = self.ord_head(fused)

        return pred_raw_unconstrained, ord_logits


# =========================
# Data
# =========================
def prepare_data(csv_paths, batch_size=256, nrows_per_dataset=2_500_000, num_workers=0):
    df, tap_cols, var_cols = load_and_merge_data(csv_paths, nrows_per_dataset)

    df = df[df["mem_len"] != 1].copy()

    df[tap_cols] = df[tap_cols].fillna(0.0)
    df[var_cols] = df[var_cols].fillna(0.0)
    df["threshold"] = df["threshold"].fillna(0.0)
    df["BER"] = df["BER"].fillna(0.0)
    df["N"] = df["N"].fillna(0.0)

    df = df[(df["threshold"] > 0) & (df["BER"] > 0) & (df["N"] > 0)].copy()
    df["BER"] = df["BER"].clip(lower=EPS, upper=0.5)

    print(f"Rows after cleaning/filtering: {len(df):,}")

    # ---- Extract raw arrays ----
    X_taps_raw = df[tap_cols].to_numpy(dtype=np.float32)
    X_vars_raw = df[var_cols].to_numpy(dtype=np.float32)
    num_molecules = df["N"].to_numpy(dtype=np.float32).reshape(-1, 1)
    mem_len = df["mem_len"].to_numpy(dtype=np.int64)

    X_thr_raw = df["threshold"].to_numpy(dtype=np.float32).reshape(-1, 1)
    y_raw = df["BER"].to_numpy(dtype=np.float32).reshape(-1, 1)

    X_thr = np.log10(X_thr_raw + EPS).astype(np.float32)
    y_log = np.log10(y_raw + EPS).astype(np.float32)

    # ---- Feature engineering ----
    X_taps_feat = (X_taps_raw * num_molecules).astype(np.float32)

    if np.any(X_vars_raw < 0):
        raise ValueError("Variance columns contain negative values.")

    X_vars_feat = X_vars_raw.astype(np.float32)
    abs_taps_raw = np.abs(X_taps_feat).astype(np.float32)
    snr_raw = np.log10((X_taps_feat ** 2) / (X_vars_raw + EPS) + EPS).astype(np.float32)

    L_full = X_taps_feat.shape[1]

    # ---- Validity masks ----
    mem_len_clamped = np.minimum(mem_len, L_full)
    valid_full = (np.arange(L_full)[None, :] < mem_len_clamped[:, None])  # [N, L_full]
    valid_past = valid_full[:, 1:]  # [N, L_full-1]
    valid_past_float = valid_past.astype(np.float32)
    past_lens = np.maximum(mem_len_clamped - 1, 0).astype(np.int64)

    # ---- Mask-aware global features ----
    first_mean = X_taps_feat[:, 0:1]
    past_means = X_taps_feat[:, 1:]
    first_var = X_vars_feat[:, 0:1]
    past_vars = X_vars_feat[:, 1:]

    past_means_masked = past_means * valid_past_float
    past_vars_masked = past_vars * valid_past_float

    mu0_raw = (0.5 * np.sum(past_means_masked, axis=1, keepdims=True)).astype(np.float32)
    mu1_raw = (first_mean + mu0_raw).astype(np.float32)

    var0_raw = (0.5 * np.sum(past_vars_masked, axis=1, keepdims=True)).astype(np.float32)
    var1_raw = (first_var + var0_raw).astype(np.float32)

    std0_raw = np.sqrt(np.maximum(var0_raw, EPS)).astype(np.float32)
    std1_raw = np.sqrt(np.maximum(var1_raw, EPS)).astype(np.float32)

    z0_raw = ((X_thr_raw - mu0_raw) / (std0_raw + EPS)).astype(np.float32)
    z1_raw = ((mu1_raw - X_thr_raw) / (std1_raw + EPS)).astype(np.float32)

    harmonic_side_z_raw = (
        2.0 / (1.0 / (z0_raw + EPS) + 1.0 / (z1_raw + EPS))
    ).astype(np.float32)
    abs_diff_side_z_raw = np.abs(z0_raw - z1_raw).astype(np.float32)
    harmonic_minus_gap_raw = (
        harmonic_side_z_raw - 0.25 * abs_diff_side_z_raw
    ).astype(np.float32)

    # ---- Scenario-level features (threshold-INDEPENDENT) ----
    # These capture intrinsic difficulty of the physical scenario.

    # 1. NSID: Normalized Signal-Interference Difference, range ~ [-1, +1]
    signal_raw = first_mean  # P_0 * N, shape [N, 1]
    isi_raw = np.sum(past_means_masked, axis=1, keepdims=True)  # [N, 1]
    nsid_raw = ((signal_raw - isi_raw) / (signal_raw + isi_raw + EPS)).astype(np.float32)

    # 2. Log Harmonic Discriminability (threshold-free)
    gap_raw = signal_raw  # mu1 - mu0 = P_0 * N
    d0_raw = (gap_raw / (std0_raw + EPS)).astype(np.float32)
    d1_raw = (gap_raw / (std1_raw + EPS)).astype(np.float32)
    harmonic_discrim_raw = (2.0 * d0_raw * d1_raw / (d0_raw + d1_raw + EPS)).astype(np.float32)
    log_harmonic_discrim_raw = np.log10(harmonic_discrim_raw + EPS).astype(np.float32)

    # 3. Discriminability Asymmetry
    discrim_asymmetry_raw = (np.abs(d0_raw - d1_raw) / (d0_raw + d1_raw + EPS)).astype(np.float32)

    # 4. Herfindahl Index of ISI concentration [1/n_past, 1]
    past_taps_sum = np.sum(past_means_masked, axis=1, keepdims=True)  # [N, 1]
    past_shares = past_means_masked / (past_taps_sum + EPS)           # [N, L_past]
    herfindahl_raw = np.sum(past_shares ** 2, axis=1, keepdims=True).astype(np.float32)  # [N, 1]

    # ---- Labels ----
    region_labels = raw_to_region_labels_np(y_raw)
    ordinal_targets = region_labels_to_ordinal_targets_np(region_labels)
    strat_labels = make_strat_bins(y_log, n_bins=10)

    # ---- Train / temp split ----
    split_args = dict(test_size=0.30, random_state=42)
    if strat_labels is not None:
        split_args["stratify"] = strat_labels

    (
        t_taps_feat, temp_taps_feat,
        t_vars_feat, temp_vars_feat,
        t_abs_raw, temp_abs_raw,
        t_snr_raw, temp_snr_raw,
        t_z0_raw, temp_z0_raw,
        t_z1_raw, temp_z1_raw,
        t_hmg_raw, temp_hmg_raw,
        t_nsid_raw, temp_nsid_raw,
        t_log_hd_raw, temp_log_hd_raw,
        t_da_raw, temp_da_raw,
        t_herf_raw, temp_herf_raw,
        t_thr, temp_thr,
        t_y_log, temp_y_log,
        t_region, temp_region,
        t_ord, temp_ord,
        t_past_lens, temp_past_lens,
    ) = train_test_split(
        X_taps_feat, X_vars_feat, abs_taps_raw, snr_raw,
        z0_raw, z1_raw, harmonic_minus_gap_raw,
        nsid_raw, log_harmonic_discrim_raw, discrim_asymmetry_raw, herfindahl_raw,
        X_thr,
        y_log, region_labels, ordinal_targets, past_lens,
        **split_args,
    )

    # ---- temp -> val / test split ----
    temp_strat = make_strat_bins(temp_y_log, n_bins=6)
    split_args2 = dict(test_size=0.50, random_state=42)
    if temp_strat is not None:
        split_args2["stratify"] = temp_strat

    (
        v_taps_feat, te_taps_feat,
        v_vars_feat, te_vars_feat,
        v_abs_raw, te_abs_raw,
        v_snr_raw, te_snr_raw,
        v_z0_raw, te_z0_raw,
        v_z1_raw, te_z1_raw,
        v_hmg_raw, te_hmg_raw,
        v_nsid_raw, te_nsid_raw,
        v_log_hd_raw, te_log_hd_raw,
        v_da_raw, te_da_raw,
        v_herf_raw, te_herf_raw,
        v_thr, te_thr,
        v_y_log, te_y_log,
        v_region, te_region,
        v_ord, te_ord,
        v_past_lens, te_past_lens,
    ) = train_test_split(
        temp_taps_feat, temp_vars_feat, temp_abs_raw, temp_snr_raw,
        temp_z0_raw, temp_z1_raw, temp_hmg_raw,
        temp_nsid_raw, temp_log_hd_raw, temp_da_raw, temp_herf_raw,
        temp_thr,
        temp_y_log, temp_region, temp_ord, temp_past_lens,
        **split_args2,
    )

    # =========================================================
    # Position-independent scaling
    # =========================================================
    # For each feature channel (taps, vars, abs, snr) we fit:
    #   - A SEPARATE mean/std for position 0 (first token)
    #   - A SHARED mean/std pooled across ALL valid past positions 1+
    #
    # This decouples the scaler from the number of columns,
    # so at inference any mem_len works with the same parameters.
    # =========================================================

    L_past = L_full - 1

    # Build per-split past validity masks
    t_valid_past = (np.arange(L_past)[None, :] < t_past_lens[:, None])

    # --- Fit first-token scalers (on train only) ---
    first_tap_mean, first_tap_std = float(t_taps_feat[:, 0].mean()), max(float(t_taps_feat[:, 0].std()), 1e-12)
    first_var_mean, first_var_std = float(t_vars_feat[:, 0].mean()), max(float(t_vars_feat[:, 0].std()), 1e-12)
    first_abs_mean, first_abs_std = float(t_abs_raw[:, 0].mean()), max(float(t_abs_raw[:, 0].std()), 1e-12)
    first_snr_mean, first_snr_std = float(t_snr_raw[:, 0].mean()), max(float(t_snr_raw[:, 0].std()), 1e-12)

    # --- Fit shared past scalers (pool all valid past entries from train) ---
    past_tap_mean, past_tap_std = fit_shared_scalar_scaler(t_taps_feat[:, 1:], t_valid_past)
    past_var_mean, past_var_std = fit_shared_scalar_scaler(t_vars_feat[:, 1:], t_valid_past)
    past_abs_mean, past_abs_std = fit_shared_scalar_scaler(t_abs_raw[:, 1:], t_valid_past)
    past_snr_mean, past_snr_std = fit_shared_scalar_scaler(t_snr_raw[:, 1:], t_valid_past)

    # --- Global feature scalers (standard, shape [N, 1]) ---
    z0_scaler = StandardScaler().fit(t_z0_raw)
    z1_scaler = StandardScaler().fit(t_z1_raw)
    hmg_scaler = StandardScaler().fit(t_hmg_raw)
    nsid_scaler = StandardScaler().fit(t_nsid_raw)
    log_hd_scaler = StandardScaler().fit(t_log_hd_raw)
    da_scaler = StandardScaler().fit(t_da_raw)
    herf_scaler = StandardScaler().fit(t_herf_raw)
    thr_scaler = StandardScaler().fit(t_thr)

    # ---- Helper: build first_token and past_tokens arrays ----
    def build_tokens(taps_feat, vars_feat, abs_raw, snr_raw, p_lens):
        """Returns first_token [N, 4] and past_tokens [N, L_past, 4], both scaled + re-zeroed."""
        N = taps_feat.shape[0]

        # Scale first token
        ft_tap = ((taps_feat[:, 0] - first_tap_mean) / first_tap_std).astype(np.float32)
        ft_var = ((vars_feat[:, 0] - first_var_mean) / first_var_std).astype(np.float32)
        ft_abs = ((abs_raw[:, 0] - first_abs_mean) / first_abs_std).astype(np.float32)
        ft_snr = ((snr_raw[:, 0] - first_snr_mean) / first_snr_std).astype(np.float32)
        first_token = np.stack([ft_tap, ft_var, ft_abs, ft_snr], axis=1).astype(np.float32)

        # Scale past tokens (shared scaler) + re-zero
        pt_tap = apply_shared_scale(taps_feat[:, 1:], past_tap_mean, past_tap_std,
                                    (np.arange(L_past)[None, :] < p_lens[:, None]))
        pt_var = apply_shared_scale(vars_feat[:, 1:], past_var_mean, past_var_std,
                                    (np.arange(L_past)[None, :] < p_lens[:, None]))
        pt_abs = apply_shared_scale(abs_raw[:, 1:], past_abs_mean, past_abs_std,
                                    (np.arange(L_past)[None, :] < p_lens[:, None]))
        pt_snr = apply_shared_scale(snr_raw[:, 1:], past_snr_mean, past_snr_std,
                                    (np.arange(L_past)[None, :] < p_lens[:, None]))

        past_tokens = np.stack([pt_tap, pt_var, pt_abs, pt_snr], axis=2).astype(np.float32)

        return first_token, past_tokens

    def build_global(z0, z1, hmg, nsid, log_hd, da, herf):
        return np.concatenate([
            z0_scaler.transform(z0),
            z1_scaler.transform(z1),
            hmg_scaler.transform(hmg),
            nsid_scaler.transform(nsid),
            log_hd_scaler.transform(log_hd),
            da_scaler.transform(da),
            herf_scaler.transform(herf),
        ], axis=1).astype(np.float32)

    def build_padding_mask(p_lens, L_max):
        return (np.arange(L_max)[None, :] >= p_lens[:, None])  # True = padding

    # ---- Build all splits ----
    t_first, t_past = build_tokens(t_taps_feat, t_vars_feat, t_abs_raw, t_snr_raw, t_past_lens)
    v_first, v_past = build_tokens(v_taps_feat, v_vars_feat, v_abs_raw, v_snr_raw, v_past_lens)
    te_first, te_past = build_tokens(te_taps_feat, te_vars_feat, te_abs_raw, te_snr_raw, te_past_lens)

    t_global = build_global(t_z0_raw, t_z1_raw, t_hmg_raw,
                            t_nsid_raw, t_log_hd_raw, t_da_raw, t_herf_raw)
    v_global = build_global(v_z0_raw, v_z1_raw, v_hmg_raw,
                            v_nsid_raw, v_log_hd_raw, v_da_raw, v_herf_raw)
    te_global = build_global(te_z0_raw, te_z1_raw, te_hmg_raw,
                             te_nsid_raw, te_log_hd_raw, te_da_raw, te_herf_raw)

    t_thr_s = thr_scaler.transform(t_thr).astype(np.float32)
    v_thr_s = thr_scaler.transform(v_thr).astype(np.float32)
    te_thr_s = thr_scaler.transform(te_thr).astype(np.float32)

    L_max_past = L_past
    t_mask = build_padding_mask(t_past_lens, L_max_past)
    v_mask = build_padding_mask(v_past_lens, L_max_past)
    te_mask = build_padding_mask(te_past_lens, L_max_past)

    # ---- TensorDatasets ----
    train_ds = TensorDataset(
        torch.from_numpy(t_first),
        torch.from_numpy(t_past),
        torch.from_numpy(t_global),
        torch.from_numpy(t_thr_s),
        torch.from_numpy(t_y_log.astype(np.float32)),
        torch.from_numpy(t_ord.astype(np.float32)),
        torch.from_numpy(t_region.astype(np.int64)),
        torch.from_numpy(t_mask),
    )
    val_ds = TensorDataset(
        torch.from_numpy(v_first),
        torch.from_numpy(v_past),
        torch.from_numpy(v_global),
        torch.from_numpy(v_thr_s),
        torch.from_numpy(v_y_log.astype(np.float32)),
        torch.from_numpy(v_ord.astype(np.float32)),
        torch.from_numpy(v_region.astype(np.int64)),
        torch.from_numpy(v_mask),
    )
    test_ds = TensorDataset(
        torch.from_numpy(te_first),
        torch.from_numpy(te_past),
        torch.from_numpy(te_global),
        torch.from_numpy(te_thr_s),
        torch.from_numpy(te_y_log.astype(np.float32)),
        torch.from_numpy(te_ord.astype(np.float32)),
        torch.from_numpy(te_region.astype(np.int64)),
        torch.from_numpy(te_mask),
    )

    # ---- Weighted sampler (region + SIR-aware) ----
    region_sample_weights = np.ones_like(t_region, dtype=np.float32)
    region_sample_weights[t_region == 6] = 2.5
    region_sample_weights[t_region == 7] = 3.0
    region_sample_weights[t_region == 8] = 5.0
    region_sample_weights[t_region == 9] = 5.0
    region_sample_weights[t_region == 10] = 5.0
    region_sample_weights[t_region == 11] = 5.0
    region_sample_weights[t_region == 12] = 3.0
    region_sample_weights[t_region == 13] = 2.5

    # SIR-aware weights (threshold-independent scenario difficulty)
    t_signal = t_taps_feat[:, 0]
    t_valid_past_for_sir = (np.arange(L_past)[None, :] < t_past_lens[:, None]).astype(np.float32)
    t_isi = np.sum(t_taps_feat[:, 1:] * t_valid_past_for_sir, axis=1)
    t_sir = t_signal / (t_isi + EPS)

    sir_sample_weights = np.ones_like(t_sir, dtype=np.float32)
    sir_sample_weights[t_sir < 0.26] = 3.0
    sir_sample_weights[(t_sir >= 0.26) & (t_sir < 0.36)] = 2.0

    combined_sample_weights = region_sample_weights * sir_sample_weights

    sampler = WeightedRandomSampler(
        weights=torch.from_numpy(combined_sample_weights),
        num_samples=len(combined_sample_weights),
        replacement=True,
    )

    pin_mem = torch.cuda.is_available()

    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler,
                              shuffle=False, pin_memory=pin_mem, num_workers=num_workers)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            pin_memory=pin_mem, num_workers=num_workers)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                             pin_memory=pin_mem, num_workers=num_workers)

    class_weights = compute_region_class_weights(t_region, NUM_REGION_CLASSES)
    ordinal_pos_weights = compute_ordinal_pos_weights_from_region_labels(
        t_region, num_thresholds=NUM_ORDINAL_THRESHOLDS, max_weight=20.0,
    )

    # ---- Save all scaler parameters (position-independent) ----
    scalers = {
        # First-token scalers (per feature channel)
        "first_tap_mean": first_tap_mean, "first_tap_std": first_tap_std,
        "first_var_mean": first_var_mean, "first_var_std": first_var_std,
        "first_abs_mean": first_abs_mean, "first_abs_std": first_abs_std,
        "first_snr_mean": first_snr_mean, "first_snr_std": first_snr_std,
        # Shared past scalers (single mean/std per feature, position-independent)
        "past_tap_mean": past_tap_mean, "past_tap_std": past_tap_std,
        "past_var_mean": past_var_mean, "past_var_std": past_var_std,
        "past_abs_mean": past_abs_mean, "past_abs_std": past_abs_std,
        "past_snr_mean": past_snr_mean, "past_snr_std": past_snr_std,
        # Global feature scalers (sklearn StandardScaler objects)
        "z0_scaler": z0_scaler,
        "z1_scaler": z1_scaler,
        "harmonic_minus_gap_scaler": hmg_scaler,
        "nsid_scaler": nsid_scaler,
        "log_hd_scaler": log_hd_scaler,
        "da_scaler": da_scaler,
        "herf_scaler": herf_scaler,
        "thr_scaler": thr_scaler,
        # Metadata
        "tap_cols": tap_cols,
        "var_cols": var_cols,
        "first_token_dim": 4,
        "past_token_dim": 4,
        "global_dim": 7,
        "train_max_past_seq_len": L_past,
        "scaling_strategy": "position_independent",
        "uses_positional_encoding": False,
        "permutation_invariance_post_first": True,
        "variable_length_support": True,
        "target_parameterization": "pred_log10_ber = log10(0.5) - softplus(raw_out)",
        "ordinal_thresholds": ORDINAL_THRESHOLDS,
        "num_region_classes": NUM_REGION_CLASSES,
        "class_weights": class_weights.tolist(),
        "ordinal_pos_weights": ordinal_pos_weights.tolist(),
        "data_paths": csv_paths,
        "nrows_per_dataset": nrows_per_dataset,
    }

    aux_info = {
        "class_weights": class_weights,
        "ordinal_pos_weights": ordinal_pos_weights,
        "t_region": t_region,
    }

    return train_loader, val_loader, test_loader, scalers, aux_info, L_past


# =========================
# Inference Helper
# =========================
def prepare_inference_batch(
    taps_raw_2d,
    vars_raw_2d,
    N_array,
    threshold_array,
    mem_len_array,
    scalers,
):
    """Prepare a batch for inference from raw numpy arrays.

    This handles ARBITRARY mem_len — the set can be larger than anything
    seen during training because scalers are position-independent.

    Args:
        taps_raw_2d:     [B, L] raw tap coefficients (padded to L with zeros)
        vars_raw_2d:     [B, L] raw variance values  (padded to L with zeros)
        N_array:         [B] or [B, 1] number of molecules
        threshold_array: [B] or [B, 1] raw threshold values
        mem_len_array:   [B] true memory length per sample
        scalers:         dict from joblib.load(SCALER_SAVE_PATH)

    Returns:
        first_token:      [B, 4] tensor
        past_tokens:      [B, L-1, 4] tensor
        global_feats:     [B, 7] tensor
        threshold_scaled: [B, 1] tensor
        key_padding_mask: [B, L-1] bool tensor (True = padding)
    """
    B, L = taps_raw_2d.shape
    N = N_array.reshape(-1, 1).astype(np.float32)
    thr_raw = threshold_array.reshape(-1, 1).astype(np.float32)
    mem_len = mem_len_array.reshape(-1).astype(np.int64)

    mem_len_clamped = np.minimum(mem_len, L)
    past_lens = np.maximum(mem_len_clamped - 1, 0).astype(np.int64)

    # Feature engineering (same as training)
    taps_feat = (taps_raw_2d * N).astype(np.float32)
    vars_feat = vars_raw_2d.astype(np.float32)
    abs_feat = np.abs(taps_feat).astype(np.float32)
    snr_feat = np.log10((taps_feat ** 2) / (vars_raw_2d + EPS) + EPS).astype(np.float32)

    L_past = L - 1

    # Validity masks
    valid_past = (np.arange(L_past)[None, :] < past_lens[:, None])
    valid_past_float = valid_past.astype(np.float32)

    # Global features (mask-aware)
    first_mean = taps_feat[:, 0:1]
    past_means = taps_feat[:, 1:] * valid_past_float
    first_var = vars_feat[:, 0:1]
    past_vars = vars_feat[:, 1:] * valid_past_float

    mu0 = (0.5 * past_means.sum(axis=1, keepdims=True)).astype(np.float32)
    mu1 = (first_mean + mu0).astype(np.float32)
    var0 = (0.5 * past_vars.sum(axis=1, keepdims=True)).astype(np.float32)
    var1 = (first_var + var0).astype(np.float32)

    std0 = np.sqrt(np.maximum(var0, EPS)).astype(np.float32)
    std1 = np.sqrt(np.maximum(var1, EPS)).astype(np.float32)

    z0 = ((thr_raw - mu0) / (std0 + EPS)).astype(np.float32)
    z1 = ((mu1 - thr_raw) / (std1 + EPS)).astype(np.float32)

    harmonic = (2.0 / (1.0 / (z0 + EPS) + 1.0 / (z1 + EPS))).astype(np.float32)
    abs_diff = np.abs(z0 - z1).astype(np.float32)
    hmg = (harmonic - 0.25 * abs_diff).astype(np.float32)

    # Scenario-level features (threshold-independent)
    signal = first_mean  # [B, 1]
    isi = past_means.sum(axis=1, keepdims=True)  # [B, 1]
    nsid = ((signal - isi) / (signal + isi + EPS)).astype(np.float32)

    gap = signal
    d0_inf = (gap / (std0 + EPS)).astype(np.float32)
    d1_inf = (gap / (std1 + EPS)).astype(np.float32)
    hd = (2.0 * d0_inf * d1_inf / (d0_inf + d1_inf + EPS)).astype(np.float32)
    log_hd = np.log10(hd + EPS).astype(np.float32)
    da = (np.abs(d0_inf - d1_inf) / (d0_inf + d1_inf + EPS)).astype(np.float32)

    past_taps_sum = past_means.sum(axis=1, keepdims=True)
    past_shares = past_means / (past_taps_sum + EPS)
    herf = (past_shares ** 2).sum(axis=1, keepdims=True).astype(np.float32)

    # Scale first token
    ft_tap = ((taps_feat[:, 0] - scalers["first_tap_mean"]) / scalers["first_tap_std"]).astype(np.float32)
    ft_var = ((vars_feat[:, 0] - scalers["first_var_mean"]) / scalers["first_var_std"]).astype(np.float32)
    ft_abs = ((abs_feat[:, 0] - scalers["first_abs_mean"]) / scalers["first_abs_std"]).astype(np.float32)
    ft_snr = ((snr_feat[:, 0] - scalers["first_snr_mean"]) / scalers["first_snr_std"]).astype(np.float32)
    first_token = np.stack([ft_tap, ft_var, ft_abs, ft_snr], axis=1).astype(np.float32)

    # Scale past tokens (position-independent shared scaler)
    pt_tap = apply_shared_scale(taps_feat[:, 1:], scalers["past_tap_mean"], scalers["past_tap_std"], valid_past)
    pt_var = apply_shared_scale(vars_feat[:, 1:], scalers["past_var_mean"], scalers["past_var_std"], valid_past)
    pt_abs = apply_shared_scale(abs_feat[:, 1:], scalers["past_abs_mean"], scalers["past_abs_std"], valid_past)
    pt_snr = apply_shared_scale(snr_feat[:, 1:], scalers["past_snr_mean"], scalers["past_snr_std"], valid_past)
    past_tokens = np.stack([pt_tap, pt_var, pt_abs, pt_snr], axis=2).astype(np.float32)

    # Scale globals (7 features)
    z0_s = scalers["z0_scaler"].transform(z0).astype(np.float32)
    z1_s = scalers["z1_scaler"].transform(z1).astype(np.float32)
    hmg_s = scalers["harmonic_minus_gap_scaler"].transform(hmg).astype(np.float32)
    nsid_s = scalers["nsid_scaler"].transform(nsid).astype(np.float32)
    log_hd_s = scalers["log_hd_scaler"].transform(log_hd).astype(np.float32)
    da_s = scalers["da_scaler"].transform(da).astype(np.float32)
    herf_s = scalers["herf_scaler"].transform(herf).astype(np.float32)
    global_feats = np.concatenate([z0_s, z1_s, hmg_s, nsid_s, log_hd_s, da_s, herf_s], axis=1).astype(np.float32)

    # Scale threshold
    thr_log = np.log10(thr_raw + EPS).astype(np.float32)
    thr_s = scalers["thr_scaler"].transform(thr_log).astype(np.float32)

    # Padding mask
    pad_mask = (np.arange(L_past)[None, :] >= past_lens[:, None])

    return (
        torch.from_numpy(first_token),
        torch.from_numpy(past_tokens),
        torch.from_numpy(global_feats),
        torch.from_numpy(thr_s),
        torch.from_numpy(pad_mask),
    )


def run_inference(model, taps_raw, vars_raw, N_arr, thr_arr, mem_len_arr, scalers, device):
    """End-to-end inference: raw arrays -> BER predictions.

    All inputs are numpy. Works with any mem_len, even values
    larger than max_past_seq_len seen during training.
    """
    first_tok, past_tok, glob, thr_s, pad_mask = prepare_inference_batch(
        taps_raw, vars_raw, N_arr, thr_arr, mem_len_arr, scalers,
    )

    model.eval()
    with torch.no_grad():
        first_tok = first_tok.to(device)
        past_tok = past_tok.to(device)
        glob = glob.to(device)
        thr_s = thr_s.to(device)
        pad_mask = pad_mask.to(device)

        pred_raw, ord_logits = model(first_tok, past_tok, glob, thr_s, key_padding_mask=pad_mask)

        pred_ber = model.raw_to_ber(pred_raw).cpu().numpy()
        pred_log = model.raw_to_log10ber(pred_raw).cpu().numpy()
        pred_region = ordinal_logits_to_region_labels_torch(ord_logits).cpu().numpy()

    return {
        "ber": pred_ber,
        "log10_ber": pred_log,
        "region": pred_region,
    }


# =========================
# Evaluation
# =========================
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_reg_loss = 0.0
    total_ord_loss = 0.0

    all_preds_log = []
    all_targets_log = []
    all_pred_regions = []
    all_true_regions = []

    with torch.no_grad():
        for b_first, b_past, b_global, b_thr, b_y_log, b_ord, b_region, b_mask in loader:
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_y_log = b_y_log.to(device, non_blocking=True)
            b_ord = b_ord.to(device, non_blocking=True)
            b_region = b_region.to(device, non_blocking=True)
            b_mask = b_mask.to(device, non_blocking=True)

            pred_raw_unconstrained, ord_logits = model(
                b_first, b_past, b_global, b_thr, key_padding_mask=b_mask
            )

            if has_nonfinite_tensor(pred_raw_unconstrained):
                raise RuntimeError("Non-finite prediction during evaluation.")
            if has_nonfinite_tensor(ord_logits):
                raise RuntimeError("Non-finite ordinal logits during evaluation.")

            loss, reg_loss, ord_loss = criterion(
                pred_raw_unconstrained, ord_logits, b_y_log, b_ord,
            )

            if has_nonfinite_tensor(loss):
                raise RuntimeError("Non-finite loss during evaluation.")

            pred_log = model.raw_to_log10ber(pred_raw_unconstrained)
            pred_region = ordinal_logits_to_region_labels_torch(ord_logits)

            total_loss += loss.item()
            total_reg_loss += reg_loss.item()
            total_ord_loss += ord_loss.item()

            all_preds_log.append(pred_log.cpu().numpy())
            all_targets_log.append(b_y_log.cpu().numpy())
            all_pred_regions.append(pred_region.cpu().numpy())
            all_true_regions.append(b_region.cpu().numpy())

    n_batches = max(len(loader), 1)
    avg_loss = total_loss / n_batches
    avg_reg_loss = total_reg_loss / n_batches
    avg_ord_loss = total_ord_loss / n_batches

    preds_log = np.vstack(all_preds_log)
    targets_log = np.vstack(all_targets_log)

    preds_raw = np.clip(10 ** preds_log, EPS, 0.5)
    targets_raw = np.clip(10 ** targets_log, EPS, 0.5)

    rmse_log = float(np.sqrt(np.mean((preds_log - targets_log) ** 2)))
    mae_log = float(np.mean(np.abs(preds_log - targets_log)))
    factor_error = float(10 ** rmse_log)

    rmse_raw = float(np.sqrt(np.mean((preds_raw - targets_raw) ** 2)))
    mae_raw = float(np.mean(np.abs(preds_raw - targets_raw)))

    rel_err = (preds_raw - targets_raw) / np.maximum(targets_raw, 1e-6)
    rmse_rel = float(np.sqrt(np.mean(rel_err ** 2)))
    mae_rel = float(np.mean(np.abs(rel_err)))

    pred_regions = np.concatenate(all_pred_regions).reshape(-1)
    true_regions = np.concatenate(all_true_regions).reshape(-1)

    region_acc = float(np.mean(pred_regions == true_regions))
    region_mae = float(np.mean(np.abs(pred_regions - true_regions)))

    return {
        "loss": avg_loss,
        "reg_loss": avg_reg_loss,
        "ord_loss": avg_ord_loss,
        "rmse_log": rmse_log,
        "mae_log": mae_log,
        "factor_error": factor_error,
        "rmse_raw": rmse_raw,
        "mae_raw": mae_raw,
        "rmse_rel": rmse_rel,
        "mae_rel": mae_rel,
        "region_acc": region_acc,
        "region_mae": region_mae,
    }


def evaluate_by_target_range(model, loader, device):
    model.eval()
    all_preds_log = []
    all_targets_log = []

    with torch.no_grad():
        for b_first, b_past, b_global, b_thr, b_y_log, _, _, b_mask in loader:
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_mask = b_mask.to(device, non_blocking=True)

            pred_raw_unconstrained, _ = model(
                b_first, b_past, b_global, b_thr, key_padding_mask=b_mask
            )

            if has_nonfinite_tensor(pred_raw_unconstrained):
                raise RuntimeError("Non-finite prediction during per-range evaluation.")

            pred_log = model.raw_to_log10ber(pred_raw_unconstrained)

            all_preds_log.append(pred_log.cpu().numpy())
            all_targets_log.append(b_y_log.numpy())

    preds_log = np.vstack(all_preds_log).reshape(-1)
    targets_log = np.vstack(all_targets_log).reshape(-1)

    preds_raw = np.clip(10 ** preds_log, EPS, 0.5)
    targets_raw = np.clip(10 ** targets_log, EPS, 0.5)

    ranges = {
        "low_BER(y<1e-6)": targets_raw < 1e-6,
        "mid_BER(1e-6<=y<1e-3)": (targets_raw >= 1e-6) & (targets_raw < 1e-3),
        "high_BER(1e-3<=y<0.10)": (targets_raw >= 1e-3) & (targets_raw < 0.10),
        "upper_BER(0.10<=y<0.20)": (targets_raw >= 0.10) & (targets_raw < 0.20),
        "target_BER(0.20<=y<0.40)": (targets_raw >= 0.20) & (targets_raw < 0.40),
        "very_high_BER(0.40<=y<=0.50)": targets_raw >= 0.40,
    }

    metrics = {}
    for name, mask in ranges.items():
        if np.any(mask):
            err_log = preds_log[mask] - targets_log[mask]
            err_raw = preds_raw[mask] - targets_raw[mask]
            rel_err = err_raw / np.maximum(targets_raw[mask], 1e-6)

            rmse_log = float(np.sqrt(np.mean(err_log ** 2)))
            mae_log = float(np.mean(np.abs(err_log)))
            rmse_raw = float(np.sqrt(np.mean(err_raw ** 2)))
            mae_raw = float(np.mean(np.abs(err_raw)))
            rmse_rel = float(np.sqrt(np.mean(rel_err ** 2)))
            mae_rel = float(np.mean(np.abs(rel_err)))

            metrics[name] = {
                "count": int(mask.sum()),
                "rmse_log": rmse_log,
                "mae_log": mae_log,
                "factor_error": float(10 ** rmse_log),
                "rmse_raw": rmse_raw,
                "mae_raw": mae_raw,
                "rmse_rel": rmse_rel,
                "mae_rel": mae_rel,
                "bias_raw": float(np.mean(err_raw)),
                "bias_log": float(np.mean(err_log)),
                "p90_abs_raw": float(np.percentile(np.abs(err_raw), 90)),
                "p95_abs_raw": float(np.percentile(np.abs(err_raw), 95)),
            }
        else:
            metrics[name] = None

    return metrics


# =========================
# Training
# =========================
def train_engine():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Executing on: {device}")

    train_loader, val_loader, test_loader, scalers, aux_info, max_past_seq_len = prepare_data(
        DATA_PATHS,
        batch_size=256,
        nrows_per_dataset=NROWS_PER_DATASET,
        num_workers=0,
    )

    joblib.dump(scalers, SCALER_SAVE_PATH)
    print(f"Scalers saved to: {SCALER_SAVE_PATH}")

    model = BERFirstTokenSetTransformerBoundedLogHalfMultiTask(
        max_past_seq_len=max_past_seq_len,
        token_dim=4,
        first_token_dim=4,
        threshold_dim=1,
        global_dim=7,
        d_model=128,
        num_set_layers=4,
        num_heads=4,
        mlp_ratio=4.0,
        dropout=0.10,
        num_ordinal_thresholds=NUM_ORDINAL_THRESHOLDS,
    ).to(device)

    print("Training target-focused first-token + set-transformer multitask model...")
    print(f"  Variable-length support: ENABLED (position-independent scaling)")
    print(f"  Max past sequence length (training): {max_past_seq_len}")
    print(f"  Scaling strategy: separate first-token / shared past-token scalers")

    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

    reg_loss = StableMultiObjectiveBERBoundedLogLoss(
        log_delta=0.35,
        raw_delta=0.006,
        rel_delta=0.03,
        alpha_log=0.45,
        beta_raw=0.35,
        gamma_rel=0.20,
        use_regime_weights=True,
    )

    boosted_pos = aux_info["ordinal_pos_weights"].copy()
    for i, thr in enumerate(ORDINAL_THRESHOLDS):
        if 0.20 <= thr <= 0.40:
            boosted_pos[i] *= 2.5
        elif 0.15 <= thr < 0.20:
            boosted_pos[i] *= 1.5
        elif 0.40 < thr <= 0.45:
            boosted_pos[i] *= 1.5

    ord_loss = OrdinalBCELoss(
        pos_weight=torch.tensor(boosted_pos, dtype=torch.float32, device=device),
        reduction="mean",
    )

    criterion = MultiTaskBERLoss(
        reg_loss=reg_loss,
        ord_loss=ord_loss,
        lambda_ord=0.25,
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=4, factor=0.5,
    )

    best_selection_score = float("inf")
    best_state = None
    patience = 12
    wait = 0
    min_epochs_before_early_stop = 60
    max_epochs = 120

    print(f"Ordinal thresholds: {ORDINAL_THRESHOLDS}")
    print(f"Boosted ordinal pos weights: {boosted_pos}")

    training_broke = False

    for epoch in range(max_epochs):
        model.train()

        for batch_idx, (b_first, b_past, b_global, b_thr, b_y_log, b_ord, _, b_mask) in enumerate(train_loader):
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_y_log = b_y_log.to(device, non_blocking=True)
            b_ord = b_ord.to(device, non_blocking=True)
            b_mask = b_mask.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            pred_raw_unconstrained, ord_logits = model(
                b_first, b_past, b_global, b_thr, key_padding_mask=b_mask,
            )

            if has_nonfinite_tensor(pred_raw_unconstrained):
                print(f"Non-finite prediction at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True
                break

            if has_nonfinite_tensor(ord_logits):
                print(f"Non-finite ordinal logits at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True
                break

            loss, reg_part, ord_part = criterion(
                pred_raw_unconstrained, ord_logits, b_y_log, b_ord,
            )

            if has_nonfinite_tensor(loss):
                print(f"Non-finite loss at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True
                break

            loss.backward()

            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            if not torch.isfinite(grad_norm):
                print(f"Non-finite gradient norm at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True
                break

            optimizer.step()

            bad_param = False
            for name, param in model.named_parameters():
                if param.requires_grad and param.data is not None and not torch.isfinite(param.data).all():
                    print(f"Non-finite parameter after optimizer step: {name}")
                    bad_param = True
                    break

            if bad_param:
                training_broke = True
                break

        if training_broke:
            print("Training stopped due to non-finite values.")
            break

        try:
            train_metrics = evaluate(model, train_loader, criterion, device)
            val_metrics = evaluate(model, val_loader, criterion, device)
            val_range_metrics = evaluate_by_target_range(model, val_loader, device)
        except RuntimeError as e:
            print(f"Evaluation failed at epoch {epoch+1}: {e}")
            break

        scheduler.step(val_metrics["loss"])
        current_lr = optimizer.param_groups[0]["lr"]

        target_key = "target_BER(0.20<=y<0.40)"
        if val_range_metrics[target_key] is not None:
            selection_score = val_range_metrics[target_key]["rmse_raw"]
        else:
            selection_score = val_metrics["rmse_raw"]

        print(
            f"Epoch {epoch+1:03d} | "
            f"LR: {current_lr:.2e} | "
            f"Train Loss: {train_metrics['loss']:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} | "
            f"Train Reg: {train_metrics['reg_loss']:.4f} | "
            f"Val Reg: {val_metrics['reg_loss']:.4f} | "
            f"Train Ord: {train_metrics['ord_loss']:.4f} | "
            f"Val Ord: {val_metrics['ord_loss']:.4f} | "
            f"Train RMSE(log10): {train_metrics['rmse_log']:.4f} "
            f"(~{train_metrics['factor_error']:.2f}x) | "
            f"Val RMSE(log10): {val_metrics['rmse_log']:.4f} "
            f"(~{val_metrics['factor_error']:.2f}x) | "
            f"Val Target 0.20-0.40 RMSE(raw): {selection_score:.6f} | "
            f"Val Target 0.20-0.40 MAE(raw): "
            f"{val_range_metrics[target_key]['mae_raw'] if val_range_metrics[target_key] is not None else float('nan'):.6f} | "
            f"Train Region Acc: {train_metrics['region_acc']:.4f} | "
            f"Val Region Acc: {val_metrics['region_acc']:.4f}"
        )

        # Print per-range tail metrics
        for rng_name, rng_stats in val_range_metrics.items():
            if rng_stats is not None:
                print(
                    f"    {rng_name}: "
                    f"n={rng_stats['count']} | "
                    f"RMSE_raw={rng_stats['rmse_raw']:.6f} | "
                    f"MAE_raw={rng_stats['mae_raw']:.6f} | "
                    f"P90={rng_stats['p90_abs_raw']:.6f} | "
                    f"P95={rng_stats['p95_abs_raw']:.6f} | "
                    f"bias={rng_stats['bias_raw']:.6f}"
                )

        if selection_score < best_selection_score:
            best_selection_score = selection_score
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
            print(
                f"  -> New best model at epoch {epoch+1} "
                f"with Val Target 0.20-0.40 RMSE(raw): {selection_score:.6f}"
            )
        else:
            if epoch + 1 >= min_epochs_before_early_stop:
                wait += 1
                if wait >= patience:
                    print("Early stopping triggered.")
                    break

    torch.save(model.state_dict(), LAST_MODEL_SAVE_PATH)
    print(f"Last model saved to: {LAST_MODEL_SAVE_PATH}")

    if best_state is not None:
        model.load_state_dict(best_state)
        torch.save(model.state_dict(), BEST_MODEL_SAVE_PATH)
        print(f"Best model saved to: {BEST_MODEL_SAVE_PATH}")
    else:
        print("Warning: no valid best checkpoint was found.")

    test_metrics = evaluate(model, test_loader, criterion, device)

    print(
        f"Test Loss: {test_metrics['loss']:.4f} | "
        f"Test Reg Loss: {test_metrics['reg_loss']:.4f} | "
        f"Test Ord Loss: {test_metrics['ord_loss']:.4f} | "
        f"Test RMSE(log10): {test_metrics['rmse_log']:.4f} | "
        f"Test MAE(log10): {test_metrics['mae_log']:.4f} | "
        f"Typical multiplicative error: ~{test_metrics['factor_error']:.2f}x | "
        f"Test RMSE(raw): {test_metrics['rmse_raw']:.6f} | "
        f"Test MAE(raw): {test_metrics['mae_raw']:.6f} | "
        f"Test RMSE(rel): {test_metrics['rmse_rel']:.6f} | "
        f"Test MAE(rel): {test_metrics['mae_rel']:.6f} | "
        f"Test Region Acc: {test_metrics['region_acc']:.4f} | "
        f"Test Region MAE: {test_metrics['region_mae']:.4f}"
    )

    range_metrics = evaluate_by_target_range(model, test_loader, device)
    print("\nPer-range test diagnostics:")
    for name, stats in range_metrics.items():
        if stats is None:
            print(f"  {name}: no samples")
        else:
            print(
                f"  {name} | count={stats['count']} | "
                f"RMSE(log10)={stats['rmse_log']:.4f} | "
                f"MAE(log10)={stats['mae_log']:.4f} | "
                f"factor~{stats['factor_error']:.2f}x | "
                f"RMSE(raw)={stats['rmse_raw']:.6f} | "
                f"MAE(raw)={stats['mae_raw']:.6f} | "
                f"RMSE(rel)={stats['rmse_rel']:.6f} | "
                f"MAE(rel)={stats['mae_rel']:.6f} | "
                f"bias_raw={stats['bias_raw']:.6f} | "
                f"bias_log={stats['bias_log']:.6f} | "
                f"P90={stats['p90_abs_raw']:.6f} | "
                f"P95={stats['p95_abs_raw']:.6f}"
            )

    return model


if __name__ == "__main__":
    os.makedirs(BASE_DIR, exist_ok=True)

    missing = [p for p in DATA_PATHS if not os.path.exists(p)]
    if missing:
        print("Critical Error: Missing dataset files:")
        for p in missing:
            print(f"  - {p}")
    else:
        print("Using training from datasets:")
        for p in DATA_PATHS:
            print(f"  - {p}")
        print(f"Row cap per dataset: {NROWS_PER_DATASET:,}")

        trained_model = train_engine()

Using training from datasets:
  - ./data_physics_with_variances_total_v2.csv
Row cap per dataset: 5,000,000
Executing on: cuda
Loading up to 5,000,000 rows from: ./data_physics_with_variances_total_v2.csv
Combined rows before filtering: 5,000,000
Rows after cleaning/filtering: 3,227,933
Scalers saved to: ./random_fulltrain_highber_scalers_v1_target020_040_transformer_multitask.pkl
Training target-focused first-token + set-transformer multitask model...
  Variable-length support: ENABLED (position-independent scaling)
  Max past sequence length (training): 13
  Scaling strategy: separate first-token / shared past-token scalers
Ordinal thresholds: [1e-06, 1e-05, 0.0001, 0.001, 0.01, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45]
Boosted ordinal pos weights: [1.        1.        1.        1.        1.        1.        1.5
 2.5       3.183598  5.136442  6.5666304 9.348731  7.836687 ]
Epoch 001 | LR: 1.00e-04 | Train Loss: 0.0496 | Val Loss: 0.1099 | Train Reg: 0.0338 | Val Reg: 0.0945 | Train

Epoch 008 | LR: 1.00e-04 | Train Loss: 0.0078 | Val Loss: 0.0130 | Train Reg: 0.0034 | Val Reg: 0.0089 | Train Ord: 0.0177 | Val Ord: 0.0165 | Train RMSE(log10): 0.1249 (~1.33x) | Val RMSE(log10): 0.2110 (~1.63x) | Val Target 0.20-0.40 RMSE(raw): 0.008547 | Val Target 0.20-0.40 MAE(raw): 0.005973 | Train Region Acc: 0.9166 | Val Region Acc: 0.9223
    low_BER(y<1e-6): n=96438 | RMSE_raw=0.000132 | MAE_raw=0.000002 | P90=0.000000 | P95=0.000000 | bias=0.000002
    mid_BER(1e-6<=y<1e-3): n=15829 | RMSE_raw=0.001343 | MAE_raw=0.000198 | P90=0.000317 | P95=0.000620 | bias=0.000142
    high_BER(1e-3<=y<0.10): n=60678 | RMSE_raw=0.010157 | MAE_raw=0.006537 | P90=0.015075 | P95=0.019388 | bias=0.004652
    upper_BER(0.10<=y<0.20): n=65355 | RMSE_raw=0.010758 | MAE_raw=0.007598 | P90=0.016288 | P95=0.022048 | bias=0.003503
    target_BER(0.20<=y<0.40): n=143565 | RMSE_raw=0.008547 | MAE_raw=0.005973 | P90=0.013459 | P95=0.017803 | bias=-0.001245
    very_high_BER(0.40<=y<=0.50): n=102325 | RMS

Epoch 016 | LR: 1.00e-04 | Train Loss: 0.0064 | Val Loss: 0.0103 | Train Reg: 0.0025 | Val Reg: 0.0063 | Train Ord: 0.0157 | Val Ord: 0.0160 | Train RMSE(log10): 0.1322 (~1.36x) | Val RMSE(log10): 0.2220 (~1.67x) | Val Target 0.20-0.40 RMSE(raw): 0.008466 | Val Target 0.20-0.40 MAE(raw): 0.005930 | Train Region Acc: 0.9346 | Val Region Acc: 0.9275
    low_BER(y<1e-6): n=96438 | RMSE_raw=0.000000 | MAE_raw=0.000000 | P90=0.000000 | P95=0.000000 | bias=0.000000
    mid_BER(1e-6<=y<1e-3): n=15829 | RMSE_raw=0.000259 | MAE_raw=0.000136 | P90=0.000426 | P95=0.000628 | bias=0.000055
    high_BER(1e-3<=y<0.10): n=60678 | RMSE_raw=0.007227 | MAE_raw=0.005024 | P90=0.011361 | P95=0.014559 | bias=0.000919
    upper_BER(0.10<=y<0.20): n=65355 | RMSE_raw=0.008564 | MAE_raw=0.005909 | P90=0.013493 | P95=0.017963 | bias=0.002265
    target_BER(0.20<=y<0.40): n=143565 | RMSE_raw=0.008466 | MAE_raw=0.005930 | P90=0.014455 | P95=0.018415 | bias=0.003179
    very_high_BER(0.40<=y<=0.50): n=102325 | RMSE

Epoch 024 | LR: 5.00e-05 | Train Loss: 0.0029 | Val Loss: 0.0047 | Train Reg: 0.0011 | Val Reg: 0.0029 | Train Ord: 0.0073 | Val Ord: 0.0072 | Train RMSE(log10): 0.0768 (~1.19x) | Val RMSE(log10): 0.1291 (~1.35x) | Val Target 0.20-0.40 RMSE(raw): 0.004837 | Val Target 0.20-0.40 MAE(raw): 0.003155 | Train Region Acc: 0.9702 | Val Region Acc: 0.9663
    low_BER(y<1e-6): n=96438 | RMSE_raw=0.000000 | MAE_raw=0.000000 | P90=0.000000 | P95=0.000000 | bias=-0.000000
    mid_BER(1e-6<=y<1e-3): n=15829 | RMSE_raw=0.000144 | MAE_raw=0.000077 | P90=0.000234 | P95=0.000341 | bias=-0.000071
    high_BER(1e-3<=y<0.10): n=60678 | RMSE_raw=0.004987 | MAE_raw=0.003648 | P90=0.007368 | P95=0.009546 | bias=-0.001007
    upper_BER(0.10<=y<0.20): n=65355 | RMSE_raw=0.006102 | MAE_raw=0.004279 | P90=0.008735 | P95=0.012848 | bias=0.000731
    target_BER(0.20<=y<0.40): n=143565 | RMSE_raw=0.004837 | MAE_raw=0.003155 | P90=0.007288 | P95=0.010408 | bias=0.000180
    very_high_BER(0.40<=y<=0.50): n=102325 | R

Epoch 032 | LR: 2.50e-05 | Train Loss: 0.0017 | Val Loss: 0.0022 | Train Reg: 0.0004 | Val Reg: 0.0009 | Train Ord: 0.0053 | Val Ord: 0.0050 | Train RMSE(log10): 0.0395 (~1.10x) | Val RMSE(log10): 0.0663 (~1.16x) | Val Target 0.20-0.40 RMSE(raw): 0.004326 | Val Target 0.20-0.40 MAE(raw): 0.002912 | Train Region Acc: 0.9806 | Val Region Acc: 0.9793
    low_BER(y<1e-6): n=96438 | RMSE_raw=0.000000 | MAE_raw=0.000000 | P90=0.000000 | P95=0.000000 | bias=-0.000000
    mid_BER(1e-6<=y<1e-3): n=15829 | RMSE_raw=0.000083 | MAE_raw=0.000029 | P90=0.000077 | P95=0.000143 | bias=-0.000018
    high_BER(1e-3<=y<0.10): n=60678 | RMSE_raw=0.003283 | MAE_raw=0.002099 | P90=0.004864 | P95=0.006437 | bias=0.000866
    upper_BER(0.10<=y<0.20): n=65355 | RMSE_raw=0.004039 | MAE_raw=0.002602 | P90=0.005698 | P95=0.008422 | bias=0.000240
    target_BER(0.20<=y<0.40): n=143565 | RMSE_raw=0.004326 | MAE_raw=0.002912 | P90=0.006255 | P95=0.008985 | bias=-0.001559
    very_high_BER(0.40<=y<=0.50): n=102325 | R

...

Epoch 048 | LR: 1.25e-05 | Train Loss: 0.0015 | Val Loss: 0.0021 | Train Reg: 0.0004 | Val Reg: 0.0009 | Train Ord: 0.0046 | Val Ord: 0.0046 | Train RMSE(log10): 0.0413 (~1.10x) | Val RMSE(log10): 0.0691 (~1.17x) | Val Target 0.20-0.40 RMSE(raw): 0.003884 | Val Target 0.20-0.40 MAE(raw): 0.002544 | Train Region Acc: 0.9798 | Val Region Acc: 0.9787
    low_BER(y<1e-6): n=96438 | RMSE_raw=0.000000 | MAE_raw=0.000000 | P90=0.000000 | P95=0.000000 | bias=0.000000
    mid_BER(1e-6<=y<1e-3): n=15829 | RMSE_raw=0.000232 | MAE_raw=0.000034 | P90=0.000066 | P95=0.000115 | bias=0.000022
    high_BER(1e-3<=y<0.10): n=60678 | RMSE_raw=0.003840 | MAE_raw=0.002773 | P90=0.005552 | P95=0.007251 | bias=0.002498
    upper_BER(0.10<=y<0.20): n=65355 | RMSE_raw=0.004111 | MAE_raw=0.003207 | P90=0.005785 | P95=0.007357 | bias=0.002555
    target_BER(0.20<=y<0.40): n=143565 | RMSE_raw=0.003884 | MAE_raw=0.002544 | P90=0.005471 | P95=0.007728 | bias=0.000994
    very_high_BER(0.40<=y<=0.50): n=102325 | RMSE

Epoch 056 | LR: 1.25e-05 | Train Loss: 0.0014 | Val Loss: 0.0018 | Train Reg: 0.0003 | Val Reg: 0.0007 | Train Ord: 0.0042 | Val Ord: 0.0042 | Train RMSE(log10): 0.0293 (~1.07x) | Val RMSE(log10): 0.0480 (~1.12x) | Val Target 0.20-0.40 RMSE(raw): 0.003910 | Val Target 0.20-0.40 MAE(raw): 0.002783 | Train Region Acc: 0.9819 | Val Region Acc: 0.9804
    low_BER(y<1e-6): n=96438 | RMSE_raw=0.000000 | MAE_raw=0.000000 | P90=0.000000 | P95=0.000000 | bias=0.000000
    mid_BER(1e-6<=y<1e-3): n=15829 | RMSE_raw=0.000077 | MAE_raw=0.000035 | P90=0.000089 | P95=0.000136 | bias=0.000032
    high_BER(1e-3<=y<0.10): n=60678 | RMSE_raw=0.003984 | MAE_raw=0.002986 | P90=0.006237 | P95=0.007808 | bias=0.002720
    upper_BER(0.10<=y<0.20): n=65355 | RMSE_raw=0.003714 | MAE_raw=0.002735 | P90=0.005317 | P95=0.006963 | bias=0.001932
    target_BER(0.20<=y<0.40): n=143565 | RMSE_raw=0.003910 | MAE_raw=0.002783 | P90=0.005279 | P95=0.007577 | bias=0.001638
    very_high_BER(0.40<=y<=0.50): n=102325 | RMSE

Epoch 064 | LR: 6.25e-06 | Train Loss: 0.0011 | Val Loss: 0.0014 | Train Reg: 0.0002 | Val Reg: 0.0005 | Train Ord: 0.0036 | Val Ord: 0.0036 | Train RMSE(log10): 0.0269 (~1.06x) | Val RMSE(log10): 0.0463 (~1.11x) | Val Target 0.20-0.40 RMSE(raw): 0.003757 | Val Target 0.20-0.40 MAE(raw): 0.002634 | Train Region Acc: 0.9853 | Val Region Acc: 0.9848
    low_BER(y<1e-6): n=96438 | RMSE_raw=0.000000 | MAE_raw=0.000000 | P90=0.000000 | P95=0.000000 | bias=-0.000000
    mid_BER(1e-6<=y<1e-3): n=15829 | RMSE_raw=0.000047 | MAE_raw=0.000017 | P90=0.000046 | P95=0.000073 | bias=0.000002
    high_BER(1e-3<=y<0.10): n=60678 | RMSE_raw=0.003793 | MAE_raw=0.002643 | P90=0.006120 | P95=0.007636 | bias=0.002415
    upper_BER(0.10<=y<0.20): n=65355 | RMSE_raw=0.004122 | MAE_raw=0.003126 | P90=0.005932 | P95=0.007076 | bias=0.002593
    target_BER(0.20<=y<0.40): n=143565 | RMSE_raw=0.003757 | MAE_raw=0.002634 | P90=0.005195 | P95=0.007540 | bias=0.001675
    very_high_BER(0.40<=y<=0.50): n=102325 | RMS

Epoch 072 | LR: 3.13e-06 | Train Loss: 0.0014 | Val Loss: 0.0016 | Train Reg: 0.0002 | Val Reg: 0.0004 | Train Ord: 0.0048 | Val Ord: 0.0046 | Train RMSE(log10): 0.0213 (~1.05x) | Val RMSE(log10): 0.0340 (~1.08x) | Val Target 0.20-0.40 RMSE(raw): 0.003775 | Val Target 0.20-0.40 MAE(raw): 0.002644 | Train Region Acc: 0.9834 | Val Region Acc: 0.9825
    low_BER(y<1e-6): n=96438 | RMSE_raw=0.000000 | MAE_raw=0.000000 | P90=0.000000 | P95=0.000000 | bias=0.000000
    mid_BER(1e-6<=y<1e-3): n=15829 | RMSE_raw=0.000061 | MAE_raw=0.000027 | P90=0.000077 | P95=0.000109 | bias=0.000023
    high_BER(1e-3<=y<0.10): n=60678 | RMSE_raw=0.003446 | MAE_raw=0.002418 | P90=0.005295 | P95=0.006758 | bias=0.002170
    upper_BER(0.10<=y<0.20): n=65355 | RMSE_raw=0.003409 | MAE_raw=0.002341 | P90=0.004765 | P95=0.006261 | bias=0.001353
    target_BER(0.20<=y<0.40): n=143565 | RMSE_raw=0.003775 | MAE_raw=0.002644 | P90=0.005414 | P95=0.007464 | bias=0.001571
    very_high_BER(0.40<=y<=0.50): n=102325 | RMSE

Epoch 080 | LR: 3.13e-06 | Train Loss: 0.0012 | Val Loss: 0.0014 | Train Reg: 0.0002 | Val Reg: 0.0004 | Train Ord: 0.0038 | Val Ord: 0.0038 | Train RMSE(log10): 0.0209 (~1.05x) | Val RMSE(log10): 0.0347 (~1.08x) | Val Target 0.20-0.40 RMSE(raw): 0.003669 | Val Target 0.20-0.40 MAE(raw): 0.002501 | Train Region Acc: 0.9846 | Val Region Acc: 0.9837
    low_BER(y<1e-6): n=96438 | RMSE_raw=0.000000 | MAE_raw=0.000000 | P90=0.000000 | P95=0.000000 | bias=0.000000
    mid_BER(1e-6<=y<1e-3): n=15829 | RMSE_raw=0.000064 | MAE_raw=0.000021 | P90=0.000055 | P95=0.000087 | bias=0.000012
    high_BER(1e-3<=y<0.10): n=60678 | RMSE_raw=0.003615 | MAE_raw=0.002646 | P90=0.005546 | P95=0.006963 | bias=0.002343
    upper_BER(0.10<=y<0.20): n=65355 | RMSE_raw=0.003635 | MAE_raw=0.002744 | P90=0.004862 | P95=0.006279 | bias=0.002051
    target_BER(0.20<=y<0.40): n=143565 | RMSE_raw=0.003669 | MAE_raw=0.002501 | P90=0.005213 | P95=0.007249 | bias=0.001400
    very_high_BER(0.40<=y<=0.50): n=102325 | RMSE

In [2]:
import os
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.special import erfc
from itertools import product as iproduct

EPS = 1e-12
PLOT_DIR = "./plots_v3_random"
os.makedirs(PLOT_DIR, exist_ok=True)

# =========================
# Paths
# =========================
MODEL_PATH = "random_fulltrain_highber_best_v1_target020_040_transformer_multitask.pth"
SCALER_PATH = "random_fulltrain_highber_scalers_v1_target020_040_transformer_multitask.pkl"

# =========================
# Config
# =========================
PHYSICS_MAX_MEM_LEN = 14
PHYSICS_MIN_MEM_LEN = 14
ARRIVAL_COVERAGE = 0.70
N_THRESHOLDS = 500
RANDOM_SEED = 60

ORDINAL_THRESHOLDS = [
    1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1,
    0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45,
]
NUM_ORDINAL_THRESHOLDS = len(ORDINAL_THRESHOLDS)
NUM_REGION_CLASSES = NUM_ORDINAL_THRESHOLDS + 1

REGION_LABELS = {
    0: "y < 1e-6",
    1: "1e-6 <= y < 1e-5",
    2: "1e-5 <= y < 1e-4",
    3: "1e-4 <= y < 1e-3",
    4: "1e-3 <= y < 1e-2",
    5: "1e-2 <= y < 1e-1",
    6: "0.10 <= y < 0.15",
    7: "0.15 <= y < 0.20",
    8: "0.20 <= y < 0.25",
    9: "0.25 <= y < 0.30",
    10: "0.30 <= y < 0.35",
    11: "0.35 <= y < 0.40",
    12: "0.40 <= y < 0.45",
    13: "0.45 <= y <= 0.50",
}


# =========================
# Physics helpers
# =========================
def Fhit_function(radius, distance, diffusionCoef, t):
    if t <= 0:
        return 0.0
    return (radius / (distance + radius)) * erfc(distance / np.sqrt(4 * diffusionCoef * t))


def calculate_hitting_probabilities(mem_len, radius, distance, diffusionCoef, Ts):
    P = np.zeros(mem_len)
    for i in range(mem_len):
        t_end = (i + 1) * Ts
        t_start = i * Ts
        P[i] = Fhit_function(radius, distance, diffusionCoef, t_end) - Fhit_function(
            radius, distance, diffusionCoef, t_start
        )
    return P


def calculate_ber_vectorized(mem_len, threshold, P_scaled, variances):
    P_arr = np.asarray(P_scaled, dtype=float)[:mem_len]
    vars_arr = np.asarray(variances, dtype=float)[:mem_len]

    seqs = np.array(list(iproduct([0, 1], repeat=mem_len)), dtype=np.float64)[:, ::-1]
    c_bit = seqs[:, 0]

    mu = (seqs * P_arr).sum(axis=1)
    var_total = (seqs * vars_arr).sum(axis=1)
    std = np.sqrt(np.maximum(var_total, 0.0))

    pe = np.empty_like(mu)
    zero_std = (std == 0)
    if np.any(zero_std):
        pe[zero_std & (c_bit == 1)] = np.where(
            mu[zero_std & (c_bit == 1)] < threshold, 1.0, 0.0)
        pe[zero_std & (c_bit == 0)] = np.where(
            mu[zero_std & (c_bit == 0)] >= threshold, 1.0, 0.0)
    nz = ~zero_std
    if np.any(nz):
        pe[nz & (c_bit == 1)] = 0.5 * erfc(
            (mu[nz & (c_bit == 1)] - threshold) / (std[nz & (c_bit == 1)] * np.sqrt(2)))
        pe[nz & (c_bit == 0)] = 0.5 * erfc(
            (threshold - mu[nz & (c_bit == 0)]) / (std[nz & (c_bit == 0)] * np.sqrt(2)))
    return float(np.mean(pe))


def raw_to_region_labels_np(y_raw):
    y = np.asarray(y_raw).reshape(-1)
    return np.searchsorted(np.asarray(ORDINAL_THRESHOLDS), y, side="right").astype(np.int64)


def ordinal_logits_to_region_labels_torch(logits):
    probs = torch.sigmoid(logits)
    passed = (probs >= 0.5).sum(dim=1)
    return passed.long()


# =========================
# Shared scaling helper
# =========================
def apply_shared_scale(data, mean, std, valid_mask=None):
    scaled = ((data - mean) / std).astype(np.float32)
    if valid_mask is not None and scaled.ndim == 2 and valid_mask.shape == scaled.shape:
        scaled = scaled * valid_mask.astype(np.float32)
    return scaled


# =========================
# Generate one physical scenario
# =========================
def generate_physical_case(rng, min_mem_len, max_mem_len, arrival_coverage, max_tries=5000):
    for _ in range(max_tries):
        radius = rng.uniform(3.0, 5.0)
        distance = rng.uniform(10.0, 15.0)
        diff = rng.uniform(50.0, 75.0)
        Ts = rng.uniform(0.5, 1.2)
        N = int(10 ** rng.uniform(3.0, 6.0))

        f_inf = radius / (radius + distance)
        target = arrival_coverage * f_inf

        cumsum, k = 0.0, 0
        while k < max_mem_len:
            pk = Fhit_function(radius, distance, diff, (k + 1) * Ts) - Fhit_function(
                radius, distance, diff, k * Ts)
            cumsum += pk
            k += 1
            if cumsum >= target:
                break

        if k < min_mem_len:
            continue

        P_ext = calculate_hitting_probabilities(k + 1, radius, distance, diff, Ts)
        P_main = P_ext[:k]
        P_extra = float(P_ext[k]) if k < len(P_ext) else 0.0

        P_scaled = P_main * N
        variances = N * P_main * (1.0 - P_main)

        return {
            "radius": radius, "distance": distance, "diffusion": diff,
            "Ts": Ts, "N": N, "mem_len": k, "P": P_main,
            "P_scaled": P_scaled, "variances": variances,
            "P_mem_len_extra": P_extra,
            "P_mem_len_extra_var": P_extra * (1.0 - P_extra),
        }

    raise RuntimeError(
        f"Could not generate a physical case with mem_len >= {min_mem_len} "
        f"after {max_tries} tries."
    )


# =========================
# Model (matches v3 training: global_dim=7, global_embed=96, cond_dim=128)
# =========================
class ConditionalFeatureModulation(nn.Module):
    def __init__(self, cond_dim, feat_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, 2 * feat_dim),
        )
        self.scale = nn.Parameter(torch.tensor(0.25, dtype=torch.float32))

    def forward(self, x, cond):
        gamma_beta = self.net(cond)
        gamma, beta = torch.chunk(gamma_beta, 2, dim=-1)
        if x.dim() == 3:
            gamma = gamma.unsqueeze(1)
            beta = beta.unsqueeze(1)
        return x * (1.0 + self.scale * gamma) + self.scale * beta


class SetSelfAttentionBlock(nn.Module):
    def __init__(self, d_model, num_heads, mlp_ratio=4.0, dropout=0.10):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=num_heads,
            dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        hidden = int(d_model * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, d_model), nn.Dropout(dropout))

    def forward(self, x, key_padding_mask=None):
        y = self.norm1(x)
        attn_out, _ = self.attn(y, y, y, need_weights=False, key_padding_mask=key_padding_mask)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x


class SetAttentionPooling(nn.Module):
    def __init__(self, d_model, hidden_dim=128):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(d_model, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, 1))

    def forward(self, x, key_padding_mask=None):
        logits = self.score(x)
        if key_padding_mask is not None:
            logits = logits.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
        weights = torch.softmax(logits, dim=1)
        weights = torch.nan_to_num(weights, nan=0.0)
        return (weights * x).sum(dim=1)


class BERFirstTokenSetTransformerBoundedLogHalfMultiTask(nn.Module):
    def __init__(self, max_past_seq_len, token_dim=4, first_token_dim=4,
                 threshold_dim=1, global_dim=7, d_model=128, num_set_layers=4,
                 num_heads=4, mlp_ratio=4.0, dropout=0.10,
                 num_ordinal_thresholds=NUM_ORDINAL_THRESHOLDS):
        super().__init__()
        self.max_past_seq_len = max_past_seq_len
        self.d_model = d_model
        self.num_ordinal_thresholds = num_ordinal_thresholds

        self.first_proj = nn.Sequential(
            nn.Linear(first_token_dim, d_model), nn.LayerNorm(d_model),
            nn.GELU(), nn.Dropout(dropout))
        self.token_proj = nn.Sequential(
            nn.Linear(token_dim, d_model), nn.LayerNorm(d_model),
            nn.GELU(), nn.Dropout(dropout))
        self.thr_embed = nn.Sequential(
            nn.Linear(threshold_dim, 32), nn.GELU(),
            nn.Linear(32, 32), nn.GELU())
        self.global_embed = nn.Sequential(
            nn.Linear(global_dim, 96), nn.GELU(),
            nn.Linear(96, 96), nn.GELU())

        cond_dim = 32 + 96
        self.first_cond_mod = ConditionalFeatureModulation(cond_dim, d_model, 256)
        self.set_cond_mod = ConditionalFeatureModulation(cond_dim, d_model, 256)

        self.set_blocks = nn.ModuleList([
            SetSelfAttentionBlock(d_model, num_heads, mlp_ratio, dropout)
            for _ in range(num_set_layers)])

        self.first_refine = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_model),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model, d_model))
        self.final_set_norm = nn.LayerNorm(d_model)
        self.set_attn_pool = SetAttentionPooling(d_model, 128)

        set_summary_dim = 3 * d_model
        head_in = d_model + set_summary_dim + cond_dim

        self.reg_head = nn.Sequential(
            nn.Linear(head_in, 256), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.15),
            nn.Linear(128, 1))
        self.ord_head = nn.Sequential(
            nn.Linear(head_in, 256), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.15),
            nn.Linear(128, num_ordinal_thresholds))

    @staticmethod
    def raw_to_log10ber(pred_raw_unconstrained):
        log10_half = torch.log10(torch.tensor(
            0.5, device=pred_raw_unconstrained.device,
            dtype=pred_raw_unconstrained.dtype))
        return log10_half - F.softplus(pred_raw_unconstrained)

    @staticmethod
    def raw_to_ber(pred_raw_unconstrained):
        pred_log = BERFirstTokenSetTransformerBoundedLogHalfMultiTask.raw_to_log10ber(
            pred_raw_unconstrained)
        return torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)

    def forward(self, first_token, past_tokens, global_feats, threshold, key_padding_mask=None):
        thr = self.thr_embed(threshold)
        g = self.global_embed(global_feats)
        cond = torch.cat([thr, g], dim=-1)

        first_x = self.first_proj(first_token)
        first_x = self.first_cond_mod(first_x, cond)
        first_x = first_x + self.first_refine(first_x)

        set_x = self.token_proj(past_tokens)
        set_x = self.set_cond_mod(set_x, cond)
        for block in self.set_blocks:
            set_x = block(set_x, key_padding_mask=key_padding_mask)
        set_x = self.final_set_norm(set_x)

        if key_padding_mask is not None:
            valid_mask = (~key_padding_mask).unsqueeze(-1).float()
        else:
            valid_mask = torch.ones(
                set_x.shape[0], set_x.shape[1], 1,
                device=set_x.device, dtype=set_x.dtype)

        pooled_attn = self.set_attn_pool(set_x, key_padding_mask=key_padding_mask)
        valid_counts = valid_mask.sum(dim=1).clamp(min=1.0)
        pooled_mean = (set_x * valid_mask).sum(dim=1) / valid_counts

        if key_padding_mask is not None:
            x_for_max = set_x.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
        else:
            x_for_max = set_x
        pooled_max = x_for_max.amax(dim=1)
        pooled_max = torch.nan_to_num(pooled_max, nan=0.0, posinf=0.0, neginf=0.0)

        set_summary = torch.cat([pooled_attn, pooled_mean, pooled_max], dim=-1)
        fused = torch.cat([first_x, set_summary, cond], dim=-1)

        pred_raw_unconstrained = self.reg_head(fused)
        ord_logits = self.ord_head(fused)
        return pred_raw_unconstrained, ord_logits


# =========================
# Build inference inputs (7 global features, position-independent scaling)
# =========================
def prepare_inference_features(case, thresholds, scalers, device):
    """Build model inputs for a threshold sweep over a single physical case.

    Returns:
        first_t, past_t, global_t, thr_t, mask_t — all torch tensors on device
        global_t has shape [B, 7]: [z0, z1, hmg, nsid, log_hd, da, herf]
    """
    B = len(thresholds)
    mem_len = case["mem_len"]
    N = float(case["N"])

    P_raw = np.asarray(case["P"], dtype=np.float32)
    var_raw = np.asarray(case["variances"], dtype=np.float32)
    thr_raw = np.asarray(thresholds, dtype=np.float32).reshape(-1, 1)

    # Feature engineering
    taps_feat = (P_raw * N).astype(np.float32)
    vars_feat = var_raw.astype(np.float32)
    abs_feat = np.abs(taps_feat).astype(np.float32)
    snr_feat = np.log10((taps_feat ** 2) / (var_raw + EPS) + EPS).astype(np.float32)

    # Broadcast to [B, mem_len]
    taps_2d = np.broadcast_to(taps_feat[None, :], (B, mem_len)).copy()
    vars_2d = np.broadcast_to(vars_feat[None, :], (B, mem_len)).copy()
    abs_2d = np.broadcast_to(abs_feat[None, :], (B, mem_len)).copy()
    snr_2d = np.broadcast_to(snr_feat[None, :], (B, mem_len)).copy()

    L_past = mem_len - 1
    valid_past = np.ones((B, L_past), dtype=bool)

    # --- Threshold-dependent global features ---
    first_mean = taps_2d[:, 0:1]
    past_means = taps_2d[:, 1:]
    first_var = vars_2d[:, 0:1]
    past_vars = vars_2d[:, 1:]

    mu0 = (0.5 * past_means.sum(axis=1, keepdims=True)).astype(np.float32)
    mu1 = (first_mean + mu0).astype(np.float32)
    var0 = (0.5 * past_vars.sum(axis=1, keepdims=True)).astype(np.float32)
    var1 = (first_var + var0).astype(np.float32)
    std0 = np.sqrt(np.maximum(var0, EPS)).astype(np.float32)
    std1 = np.sqrt(np.maximum(var1, EPS)).astype(np.float32)

    z0 = ((thr_raw - mu0) / (std0 + EPS)).astype(np.float32)
    z1 = ((mu1 - thr_raw) / (std1 + EPS)).astype(np.float32)
    harmonic = (2.0 / (1.0 / (z0 + EPS) + 1.0 / (z1 + EPS))).astype(np.float32)
    abs_diff = np.abs(z0 - z1).astype(np.float32)
    hmg = (harmonic - 0.25 * abs_diff).astype(np.float32)

    # --- Scenario-level features (threshold-independent) ---
    signal = first_mean  # [B, 1]
    isi = past_means.sum(axis=1, keepdims=True)
    nsid = ((signal - isi) / (signal + isi + EPS)).astype(np.float32)

    gap = signal
    d0_inf = (gap / (std0 + EPS)).astype(np.float32)
    d1_inf = (gap / (std1 + EPS)).astype(np.float32)
    hd = (2.0 * d0_inf * d1_inf / (d0_inf + d1_inf + EPS)).astype(np.float32)
    log_hd = np.log10(hd + EPS).astype(np.float32)
    da = (np.abs(d0_inf - d1_inf) / (d0_inf + d1_inf + EPS)).astype(np.float32)

    past_taps_sum = past_means.sum(axis=1, keepdims=True)
    past_shares = past_means / (past_taps_sum + EPS)
    herf = (past_shares ** 2).sum(axis=1, keepdims=True).astype(np.float32)

    # --- Scale first token ---
    ft_tap = ((taps_2d[:, 0] - scalers["first_tap_mean"]) / scalers["first_tap_std"]).astype(np.float32)
    ft_var = ((vars_2d[:, 0] - scalers["first_var_mean"]) / scalers["first_var_std"]).astype(np.float32)
    ft_abs = ((abs_2d[:, 0] - scalers["first_abs_mean"]) / scalers["first_abs_std"]).astype(np.float32)
    ft_snr = ((snr_2d[:, 0] - scalers["first_snr_mean"]) / scalers["first_snr_std"]).astype(np.float32)
    first_token = np.stack([ft_tap, ft_var, ft_abs, ft_snr], axis=1).astype(np.float32)

    # --- Scale past tokens ---
    pt_tap = apply_shared_scale(taps_2d[:, 1:], scalers["past_tap_mean"], scalers["past_tap_std"], valid_past)
    pt_var = apply_shared_scale(vars_2d[:, 1:], scalers["past_var_mean"], scalers["past_var_std"], valid_past)
    pt_abs = apply_shared_scale(abs_2d[:, 1:], scalers["past_abs_mean"], scalers["past_abs_std"], valid_past)
    pt_snr = apply_shared_scale(snr_2d[:, 1:], scalers["past_snr_mean"], scalers["past_snr_std"], valid_past)
    past_tokens = np.stack([pt_tap, pt_var, pt_abs, pt_snr], axis=2).astype(np.float32)

    # --- Scale globals (7 features) ---
    z0_s = scalers["z0_scaler"].transform(z0).astype(np.float32)
    z1_s = scalers["z1_scaler"].transform(z1).astype(np.float32)
    hmg_s = scalers["harmonic_minus_gap_scaler"].transform(hmg).astype(np.float32)
    nsid_s = scalers["nsid_scaler"].transform(nsid).astype(np.float32)
    log_hd_s = scalers["log_hd_scaler"].transform(log_hd).astype(np.float32)
    da_s = scalers["da_scaler"].transform(da).astype(np.float32)
    herf_s = scalers["herf_scaler"].transform(herf).astype(np.float32)
    global_feats = np.concatenate(
        [z0_s, z1_s, hmg_s, nsid_s, log_hd_s, da_s, herf_s], axis=1
    ).astype(np.float32)

    # --- Scale threshold ---
    thr_log = np.log10(thr_raw + EPS).astype(np.float32)
    thr_s = scalers["thr_scaler"].transform(thr_log).astype(np.float32)

    # --- Padding mask (no padding needed here) ---
    pad_mask = np.zeros((B, L_past), dtype=bool)

    first_t = torch.from_numpy(first_token).to(device)
    past_t = torch.from_numpy(past_tokens).to(device)
    global_t = torch.from_numpy(global_feats).to(device)
    thr_t = torch.from_numpy(thr_s).to(device)
    mask_t = torch.from_numpy(pad_mask).to(device)

    return first_t, past_t, global_t, thr_t, mask_t


# =========================
# Generate scenario
# =========================
rng = np.random.default_rng(RANDOM_SEED)
case = generate_physical_case(
    rng,
    min_mem_len=PHYSICS_MIN_MEM_LEN,
    max_mem_len=PHYSICS_MAX_MEM_LEN,
    arrival_coverage=ARRIVAL_COVERAGE,
)

print("Generated physical scenario")
print("radius    =", case["radius"])
print("distance  =", case["distance"])
print("diffusion =", case["diffusion"])
print("Ts        =", case["Ts"])
print("N         =", case["N"])
print("mem_len   =", case["mem_len"])
print("P         =", case["P"])
print("P_scaled  =", case["P_scaled"])
print("variances =", case["variances"])

# =========================
# Threshold sweep — ground truth
# =========================
thr_min = 0.0
thr_max = float(np.sum(case["P_scaled"]))
thresholds = np.linspace(thr_min, thr_max, N_THRESHOLDS)

print("Threshold search interval:", thr_min, "to", thr_max)
print("sum(P_scaled) =", np.sum(case["P_scaled"]))

real_bers = np.array([
    calculate_ber_vectorized(
        mem_len=case["mem_len"],
        threshold=thr,
        P_scaled=case["P_scaled"],
        variances=case["variances"],
    )
    for thr in thresholds
])

real_bers = np.clip(real_bers, EPS, 0.5)
real_regions = raw_to_region_labels_np(real_bers)

# =========================
# Load model + scalers
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scalers = joblib.load(SCALER_PATH)

max_past_seq_len = scalers.get("train_max_past_seq_len", scalers.get("max_past_seq_len", case["mem_len"] - 1))
ordinal_thresholds = scalers.get("ordinal_thresholds", ORDINAL_THRESHOLDS)
num_ordinal = len(ordinal_thresholds)
global_dim = scalers.get("global_dim", 7)

model = BERFirstTokenSetTransformerBoundedLogHalfMultiTask(
    max_past_seq_len=max_past_seq_len,
    token_dim=scalers.get("past_token_dim", 4),
    first_token_dim=scalers.get("first_token_dim", 4),
    threshold_dim=1,
    global_dim=global_dim,
    d_model=128,
    num_set_layers=4,
    num_heads=4,
    mlp_ratio=4.0,
    dropout=0.10,
    num_ordinal_thresholds=num_ordinal,
).to(device)

state = torch.load(MODEL_PATH, map_location=device, weights_only=True)
model.load_state_dict(state)
model.eval()

print(f"\nModel loaded: max_past_seq_len={max_past_seq_len}, "
      f"global_dim={global_dim}, num_ordinal={num_ordinal}")
print(f"Scaling strategy: {scalers.get('scaling_strategy', 'unknown')}")

# =========================
# Predict across thresholds
# =========================
first_t, past_t, global_t, thr_t, mask_t = prepare_inference_features(
    case, thresholds, scalers, device,
)

with torch.no_grad():
    pred_raw_out, ord_logits = model(
        first_t, past_t, global_t, thr_t, key_padding_mask=mask_t,
    )
    pred_bers = model.raw_to_ber(pred_raw_out).cpu().numpy().reshape(-1)
    pred_log = model.raw_to_log10ber(pred_raw_out).cpu().numpy().reshape(-1)
    pred_regions = ordinal_logits_to_region_labels_torch(ord_logits).cpu().numpy().reshape(-1)
    pred_region_probs = torch.sigmoid(ord_logits).cpu().numpy()

pred_bers = np.clip(pred_bers, EPS, 0.5)

# =========================
# Comparison table
# =========================
results = pd.DataFrame({
    "threshold": thresholds,
    "real_BER": real_bers,
    "estimated_BER": pred_bers,
    "real_region": real_regions,
    "predicted_region": pred_regions,
    "abs_error": np.abs(pred_bers - real_bers),
    "abs_log10_error": np.abs(
        np.log10(np.clip(pred_bers, EPS, 0.5)) -
        np.log10(np.clip(real_bers, EPS, 0.5))
    ),
    "region_abs_error": np.abs(pred_regions.astype(np.int64) - real_regions.astype(np.int64)),
})

print(results.head(15))

print("\nSummary")
print("Mean abs raw error        :", results["abs_error"].mean())
print("Mean abs log10 error      :", results["abs_log10_error"].mean())
print("Max  abs log10 error      :", results["abs_log10_error"].max())
print("Mean region abs error     :", results["region_abs_error"].mean())
print("Exact region accuracy     :", np.mean(results["real_region"] == results["predicted_region"]))

best_real_idx = np.argmin(real_bers)
best_est_idx = np.argmin(pred_bers)

print("\nBest threshold from real BER      :", thresholds[best_real_idx])
print("Minimum real BER                  :", real_bers[best_real_idx])
print("Real BER region there             :", REGION_LABELS.get(int(real_regions[best_real_idx]), "?"))

print("Best threshold from estimated BER :", thresholds[best_est_idx])
print("Estimated BER at that threshold   :", pred_bers[best_est_idx])
print("Predicted BER region there        :", REGION_LABELS.get(int(pred_regions[best_est_idx]), "?"))

# =========================
# Plot 1: log-scale BER
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["estimated_BER"], label="Estimated BER")
plt.plot(results["threshold"], results["real_BER"], label="Real BER")
plt.yscale("log")
plt.xlabel("Threshold")
plt.ylabel("BER")
plt.title(f"Threshold vs BER (log scale) — mem_len={case['mem_len']}")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "01_ber_log_scale.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/01_ber_log_scale.png")

# =========================
# Plot 2: linear-scale BER
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["estimated_BER"], label="Estimated BER")
plt.plot(results["threshold"], results["real_BER"], label="Real BER")
plt.yscale("linear")
plt.xlabel("Threshold")
plt.ylabel("BER")
plt.title(f"Threshold vs BER (linear scale) — mem_len={case['mem_len']}")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "02_ber_linear_scale.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/02_ber_linear_scale.png")

# =========================
# Plot 3: predicted vs true region
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["predicted_region"], label="Predicted region")
plt.plot(results["threshold"], results["real_region"], label="True region")
plt.xlabel("Threshold")
plt.ylabel("BER Region Class")
plt.title(f"Threshold vs BER Region — mem_len={case['mem_len']}")
plt.yticks(list(REGION_LABELS.keys()),
           [REGION_LABELS[k] for k in sorted(REGION_LABELS.keys())],
           fontsize=7)
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "03_region_comparison.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/03_region_comparison.png")

# =========================
# Plot 4: ordinal threshold probabilities
# =========================
plt.figure(figsize=(12, 7))
for i, thr_val in enumerate(ordinal_thresholds):
    plt.plot(thresholds, pred_region_probs[:, i], label=f"P(y >= {thr_val:g})")
plt.xlabel("Threshold")
plt.ylabel("Ordinal Probability")
plt.title(f"Ordinal Head Outputs Across Threshold Sweep — mem_len={case['mem_len']}")
plt.legend(fontsize=7, ncol=2)
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "04_ordinal_probabilities.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/04_ordinal_probabilities.png")

# =========================
# Plot 5: absolute error by threshold
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["abs_error"], label="Abs raw error", alpha=0.8)
plt.plot(results["threshold"], results["abs_log10_error"], label="Abs log10 error", alpha=0.8)
plt.yscale("log")
plt.xlabel("Threshold")
plt.ylabel("Error")
plt.title(f"Prediction Error Across Threshold Sweep — mem_len={case['mem_len']}")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "05_error_by_threshold.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/05_error_by_threshold.png")

Generated physical scenario
radius    = 4.999903475776163
distance  = 13.063595613360427
diffusion = 63.15707394395149
Ts        = 0.5834416162676823
N         = 2026
mem_len   = 14
P         = [0.03545102 0.042582   0.02704795 0.01857409 0.01368091 0.01059455
 0.00850978 0.00702648 0.00592783 0.00508774 0.00442855 0.00390015
 0.00346895 0.00311165]
P_scaled  = [71.82377081 86.27113641 54.79914164 37.63111579 27.71752004 21.46456524
 17.24082032 14.23564842 12.00979138 10.30776824  8.9722469   7.90171201
  7.0280834   6.304198  ]
variances = [69.27754472 82.59753869 53.31693734 36.93215188 27.33831919 21.23715776
 17.09410468 14.13562193 11.93859933 10.25532496  8.93251283  7.87089412
  7.00370336  6.28458156]
Threshold search interval: 0.0 to 383.7075186062634
sum(P_scaled) = 383.7075186062634

Model loaded: max_past_seq_len=13, global_dim=7, num_ordinal=13
Scaling strategy: position_independent
    threshold  real_BER  estimated_BER  real_region  predicted_region  \
0    0.000000  0.

In [3]:
import os
import re
import copy
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.multiprocessing as mp
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

mp.set_sharing_strategy("file_system")

# =========================
# Configuration
# =========================
BASE_DIR = "./"

DATA_PATHS = [
    os.path.join(BASE_DIR, "data_physics_with_variances_total.csv"),
]

NROWS_PER_DATASET = 5_000_000

BEST_MODEL_SAVE_PATH = os.path.join(
    BASE_DIR,
    "physics_fulltrain_highber_best_v2_target020_040_transformer_multitask.pth",
)
LAST_MODEL_SAVE_PATH = os.path.join(
    BASE_DIR,
    "physics_fulltrain_highber_last_v2_target020_040_transformer_multitask.pth",
)
SCALER_SAVE_PATH = os.path.join(
    BASE_DIR,
    "physics_fulltrain_highber_scalers_v2_target020_040_transformer_multitask.pkl",
)

EPS = 1e-12
LOG10_HALF = float(np.log10(0.5))

ORDINAL_THRESHOLDS = [
    1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1,
    0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45,
]
NUM_ORDINAL_THRESHOLDS = len(ORDINAL_THRESHOLDS)
NUM_REGION_CLASSES = NUM_ORDINAL_THRESHOLDS + 1


# =========================
# Utilities
# =========================
def get_sorted_seq_cols(columns, prefix):
    pattern = re.compile(rf"^{re.escape(prefix)}_(\d+)$")
    matched = []
    for col in columns:
        m = pattern.match(col)
        if m:
            matched.append((int(m.group(1)), col))
    matched.sort(key=lambda x: x[0])
    return [col for _, col in matched]


def make_strat_bins(y_log, n_bins=10):
    y_flat = y_log.reshape(-1)
    quantiles = np.linspace(0, 1, n_bins + 1)
    edges = np.quantile(y_flat, quantiles)
    edges = np.unique(edges)

    if len(edges) < 3:
        return None

    bins = np.digitize(y_flat, edges[1:-1], right=True)
    counts = np.bincount(bins)
    if np.any(counts < 2):
        return None
    return bins


def has_nonfinite_tensor(x):
    return not torch.isfinite(x).all().item()


def validate_required_columns(df, file_path):
    if "mem_len" not in df.columns:
        raise ValueError(f"Required column 'mem_len' not found in {file_path}")
    if "N" not in df.columns:
        raise ValueError(f"Required column 'N' not found in {file_path}")

    tap_cols = get_sorted_seq_cols(df.columns, "tap")
    var_cols = get_sorted_seq_cols(df.columns, "var")

    if not tap_cols:
        raise ValueError(f"No tap_* columns found in {file_path}")
    if not var_cols:
        raise ValueError(f"No var_* columns found in {file_path}")
    if len(tap_cols) != len(var_cols):
        raise ValueError(
            f"tap/var length mismatch in {file_path}: "
            f"{len(tap_cols)} tap cols vs {len(var_cols)} var cols"
        )

    required_cols = tap_cols + var_cols + ["threshold", "BER", "N"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in {file_path}: {missing}")

    return tap_cols, var_cols


def load_and_merge_data(csv_paths, nrows_per_dataset):
    dfs = []
    reference_tap_cols = None
    reference_var_cols = None

    for path in csv_paths:
        if not os.path.exists(path):
            raise FileNotFoundError(f"Dataset not found: {path}")

        print(f"Loading up to {nrows_per_dataset:,} rows from: {path}")
        df = pd.read_csv(path, nrows=nrows_per_dataset)

        tap_cols, var_cols = validate_required_columns(df, path)

        if reference_tap_cols is None:
            reference_tap_cols = tap_cols
            reference_var_cols = var_cols
        else:
            if tap_cols != reference_tap_cols:
                raise ValueError("tap columns do not match across files.")
            if var_cols != reference_var_cols:
                raise ValueError("var columns do not match across files.")

        df["source_dataset"] = os.path.basename(path)
        dfs.append(df)

    merged = pd.concat(dfs, ignore_index=True)
    print(f"Combined rows before filtering: {len(merged):,}")

    return merged, reference_tap_cols, reference_var_cols


def y_log_to_raw_np(y_log):
    return np.clip(10 ** y_log, EPS, 0.5)


def raw_to_region_labels_np(y_raw):
    y = np.asarray(y_raw).reshape(-1)
    labels = np.searchsorted(np.asarray(ORDINAL_THRESHOLDS), y, side="right")
    return labels.astype(np.int64)


def region_labels_to_ordinal_targets_np(labels, num_thresholds=NUM_ORDINAL_THRESHOLDS):
    labels = np.asarray(labels).reshape(-1)
    thresholds = np.arange(1, num_thresholds + 1, dtype=np.int64)
    ordinal = (labels[:, None] >= thresholds[None, :]).astype(np.float32)
    return ordinal


def ordinal_logits_to_region_labels_torch(logits):
    probs = torch.sigmoid(logits)
    passed = (probs >= 0.5).sum(dim=1)
    return passed.long()


def compute_region_class_weights(labels_np, num_classes=NUM_REGION_CLASSES, max_weight=8.0):
    counts = np.bincount(labels_np.reshape(-1), minlength=num_classes).astype(np.float64)
    counts = np.maximum(counts, 1.0)
    weights = counts.sum() / (num_classes * counts)
    weights = np.sqrt(weights)
    weights = np.clip(weights, 1.0, max_weight)
    return weights.astype(np.float32)


def compute_ordinal_pos_weights_from_region_labels(
    region_labels_np,
    num_thresholds=NUM_ORDINAL_THRESHOLDS,
    max_weight=20.0,
):
    ordinal_targets = region_labels_to_ordinal_targets_np(region_labels_np, num_thresholds)
    pos_counts = ordinal_targets.sum(axis=0)
    neg_counts = ordinal_targets.shape[0] - pos_counts
    pos_counts = np.maximum(pos_counts, 1.0)
    pos_weight = neg_counts / pos_counts
    pos_weight = np.clip(pos_weight, 1.0, max_weight)
    return pos_weight.astype(np.float32)


# =========================
# Position-Independent Scaling
# =========================
class SharedFeatureScaler:
    """A scaler that stores a single (mean, std) per feature channel,
    shared across all sequence positions.

    For the first token (always position 0) we keep a separate scaler,
    since the current-symbol tap has a genuinely different distribution
    than the ISI taps at positions 1+.

    For all past positions we pool every valid entry into one distribution
    and fit a single mean/std.  This makes the scaler independent of
    sequence length — at inference time, any number of past taps can be
    scaled with the same parameters.
    """

    def __init__(self):
        self.first_mean = None  # shape [n_features]
        self.first_std = None
        self.past_mean = None   # shape [n_features]
        self.past_std = None

    def fit(self, first_data_list, past_data_list, past_valid_list):
        """
        Args:
            first_data_list: list of arrays, each [N, n_features] for the
                             first-token features (taps[:,0], vars[:,0], ...).
                             Concatenated across splits if desired, or just train.
            past_data_list:  list of arrays, each [N, L_past, n_features] or
                             [N, L_past] for a single feature channel.
            past_valid_list: list of bool arrays, each [N, L_past], True = valid.
        """
        # --- First token ---
        first_all = np.concatenate(first_data_list, axis=0)  # [N_total, F]
        self.first_mean = first_all.mean(axis=0).astype(np.float64)
        self.first_std = first_all.std(axis=0).astype(np.float64)
        self.first_std = np.maximum(self.first_std, 1e-12)

        # --- Past tokens (pool all valid entries per feature) ---
        valid_entries = []
        for data, valid in zip(past_data_list, past_valid_list):
            if data.ndim == 2:
                # single feature: [N, L] -> expand to [N, L, 1]
                data = data[:, :, None]
            # data: [N, L, F], valid: [N, L]
            valid_expanded = valid[:, :, None]  # [N, L, 1]
            # Gather valid entries: [?, F]
            valid_entries.append(data[np.broadcast_to(valid_expanded, data.shape)].reshape(-1, data.shape[-1]))

        pooled = np.concatenate(valid_entries, axis=0)  # [total_valid, F]
        self.past_mean = pooled.mean(axis=0).astype(np.float64)
        self.past_std = pooled.std(axis=0).astype(np.float64)
        self.past_std = np.maximum(self.past_std, 1e-12)

    def transform_first(self, first_data):
        """first_data: [N, F] -> scaled [N, F]"""
        return ((first_data - self.first_mean) / self.first_std).astype(np.float32)

    def transform_past(self, past_data, past_lens):
        """past_data: [N, L, F] or [N, L] -> scaled + re-zeroed [N, L, ...].
        past_lens: [N] int, number of valid past positions per row."""
        shape = past_data.shape
        if past_data.ndim == 2:
            scaled = ((past_data - self.past_mean[0]) / self.past_std[0]).astype(np.float32)
            L = shape[1]
        else:
            scaled = ((past_data - self.past_mean) / self.past_std).astype(np.float32)
            L = shape[1]

        # Re-zero padding positions
        valid = (np.arange(L)[None, :] < past_lens[:, None])  # [N, L]
        if scaled.ndim == 3:
            valid = valid[:, :, None]
        scaled = scaled * valid.astype(np.float32)
        return scaled


def fit_shared_scalar_scaler(train_data, train_valid_mask):
    """Fit a single-feature StandardScaler on pooled valid entries.
    train_data: [N, L], train_valid_mask: [N, L] bool.
    Returns (mean, std) as floats."""
    entries = train_data[train_valid_mask].reshape(-1)
    if len(entries) == 0:
        entries = train_data.reshape(-1)
    mean = float(entries.mean())
    std = float(max(entries.std(), 1e-12))
    return mean, std


def apply_shared_scale(data, mean, std, valid_mask=None):
    """Scale data with a single (mean, std), optionally re-zero invalid positions.
    data: [N, L] or [N, 1], valid_mask: [N, L] bool or None."""
    scaled = ((data - mean) / std).astype(np.float32)
    if valid_mask is not None and scaled.ndim == 2 and valid_mask.shape == scaled.shape:
        scaled = scaled * valid_mask.astype(np.float32)
    return scaled


# =========================
# Targeted Regression Loss
# =========================
class StableMultiObjectiveBERBoundedLogLoss(nn.Module):
    def __init__(
        self,
        log_delta=0.35,
        raw_delta=0.006,
        rel_delta=0.03,
        alpha_log=0.45,
        beta_raw=0.35,
        gamma_rel=0.20,
        use_regime_weights=True,
    ):
        super().__init__()
        self.log_delta = log_delta
        self.raw_delta = raw_delta
        self.rel_delta = rel_delta
        self.alpha_log = alpha_log
        self.beta_raw = beta_raw
        self.gamma_rel = gamma_rel
        self.use_regime_weights = use_regime_weights

    @staticmethod
    def huber_elementwise(pred, target, delta):
        err = pred - target
        abs_err = err.abs()
        return torch.where(
            abs_err < delta,
            0.5 * err * err,
            delta * (abs_err - 0.5 * delta),
        )

    @staticmethod
    def raw_to_pred_log(pred_raw_unconstrained):
        log10_half = torch.log10(
            torch.tensor(
                0.5,
                device=pred_raw_unconstrained.device,
                dtype=pred_raw_unconstrained.dtype,
            )
        )
        return log10_half - F.softplus(pred_raw_unconstrained)

    @staticmethod
    def raw_to_pred_ber(pred_raw_unconstrained):
        pred_log = StableMultiObjectiveBERBoundedLogLoss.raw_to_pred_log(
            pred_raw_unconstrained
        )
        return torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)

    def forward(self, pred_raw_unconstrained, target_log):
        pred_log = self.raw_to_pred_log(pred_raw_unconstrained)
        pred_raw = torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)

        target_raw = torch.pow(10.0, target_log).clamp(min=EPS, max=0.5)
        target_log_for_loss = torch.log10(target_raw)

        log_loss = self.huber_elementwise(pred_log, target_log_for_loss, self.log_delta)
        raw_loss = self.huber_elementwise(pred_raw, target_raw, self.raw_delta)
        rel_err = (pred_raw - target_raw) / torch.clamp(target_raw, min=1e-6)
        rel_loss = self.huber_elementwise(rel_err, torch.zeros_like(rel_err), self.rel_delta)

        total = (
            self.alpha_log * log_loss
            + self.beta_raw * raw_loss
            + self.gamma_rel * rel_loss
        )

        if self.use_regime_weights:
            weights = torch.ones_like(target_raw)
            weights = torch.where(
                (target_raw >= 1e-2) & (target_raw < 0.1),
                torch.full_like(weights, 1.5), weights,
            )
            weights = torch.where(
                (target_raw >= 0.1) & (target_raw < 0.15),
                torch.full_like(weights, 2.5), weights,
            )
            weights = torch.where(
                (target_raw >= 0.15) & (target_raw < 0.2),
                torch.full_like(weights, 3.0), weights,
            )
            weights = torch.where(
                (target_raw >= 0.2) & (target_raw < 0.3),
                torch.full_like(weights, 4.0), weights,
            )
            weights = torch.where(
                (target_raw >= 0.3) & (target_raw < 0.4),
                torch.full_like(weights, 4.5), weights,
            )
            weights = torch.where(
                (target_raw >= 0.4) & (target_raw < 0.45),
                torch.full_like(weights, 3.0), weights,
            )
            weights = torch.where(
                target_raw >= 0.45,
                torch.full_like(weights, 2.5), weights,
            )
            total = total * weights

        return total.mean()


class OrdinalBCELoss(nn.Module):
    def __init__(self, pos_weight=None, reduction="mean"):
        super().__init__()
        if pos_weight is not None and not isinstance(pos_weight, torch.Tensor):
            pos_weight = torch.tensor(pos_weight, dtype=torch.float32)
        self.register_buffer(
            "pos_weight", pos_weight if pos_weight is not None else None
        )
        self.reduction = reduction

    def forward(self, logits, ordinal_targets):
        return F.binary_cross_entropy_with_logits(
            logits,
            ordinal_targets,
            pos_weight=self.pos_weight,
            reduction=self.reduction,
        )


class MultiTaskBERLoss(nn.Module):
    def __init__(self, reg_loss, ord_loss, lambda_ord=0.25):
        super().__init__()
        self.reg_loss = reg_loss
        self.ord_loss = ord_loss
        self.lambda_ord = lambda_ord

    def forward(self, pred_raw_unconstrained, ord_logits, target_log, target_ord):
        reg = self.reg_loss(pred_raw_unconstrained, target_log)
        ordl = self.ord_loss(ord_logits, target_ord)
        total = reg + self.lambda_ord * ordl
        return total, reg.detach(), ordl.detach()


# =========================
# Set Transformer Blocks
# =========================
class SetSelfAttentionBlock(nn.Module):
    def __init__(self, d_model, num_heads, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norm2 = nn.LayerNorm(d_model)

        hidden = int(d_model * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x, key_padding_mask=None):
        """
        Args:
            x: [B, L, D]
            key_padding_mask: [B, L] bool, True = padding (ignore)
        """
        y = self.norm1(x)
        attn_out, _ = self.attn(
            y, y, y,
            need_weights=False,
            key_padding_mask=key_padding_mask,
        )
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x


class ConditionalFeatureModulation(nn.Module):
    def __init__(self, cond_dim, feat_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, 2 * feat_dim),
        )
        self.scale = nn.Parameter(torch.tensor(0.25, dtype=torch.float32))

    def forward(self, x, cond):
        gamma_beta = self.net(cond)
        gamma, beta = torch.chunk(gamma_beta, 2, dim=-1)

        if x.dim() == 3:
            gamma = gamma.unsqueeze(1)
            beta = beta.unsqueeze(1)

        return x * (1.0 + self.scale * gamma) + self.scale * beta


class SetAttentionPooling(nn.Module):
    def __init__(self, d_model, hidden_dim=128):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x, key_padding_mask=None):
        """
        Args:
            x: [B, L, D]
            key_padding_mask: [B, L] bool, True = padding
        Returns:
            pooled: [B, D]
        """
        logits = self.score(x)  # [B, L, 1]

        if key_padding_mask is not None:
            logits = logits.masked_fill(
                key_padding_mask.unsqueeze(-1), float("-inf")
            )

        weights = torch.softmax(logits, dim=1)  # [B, L, 1]
        # Safety: all-masked rows produce NaN from softmax(-inf); replace with 0
        weights = torch.nan_to_num(weights, nan=0.0)

        pooled = (weights * x).sum(dim=1)  # [B, D]
        return pooled


# =========================
# Model
# =========================
class BERFirstTokenSetTransformerBoundedLogHalfMultiTask(nn.Module):
    def __init__(
        self,
        max_past_seq_len,
        token_dim=4,
        first_token_dim=4,
        threshold_dim=1,
        global_dim=7,
        d_model=128,
        num_set_layers=4,
        num_heads=4,
        mlp_ratio=4.0,
        dropout=0.10,
        num_ordinal_thresholds=NUM_ORDINAL_THRESHOLDS,
    ):
        super().__init__()

        self.max_past_seq_len = max_past_seq_len
        self.token_dim = token_dim
        self.first_token_dim = first_token_dim
        self.d_model = d_model
        self.num_ordinal_thresholds = num_ordinal_thresholds

        self.first_proj = nn.Sequential(
            nn.Linear(first_token_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.token_proj = nn.Sequential(
            nn.Linear(token_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.thr_embed = nn.Sequential(
            nn.Linear(threshold_dim, 32),
            nn.GELU(),
            nn.Linear(32, 32),
            nn.GELU(),
        )

        self.global_embed = nn.Sequential(
            nn.Linear(global_dim, 96),
            nn.GELU(),
            nn.Linear(96, 96),
            nn.GELU(),
        )

        cond_dim = 32 + 96

        self.first_cond_mod = ConditionalFeatureModulation(
            cond_dim=cond_dim, feat_dim=d_model, hidden_dim=256,
        )
        self.set_cond_mod = ConditionalFeatureModulation(
            cond_dim=cond_dim, feat_dim=d_model, hidden_dim=256,
        )

        self.set_blocks = nn.ModuleList(
            [
                SetSelfAttentionBlock(
                    d_model=d_model,
                    num_heads=num_heads,
                    mlp_ratio=mlp_ratio,
                    dropout=dropout,
                )
                for _ in range(num_set_layers)
            ]
        )

        self.first_refine = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
        )

        self.final_set_norm = nn.LayerNorm(d_model)
        self.set_attn_pool = SetAttentionPooling(d_model=d_model, hidden_dim=128)

        set_summary_dim = 3 * d_model
        head_in = d_model + set_summary_dim + cond_dim

        self.reg_head = nn.Sequential(
            nn.Linear(head_in, 256),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(128, 1),
        )

        self.ord_head = nn.Sequential(
            nn.Linear(head_in, 256),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(128, num_ordinal_thresholds),
        )

    @staticmethod
    def raw_to_log10ber(pred_raw_unconstrained):
        log10_half = torch.log10(
            torch.tensor(
                0.5,
                device=pred_raw_unconstrained.device,
                dtype=pred_raw_unconstrained.dtype,
            )
        )
        return log10_half - F.softplus(pred_raw_unconstrained)

    @staticmethod
    def raw_to_ber(pred_raw_unconstrained):
        pred_log = BERFirstTokenSetTransformerBoundedLogHalfMultiTask.raw_to_log10ber(
            pred_raw_unconstrained
        )
        return torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)

    def forward(self, first_token, past_tokens, global_feats, threshold, key_padding_mask=None):
        """
        Args:
            first_token:       [B, first_token_dim]
            past_tokens:       [B, L, token_dim]  (L can vary between training and inference)
            global_feats:      [B, global_dim]
            threshold:         [B, 1]
            key_padding_mask:  [B, L] bool, True = padding position
        """
        thr = self.thr_embed(threshold)
        g = self.global_embed(global_feats)
        cond = torch.cat([thr, g], dim=-1)

        # First token path
        first_x = self.first_proj(first_token)
        first_x = self.first_cond_mod(first_x, cond)
        first_x = first_x + self.first_refine(first_x)

        # Past tokens path with mask
        set_x = self.token_proj(past_tokens)
        set_x = self.set_cond_mod(set_x, cond)

        for block in self.set_blocks:
            set_x = block(set_x, key_padding_mask=key_padding_mask)

        set_x = self.final_set_norm(set_x)

        # --- Masked pooling ---
        if key_padding_mask is not None:
            valid_mask = (~key_padding_mask).unsqueeze(-1).float()  # [B, L, 1]
        else:
            valid_mask = torch.ones(
                set_x.shape[0], set_x.shape[1], 1,
                device=set_x.device, dtype=set_x.dtype,
            )

        # Attention pooling
        pooled_attn = self.set_attn_pool(set_x, key_padding_mask=key_padding_mask)

        # Masked mean pooling
        valid_counts = valid_mask.sum(dim=1).clamp(min=1.0)  # [B, 1]
        pooled_mean = (set_x * valid_mask).sum(dim=1) / valid_counts

        # Masked max pooling
        if key_padding_mask is not None:
            x_for_max = set_x.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
        else:
            x_for_max = set_x
        pooled_max = x_for_max.amax(dim=1)
        pooled_max = torch.nan_to_num(pooled_max, nan=0.0, posinf=0.0, neginf=0.0)

        set_summary = torch.cat([pooled_attn, pooled_mean, pooled_max], dim=-1)
        fused = torch.cat([first_x, set_summary, cond], dim=-1)

        pred_raw_unconstrained = self.reg_head(fused)
        ord_logits = self.ord_head(fused)

        return pred_raw_unconstrained, ord_logits


# =========================
# Data
# =========================
def prepare_data(csv_paths, batch_size=256, nrows_per_dataset=2_500_000, num_workers=0):
    df, tap_cols, var_cols = load_and_merge_data(csv_paths, nrows_per_dataset)

    df = df[df["mem_len"] != 1].copy()

    df[tap_cols] = df[tap_cols].fillna(0.0)
    df[var_cols] = df[var_cols].fillna(0.0)
    df["threshold"] = df["threshold"].fillna(0.0)
    df["BER"] = df["BER"].fillna(0.0)
    df["N"] = df["N"].fillna(0.0)

    df = df[(df["threshold"] > 0) & (df["BER"] > 0) & (df["N"] > 0)].copy()
    df["BER"] = df["BER"].clip(lower=EPS, upper=0.5)

    print(f"Rows after cleaning/filtering: {len(df):,}")

    # ---- Extract raw arrays ----
    X_taps_raw = df[tap_cols].to_numpy(dtype=np.float32)
    X_vars_raw = df[var_cols].to_numpy(dtype=np.float32)
    num_molecules = df["N"].to_numpy(dtype=np.float32).reshape(-1, 1)
    mem_len = df["mem_len"].to_numpy(dtype=np.int64)

    X_thr_raw = df["threshold"].to_numpy(dtype=np.float32).reshape(-1, 1)
    y_raw = df["BER"].to_numpy(dtype=np.float32).reshape(-1, 1)

    X_thr = np.log10(X_thr_raw + EPS).astype(np.float32)
    y_log = np.log10(y_raw + EPS).astype(np.float32)

    # ---- Feature engineering ----
    X_taps_feat = (X_taps_raw * num_molecules).astype(np.float32)

    if np.any(X_vars_raw < 0):
        raise ValueError("Variance columns contain negative values.")

    X_vars_feat = X_vars_raw.astype(np.float32)
    abs_taps_raw = np.abs(X_taps_feat).astype(np.float32)
    snr_raw = np.log10((X_taps_feat ** 2) / (X_vars_raw + EPS) + EPS).astype(np.float32)

    L_full = X_taps_feat.shape[1]

    # ---- Validity masks ----
    mem_len_clamped = np.minimum(mem_len, L_full)
    valid_full = (np.arange(L_full)[None, :] < mem_len_clamped[:, None])  # [N, L_full]
    valid_past = valid_full[:, 1:]  # [N, L_full-1]
    valid_past_float = valid_past.astype(np.float32)
    past_lens = np.maximum(mem_len_clamped - 1, 0).astype(np.int64)

    # ---- Mask-aware global features ----
    first_mean = X_taps_feat[:, 0:1]
    past_means = X_taps_feat[:, 1:]
    first_var = X_vars_feat[:, 0:1]
    past_vars = X_vars_feat[:, 1:]

    past_means_masked = past_means * valid_past_float
    past_vars_masked = past_vars * valid_past_float

    mu0_raw = (0.5 * np.sum(past_means_masked, axis=1, keepdims=True)).astype(np.float32)
    mu1_raw = (first_mean + mu0_raw).astype(np.float32)

    var0_raw = (0.5 * np.sum(past_vars_masked, axis=1, keepdims=True)).astype(np.float32)
    var1_raw = (first_var + var0_raw).astype(np.float32)

    std0_raw = np.sqrt(np.maximum(var0_raw, EPS)).astype(np.float32)
    std1_raw = np.sqrt(np.maximum(var1_raw, EPS)).astype(np.float32)

    z0_raw = ((X_thr_raw - mu0_raw) / (std0_raw + EPS)).astype(np.float32)
    z1_raw = ((mu1_raw - X_thr_raw) / (std1_raw + EPS)).astype(np.float32)

    harmonic_side_z_raw = (
        2.0 / (1.0 / (z0_raw + EPS) + 1.0 / (z1_raw + EPS))
    ).astype(np.float32)
    abs_diff_side_z_raw = np.abs(z0_raw - z1_raw).astype(np.float32)
    harmonic_minus_gap_raw = (
        harmonic_side_z_raw - 0.25 * abs_diff_side_z_raw
    ).astype(np.float32)

    # ---- Scenario-level features (threshold-INDEPENDENT) ----
    # These capture intrinsic difficulty of the physical scenario.

    # 1. NSID: Normalized Signal-Interference Difference, range ~ [-1, +1]
    signal_raw = first_mean  # P_0 * N, shape [N, 1]
    isi_raw = np.sum(past_means_masked, axis=1, keepdims=True)  # [N, 1]
    nsid_raw = ((signal_raw - isi_raw) / (signal_raw + isi_raw + EPS)).astype(np.float32)

    # 2. Log Harmonic Discriminability (threshold-free)
    gap_raw = signal_raw  # mu1 - mu0 = P_0 * N
    d0_raw = (gap_raw / (std0_raw + EPS)).astype(np.float32)
    d1_raw = (gap_raw / (std1_raw + EPS)).astype(np.float32)
    harmonic_discrim_raw = (2.0 * d0_raw * d1_raw / (d0_raw + d1_raw + EPS)).astype(np.float32)
    log_harmonic_discrim_raw = np.log10(harmonic_discrim_raw + EPS).astype(np.float32)

    # 3. Discriminability Asymmetry
    discrim_asymmetry_raw = (np.abs(d0_raw - d1_raw) / (d0_raw + d1_raw + EPS)).astype(np.float32)

    # 4. Herfindahl Index of ISI concentration [1/n_past, 1]
    past_taps_sum = np.sum(past_means_masked, axis=1, keepdims=True)  # [N, 1]
    past_shares = past_means_masked / (past_taps_sum + EPS)           # [N, L_past]
    herfindahl_raw = np.sum(past_shares ** 2, axis=1, keepdims=True).astype(np.float32)  # [N, 1]

    # ---- Labels ----
    region_labels = raw_to_region_labels_np(y_raw)
    ordinal_targets = region_labels_to_ordinal_targets_np(region_labels)
    strat_labels = make_strat_bins(y_log, n_bins=10)

    # ---- Train / temp split ----
    split_args = dict(test_size=0.30, random_state=42)
    if strat_labels is not None:
        split_args["stratify"] = strat_labels

    (
        t_taps_feat, temp_taps_feat,
        t_vars_feat, temp_vars_feat,
        t_abs_raw, temp_abs_raw,
        t_snr_raw, temp_snr_raw,
        t_z0_raw, temp_z0_raw,
        t_z1_raw, temp_z1_raw,
        t_hmg_raw, temp_hmg_raw,
        t_nsid_raw, temp_nsid_raw,
        t_log_hd_raw, temp_log_hd_raw,
        t_da_raw, temp_da_raw,
        t_herf_raw, temp_herf_raw,
        t_thr, temp_thr,
        t_y_log, temp_y_log,
        t_region, temp_region,
        t_ord, temp_ord,
        t_past_lens, temp_past_lens,
    ) = train_test_split(
        X_taps_feat, X_vars_feat, abs_taps_raw, snr_raw,
        z0_raw, z1_raw, harmonic_minus_gap_raw,
        nsid_raw, log_harmonic_discrim_raw, discrim_asymmetry_raw, herfindahl_raw,
        X_thr,
        y_log, region_labels, ordinal_targets, past_lens,
        **split_args,
    )

    # ---- temp -> val / test split ----
    temp_strat = make_strat_bins(temp_y_log, n_bins=6)
    split_args2 = dict(test_size=0.50, random_state=42)
    if temp_strat is not None:
        split_args2["stratify"] = temp_strat

    (
        v_taps_feat, te_taps_feat,
        v_vars_feat, te_vars_feat,
        v_abs_raw, te_abs_raw,
        v_snr_raw, te_snr_raw,
        v_z0_raw, te_z0_raw,
        v_z1_raw, te_z1_raw,
        v_hmg_raw, te_hmg_raw,
        v_nsid_raw, te_nsid_raw,
        v_log_hd_raw, te_log_hd_raw,
        v_da_raw, te_da_raw,
        v_herf_raw, te_herf_raw,
        v_thr, te_thr,
        v_y_log, te_y_log,
        v_region, te_region,
        v_ord, te_ord,
        v_past_lens, te_past_lens,
    ) = train_test_split(
        temp_taps_feat, temp_vars_feat, temp_abs_raw, temp_snr_raw,
        temp_z0_raw, temp_z1_raw, temp_hmg_raw,
        temp_nsid_raw, temp_log_hd_raw, temp_da_raw, temp_herf_raw,
        temp_thr,
        temp_y_log, temp_region, temp_ord, temp_past_lens,
        **split_args2,
    )

    # =========================================================
    # Position-independent scaling
    # =========================================================
    # For each feature channel (taps, vars, abs, snr) we fit:
    #   - A SEPARATE mean/std for position 0 (first token)
    #   - A SHARED mean/std pooled across ALL valid past positions 1+
    #
    # This decouples the scaler from the number of columns,
    # so at inference any mem_len works with the same parameters.
    # =========================================================

    L_past = L_full - 1

    # Build per-split past validity masks
    t_valid_past = (np.arange(L_past)[None, :] < t_past_lens[:, None])

    # --- Fit first-token scalers (on train only) ---
    first_tap_mean, first_tap_std = float(t_taps_feat[:, 0].mean()), max(float(t_taps_feat[:, 0].std()), 1e-12)
    first_var_mean, first_var_std = float(t_vars_feat[:, 0].mean()), max(float(t_vars_feat[:, 0].std()), 1e-12)
    first_abs_mean, first_abs_std = float(t_abs_raw[:, 0].mean()), max(float(t_abs_raw[:, 0].std()), 1e-12)
    first_snr_mean, first_snr_std = float(t_snr_raw[:, 0].mean()), max(float(t_snr_raw[:, 0].std()), 1e-12)

    # --- Fit shared past scalers (pool all valid past entries from train) ---
    past_tap_mean, past_tap_std = fit_shared_scalar_scaler(t_taps_feat[:, 1:], t_valid_past)
    past_var_mean, past_var_std = fit_shared_scalar_scaler(t_vars_feat[:, 1:], t_valid_past)
    past_abs_mean, past_abs_std = fit_shared_scalar_scaler(t_abs_raw[:, 1:], t_valid_past)
    past_snr_mean, past_snr_std = fit_shared_scalar_scaler(t_snr_raw[:, 1:], t_valid_past)

    # --- Global feature scalers (standard, shape [N, 1]) ---
    z0_scaler = StandardScaler().fit(t_z0_raw)
    z1_scaler = StandardScaler().fit(t_z1_raw)
    hmg_scaler = StandardScaler().fit(t_hmg_raw)
    nsid_scaler = StandardScaler().fit(t_nsid_raw)
    log_hd_scaler = StandardScaler().fit(t_log_hd_raw)
    da_scaler = StandardScaler().fit(t_da_raw)
    herf_scaler = StandardScaler().fit(t_herf_raw)
    thr_scaler = StandardScaler().fit(t_thr)

    # ---- Helper: build first_token and past_tokens arrays ----
    def build_tokens(taps_feat, vars_feat, abs_raw, snr_raw, p_lens):
        """Returns first_token [N, 4] and past_tokens [N, L_past, 4], both scaled + re-zeroed."""
        N = taps_feat.shape[0]

        # Scale first token
        ft_tap = ((taps_feat[:, 0] - first_tap_mean) / first_tap_std).astype(np.float32)
        ft_var = ((vars_feat[:, 0] - first_var_mean) / first_var_std).astype(np.float32)
        ft_abs = ((abs_raw[:, 0] - first_abs_mean) / first_abs_std).astype(np.float32)
        ft_snr = ((snr_raw[:, 0] - first_snr_mean) / first_snr_std).astype(np.float32)
        first_token = np.stack([ft_tap, ft_var, ft_abs, ft_snr], axis=1).astype(np.float32)

        # Scale past tokens (shared scaler) + re-zero
        pt_tap = apply_shared_scale(taps_feat[:, 1:], past_tap_mean, past_tap_std,
                                    (np.arange(L_past)[None, :] < p_lens[:, None]))
        pt_var = apply_shared_scale(vars_feat[:, 1:], past_var_mean, past_var_std,
                                    (np.arange(L_past)[None, :] < p_lens[:, None]))
        pt_abs = apply_shared_scale(abs_raw[:, 1:], past_abs_mean, past_abs_std,
                                    (np.arange(L_past)[None, :] < p_lens[:, None]))
        pt_snr = apply_shared_scale(snr_raw[:, 1:], past_snr_mean, past_snr_std,
                                    (np.arange(L_past)[None, :] < p_lens[:, None]))

        past_tokens = np.stack([pt_tap, pt_var, pt_abs, pt_snr], axis=2).astype(np.float32)

        return first_token, past_tokens

    def build_global(z0, z1, hmg, nsid, log_hd, da, herf):
        return np.concatenate([
            z0_scaler.transform(z0),
            z1_scaler.transform(z1),
            hmg_scaler.transform(hmg),
            nsid_scaler.transform(nsid),
            log_hd_scaler.transform(log_hd),
            da_scaler.transform(da),
            herf_scaler.transform(herf),
        ], axis=1).astype(np.float32)

    def build_padding_mask(p_lens, L_max):
        return (np.arange(L_max)[None, :] >= p_lens[:, None])  # True = padding

    # ---- Build all splits ----
    t_first, t_past = build_tokens(t_taps_feat, t_vars_feat, t_abs_raw, t_snr_raw, t_past_lens)
    v_first, v_past = build_tokens(v_taps_feat, v_vars_feat, v_abs_raw, v_snr_raw, v_past_lens)
    te_first, te_past = build_tokens(te_taps_feat, te_vars_feat, te_abs_raw, te_snr_raw, te_past_lens)

    t_global = build_global(t_z0_raw, t_z1_raw, t_hmg_raw,
                            t_nsid_raw, t_log_hd_raw, t_da_raw, t_herf_raw)
    v_global = build_global(v_z0_raw, v_z1_raw, v_hmg_raw,
                            v_nsid_raw, v_log_hd_raw, v_da_raw, v_herf_raw)
    te_global = build_global(te_z0_raw, te_z1_raw, te_hmg_raw,
                             te_nsid_raw, te_log_hd_raw, te_da_raw, te_herf_raw)

    t_thr_s = thr_scaler.transform(t_thr).astype(np.float32)
    v_thr_s = thr_scaler.transform(v_thr).astype(np.float32)
    te_thr_s = thr_scaler.transform(te_thr).astype(np.float32)

    L_max_past = L_past
    t_mask = build_padding_mask(t_past_lens, L_max_past)
    v_mask = build_padding_mask(v_past_lens, L_max_past)
    te_mask = build_padding_mask(te_past_lens, L_max_past)

    # ---- TensorDatasets ----
    train_ds = TensorDataset(
        torch.from_numpy(t_first),
        torch.from_numpy(t_past),
        torch.from_numpy(t_global),
        torch.from_numpy(t_thr_s),
        torch.from_numpy(t_y_log.astype(np.float32)),
        torch.from_numpy(t_ord.astype(np.float32)),
        torch.from_numpy(t_region.astype(np.int64)),
        torch.from_numpy(t_mask),
    )
    val_ds = TensorDataset(
        torch.from_numpy(v_first),
        torch.from_numpy(v_past),
        torch.from_numpy(v_global),
        torch.from_numpy(v_thr_s),
        torch.from_numpy(v_y_log.astype(np.float32)),
        torch.from_numpy(v_ord.astype(np.float32)),
        torch.from_numpy(v_region.astype(np.int64)),
        torch.from_numpy(v_mask),
    )
    test_ds = TensorDataset(
        torch.from_numpy(te_first),
        torch.from_numpy(te_past),
        torch.from_numpy(te_global),
        torch.from_numpy(te_thr_s),
        torch.from_numpy(te_y_log.astype(np.float32)),
        torch.from_numpy(te_ord.astype(np.float32)),
        torch.from_numpy(te_region.astype(np.int64)),
        torch.from_numpy(te_mask),
    )

    # ---- Weighted sampler (region + SIR-aware) ----
    region_sample_weights = np.ones_like(t_region, dtype=np.float32)
    region_sample_weights[t_region == 6] = 2.5
    region_sample_weights[t_region == 7] = 3.0
    region_sample_weights[t_region == 8] = 5.0
    region_sample_weights[t_region == 9] = 5.0
    region_sample_weights[t_region == 10] = 5.0
    region_sample_weights[t_region == 11] = 5.0
    region_sample_weights[t_region == 12] = 3.0
    region_sample_weights[t_region == 13] = 2.5

    # SIR-aware weights (threshold-independent scenario difficulty)
    t_signal = t_taps_feat[:, 0]
    t_valid_past_for_sir = (np.arange(L_past)[None, :] < t_past_lens[:, None]).astype(np.float32)
    t_isi = np.sum(t_taps_feat[:, 1:] * t_valid_past_for_sir, axis=1)
    t_sir = t_signal / (t_isi + EPS)

    sir_sample_weights = np.ones_like(t_sir, dtype=np.float32)
    sir_sample_weights[t_sir < 0.26] = 3.0
    sir_sample_weights[(t_sir >= 0.26) & (t_sir < 0.36)] = 2.0

    combined_sample_weights = region_sample_weights * sir_sample_weights

    sampler = WeightedRandomSampler(
        weights=torch.from_numpy(combined_sample_weights),
        num_samples=len(combined_sample_weights),
        replacement=True,
    )

    pin_mem = torch.cuda.is_available()

    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler,
                              shuffle=False, pin_memory=pin_mem, num_workers=num_workers)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            pin_memory=pin_mem, num_workers=num_workers)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                             pin_memory=pin_mem, num_workers=num_workers)

    class_weights = compute_region_class_weights(t_region, NUM_REGION_CLASSES)
    ordinal_pos_weights = compute_ordinal_pos_weights_from_region_labels(
        t_region, num_thresholds=NUM_ORDINAL_THRESHOLDS, max_weight=20.0,
    )

    # ---- Save all scaler parameters (position-independent) ----
    scalers = {
        # First-token scalers (per feature channel)
        "first_tap_mean": first_tap_mean, "first_tap_std": first_tap_std,
        "first_var_mean": first_var_mean, "first_var_std": first_var_std,
        "first_abs_mean": first_abs_mean, "first_abs_std": first_abs_std,
        "first_snr_mean": first_snr_mean, "first_snr_std": first_snr_std,
        # Shared past scalers (single mean/std per feature, position-independent)
        "past_tap_mean": past_tap_mean, "past_tap_std": past_tap_std,
        "past_var_mean": past_var_mean, "past_var_std": past_var_std,
        "past_abs_mean": past_abs_mean, "past_abs_std": past_abs_std,
        "past_snr_mean": past_snr_mean, "past_snr_std": past_snr_std,
        # Global feature scalers (sklearn StandardScaler objects)
        "z0_scaler": z0_scaler,
        "z1_scaler": z1_scaler,
        "harmonic_minus_gap_scaler": hmg_scaler,
        "nsid_scaler": nsid_scaler,
        "log_hd_scaler": log_hd_scaler,
        "da_scaler": da_scaler,
        "herf_scaler": herf_scaler,
        "thr_scaler": thr_scaler,
        # Metadata
        "tap_cols": tap_cols,
        "var_cols": var_cols,
        "first_token_dim": 4,
        "past_token_dim": 4,
        "global_dim": 7,
        "train_max_past_seq_len": L_past,
        "scaling_strategy": "position_independent",
        "uses_positional_encoding": False,
        "permutation_invariance_post_first": True,
        "variable_length_support": True,
        "target_parameterization": "pred_log10_ber = log10(0.5) - softplus(raw_out)",
        "ordinal_thresholds": ORDINAL_THRESHOLDS,
        "num_region_classes": NUM_REGION_CLASSES,
        "class_weights": class_weights.tolist(),
        "ordinal_pos_weights": ordinal_pos_weights.tolist(),
        "data_paths": csv_paths,
        "nrows_per_dataset": nrows_per_dataset,
    }

    aux_info = {
        "class_weights": class_weights,
        "ordinal_pos_weights": ordinal_pos_weights,
        "t_region": t_region,
    }

    return train_loader, val_loader, test_loader, scalers, aux_info, L_past


# =========================
# Inference Helper
# =========================
def prepare_inference_batch(
    taps_raw_2d,
    vars_raw_2d,
    N_array,
    threshold_array,
    mem_len_array,
    scalers,
):
    """Prepare a batch for inference from raw numpy arrays.

    This handles ARBITRARY mem_len — the set can be larger than anything
    seen during training because scalers are position-independent.

    Args:
        taps_raw_2d:     [B, L] raw tap coefficients (padded to L with zeros)
        vars_raw_2d:     [B, L] raw variance values  (padded to L with zeros)
        N_array:         [B] or [B, 1] number of molecules
        threshold_array: [B] or [B, 1] raw threshold values
        mem_len_array:   [B] true memory length per sample
        scalers:         dict from joblib.load(SCALER_SAVE_PATH)

    Returns:
        first_token:      [B, 4] tensor
        past_tokens:      [B, L-1, 4] tensor
        global_feats:     [B, 7] tensor
        threshold_scaled: [B, 1] tensor
        key_padding_mask: [B, L-1] bool tensor (True = padding)
    """
    B, L = taps_raw_2d.shape
    N = N_array.reshape(-1, 1).astype(np.float32)
    thr_raw = threshold_array.reshape(-1, 1).astype(np.float32)
    mem_len = mem_len_array.reshape(-1).astype(np.int64)

    mem_len_clamped = np.minimum(mem_len, L)
    past_lens = np.maximum(mem_len_clamped - 1, 0).astype(np.int64)

    # Feature engineering (same as training)
    taps_feat = (taps_raw_2d * N).astype(np.float32)
    vars_feat = vars_raw_2d.astype(np.float32)
    abs_feat = np.abs(taps_feat).astype(np.float32)
    snr_feat = np.log10((taps_feat ** 2) / (vars_raw_2d + EPS) + EPS).astype(np.float32)

    L_past = L - 1

    # Validity masks
    valid_past = (np.arange(L_past)[None, :] < past_lens[:, None])
    valid_past_float = valid_past.astype(np.float32)

    # Global features (mask-aware)
    first_mean = taps_feat[:, 0:1]
    past_means = taps_feat[:, 1:] * valid_past_float
    first_var = vars_feat[:, 0:1]
    past_vars = vars_feat[:, 1:] * valid_past_float

    mu0 = (0.5 * past_means.sum(axis=1, keepdims=True)).astype(np.float32)
    mu1 = (first_mean + mu0).astype(np.float32)
    var0 = (0.5 * past_vars.sum(axis=1, keepdims=True)).astype(np.float32)
    var1 = (first_var + var0).astype(np.float32)

    std0 = np.sqrt(np.maximum(var0, EPS)).astype(np.float32)
    std1 = np.sqrt(np.maximum(var1, EPS)).astype(np.float32)

    z0 = ((thr_raw - mu0) / (std0 + EPS)).astype(np.float32)
    z1 = ((mu1 - thr_raw) / (std1 + EPS)).astype(np.float32)

    harmonic = (2.0 / (1.0 / (z0 + EPS) + 1.0 / (z1 + EPS))).astype(np.float32)
    abs_diff = np.abs(z0 - z1).astype(np.float32)
    hmg = (harmonic - 0.25 * abs_diff).astype(np.float32)

    # Scenario-level features (threshold-independent)
    signal = first_mean  # [B, 1]
    isi = past_means.sum(axis=1, keepdims=True)  # [B, 1]
    nsid = ((signal - isi) / (signal + isi + EPS)).astype(np.float32)

    gap = signal
    d0_inf = (gap / (std0 + EPS)).astype(np.float32)
    d1_inf = (gap / (std1 + EPS)).astype(np.float32)
    hd = (2.0 * d0_inf * d1_inf / (d0_inf + d1_inf + EPS)).astype(np.float32)
    log_hd = np.log10(hd + EPS).astype(np.float32)
    da = (np.abs(d0_inf - d1_inf) / (d0_inf + d1_inf + EPS)).astype(np.float32)

    past_taps_sum = past_means.sum(axis=1, keepdims=True)
    past_shares = past_means / (past_taps_sum + EPS)
    herf = (past_shares ** 2).sum(axis=1, keepdims=True).astype(np.float32)

    # Scale first token
    ft_tap = ((taps_feat[:, 0] - scalers["first_tap_mean"]) / scalers["first_tap_std"]).astype(np.float32)
    ft_var = ((vars_feat[:, 0] - scalers["first_var_mean"]) / scalers["first_var_std"]).astype(np.float32)
    ft_abs = ((abs_feat[:, 0] - scalers["first_abs_mean"]) / scalers["first_abs_std"]).astype(np.float32)
    ft_snr = ((snr_feat[:, 0] - scalers["first_snr_mean"]) / scalers["first_snr_std"]).astype(np.float32)
    first_token = np.stack([ft_tap, ft_var, ft_abs, ft_snr], axis=1).astype(np.float32)

    # Scale past tokens (position-independent shared scaler)
    pt_tap = apply_shared_scale(taps_feat[:, 1:], scalers["past_tap_mean"], scalers["past_tap_std"], valid_past)
    pt_var = apply_shared_scale(vars_feat[:, 1:], scalers["past_var_mean"], scalers["past_var_std"], valid_past)
    pt_abs = apply_shared_scale(abs_feat[:, 1:], scalers["past_abs_mean"], scalers["past_abs_std"], valid_past)
    pt_snr = apply_shared_scale(snr_feat[:, 1:], scalers["past_snr_mean"], scalers["past_snr_std"], valid_past)
    past_tokens = np.stack([pt_tap, pt_var, pt_abs, pt_snr], axis=2).astype(np.float32)

    # Scale globals (7 features)
    z0_s = scalers["z0_scaler"].transform(z0).astype(np.float32)
    z1_s = scalers["z1_scaler"].transform(z1).astype(np.float32)
    hmg_s = scalers["harmonic_minus_gap_scaler"].transform(hmg).astype(np.float32)
    nsid_s = scalers["nsid_scaler"].transform(nsid).astype(np.float32)
    log_hd_s = scalers["log_hd_scaler"].transform(log_hd).astype(np.float32)
    da_s = scalers["da_scaler"].transform(da).astype(np.float32)
    herf_s = scalers["herf_scaler"].transform(herf).astype(np.float32)
    global_feats = np.concatenate([z0_s, z1_s, hmg_s, nsid_s, log_hd_s, da_s, herf_s], axis=1).astype(np.float32)

    # Scale threshold
    thr_log = np.log10(thr_raw + EPS).astype(np.float32)
    thr_s = scalers["thr_scaler"].transform(thr_log).astype(np.float32)

    # Padding mask
    pad_mask = (np.arange(L_past)[None, :] >= past_lens[:, None])

    return (
        torch.from_numpy(first_token),
        torch.from_numpy(past_tokens),
        torch.from_numpy(global_feats),
        torch.from_numpy(thr_s),
        torch.from_numpy(pad_mask),
    )


def run_inference(model, taps_raw, vars_raw, N_arr, thr_arr, mem_len_arr, scalers, device):
    """End-to-end inference: raw arrays -> BER predictions.

    All inputs are numpy. Works with any mem_len, even values
    larger than max_past_seq_len seen during training.
    """
    first_tok, past_tok, glob, thr_s, pad_mask = prepare_inference_batch(
        taps_raw, vars_raw, N_arr, thr_arr, mem_len_arr, scalers,
    )

    model.eval()
    with torch.no_grad():
        first_tok = first_tok.to(device)
        past_tok = past_tok.to(device)
        glob = glob.to(device)
        thr_s = thr_s.to(device)
        pad_mask = pad_mask.to(device)

        pred_raw, ord_logits = model(first_tok, past_tok, glob, thr_s, key_padding_mask=pad_mask)

        pred_ber = model.raw_to_ber(pred_raw).cpu().numpy()
        pred_log = model.raw_to_log10ber(pred_raw).cpu().numpy()
        pred_region = ordinal_logits_to_region_labels_torch(ord_logits).cpu().numpy()

    return {
        "ber": pred_ber,
        "log10_ber": pred_log,
        "region": pred_region,
    }


# =========================
# Evaluation
# =========================
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_reg_loss = 0.0
    total_ord_loss = 0.0

    all_preds_log = []
    all_targets_log = []
    all_pred_regions = []
    all_true_regions = []

    with torch.no_grad():
        for b_first, b_past, b_global, b_thr, b_y_log, b_ord, b_region, b_mask in loader:
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_y_log = b_y_log.to(device, non_blocking=True)
            b_ord = b_ord.to(device, non_blocking=True)
            b_region = b_region.to(device, non_blocking=True)
            b_mask = b_mask.to(device, non_blocking=True)

            pred_raw_unconstrained, ord_logits = model(
                b_first, b_past, b_global, b_thr, key_padding_mask=b_mask
            )

            if has_nonfinite_tensor(pred_raw_unconstrained):
                raise RuntimeError("Non-finite prediction during evaluation.")
            if has_nonfinite_tensor(ord_logits):
                raise RuntimeError("Non-finite ordinal logits during evaluation.")

            loss, reg_loss, ord_loss = criterion(
                pred_raw_unconstrained, ord_logits, b_y_log, b_ord,
            )

            if has_nonfinite_tensor(loss):
                raise RuntimeError("Non-finite loss during evaluation.")

            pred_log = model.raw_to_log10ber(pred_raw_unconstrained)
            pred_region = ordinal_logits_to_region_labels_torch(ord_logits)

            total_loss += loss.item()
            total_reg_loss += reg_loss.item()
            total_ord_loss += ord_loss.item()

            all_preds_log.append(pred_log.cpu().numpy())
            all_targets_log.append(b_y_log.cpu().numpy())
            all_pred_regions.append(pred_region.cpu().numpy())
            all_true_regions.append(b_region.cpu().numpy())

    n_batches = max(len(loader), 1)
    avg_loss = total_loss / n_batches
    avg_reg_loss = total_reg_loss / n_batches
    avg_ord_loss = total_ord_loss / n_batches

    preds_log = np.vstack(all_preds_log)
    targets_log = np.vstack(all_targets_log)

    preds_raw = np.clip(10 ** preds_log, EPS, 0.5)
    targets_raw = np.clip(10 ** targets_log, EPS, 0.5)

    rmse_log = float(np.sqrt(np.mean((preds_log - targets_log) ** 2)))
    mae_log = float(np.mean(np.abs(preds_log - targets_log)))
    factor_error = float(10 ** rmse_log)

    rmse_raw = float(np.sqrt(np.mean((preds_raw - targets_raw) ** 2)))
    mae_raw = float(np.mean(np.abs(preds_raw - targets_raw)))

    rel_err = (preds_raw - targets_raw) / np.maximum(targets_raw, 1e-6)
    rmse_rel = float(np.sqrt(np.mean(rel_err ** 2)))
    mae_rel = float(np.mean(np.abs(rel_err)))

    pred_regions = np.concatenate(all_pred_regions).reshape(-1)
    true_regions = np.concatenate(all_true_regions).reshape(-1)

    region_acc = float(np.mean(pred_regions == true_regions))
    region_mae = float(np.mean(np.abs(pred_regions - true_regions)))

    return {
        "loss": avg_loss,
        "reg_loss": avg_reg_loss,
        "ord_loss": avg_ord_loss,
        "rmse_log": rmse_log,
        "mae_log": mae_log,
        "factor_error": factor_error,
        "rmse_raw": rmse_raw,
        "mae_raw": mae_raw,
        "rmse_rel": rmse_rel,
        "mae_rel": mae_rel,
        "region_acc": region_acc,
        "region_mae": region_mae,
    }


def evaluate_by_target_range(model, loader, device):
    model.eval()
    all_preds_log = []
    all_targets_log = []

    with torch.no_grad():
        for b_first, b_past, b_global, b_thr, b_y_log, _, _, b_mask in loader:
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_mask = b_mask.to(device, non_blocking=True)

            pred_raw_unconstrained, _ = model(
                b_first, b_past, b_global, b_thr, key_padding_mask=b_mask
            )

            if has_nonfinite_tensor(pred_raw_unconstrained):
                raise RuntimeError("Non-finite prediction during per-range evaluation.")

            pred_log = model.raw_to_log10ber(pred_raw_unconstrained)

            all_preds_log.append(pred_log.cpu().numpy())
            all_targets_log.append(b_y_log.numpy())

    preds_log = np.vstack(all_preds_log).reshape(-1)
    targets_log = np.vstack(all_targets_log).reshape(-1)

    preds_raw = np.clip(10 ** preds_log, EPS, 0.5)
    targets_raw = np.clip(10 ** targets_log, EPS, 0.5)

    ranges = {
        "low_BER(y<1e-6)": targets_raw < 1e-6,
        "mid_BER(1e-6<=y<1e-3)": (targets_raw >= 1e-6) & (targets_raw < 1e-3),
        "high_BER(1e-3<=y<0.10)": (targets_raw >= 1e-3) & (targets_raw < 0.10),
        "upper_BER(0.10<=y<0.20)": (targets_raw >= 0.10) & (targets_raw < 0.20),
        "target_BER(0.20<=y<0.40)": (targets_raw >= 0.20) & (targets_raw < 0.40),
        "very_high_BER(0.40<=y<=0.50)": targets_raw >= 0.40,
    }

    metrics = {}
    for name, mask in ranges.items():
        if np.any(mask):
            err_log = preds_log[mask] - targets_log[mask]
            err_raw = preds_raw[mask] - targets_raw[mask]
            rel_err = err_raw / np.maximum(targets_raw[mask], 1e-6)

            rmse_log = float(np.sqrt(np.mean(err_log ** 2)))
            mae_log = float(np.mean(np.abs(err_log)))
            rmse_raw = float(np.sqrt(np.mean(err_raw ** 2)))
            mae_raw = float(np.mean(np.abs(err_raw)))
            rmse_rel = float(np.sqrt(np.mean(rel_err ** 2)))
            mae_rel = float(np.mean(np.abs(rel_err)))

            metrics[name] = {
                "count": int(mask.sum()),
                "rmse_log": rmse_log,
                "mae_log": mae_log,
                "factor_error": float(10 ** rmse_log),
                "rmse_raw": rmse_raw,
                "mae_raw": mae_raw,
                "rmse_rel": rmse_rel,
                "mae_rel": mae_rel,
                "bias_raw": float(np.mean(err_raw)),
                "bias_log": float(np.mean(err_log)),
                "p90_abs_raw": float(np.percentile(np.abs(err_raw), 90)),
                "p95_abs_raw": float(np.percentile(np.abs(err_raw), 95)),
            }
        else:
            metrics[name] = None

    return metrics


# =========================
# Multi-Component Selection Scoring
# =========================
# Selection rationale:
#   BER spans 6+ orders of magnitude (1e-6 to 0.5), so multiplicative error
#   is the natural metric. We use weighted log-RMSE per region (already in log
#   space, so equivalent to weighted geometric mean of factor errors), then
#   add a bias penalty (also in log space, so multiplicative bias) and a tail
#   penalty (P95/RMSE ratio in raw space, capped, to penalize heavy tails).

# Weights for each BER region in the composite score.
# Higher weight = more important to get this region right.
SELECTION_REGION_WEIGHTS = {
    "low_BER(y<1e-6)": 0.25,
    "mid_BER(1e-6<=y<1e-3)": 0.50,
    "high_BER(1e-3<=y<0.10)": 1.00,
    "upper_BER(0.10<=y<0.20)": 2.00,
    "target_BER(0.20<=y<0.40)": 3.00,
    "very_high_BER(0.40<=y<=0.50)": 1.50,
}

# Composite weights: how much each component contributes
SELECTION_W_LOG_RMSE = 1.00   # Primary: per-region log-RMSE
SELECTION_W_BIAS = 0.50       # Secondary: bias in log space (factor bias)
SELECTION_W_TAIL = 0.05       # Tertiary: P95/RMSE ratio (heavy tails)
SELECTION_TAIL_CAP = 5.0      # Cap tail ratio so one bad region doesn't dominate


def compute_selection_score(val_range_metrics, val_metrics, min_count=50):
    """Compute a multi-component selection score, lower is better.

    Components:
      1. Weighted average of per-region RMSE(log10).
         (This is equivalent to a weighted geometric mean of factor errors.)
      2. Weighted average of |bias_log| per region (multiplicative bias).
      3. Weighted average of capped P95/RMSE ratio per region (tail penalty).

    Returns a dict with all components and the composite.
    """
    log_rmse_sum = 0.0
    bias_sum = 0.0
    tail_sum = 0.0
    total_w = 0.0

    per_region_factor = {}

    for region_name, w in SELECTION_REGION_WEIGHTS.items():
        m = val_range_metrics.get(region_name)
        if m is None or m["count"] < min_count:
            continue

        # 1. Log-RMSE component (already log-space)
        log_rmse_sum += w * m["rmse_log"]

        # 2. Bias component in log space (multiplicative bias)
        bias_sum += w * abs(m["bias_log"])

        # 3. Tail component: P95 / RMSE in raw space, capped
        if m["rmse_raw"] > 1e-9:
            tail_ratio = m["p95_abs_raw"] / (m["rmse_raw"] + 1e-9)
            tail_sum += w * min(tail_ratio, SELECTION_TAIL_CAP)
        else:
            tail_sum += w * 1.0

        total_w += w
        per_region_factor[region_name] = m["factor_error"]

    if total_w == 0:
        # Fallback to global metrics
        return {
            "composite": float(val_metrics["rmse_log"]),
            "log_rmse_weighted": float(val_metrics["rmse_log"]),
            "bias_weighted": 0.0,
            "tail_weighted": 0.0,
            "geometric_factor_error": float(val_metrics["factor_error"]),
            "fallback": True,
        }

    log_rmse_weighted = log_rmse_sum / total_w
    bias_weighted = bias_sum / total_w
    tail_weighted = tail_sum / total_w

    composite = (
        SELECTION_W_LOG_RMSE * log_rmse_weighted
        + SELECTION_W_BIAS * bias_weighted
        + SELECTION_W_TAIL * tail_weighted
    )

    # The geometric-mean factor error: 10^(weighted log_rmse)
    # Reported for human readability — "this model is off by ~Nx on average"
    geometric_factor = float(10 ** log_rmse_weighted)

    return {
        "composite": float(composite),
        "log_rmse_weighted": float(log_rmse_weighted),
        "bias_weighted": float(bias_weighted),
        "tail_weighted": float(tail_weighted),
        "geometric_factor_error": geometric_factor,
        "fallback": False,
    }


def is_acceptable_checkpoint(val_range_metrics, val_metrics):
    """Hard constraints: model must meet these to be considered for 'best'.

    These prevent pathological epochs from being selected just because
    they happen to score well on the composite (e.g., if a region has
    almost no samples and dominates by luck).
    """
    target = val_range_metrics.get("target_BER(0.20<=y<0.40)")
    if target is None or target["count"] < 100:
        return False, "target region too small"

    # No region's bias_log can exceed 0.15 (i.e., factor bias of ~1.4x)
    for region_name, m in val_range_metrics.items():
        if m is None or m["count"] < 50:
            continue
        if abs(m["bias_log"]) > 0.15:
            return False, f"{region_name} has bias_log={m['bias_log']:.3f}"

    # Overall log RMSE must be reasonable
    if val_metrics["rmse_log"] > 0.30:
        return False, f"overall rmse_log={val_metrics['rmse_log']:.3f} too high"

    return True, "ok"


# =========================
# Training
# =========================
def train_engine():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Executing on: {device}")

    train_loader, val_loader, test_loader, scalers, aux_info, max_past_seq_len = prepare_data(
        DATA_PATHS,
        batch_size=256,
        nrows_per_dataset=NROWS_PER_DATASET,
        num_workers=0,
    )

    joblib.dump(scalers, SCALER_SAVE_PATH)
    print(f"Scalers saved to: {SCALER_SAVE_PATH}")

    model = BERFirstTokenSetTransformerBoundedLogHalfMultiTask(
        max_past_seq_len=max_past_seq_len,
        token_dim=4,
        first_token_dim=4,
        threshold_dim=1,
        global_dim=7,
        d_model=128,
        num_set_layers=4,
        num_heads=4,
        mlp_ratio=4.0,
        dropout=0.10,
        num_ordinal_thresholds=NUM_ORDINAL_THRESHOLDS,
    ).to(device)

    print("Training target-focused first-token + set-transformer multitask model...")
    print(f"  Variable-length support: ENABLED (position-independent scaling)")
    print(f"  Max past sequence length (training): {max_past_seq_len}")
    print(f"  Scaling strategy: separate first-token / shared past-token scalers")

    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

    reg_loss = StableMultiObjectiveBERBoundedLogLoss(
        log_delta=0.35,
        raw_delta=0.006,
        rel_delta=0.03,
        alpha_log=0.45,
        beta_raw=0.35,
        gamma_rel=0.20,
        use_regime_weights=True,
    )

    boosted_pos = aux_info["ordinal_pos_weights"].copy()
    for i, thr in enumerate(ORDINAL_THRESHOLDS):
        if 0.20 <= thr <= 0.40:
            boosted_pos[i] *= 2.5
        elif 0.15 <= thr < 0.20:
            boosted_pos[i] *= 1.5
        elif 0.40 < thr <= 0.45:
            boosted_pos[i] *= 1.5

    ord_loss = OrdinalBCELoss(
        pos_weight=torch.tensor(boosted_pos, dtype=torch.float32, device=device),
        reduction="mean",
    )

    criterion = MultiTaskBERLoss(
        reg_loss=reg_loss,
        ord_loss=ord_loss,
        lambda_ord=0.25,
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=4, factor=0.5,
    )

    # ---- Multi-component selection state ----
    # Track multiple "best" checkpoints under different criteria.
    # We'll save all of them and the user can choose the right tradeoff at inference.
    best_checkpoints = {
        "composite": {"score": float("inf"), "state": None, "epoch": -1},   # primary: composite score
        "composite_ema": {"score": float("inf"), "state": None, "epoch": -1},  # EMA-smoothed composite
        "log_rmse": {"score": float("inf"), "state": None, "epoch": -1},    # weighted log-RMSE only
        "target_mae": {"score": float("inf"), "state": None, "epoch": -1},  # target region MAE
        "low_bias": {"score": float("inf"), "state": None, "epoch": -1},    # min weighted |bias_log|
    }

    # EMA smoothing of the composite score to reduce epoch-to-epoch noise
    EMA_ALPHA = 0.5
    ema_composite = None

    patience = 12
    wait = 0
    min_epochs_before_early_stop = 60
    max_epochs = 120

    print(f"Ordinal thresholds: {ORDINAL_THRESHOLDS}")
    print(f"Boosted ordinal pos weights: {boosted_pos}")
    print(f"Selection: weighted log-RMSE (geometric factor mean) + bias + tail penalties")
    print(f"  Region weights: {SELECTION_REGION_WEIGHTS}")
    print(f"  Composite weights: log_rmse={SELECTION_W_LOG_RMSE}, "
          f"bias={SELECTION_W_BIAS}, tail={SELECTION_W_TAIL}")
    print(f"  EMA alpha for composite smoothing: {EMA_ALPHA}")

    training_broke = False

    for epoch in range(max_epochs):
        model.train()

        for batch_idx, (b_first, b_past, b_global, b_thr, b_y_log, b_ord, _, b_mask) in enumerate(train_loader):
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_y_log = b_y_log.to(device, non_blocking=True)
            b_ord = b_ord.to(device, non_blocking=True)
            b_mask = b_mask.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            pred_raw_unconstrained, ord_logits = model(
                b_first, b_past, b_global, b_thr, key_padding_mask=b_mask,
            )

            if has_nonfinite_tensor(pred_raw_unconstrained):
                print(f"Non-finite prediction at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True
                break

            if has_nonfinite_tensor(ord_logits):
                print(f"Non-finite ordinal logits at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True
                break

            loss, reg_part, ord_part = criterion(
                pred_raw_unconstrained, ord_logits, b_y_log, b_ord,
            )

            if has_nonfinite_tensor(loss):
                print(f"Non-finite loss at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True
                break

            loss.backward()

            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            if not torch.isfinite(grad_norm):
                print(f"Non-finite gradient norm at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True
                break

            optimizer.step()

            bad_param = False
            for name, param in model.named_parameters():
                if param.requires_grad and param.data is not None and not torch.isfinite(param.data).all():
                    print(f"Non-finite parameter after optimizer step: {name}")
                    bad_param = True
                    break

            if bad_param:
                training_broke = True
                break

        if training_broke:
            print("Training stopped due to non-finite values.")
            break

        try:
            train_metrics = evaluate(model, train_loader, criterion, device)
            val_metrics = evaluate(model, val_loader, criterion, device)
            val_range_metrics = evaluate_by_target_range(model, val_loader, device)
        except RuntimeError as e:
            print(f"Evaluation failed at epoch {epoch+1}: {e}")
            break

        scheduler.step(val_metrics["loss"])
        current_lr = optimizer.param_groups[0]["lr"]

        # ---- Compute multi-component selection score ----
        sel = compute_selection_score(val_range_metrics, val_metrics)
        composite = sel["composite"]

        # EMA-smoothed composite (reduces noise from single bad/lucky epochs)
        if ema_composite is None:
            ema_composite = composite
        else:
            ema_composite = EMA_ALPHA * composite + (1.0 - EMA_ALPHA) * ema_composite

        # Components for individual best-checkpoints
        target_key = "target_BER(0.20<=y<0.40)"
        target_mae = (
            val_range_metrics[target_key]["mae_raw"]
            if val_range_metrics[target_key] is not None
            else float("inf")
        )

        # Acceptability gate
        is_acceptable, reason = is_acceptable_checkpoint(val_range_metrics, val_metrics)

        # ---- Logging ----
        print(
            f"Epoch {epoch+1:03d} | "
            f"LR: {current_lr:.2e} | "
            f"Train Loss: {train_metrics['loss']:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} | "
            f"Val RMSE(log10): {val_metrics['rmse_log']:.4f} "
            f"(~{val_metrics['factor_error']:.3f}x) | "
            f"Val Region Acc: {val_metrics['region_acc']:.4f}"
        )
        print(
            f"  SELECTION | composite={composite:.5f} | "
            f"ema_composite={ema_composite:.5f} | "
            f"weighted log-RMSE={sel['log_rmse_weighted']:.5f} "
            f"(~{sel['geometric_factor_error']:.3f}x avg factor) | "
            f"weighted |bias_log|={sel['bias_weighted']:.5f} | "
            f"tail penalty={sel['tail_weighted']:.3f} | "
            f"acceptable={is_acceptable}"
            + ("" if is_acceptable else f" ({reason})")
        )

        # Per-range tail metrics
        for rng_name, rng_stats in val_range_metrics.items():
            if rng_stats is not None:
                print(
                    f"    {rng_name}: "
                    f"n={rng_stats['count']} | "
                    f"factor~{rng_stats['factor_error']:.3f}x | "
                    f"RMSE_log={rng_stats['rmse_log']:.4f} | "
                    f"MAE_raw={rng_stats['mae_raw']:.6f} | "
                    f"P90={rng_stats['p90_abs_raw']:.6f} | "
                    f"P95={rng_stats['p95_abs_raw']:.6f} | "
                    f"bias_raw={rng_stats['bias_raw']:+.6f} | "
                    f"bias_log={rng_stats['bias_log']:+.4f}"
                )

        # ---- Update each best-checkpoint (only if acceptable) ----
        improved_any = False

        if is_acceptable:
            current_state_snapshot = None  # lazy deepcopy

            checkpoint_candidates = [
                ("composite", composite),
                ("composite_ema", ema_composite),
                ("log_rmse", sel["log_rmse_weighted"]),
                ("target_mae", target_mae),
                ("low_bias", sel["bias_weighted"]),
            ]

            for ckpt_name, score in checkpoint_candidates:
                if score < best_checkpoints[ckpt_name]["score"]:
                    if current_state_snapshot is None:
                        current_state_snapshot = copy.deepcopy(model.state_dict())
                    best_checkpoints[ckpt_name] = {
                        "score": float(score),
                        "state": current_state_snapshot,
                        "epoch": epoch + 1,
                    }
                    improved_any = True
                    print(f"  -> New best [{ckpt_name}] at epoch {epoch+1}: {score:.6f}")

        # ---- Early stopping driven by EMA composite ----
        # We use the EMA score so we don't stop on a single noisy spike
        ema_best = best_checkpoints["composite_ema"]["score"]
        if ema_composite < ema_best + 1e-9 or improved_any:
            wait = 0
        else:
            if epoch + 1 >= min_epochs_before_early_stop:
                wait += 1
                if wait >= patience:
                    print(f"Early stopping triggered after {wait} non-improving epochs (EMA basis).")
                    break

    # ---- Save last model ----
    torch.save(model.state_dict(), LAST_MODEL_SAVE_PATH)
    print(f"\nLast model saved to: {LAST_MODEL_SAVE_PATH}")

    # ---- Save all best checkpoints ----
    print("\n" + "=" * 80)
    print("BEST CHECKPOINTS SUMMARY")
    print("=" * 80)
    for ckpt_name, info in best_checkpoints.items():
        if info["state"] is None:
            print(f"  [{ckpt_name:>14s}] never updated")
            continue

        path = BEST_MODEL_SAVE_PATH.replace(".pth", f"_{ckpt_name}.pth")
        torch.save(info["state"], path)
        print(
            f"  [{ckpt_name:>14s}] epoch={info['epoch']:>3d} | "
            f"score={info['score']:.6f} | saved -> {path}"
        )

    # ---- Choose primary best for evaluation: composite_ema is most stable ----
    primary_choice = "composite_ema"
    if best_checkpoints[primary_choice]["state"] is None:
        # Fall back to composite if EMA never updated (shouldn't happen but safe)
        primary_choice = "composite"
    if best_checkpoints[primary_choice]["state"] is None:
        # Fall back to log_rmse
        primary_choice = "log_rmse"

    if best_checkpoints[primary_choice]["state"] is not None:
        model.load_state_dict(best_checkpoints[primary_choice]["state"])
        torch.save(model.state_dict(), BEST_MODEL_SAVE_PATH)
        print(
            f"\nPrimary best ({primary_choice}, epoch "
            f"{best_checkpoints[primary_choice]['epoch']}) "
            f"saved to: {BEST_MODEL_SAVE_PATH}"
        )
    else:
        print("\nWarning: no valid best checkpoint was found.")

    test_metrics = evaluate(model, test_loader, criterion, device)

    print(
        f"Test Loss: {test_metrics['loss']:.4f} | "
        f"Test Reg Loss: {test_metrics['reg_loss']:.4f} | "
        f"Test Ord Loss: {test_metrics['ord_loss']:.4f} | "
        f"Test RMSE(log10): {test_metrics['rmse_log']:.4f} | "
        f"Test MAE(log10): {test_metrics['mae_log']:.4f} | "
        f"Typical multiplicative error: ~{test_metrics['factor_error']:.2f}x | "
        f"Test RMSE(raw): {test_metrics['rmse_raw']:.6f} | "
        f"Test MAE(raw): {test_metrics['mae_raw']:.6f} | "
        f"Test RMSE(rel): {test_metrics['rmse_rel']:.6f} | "
        f"Test MAE(rel): {test_metrics['mae_rel']:.6f} | "
        f"Test Region Acc: {test_metrics['region_acc']:.4f} | "
        f"Test Region MAE: {test_metrics['region_mae']:.4f}"
    )

    range_metrics = evaluate_by_target_range(model, test_loader, device)
    print("\nPer-range test diagnostics:")
    for name, stats in range_metrics.items():
        if stats is None:
            print(f"  {name}: no samples")
        else:
            print(
                f"  {name} | count={stats['count']} | "
                f"RMSE(log10)={stats['rmse_log']:.4f} | "
                f"MAE(log10)={stats['mae_log']:.4f} | "
                f"factor~{stats['factor_error']:.2f}x | "
                f"RMSE(raw)={stats['rmse_raw']:.6f} | "
                f"MAE(raw)={stats['mae_raw']:.6f} | "
                f"RMSE(rel)={stats['rmse_rel']:.6f} | "
                f"MAE(rel)={stats['mae_rel']:.6f} | "
                f"bias_raw={stats['bias_raw']:.6f} | "
                f"bias_log={stats['bias_log']:.6f} | "
                f"P90={stats['p90_abs_raw']:.6f} | "
                f"P95={stats['p95_abs_raw']:.6f}"
            )

    return model


if __name__ == "__main__":
    os.makedirs(BASE_DIR, exist_ok=True)

    missing = [p for p in DATA_PATHS if not os.path.exists(p)]
    if missing:
        print("Critical Error: Missing dataset files:")
        for p in missing:
            print(f"  - {p}")
    else:
        print("Using training from datasets:")
        for p in DATA_PATHS:
            print(f"  - {p}")
        print(f"Row cap per dataset: {NROWS_PER_DATASET:,}")

        trained_model = train_engine()

Using training from datasets:
  - ./data_physics_with_variances_total.csv
Row cap per dataset: 5,000,000
Executing on: cuda
Loading up to 5,000,000 rows from: ./data_physics_with_variances_total.csv
Combined rows before filtering: 5,000,000
Rows after cleaning/filtering: 3,227,965
Scalers saved to: ./physics_fulltrain_highber_scalers_v2_target020_040_transformer_multitask.pkl
Training target-focused first-token + set-transformer multitask model...
  Variable-length support: ENABLED (position-independent scaling)
  Max past sequence length (training): 13
  Scaling strategy: separate first-token / shared past-token scalers
Ordinal thresholds: [1e-06, 1e-05, 0.0001, 0.001, 0.01, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45]
Boosted ordinal pos weights: [1.        1.        1.        1.        1.        1.        1.5
 2.5       3.185707  5.14224   6.5747147 9.351631  7.8258343]
Selection: weighted log-RMSE (geometric factor mean) + bias + tail penalties
  Region weights: {'low_BER(y<1e-6)': 

Epoch 007 | LR: 1.00e-04 | Train Loss: 0.0097 | Val Loss: 0.0181 | Val RMSE(log10): 0.3742 (~2.367x) | Val Region Acc: 0.9214
  SELECTION | composite=0.30354 | ema_composite=0.27659 | weighted log-RMSE=0.17274 (~1.488x avg factor) | weighted |bias_log|=0.08024 | tail penalty=1.813 | acceptable=False (low_BER(y<1e-6) has bias_log=-0.218)
    low_BER(y<1e-6): n=96213 | factor~3.285x | RMSE_log=0.5165 | MAE_raw=0.000001 | P90=0.000000 | P95=0.000000 | bias_raw=+0.000000 | bias_log=-0.2181
    mid_BER(1e-6<=y<1e-3): n=15895 | factor~19.417x | RMSE_log=1.2882 | MAE_raw=0.000227 | P90=0.000425 | P95=0.000590 | bias_raw=-0.000003 | bias_log=-0.8006
    high_BER(1e-3<=y<0.10): n=60975 | factor~3.204x | RMSE_log=0.5057 | MAE_raw=0.006194 | P90=0.013343 | P95=0.017359 | bias_raw=-0.001032 | bias_log=-0.1497
    upper_BER(0.10<=y<0.20): n=65482 | factor~1.102x | RMSE_log=0.0422 | MAE_raw=0.010466 | P90=0.021121 | P95=0.028093 | bias_raw=+0.008168 | bias_log=+0.0218
    target_BER(0.20<=y<0.40): n

Epoch 013 | LR: 1.00e-04 | Train Loss: 0.0048 | Val Loss: 0.0072 | Val RMSE(log10): 0.1695 (~1.477x) | Val Region Acc: 0.9371
  SELECTION | composite=0.16376 | ema_composite=0.17613 | weighted log-RMSE=0.06021 (~1.149x avg factor) | weighted |bias_log|=0.01620 | tail penalty=1.909 | acceptable=True
    low_BER(y<1e-6): n=96213 | factor~2.167x | RMSE_log=0.3358 | MAE_raw=0.000000 | P90=0.000000 | P95=0.000000 | bias_raw=+0.000000 | bias_log=+0.0524
    mid_BER(1e-6<=y<1e-3): n=15895 | factor~2.231x | RMSE_log=0.3484 | MAE_raw=0.000088 | P90=0.000204 | P95=0.000339 | bias_raw=+0.000016 | bias_log=-0.0805
    high_BER(1e-3<=y<0.10): n=60975 | factor~1.350x | RMSE_log=0.1302 | MAE_raw=0.006143 | P90=0.013854 | P95=0.016911 | bias_raw=+0.005059 | bias_log=+0.0361
    upper_BER(0.10<=y<0.20): n=65482 | factor~1.079x | RMSE_log=0.0330 | MAE_raw=0.008033 | P90=0.015816 | P95=0.020569 | bias_raw=+0.007127 | bias_log=+0.0205
    target_BER(0.20<=y<0.40): n=143374 | factor~1.026x | RMSE_log=0.011

Epoch 020 | LR: 1.00e-04 | Train Loss: 0.0045 | Val Loss: 0.0072 | Val RMSE(log10): 0.1777 (~1.506x) | Val Region Acc: 0.9557
  SELECTION | composite=0.16453 | ema_composite=0.17221 | weighted log-RMSE=0.06558 (~1.163x avg factor) | weighted |bias_log|=0.00962 | tail penalty=1.883 | acceptable=True
    low_BER(y<1e-6): n=96213 | factor~2.160x | RMSE_log=0.3344 | MAE_raw=0.000000 | P90=0.000000 | P95=0.000000 | bias_raw=+0.000000 | bias_log=-0.0028
    mid_BER(1e-6<=y<1e-3): n=15895 | factor~2.712x | RMSE_log=0.4334 | MAE_raw=0.000143 | P90=0.000275 | P95=0.000482 | bias_raw=+0.000092 | bias_log=-0.0125
    high_BER(1e-3<=y<0.10): n=60975 | factor~1.434x | RMSE_log=0.1565 | MAE_raw=0.005714 | P90=0.012763 | P95=0.015767 | bias_raw=+0.004923 | bias_log=+0.0454
    upper_BER(0.10<=y<0.20): n=65482 | factor~1.062x | RMSE_log=0.0261 | MAE_raw=0.006071 | P90=0.012205 | P95=0.017357 | bias_raw=+0.004536 | bias_log=+0.0129
    target_BER(0.20<=y<0.40): n=143374 | factor~1.020x | RMSE_log=0.008

Epoch 027 | LR: 5.00e-05 | Train Loss: 0.0037 | Val Loss: 0.0057 | Val RMSE(log10): 0.1346 (~1.363x) | Val Region Acc: 0.9585
  SELECTION | composite=0.14339 | ema_composite=0.14995 | weighted log-RMSE=0.04966 (~1.121x avg factor) | weighted |bias_log|=0.00876 | tail penalty=1.787 | acceptable=True
    low_BER(y<1e-6): n=96213 | factor~1.814x | RMSE_log=0.2585 | MAE_raw=0.000000 | P90=0.000000 | P95=0.000000 | bias_raw=+0.000000 | bias_log=+0.0668
    mid_BER(1e-6<=y<1e-3): n=15895 | factor~2.096x | RMSE_log=0.3213 | MAE_raw=0.000112 | P90=0.000227 | P95=0.000353 | bias_raw=+0.000071 | bias_log=+0.0253
    high_BER(1e-3<=y<0.10): n=60975 | factor~1.267x | RMSE_log=0.1028 | MAE_raw=0.004576 | P90=0.009686 | P95=0.011985 | bias_raw=+0.003096 | bias_log=+0.0253
    upper_BER(0.10<=y<0.20): n=65482 | factor~1.056x | RMSE_log=0.0238 | MAE_raw=0.005340 | P90=0.010968 | P95=0.015025 | bias_raw=+0.002764 | bias_log=+0.0078
    target_BER(0.20<=y<0.40): n=143374 | factor~1.021x | RMSE_log=0.008

...

Epoch 090 | LR: 3.13e-06 | Train Loss: 0.0009 | Val Loss: 0.0011 | Val RMSE(log10): 0.0364 (~1.087x) | Val Region Acc: 0.9857
  SELECTION | composite=0.11467 | ema_composite=0.11352 | weighted log-RMSE=0.01778 (~1.042x avg factor) | weighted |bias_log|=0.00950 | tail penalty=1.843 | acceptable=True
    low_BER(y<1e-6): n=96213 | factor~1.156x | RMSE_log=0.0628 | MAE_raw=0.000000 | P90=0.000000 | P95=0.000000 | bias_raw=+0.000000 | bias_log=+0.0257
    mid_BER(1e-6<=y<1e-3): n=15895 | factor~1.274x | RMSE_log=0.1051 | MAE_raw=0.000016 | P90=0.000041 | P95=0.000060 | bias_raw=+0.000007 | bias_log=+0.0537
    high_BER(1e-3<=y<0.10): n=60975 | factor~1.082x | RMSE_log=0.0342 | MAE_raw=0.002461 | P90=0.005857 | P95=0.007297 | bias_raw=+0.002151 | bias_log=+0.0183
    upper_BER(0.10<=y<0.20): n=65482 | factor~1.028x | RMSE_log=0.0120 | MAE_raw=0.003326 | P90=0.005660 | P95=0.006861 | bias_raw=+0.003094 | bias_log=+0.0093
    target_BER(0.20<=y<0.40): n=143374 | factor~1.012x | RMSE_log=0.005

Epoch 097 | LR: 3.13e-06 | Train Loss: 0.0008 | Val Loss: 0.0010 | Val RMSE(log10): 0.0285 (~1.068x) | Val Region Acc: 0.9878
  SELECTION | composite=0.11127 | ema_composite=0.11094 | weighted log-RMSE=0.01553 (~1.036x avg factor) | weighted |bias_log|=0.00730 | tail penalty=1.842 | acceptable=True
    low_BER(y<1e-6): n=96213 | factor~1.115x | RMSE_log=0.0474 | MAE_raw=0.000000 | P90=0.000000 | P95=0.000000 | bias_raw=+0.000000 | bias_log=+0.0086
    mid_BER(1e-6<=y<1e-3): n=15895 | factor~1.190x | RMSE_log=0.0755 | MAE_raw=0.000014 | P90=0.000038 | P95=0.000065 | bias_raw=+0.000006 | bias_log=+0.0216
    high_BER(1e-3<=y<0.10): n=60975 | factor~1.083x | RMSE_log=0.0346 | MAE_raw=0.002517 | P90=0.006060 | P95=0.007375 | bias_raw=+0.002262 | bias_log=+0.0214
    upper_BER(0.10<=y<0.20): n=65482 | factor~1.028x | RMSE_log=0.0121 | MAE_raw=0.003271 | P90=0.005562 | P95=0.006796 | bias_raw=+0.003015 | bias_log=+0.0091
    target_BER(0.20<=y<0.40): n=143374 | factor~1.012x | RMSE_log=0.005

Epoch 104 | LR: 1.56e-06 | Train Loss: 0.0008 | Val Loss: 0.0010 | Val RMSE(log10): 0.0275 (~1.065x) | Val Region Acc: 0.9879
  SELECTION | composite=0.11093 | ema_composite=0.11094 | weighted log-RMSE=0.01433 (~1.034x avg factor) | weighted |bias_log|=0.00509 | tail penalty=1.881 | acceptable=True
    low_BER(y<1e-6): n=96213 | factor~1.116x | RMSE_log=0.0476 | MAE_raw=0.000000 | P90=0.000000 | P95=0.000000 | bias_raw=-0.000000 | bias_log=-0.0079
    mid_BER(1e-6<=y<1e-3): n=15895 | factor~1.165x | RMSE_log=0.0665 | MAE_raw=0.000014 | P90=0.000041 | P95=0.000063 | bias_raw=-0.000006 | bias_log=-0.0079
    high_BER(1e-3<=y<0.10): n=60975 | factor~1.079x | RMSE_log=0.0330 | MAE_raw=0.002401 | P90=0.005866 | P95=0.007240 | bias_raw=+0.002057 | bias_log=+0.0154
    upper_BER(0.10<=y<0.20): n=65482 | factor~1.025x | RMSE_log=0.0107 | MAE_raw=0.002776 | P90=0.005056 | P95=0.006145 | bias_raw=+0.002449 | bias_log=+0.0075
    target_BER(0.20<=y<0.40): n=143374 | factor~1.011x | RMSE_log=0.004

Epoch 111 | LR: 7.81e-07 | Train Loss: 0.0008 | Val Loss: 0.0010 | Val RMSE(log10): 0.0270 (~1.064x) | Val Region Acc: 0.9878
  SELECTION | composite=0.11105 | ema_composite=0.11036 | weighted log-RMSE=0.01487 (~1.035x avg factor) | weighted |bias_log|=0.00775 | tail penalty=1.846 | acceptable=True
    low_BER(y<1e-6): n=96213 | factor~1.106x | RMSE_log=0.0437 | MAE_raw=0.000000 | P90=0.000000 | P95=0.000000 | bias_raw=+0.000000 | bias_log=+0.0042
    mid_BER(1e-6<=y<1e-3): n=15895 | factor~1.196x | RMSE_log=0.0776 | MAE_raw=0.000012 | P90=0.000033 | P95=0.000051 | bias_raw=+0.000009 | bias_log=+0.0420
    high_BER(1e-3<=y<0.10): n=60975 | factor~1.077x | RMSE_log=0.0322 | MAE_raw=0.002345 | P90=0.005745 | P95=0.007054 | bias_raw=+0.002077 | bias_log=+0.0189
    upper_BER(0.10<=y<0.20): n=65482 | factor~1.025x | RMSE_log=0.0107 | MAE_raw=0.002826 | P90=0.005032 | P95=0.006223 | bias_raw=+0.002541 | bias_log=+0.0077
    target_BER(0.20<=y<0.40): n=143374 | factor~1.011x | RMSE_log=0.004

Epoch 118 | LR: 1.95e-07 | Train Loss: 0.0008 | Val Loss: 0.0009 | Val RMSE(log10): 0.0271 (~1.064x) | Val Region Acc: 0.9895
  SELECTION | composite=0.10991 | ema_composite=0.10983 | weighted log-RMSE=0.01437 (~1.034x avg factor) | weighted |bias_log|=0.00574 | tail penalty=1.853 | acceptable=True
    low_BER(y<1e-6): n=96213 | factor~1.113x | RMSE_log=0.0466 | MAE_raw=0.000000 | P90=0.000000 | P95=0.000000 | bias_raw=+0.000000 | bias_log=+0.0047
    mid_BER(1e-6<=y<1e-3): n=15895 | factor~1.166x | RMSE_log=0.0667 | MAE_raw=0.000009 | P90=0.000025 | P95=0.000041 | bias_raw=-0.000001 | bias_log=+0.0138
    high_BER(1e-3<=y<0.10): n=60975 | factor~1.076x | RMSE_log=0.0319 | MAE_raw=0.002358 | P90=0.006021 | P95=0.007420 | bias_raw=+0.002032 | bias_log=+0.0162
    upper_BER(0.10<=y<0.20): n=65482 | factor~1.026x | RMSE_log=0.0113 | MAE_raw=0.002974 | P90=0.005327 | P95=0.006493 | bias_raw=+0.002710 | bias_log=+0.0082
    target_BER(0.20<=y<0.40): n=143374 | factor~1.011x | RMSE_log=0.004

In [4]:
import os
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.special import erfc
from itertools import product as iproduct

EPS = 1e-12
PLOT_DIR = "./plots_v4_random"
os.makedirs(PLOT_DIR, exist_ok=True)

# =========================
# Paths
# =========================
MODEL_PATH = "physics_fulltrain_highber_best_v2_target020_040_transformer_multitask.pth"
SCALER_PATH = "physics_fulltrain_highber_scalers_v2_target020_040_transformer_multitask.pkl"

# =========================
# Config
# =========================
PHYSICS_MAX_MEM_LEN = 14
PHYSICS_MIN_MEM_LEN = 14
ARRIVAL_COVERAGE = 0.70
N_THRESHOLDS = 500
RANDOM_SEED = 60

ORDINAL_THRESHOLDS = [
    1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1,
    0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45,
]
NUM_ORDINAL_THRESHOLDS = len(ORDINAL_THRESHOLDS)
NUM_REGION_CLASSES = NUM_ORDINAL_THRESHOLDS + 1

REGION_LABELS = {
    0: "y < 1e-6",
    1: "1e-6 <= y < 1e-5",
    2: "1e-5 <= y < 1e-4",
    3: "1e-4 <= y < 1e-3",
    4: "1e-3 <= y < 1e-2",
    5: "1e-2 <= y < 1e-1",
    6: "0.10 <= y < 0.15",
    7: "0.15 <= y < 0.20",
    8: "0.20 <= y < 0.25",
    9: "0.25 <= y < 0.30",
    10: "0.30 <= y < 0.35",
    11: "0.35 <= y < 0.40",
    12: "0.40 <= y < 0.45",
    13: "0.45 <= y <= 0.50",
}


# =========================
# Physics helpers
# =========================
def Fhit_function(radius, distance, diffusionCoef, t):
    if t <= 0:
        return 0.0
    return (radius / (distance + radius)) * erfc(distance / np.sqrt(4 * diffusionCoef * t))


def calculate_hitting_probabilities(mem_len, radius, distance, diffusionCoef, Ts):
    P = np.zeros(mem_len)
    for i in range(mem_len):
        t_end = (i + 1) * Ts
        t_start = i * Ts
        P[i] = Fhit_function(radius, distance, diffusionCoef, t_end) - Fhit_function(
            radius, distance, diffusionCoef, t_start
        )
    return P


def calculate_ber_vectorized(mem_len, threshold, P_scaled, variances):
    P_arr = np.asarray(P_scaled, dtype=float)[:mem_len]
    vars_arr = np.asarray(variances, dtype=float)[:mem_len]

    seqs = np.array(list(iproduct([0, 1], repeat=mem_len)), dtype=np.float64)[:, ::-1]
    c_bit = seqs[:, 0]

    mu = (seqs * P_arr).sum(axis=1)
    var_total = (seqs * vars_arr).sum(axis=1)
    std = np.sqrt(np.maximum(var_total, 0.0))

    pe = np.empty_like(mu)
    zero_std = (std == 0)
    if np.any(zero_std):
        pe[zero_std & (c_bit == 1)] = np.where(
            mu[zero_std & (c_bit == 1)] < threshold, 1.0, 0.0)
        pe[zero_std & (c_bit == 0)] = np.where(
            mu[zero_std & (c_bit == 0)] >= threshold, 1.0, 0.0)
    nz = ~zero_std
    if np.any(nz):
        pe[nz & (c_bit == 1)] = 0.5 * erfc(
            (mu[nz & (c_bit == 1)] - threshold) / (std[nz & (c_bit == 1)] * np.sqrt(2)))
        pe[nz & (c_bit == 0)] = 0.5 * erfc(
            (threshold - mu[nz & (c_bit == 0)]) / (std[nz & (c_bit == 0)] * np.sqrt(2)))
    return float(np.mean(pe))


def raw_to_region_labels_np(y_raw):
    y = np.asarray(y_raw).reshape(-1)
    return np.searchsorted(np.asarray(ORDINAL_THRESHOLDS), y, side="right").astype(np.int64)


def ordinal_logits_to_region_labels_torch(logits):
    probs = torch.sigmoid(logits)
    passed = (probs >= 0.5).sum(dim=1)
    return passed.long()


# =========================
# Shared scaling helper
# =========================
def apply_shared_scale(data, mean, std, valid_mask=None):
    scaled = ((data - mean) / std).astype(np.float32)
    if valid_mask is not None and scaled.ndim == 2 and valid_mask.shape == scaled.shape:
        scaled = scaled * valid_mask.astype(np.float32)
    return scaled


# =========================
# Generate one physical scenario
# =========================
def generate_physical_case(rng, min_mem_len, max_mem_len, arrival_coverage, max_tries=5000):
    for _ in range(max_tries):
        radius = rng.uniform(3.0, 5.0)
        distance = rng.uniform(10.0, 15.0)
        diff = rng.uniform(50.0, 75.0)
        Ts = rng.uniform(0.5, 1.2)
        N = int(10 ** rng.uniform(3.0, 6.0))

        f_inf = radius / (radius + distance)
        target = arrival_coverage * f_inf

        cumsum, k = 0.0, 0
        while k < max_mem_len:
            pk = Fhit_function(radius, distance, diff, (k + 1) * Ts) - Fhit_function(
                radius, distance, diff, k * Ts)
            cumsum += pk
            k += 1
            if cumsum >= target:
                break

        if k < min_mem_len:
            continue

        P_ext = calculate_hitting_probabilities(k + 1, radius, distance, diff, Ts)
        P_main = P_ext[:k]
        P_extra = float(P_ext[k]) if k < len(P_ext) else 0.0

        P_scaled = P_main * N
        variances = N * P_main * (1.0 - P_main)

        return {
            "radius": radius, "distance": distance, "diffusion": diff,
            "Ts": Ts, "N": N, "mem_len": k, "P": P_main,
            "P_scaled": P_scaled, "variances": variances,
            "P_mem_len_extra": P_extra,
            "P_mem_len_extra_var": P_extra * (1.0 - P_extra),
        }

    raise RuntimeError(
        f"Could not generate a physical case with mem_len >= {min_mem_len} "
        f"after {max_tries} tries."
    )


# =========================
# Model (matches v3 training: global_dim=7, global_embed=96, cond_dim=128)
# =========================
class ConditionalFeatureModulation(nn.Module):
    def __init__(self, cond_dim, feat_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, 2 * feat_dim),
        )
        self.scale = nn.Parameter(torch.tensor(0.25, dtype=torch.float32))

    def forward(self, x, cond):
        gamma_beta = self.net(cond)
        gamma, beta = torch.chunk(gamma_beta, 2, dim=-1)
        if x.dim() == 3:
            gamma = gamma.unsqueeze(1)
            beta = beta.unsqueeze(1)
        return x * (1.0 + self.scale * gamma) + self.scale * beta


class SetSelfAttentionBlock(nn.Module):
    def __init__(self, d_model, num_heads, mlp_ratio=4.0, dropout=0.10):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=num_heads,
            dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        hidden = int(d_model * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, d_model), nn.Dropout(dropout))

    def forward(self, x, key_padding_mask=None):
        y = self.norm1(x)
        attn_out, _ = self.attn(y, y, y, need_weights=False, key_padding_mask=key_padding_mask)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x


class SetAttentionPooling(nn.Module):
    def __init__(self, d_model, hidden_dim=128):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(d_model, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, 1))

    def forward(self, x, key_padding_mask=None):
        logits = self.score(x)
        if key_padding_mask is not None:
            logits = logits.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
        weights = torch.softmax(logits, dim=1)
        weights = torch.nan_to_num(weights, nan=0.0)
        return (weights * x).sum(dim=1)


class BERFirstTokenSetTransformerBoundedLogHalfMultiTask(nn.Module):
    def __init__(self, max_past_seq_len, token_dim=4, first_token_dim=4,
                 threshold_dim=1, global_dim=7, d_model=128, num_set_layers=4,
                 num_heads=4, mlp_ratio=4.0, dropout=0.10,
                 num_ordinal_thresholds=NUM_ORDINAL_THRESHOLDS):
        super().__init__()
        self.max_past_seq_len = max_past_seq_len
        self.d_model = d_model
        self.num_ordinal_thresholds = num_ordinal_thresholds

        self.first_proj = nn.Sequential(
            nn.Linear(first_token_dim, d_model), nn.LayerNorm(d_model),
            nn.GELU(), nn.Dropout(dropout))
        self.token_proj = nn.Sequential(
            nn.Linear(token_dim, d_model), nn.LayerNorm(d_model),
            nn.GELU(), nn.Dropout(dropout))
        self.thr_embed = nn.Sequential(
            nn.Linear(threshold_dim, 32), nn.GELU(),
            nn.Linear(32, 32), nn.GELU())
        self.global_embed = nn.Sequential(
            nn.Linear(global_dim, 96), nn.GELU(),
            nn.Linear(96, 96), nn.GELU())

        cond_dim = 32 + 96
        self.first_cond_mod = ConditionalFeatureModulation(cond_dim, d_model, 256)
        self.set_cond_mod = ConditionalFeatureModulation(cond_dim, d_model, 256)

        self.set_blocks = nn.ModuleList([
            SetSelfAttentionBlock(d_model, num_heads, mlp_ratio, dropout)
            for _ in range(num_set_layers)])

        self.first_refine = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_model),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model, d_model))
        self.final_set_norm = nn.LayerNorm(d_model)
        self.set_attn_pool = SetAttentionPooling(d_model, 128)

        set_summary_dim = 3 * d_model
        head_in = d_model + set_summary_dim + cond_dim

        self.reg_head = nn.Sequential(
            nn.Linear(head_in, 256), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.15),
            nn.Linear(128, 1))
        self.ord_head = nn.Sequential(
            nn.Linear(head_in, 256), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.15),
            nn.Linear(128, num_ordinal_thresholds))

    @staticmethod
    def raw_to_log10ber(pred_raw_unconstrained):
        log10_half = torch.log10(torch.tensor(
            0.5, device=pred_raw_unconstrained.device,
            dtype=pred_raw_unconstrained.dtype))
        return log10_half - F.softplus(pred_raw_unconstrained)

    @staticmethod
    def raw_to_ber(pred_raw_unconstrained):
        pred_log = BERFirstTokenSetTransformerBoundedLogHalfMultiTask.raw_to_log10ber(
            pred_raw_unconstrained)
        return torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)

    def forward(self, first_token, past_tokens, global_feats, threshold, key_padding_mask=None):
        thr = self.thr_embed(threshold)
        g = self.global_embed(global_feats)
        cond = torch.cat([thr, g], dim=-1)

        first_x = self.first_proj(first_token)
        first_x = self.first_cond_mod(first_x, cond)
        first_x = first_x + self.first_refine(first_x)

        set_x = self.token_proj(past_tokens)
        set_x = self.set_cond_mod(set_x, cond)
        for block in self.set_blocks:
            set_x = block(set_x, key_padding_mask=key_padding_mask)
        set_x = self.final_set_norm(set_x)

        if key_padding_mask is not None:
            valid_mask = (~key_padding_mask).unsqueeze(-1).float()
        else:
            valid_mask = torch.ones(
                set_x.shape[0], set_x.shape[1], 1,
                device=set_x.device, dtype=set_x.dtype)

        pooled_attn = self.set_attn_pool(set_x, key_padding_mask=key_padding_mask)
        valid_counts = valid_mask.sum(dim=1).clamp(min=1.0)
        pooled_mean = (set_x * valid_mask).sum(dim=1) / valid_counts

        if key_padding_mask is not None:
            x_for_max = set_x.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
        else:
            x_for_max = set_x
        pooled_max = x_for_max.amax(dim=1)
        pooled_max = torch.nan_to_num(pooled_max, nan=0.0, posinf=0.0, neginf=0.0)

        set_summary = torch.cat([pooled_attn, pooled_mean, pooled_max], dim=-1)
        fused = torch.cat([first_x, set_summary, cond], dim=-1)

        pred_raw_unconstrained = self.reg_head(fused)
        ord_logits = self.ord_head(fused)
        return pred_raw_unconstrained, ord_logits


# =========================
# Build inference inputs (7 global features, position-independent scaling)
# =========================
def prepare_inference_features(case, thresholds, scalers, device):
    """Build model inputs for a threshold sweep over a single physical case.

    Returns:
        first_t, past_t, global_t, thr_t, mask_t — all torch tensors on device
        global_t has shape [B, 7]: [z0, z1, hmg, nsid, log_hd, da, herf]
    """
    B = len(thresholds)
    mem_len = case["mem_len"]
    N = float(case["N"])

    P_raw = np.asarray(case["P"], dtype=np.float32)
    var_raw = np.asarray(case["variances"], dtype=np.float32)
    thr_raw = np.asarray(thresholds, dtype=np.float32).reshape(-1, 1)

    # Feature engineering
    taps_feat = (P_raw * N).astype(np.float32)
    vars_feat = var_raw.astype(np.float32)
    abs_feat = np.abs(taps_feat).astype(np.float32)
    snr_feat = np.log10((taps_feat ** 2) / (var_raw + EPS) + EPS).astype(np.float32)

    # Broadcast to [B, mem_len]
    taps_2d = np.broadcast_to(taps_feat[None, :], (B, mem_len)).copy()
    vars_2d = np.broadcast_to(vars_feat[None, :], (B, mem_len)).copy()
    abs_2d = np.broadcast_to(abs_feat[None, :], (B, mem_len)).copy()
    snr_2d = np.broadcast_to(snr_feat[None, :], (B, mem_len)).copy()

    L_past = mem_len - 1
    valid_past = np.ones((B, L_past), dtype=bool)

    # --- Threshold-dependent global features ---
    first_mean = taps_2d[:, 0:1]
    past_means = taps_2d[:, 1:]
    first_var = vars_2d[:, 0:1]
    past_vars = vars_2d[:, 1:]

    mu0 = (0.5 * past_means.sum(axis=1, keepdims=True)).astype(np.float32)
    mu1 = (first_mean + mu0).astype(np.float32)
    var0 = (0.5 * past_vars.sum(axis=1, keepdims=True)).astype(np.float32)
    var1 = (first_var + var0).astype(np.float32)
    std0 = np.sqrt(np.maximum(var0, EPS)).astype(np.float32)
    std1 = np.sqrt(np.maximum(var1, EPS)).astype(np.float32)

    z0 = ((thr_raw - mu0) / (std0 + EPS)).astype(np.float32)
    z1 = ((mu1 - thr_raw) / (std1 + EPS)).astype(np.float32)
    harmonic = (2.0 / (1.0 / (z0 + EPS) + 1.0 / (z1 + EPS))).astype(np.float32)
    abs_diff = np.abs(z0 - z1).astype(np.float32)
    hmg = (harmonic - 0.25 * abs_diff).astype(np.float32)

    # --- Scenario-level features (threshold-independent) ---
    signal = first_mean  # [B, 1]
    isi = past_means.sum(axis=1, keepdims=True)
    nsid = ((signal - isi) / (signal + isi + EPS)).astype(np.float32)

    gap = signal
    d0_inf = (gap / (std0 + EPS)).astype(np.float32)
    d1_inf = (gap / (std1 + EPS)).astype(np.float32)
    hd = (2.0 * d0_inf * d1_inf / (d0_inf + d1_inf + EPS)).astype(np.float32)
    log_hd = np.log10(hd + EPS).astype(np.float32)
    da = (np.abs(d0_inf - d1_inf) / (d0_inf + d1_inf + EPS)).astype(np.float32)

    past_taps_sum = past_means.sum(axis=1, keepdims=True)
    past_shares = past_means / (past_taps_sum + EPS)
    herf = (past_shares ** 2).sum(axis=1, keepdims=True).astype(np.float32)

    # --- Scale first token ---
    ft_tap = ((taps_2d[:, 0] - scalers["first_tap_mean"]) / scalers["first_tap_std"]).astype(np.float32)
    ft_var = ((vars_2d[:, 0] - scalers["first_var_mean"]) / scalers["first_var_std"]).astype(np.float32)
    ft_abs = ((abs_2d[:, 0] - scalers["first_abs_mean"]) / scalers["first_abs_std"]).astype(np.float32)
    ft_snr = ((snr_2d[:, 0] - scalers["first_snr_mean"]) / scalers["first_snr_std"]).astype(np.float32)
    first_token = np.stack([ft_tap, ft_var, ft_abs, ft_snr], axis=1).astype(np.float32)

    # --- Scale past tokens ---
    pt_tap = apply_shared_scale(taps_2d[:, 1:], scalers["past_tap_mean"], scalers["past_tap_std"], valid_past)
    pt_var = apply_shared_scale(vars_2d[:, 1:], scalers["past_var_mean"], scalers["past_var_std"], valid_past)
    pt_abs = apply_shared_scale(abs_2d[:, 1:], scalers["past_abs_mean"], scalers["past_abs_std"], valid_past)
    pt_snr = apply_shared_scale(snr_2d[:, 1:], scalers["past_snr_mean"], scalers["past_snr_std"], valid_past)
    past_tokens = np.stack([pt_tap, pt_var, pt_abs, pt_snr], axis=2).astype(np.float32)

    # --- Scale globals (7 features) ---
    z0_s = scalers["z0_scaler"].transform(z0).astype(np.float32)
    z1_s = scalers["z1_scaler"].transform(z1).astype(np.float32)
    hmg_s = scalers["harmonic_minus_gap_scaler"].transform(hmg).astype(np.float32)
    nsid_s = scalers["nsid_scaler"].transform(nsid).astype(np.float32)
    log_hd_s = scalers["log_hd_scaler"].transform(log_hd).astype(np.float32)
    da_s = scalers["da_scaler"].transform(da).astype(np.float32)
    herf_s = scalers["herf_scaler"].transform(herf).astype(np.float32)
    global_feats = np.concatenate(
        [z0_s, z1_s, hmg_s, nsid_s, log_hd_s, da_s, herf_s], axis=1
    ).astype(np.float32)

    # --- Scale threshold ---
    thr_log = np.log10(thr_raw + EPS).astype(np.float32)
    thr_s = scalers["thr_scaler"].transform(thr_log).astype(np.float32)

    # --- Padding mask (no padding needed here) ---
    pad_mask = np.zeros((B, L_past), dtype=bool)

    first_t = torch.from_numpy(first_token).to(device)
    past_t = torch.from_numpy(past_tokens).to(device)
    global_t = torch.from_numpy(global_feats).to(device)
    thr_t = torch.from_numpy(thr_s).to(device)
    mask_t = torch.from_numpy(pad_mask).to(device)

    return first_t, past_t, global_t, thr_t, mask_t


# =========================
# Generate scenario
# =========================
rng = np.random.default_rng(RANDOM_SEED)
case = generate_physical_case(
    rng,
    min_mem_len=PHYSICS_MIN_MEM_LEN,
    max_mem_len=PHYSICS_MAX_MEM_LEN,
    arrival_coverage=ARRIVAL_COVERAGE,
)

print("Generated physical scenario")
print("radius    =", case["radius"])
print("distance  =", case["distance"])
print("diffusion =", case["diffusion"])
print("Ts        =", case["Ts"])
print("N         =", case["N"])
print("mem_len   =", case["mem_len"])
print("P         =", case["P"])
print("P_scaled  =", case["P_scaled"])
print("variances =", case["variances"])

# =========================
# Threshold sweep — ground truth
# =========================
thr_min = 0.0
thr_max = float(np.sum(case["P_scaled"]))
thresholds = np.linspace(thr_min, thr_max, N_THRESHOLDS)

print("Threshold search interval:", thr_min, "to", thr_max)
print("sum(P_scaled) =", np.sum(case["P_scaled"]))

real_bers = np.array([
    calculate_ber_vectorized(
        mem_len=case["mem_len"],
        threshold=thr,
        P_scaled=case["P_scaled"],
        variances=case["variances"],
    )
    for thr in thresholds
])

real_bers = np.clip(real_bers, EPS, 0.5)
real_regions = raw_to_region_labels_np(real_bers)

# =========================
# Load model + scalers
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scalers = joblib.load(SCALER_PATH)

max_past_seq_len = scalers.get("train_max_past_seq_len", scalers.get("max_past_seq_len", case["mem_len"] - 1))
ordinal_thresholds = scalers.get("ordinal_thresholds", ORDINAL_THRESHOLDS)
num_ordinal = len(ordinal_thresholds)
global_dim = scalers.get("global_dim", 7)

model = BERFirstTokenSetTransformerBoundedLogHalfMultiTask(
    max_past_seq_len=max_past_seq_len,
    token_dim=scalers.get("past_token_dim", 4),
    first_token_dim=scalers.get("first_token_dim", 4),
    threshold_dim=1,
    global_dim=global_dim,
    d_model=128,
    num_set_layers=4,
    num_heads=4,
    mlp_ratio=4.0,
    dropout=0.10,
    num_ordinal_thresholds=num_ordinal,
).to(device)

state = torch.load(MODEL_PATH, map_location=device, weights_only=True)
model.load_state_dict(state)
model.eval()

print(f"\nModel loaded: max_past_seq_len={max_past_seq_len}, "
      f"global_dim={global_dim}, num_ordinal={num_ordinal}")
print(f"Scaling strategy: {scalers.get('scaling_strategy', 'unknown')}")

# =========================
# Predict across thresholds
# =========================
first_t, past_t, global_t, thr_t, mask_t = prepare_inference_features(
    case, thresholds, scalers, device,
)

with torch.no_grad():
    pred_raw_out, ord_logits = model(
        first_t, past_t, global_t, thr_t, key_padding_mask=mask_t,
    )
    pred_bers = model.raw_to_ber(pred_raw_out).cpu().numpy().reshape(-1)
    pred_log = model.raw_to_log10ber(pred_raw_out).cpu().numpy().reshape(-1)
    pred_regions = ordinal_logits_to_region_labels_torch(ord_logits).cpu().numpy().reshape(-1)
    pred_region_probs = torch.sigmoid(ord_logits).cpu().numpy()

pred_bers = np.clip(pred_bers, EPS, 0.5)

# =========================
# Comparison table
# =========================
results = pd.DataFrame({
    "threshold": thresholds,
    "real_BER": real_bers,
    "estimated_BER": pred_bers,
    "real_region": real_regions,
    "predicted_region": pred_regions,
    "abs_error": np.abs(pred_bers - real_bers),
    "abs_log10_error": np.abs(
        np.log10(np.clip(pred_bers, EPS, 0.5)) -
        np.log10(np.clip(real_bers, EPS, 0.5))
    ),
    "region_abs_error": np.abs(pred_regions.astype(np.int64) - real_regions.astype(np.int64)),
})

print(results.head(15))

print("\nSummary")
print("Mean abs raw error        :", results["abs_error"].mean())
print("Mean abs log10 error      :", results["abs_log10_error"].mean())
print("Max  abs log10 error      :", results["abs_log10_error"].max())
print("Mean region abs error     :", results["region_abs_error"].mean())
print("Exact region accuracy     :", np.mean(results["real_region"] == results["predicted_region"]))

best_real_idx = np.argmin(real_bers)
best_est_idx = np.argmin(pred_bers)

print("\nBest threshold from real BER      :", thresholds[best_real_idx])
print("Minimum real BER                  :", real_bers[best_real_idx])
print("Real BER region there             :", REGION_LABELS.get(int(real_regions[best_real_idx]), "?"))

print("Best threshold from estimated BER :", thresholds[best_est_idx])
print("Estimated BER at that threshold   :", pred_bers[best_est_idx])
print("Predicted BER region there        :", REGION_LABELS.get(int(pred_regions[best_est_idx]), "?"))

# =========================
# Plot 1: log-scale BER
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["estimated_BER"], label="Estimated BER")
plt.plot(results["threshold"], results["real_BER"], label="Real BER")
plt.yscale("log")
plt.xlabel("Threshold")
plt.ylabel("BER")
plt.title(f"Threshold vs BER (log scale) — mem_len={case['mem_len']}")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "01_ber_log_scale.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/01_ber_log_scale.png")

# =========================
# Plot 2: linear-scale BER
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["estimated_BER"], label="Estimated BER")
plt.plot(results["threshold"], results["real_BER"], label="Real BER")
plt.yscale("linear")
plt.xlabel("Threshold")
plt.ylabel("BER")
plt.title(f"Threshold vs BER (linear scale) — mem_len={case['mem_len']}")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "02_ber_linear_scale.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/02_ber_linear_scale.png")

# =========================
# Plot 3: predicted vs true region
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["predicted_region"], label="Predicted region")
plt.plot(results["threshold"], results["real_region"], label="True region")
plt.xlabel("Threshold")
plt.ylabel("BER Region Class")
plt.title(f"Threshold vs BER Region — mem_len={case['mem_len']}")
plt.yticks(list(REGION_LABELS.keys()),
           [REGION_LABELS[k] for k in sorted(REGION_LABELS.keys())],
           fontsize=7)
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "03_region_comparison.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/03_region_comparison.png")

# =========================
# Plot 4: ordinal threshold probabilities
# =========================
plt.figure(figsize=(12, 7))
for i, thr_val in enumerate(ordinal_thresholds):
    plt.plot(thresholds, pred_region_probs[:, i], label=f"P(y >= {thr_val:g})")
plt.xlabel("Threshold")
plt.ylabel("Ordinal Probability")
plt.title(f"Ordinal Head Outputs Across Threshold Sweep — mem_len={case['mem_len']}")
plt.legend(fontsize=7, ncol=2)
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "04_ordinal_probabilities.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/04_ordinal_probabilities.png")

# =========================
# Plot 5: absolute error by threshold
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["abs_error"], label="Abs raw error", alpha=0.8)
plt.plot(results["threshold"], results["abs_log10_error"], label="Abs log10 error", alpha=0.8)
plt.yscale("log")
plt.xlabel("Threshold")
plt.ylabel("Error")
plt.title(f"Prediction Error Across Threshold Sweep — mem_len={case['mem_len']}")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "05_error_by_threshold.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/05_error_by_threshold.png")

Generated physical scenario
radius    = 4.999903475776163
distance  = 13.063595613360427
diffusion = 63.15707394395149
Ts        = 0.5834416162676823
N         = 2026
mem_len   = 14
P         = [0.03545102 0.042582   0.02704795 0.01857409 0.01368091 0.01059455
 0.00850978 0.00702648 0.00592783 0.00508774 0.00442855 0.00390015
 0.00346895 0.00311165]
P_scaled  = [71.82377081 86.27113641 54.79914164 37.63111579 27.71752004 21.46456524
 17.24082032 14.23564842 12.00979138 10.30776824  8.9722469   7.90171201
  7.0280834   6.304198  ]
variances = [69.27754472 82.59753869 53.31693734 36.93215188 27.33831919 21.23715776
 17.09410468 14.13562193 11.93859933 10.25532496  8.93251283  7.87089412
  7.00370336  6.28458156]
Threshold search interval: 0.0 to 383.7075186062634
sum(P_scaled) = 383.7075186062634

Model loaded: max_past_seq_len=13, global_dim=7, num_ordinal=13
Scaling strategy: position_independent
    threshold  real_BER  estimated_BER  real_region  predicted_region  \
0    0.000000  0.

In [6]:
import os
import re
import copy
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.multiprocessing as mp
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

mp.set_sharing_strategy("file_system")

# =========================
# Configuration
# =========================
BASE_DIR = "./"

DATA_PATHS = [
    os.path.join(BASE_DIR, "data_random_with_random_variances_total_v2.csv"),
]

NROWS_PER_DATASET = 5_000_000

BEST_MODEL_SAVE_PATH = os.path.join(
    BASE_DIR,
    "random_extra_multitask_best.pth",
)
LAST_MODEL_SAVE_PATH = os.path.join(
    BASE_DIR,
    "random_extra_multitask_last.pth",
)
SCALER_SAVE_PATH = os.path.join(
    BASE_DIR,
    "random_extra_multitask_scalers.pkl",
)

EPS = 1e-12
LOG10_HALF = float(np.log10(0.5))

ORDINAL_THRESHOLDS = [
    1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1,
    0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45,
]
NUM_ORDINAL_THRESHOLDS = len(ORDINAL_THRESHOLDS)
NUM_REGION_CLASSES = NUM_ORDINAL_THRESHOLDS + 1


# =========================
# Utilities
# =========================
def get_sorted_seq_cols(columns, prefix):
    pattern = re.compile(rf"^{re.escape(prefix)}_(\d+)$")
    matched = []
    for col in columns:
        m = pattern.match(col)
        if m:
            matched.append((int(m.group(1)), col))
    matched.sort(key=lambda x: x[0])
    return [col for _, col in matched]


def make_strat_bins(y_log, n_bins=10):
    y_flat = y_log.reshape(-1)
    quantiles = np.linspace(0, 1, n_bins + 1)
    edges = np.quantile(y_flat, quantiles)
    edges = np.unique(edges)

    if len(edges) < 3:
        return None

    bins = np.digitize(y_flat, edges[1:-1], right=True)
    counts = np.bincount(bins)
    if np.any(counts < 2):
        return None
    return bins


def has_nonfinite_tensor(x):
    return not torch.isfinite(x).all().item()


def validate_required_columns(df, file_path):
    if "mem_len" not in df.columns:
        raise ValueError(f"Required column 'mem_len' not found in {file_path}")
    if "N" not in df.columns:
        raise ValueError(f"Required column 'N' not found in {file_path}")

    tap_cols = get_sorted_seq_cols(df.columns, "tap")
    var_cols = get_sorted_seq_cols(df.columns, "var")

    if not tap_cols:
        raise ValueError(f"No tap_* columns found in {file_path}")
    if not var_cols:
        raise ValueError(f"No var_* columns found in {file_path}")
    if len(tap_cols) != len(var_cols):
        raise ValueError(
            f"tap/var length mismatch in {file_path}: "
            f"{len(tap_cols)} tap cols vs {len(var_cols)} var cols"
        )

    required_cols = tap_cols + var_cols + ["threshold", "BER", "N"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in {file_path}: {missing}")

    return tap_cols, var_cols


def load_and_merge_data(csv_paths, nrows_per_dataset):
    dfs = []
    reference_tap_cols = None
    reference_var_cols = None

    for path in csv_paths:
        if not os.path.exists(path):
            raise FileNotFoundError(f"Dataset not found: {path}")

        print(f"Loading up to {nrows_per_dataset:,} rows from: {path}")
        df = pd.read_csv(path, nrows=nrows_per_dataset)

        tap_cols, var_cols = validate_required_columns(df, path)

        if reference_tap_cols is None:
            reference_tap_cols = tap_cols
            reference_var_cols = var_cols
        else:
            if tap_cols != reference_tap_cols:
                raise ValueError("tap columns do not match across files.")
            if var_cols != reference_var_cols:
                raise ValueError("var columns do not match across files.")

        df["source_dataset"] = os.path.basename(path)
        dfs.append(df)

    merged = pd.concat(dfs, ignore_index=True)
    print(f"Combined rows before filtering: {len(merged):,}")

    return merged, reference_tap_cols, reference_var_cols


def y_log_to_raw_np(y_log):
    return np.clip(10 ** y_log, EPS, 0.5)


def raw_to_region_labels_np(y_raw):
    y = np.asarray(y_raw).reshape(-1)
    labels = np.searchsorted(np.asarray(ORDINAL_THRESHOLDS), y, side="right")
    return labels.astype(np.int64)


def region_labels_to_ordinal_targets_np(labels, num_thresholds=NUM_ORDINAL_THRESHOLDS):
    labels = np.asarray(labels).reshape(-1)
    thresholds = np.arange(1, num_thresholds + 1, dtype=np.int64)
    ordinal = (labels[:, None] >= thresholds[None, :]).astype(np.float32)
    return ordinal


def ordinal_logits_to_region_labels_torch(logits):
    probs = torch.sigmoid(logits)
    passed = (probs >= 0.5).sum(dim=1)
    return passed.long()


def compute_region_class_weights(labels_np, num_classes=NUM_REGION_CLASSES, max_weight=8.0):
    counts = np.bincount(labels_np.reshape(-1), minlength=num_classes).astype(np.float64)
    counts = np.maximum(counts, 1.0)
    weights = counts.sum() / (num_classes * counts)
    weights = np.sqrt(weights)
    weights = np.clip(weights, 1.0, max_weight)
    return weights.astype(np.float32)


def compute_ordinal_pos_weights_from_region_labels(
    region_labels_np,
    num_thresholds=NUM_ORDINAL_THRESHOLDS,
    max_weight=20.0,
):
    ordinal_targets = region_labels_to_ordinal_targets_np(region_labels_np, num_thresholds)
    pos_counts = ordinal_targets.sum(axis=0)
    neg_counts = ordinal_targets.shape[0] - pos_counts
    pos_counts = np.maximum(pos_counts, 1.0)
    pos_weight = neg_counts / pos_counts
    pos_weight = np.clip(pos_weight, 1.0, max_weight)
    return pos_weight.astype(np.float32)


# =========================
# Position-Independent Scaling
# =========================
class SharedFeatureScaler:
    """A scaler that stores a single (mean, std) per feature channel,
    shared across all sequence positions.

    For the first token (always position 0) we keep a separate scaler,
    since the current-symbol tap has a genuinely different distribution
    than the ISI taps at positions 1+.

    For all past positions we pool every valid entry into one distribution
    and fit a single mean/std.  This makes the scaler independent of
    sequence length — at inference time, any number of past taps can be
    scaled with the same parameters.
    """

    def __init__(self):
        self.first_mean = None  # shape [n_features]
        self.first_std = None
        self.past_mean = None   # shape [n_features]
        self.past_std = None

    def fit(self, first_data_list, past_data_list, past_valid_list):
        """
        Args:
            first_data_list: list of arrays, each [N, n_features] for the
                             first-token features (taps[:,0], vars[:,0], ...).
                             Concatenated across splits if desired, or just train.
            past_data_list:  list of arrays, each [N, L_past, n_features] or
                             [N, L_past] for a single feature channel.
            past_valid_list: list of bool arrays, each [N, L_past], True = valid.
        """
        # --- First token ---
        first_all = np.concatenate(first_data_list, axis=0)  # [N_total, F]
        self.first_mean = first_all.mean(axis=0).astype(np.float64)
        self.first_std = first_all.std(axis=0).astype(np.float64)
        self.first_std = np.maximum(self.first_std, 1e-12)

        # --- Past tokens (pool all valid entries per feature) ---
        valid_entries = []
        for data, valid in zip(past_data_list, past_valid_list):
            if data.ndim == 2:
                # single feature: [N, L] -> expand to [N, L, 1]
                data = data[:, :, None]
            # data: [N, L, F], valid: [N, L]
            valid_expanded = valid[:, :, None]  # [N, L, 1]
            # Gather valid entries: [?, F]
            valid_entries.append(data[np.broadcast_to(valid_expanded, data.shape)].reshape(-1, data.shape[-1]))

        pooled = np.concatenate(valid_entries, axis=0)  # [total_valid, F]
        self.past_mean = pooled.mean(axis=0).astype(np.float64)
        self.past_std = pooled.std(axis=0).astype(np.float64)
        self.past_std = np.maximum(self.past_std, 1e-12)

    def transform_first(self, first_data):
        """first_data: [N, F] -> scaled [N, F]"""
        return ((first_data - self.first_mean) / self.first_std).astype(np.float32)

    def transform_past(self, past_data, past_lens):
        """past_data: [N, L, F] or [N, L] -> scaled + re-zeroed [N, L, ...].
        past_lens: [N] int, number of valid past positions per row."""
        shape = past_data.shape
        if past_data.ndim == 2:
            scaled = ((past_data - self.past_mean[0]) / self.past_std[0]).astype(np.float32)
            L = shape[1]
        else:
            scaled = ((past_data - self.past_mean) / self.past_std).astype(np.float32)
            L = shape[1]

        # Re-zero padding positions
        valid = (np.arange(L)[None, :] < past_lens[:, None])  # [N, L]
        if scaled.ndim == 3:
            valid = valid[:, :, None]
        scaled = scaled * valid.astype(np.float32)
        return scaled


def fit_shared_scalar_scaler(train_data, train_valid_mask):
    """Fit a single-feature StandardScaler on pooled valid entries.
    train_data: [N, L], train_valid_mask: [N, L] bool.
    Returns (mean, std) as floats."""
    entries = train_data[train_valid_mask].reshape(-1)
    if len(entries) == 0:
        entries = train_data.reshape(-1)
    mean = float(entries.mean())
    std = float(max(entries.std(), 1e-12))
    return mean, std


def apply_shared_scale(data, mean, std, valid_mask=None):
    """Scale data with a single (mean, std), optionally re-zero invalid positions.
    data: [N, L] or [N, 1], valid_mask: [N, L] bool or None."""
    scaled = ((data - mean) / std).astype(np.float32)
    if valid_mask is not None and scaled.ndim == 2 and valid_mask.shape == scaled.shape:
        scaled = scaled * valid_mask.astype(np.float32)
    return scaled


# =========================
# Targeted Regression Loss
# =========================
class StableMultiObjectiveBERBoundedLogLoss(nn.Module):
    def __init__(
        self,
        log_delta=0.35,
        raw_delta=0.006,
        rel_delta=0.03,
        alpha_log=0.45,
        beta_raw=0.35,
        gamma_rel=0.20,
        use_regime_weights=True,
    ):
        super().__init__()
        self.log_delta = log_delta
        self.raw_delta = raw_delta
        self.rel_delta = rel_delta
        self.alpha_log = alpha_log
        self.beta_raw = beta_raw
        self.gamma_rel = gamma_rel
        self.use_regime_weights = use_regime_weights

    @staticmethod
    def huber_elementwise(pred, target, delta):
        err = pred - target
        abs_err = err.abs()
        return torch.where(
            abs_err < delta,
            0.5 * err * err,
            delta * (abs_err - 0.5 * delta),
        )

    @staticmethod
    def raw_to_pred_log(pred_raw_unconstrained):
        log10_half = torch.log10(
            torch.tensor(
                0.5,
                device=pred_raw_unconstrained.device,
                dtype=pred_raw_unconstrained.dtype,
            )
        )
        return log10_half - F.softplus(pred_raw_unconstrained)

    @staticmethod
    def raw_to_pred_ber(pred_raw_unconstrained):
        pred_log = StableMultiObjectiveBERBoundedLogLoss.raw_to_pred_log(
            pred_raw_unconstrained
        )
        return torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)

    def forward(self, pred_raw_unconstrained, target_log):
        pred_log = self.raw_to_pred_log(pred_raw_unconstrained)
        pred_raw = torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)

        target_raw = torch.pow(10.0, target_log).clamp(min=EPS, max=0.5)
        target_log_for_loss = torch.log10(target_raw)

        log_loss = self.huber_elementwise(pred_log, target_log_for_loss, self.log_delta)
        raw_loss = self.huber_elementwise(pred_raw, target_raw, self.raw_delta)
        rel_err = (pred_raw - target_raw) / torch.clamp(target_raw, min=1e-6)
        rel_loss = self.huber_elementwise(rel_err, torch.zeros_like(rel_err), self.rel_delta)

        total = (
            self.alpha_log * log_loss
            + self.beta_raw * raw_loss
            + self.gamma_rel * rel_loss
        )

        if self.use_regime_weights:
            weights = torch.ones_like(target_raw)
            weights = torch.where(
                (target_raw >= 1e-2) & (target_raw < 0.1),
                torch.full_like(weights, 1.5), weights,
            )
            weights = torch.where(
                (target_raw >= 0.1) & (target_raw < 0.15),
                torch.full_like(weights, 2.5), weights,
            )
            weights = torch.where(
                (target_raw >= 0.15) & (target_raw < 0.2),
                torch.full_like(weights, 3.0), weights,
            )
            weights = torch.where(
                (target_raw >= 0.2) & (target_raw < 0.3),
                torch.full_like(weights, 4.0), weights,
            )
            weights = torch.where(
                (target_raw >= 0.3) & (target_raw < 0.4),
                torch.full_like(weights, 4.5), weights,
            )
            weights = torch.where(
                (target_raw >= 0.4) & (target_raw < 0.45),
                torch.full_like(weights, 3.0), weights,
            )
            weights = torch.where(
                target_raw >= 0.45,
                torch.full_like(weights, 2.5), weights,
            )
            total = total * weights

        return total.mean()


class OrdinalBCELoss(nn.Module):
    def __init__(self, pos_weight=None, reduction="mean"):
        super().__init__()
        if pos_weight is not None and not isinstance(pos_weight, torch.Tensor):
            pos_weight = torch.tensor(pos_weight, dtype=torch.float32)
        self.register_buffer(
            "pos_weight", pos_weight if pos_weight is not None else None
        )
        self.reduction = reduction

    def forward(self, logits, ordinal_targets):
        return F.binary_cross_entropy_with_logits(
            logits,
            ordinal_targets,
            pos_weight=self.pos_weight,
            reduction=self.reduction,
        )


class MultiTaskBERLoss(nn.Module):
    def __init__(self, reg_loss, ord_loss, lambda_ord=0.25):
        super().__init__()
        self.reg_loss = reg_loss
        self.ord_loss = ord_loss
        self.lambda_ord = lambda_ord

    def forward(self, pred_raw_unconstrained, ord_logits, target_log, target_ord):
        reg = self.reg_loss(pred_raw_unconstrained, target_log)
        ordl = self.ord_loss(ord_logits, target_ord)
        total = reg + self.lambda_ord * ordl
        return total, reg.detach(), ordl.detach()


# =========================
# Set Transformer Blocks
# =========================
class SetSelfAttentionBlock(nn.Module):
    def __init__(self, d_model, num_heads, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norm2 = nn.LayerNorm(d_model)

        hidden = int(d_model * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x, key_padding_mask=None):
        """
        Args:
            x: [B, L, D]
            key_padding_mask: [B, L] bool, True = padding (ignore)
        """
        y = self.norm1(x)
        attn_out, _ = self.attn(
            y, y, y,
            need_weights=False,
            key_padding_mask=key_padding_mask,
        )
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x


class ConditionalFeatureModulation(nn.Module):
    def __init__(self, cond_dim, feat_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, 2 * feat_dim),
        )
        self.scale = nn.Parameter(torch.tensor(0.25, dtype=torch.float32))

    def forward(self, x, cond):
        gamma_beta = self.net(cond)
        gamma, beta = torch.chunk(gamma_beta, 2, dim=-1)

        if x.dim() == 3:
            gamma = gamma.unsqueeze(1)
            beta = beta.unsqueeze(1)

        return x * (1.0 + self.scale * gamma) + self.scale * beta


class SetAttentionPooling(nn.Module):
    def __init__(self, d_model, hidden_dim=128):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x, key_padding_mask=None):
        """
        Args:
            x: [B, L, D]
            key_padding_mask: [B, L] bool, True = padding
        Returns:
            pooled: [B, D]
        """
        logits = self.score(x)  # [B, L, 1]

        if key_padding_mask is not None:
            logits = logits.masked_fill(
                key_padding_mask.unsqueeze(-1), float("-inf")
            )

        weights = torch.softmax(logits, dim=1)  # [B, L, 1]
        # Safety: all-masked rows produce NaN from softmax(-inf); replace with 0
        weights = torch.nan_to_num(weights, nan=0.0)

        pooled = (weights * x).sum(dim=1)  # [B, D]
        return pooled


# =========================
# Model
# =========================
class BERFirstTokenSetTransformerBoundedLogHalfMultiTask(nn.Module):
    def __init__(
        self,
        max_past_seq_len,
        token_dim=4,
        first_token_dim=4,
        threshold_dim=1,
        global_dim=7,
        d_model=128,
        num_set_layers=4,
        num_heads=4,
        mlp_ratio=4.0,
        dropout=0.10,
        num_ordinal_thresholds=NUM_ORDINAL_THRESHOLDS,
    ):
        super().__init__()

        self.max_past_seq_len = max_past_seq_len
        self.token_dim = token_dim
        self.first_token_dim = first_token_dim
        self.d_model = d_model
        self.num_ordinal_thresholds = num_ordinal_thresholds

        self.first_proj = nn.Sequential(
            nn.Linear(first_token_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.token_proj = nn.Sequential(
            nn.Linear(token_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.thr_embed = nn.Sequential(
            nn.Linear(threshold_dim, 32),
            nn.GELU(),
            nn.Linear(32, 32),
            nn.GELU(),
        )

        self.global_embed = nn.Sequential(
            nn.Linear(global_dim, 96),
            nn.GELU(),
            nn.Linear(96, 96),
            nn.GELU(),
        )

        cond_dim = 32 + 96

        self.first_cond_mod = ConditionalFeatureModulation(
            cond_dim=cond_dim, feat_dim=d_model, hidden_dim=256,
        )
        self.set_cond_mod = ConditionalFeatureModulation(
            cond_dim=cond_dim, feat_dim=d_model, hidden_dim=256,
        )

        self.set_blocks = nn.ModuleList(
            [
                SetSelfAttentionBlock(
                    d_model=d_model,
                    num_heads=num_heads,
                    mlp_ratio=mlp_ratio,
                    dropout=dropout,
                )
                for _ in range(num_set_layers)
            ]
        )

        self.first_refine = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
        )

        self.final_set_norm = nn.LayerNorm(d_model)
        self.set_attn_pool = SetAttentionPooling(d_model=d_model, hidden_dim=128)

        set_summary_dim = 3 * d_model
        head_in = d_model + set_summary_dim + cond_dim

        self.reg_head = nn.Sequential(
            nn.Linear(head_in, 256),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(128, 1),
        )

        self.ord_head = nn.Sequential(
            nn.Linear(head_in, 256),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(128, num_ordinal_thresholds),
        )

    @staticmethod
    def raw_to_log10ber(pred_raw_unconstrained):
        log10_half = torch.log10(
            torch.tensor(
                0.5,
                device=pred_raw_unconstrained.device,
                dtype=pred_raw_unconstrained.dtype,
            )
        )
        return log10_half - F.softplus(pred_raw_unconstrained)

    @staticmethod
    def raw_to_ber(pred_raw_unconstrained):
        pred_log = BERFirstTokenSetTransformerBoundedLogHalfMultiTask.raw_to_log10ber(
            pred_raw_unconstrained
        )
        return torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)

    def forward(self, first_token, past_tokens, global_feats, threshold, key_padding_mask=None):
        """
        Args:
            first_token:       [B, first_token_dim]
            past_tokens:       [B, L, token_dim]  (L can vary between training and inference)
            global_feats:      [B, global_dim]
            threshold:         [B, 1]
            key_padding_mask:  [B, L] bool, True = padding position
        """
        thr = self.thr_embed(threshold)
        g = self.global_embed(global_feats)
        cond = torch.cat([thr, g], dim=-1)

        # First token path
        first_x = self.first_proj(first_token)
        first_x = self.first_cond_mod(first_x, cond)
        first_x = first_x + self.first_refine(first_x)

        # Past tokens path with mask
        set_x = self.token_proj(past_tokens)
        set_x = self.set_cond_mod(set_x, cond)

        for block in self.set_blocks:
            set_x = block(set_x, key_padding_mask=key_padding_mask)

        set_x = self.final_set_norm(set_x)

        # --- Masked pooling ---
        if key_padding_mask is not None:
            valid_mask = (~key_padding_mask).unsqueeze(-1).float()  # [B, L, 1]
        else:
            valid_mask = torch.ones(
                set_x.shape[0], set_x.shape[1], 1,
                device=set_x.device, dtype=set_x.dtype,
            )

        # Attention pooling
        pooled_attn = self.set_attn_pool(set_x, key_padding_mask=key_padding_mask)

        # Masked mean pooling
        valid_counts = valid_mask.sum(dim=1).clamp(min=1.0)  # [B, 1]
        pooled_mean = (set_x * valid_mask).sum(dim=1) / valid_counts

        # Masked max pooling
        if key_padding_mask is not None:
            x_for_max = set_x.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
        else:
            x_for_max = set_x
        pooled_max = x_for_max.amax(dim=1)
        pooled_max = torch.nan_to_num(pooled_max, nan=0.0, posinf=0.0, neginf=0.0)

        set_summary = torch.cat([pooled_attn, pooled_mean, pooled_max], dim=-1)
        fused = torch.cat([first_x, set_summary, cond], dim=-1)

        pred_raw_unconstrained = self.reg_head(fused)
        ord_logits = self.ord_head(fused)

        return pred_raw_unconstrained, ord_logits


# =========================
# Data
# =========================
def prepare_data(csv_paths, batch_size=256, nrows_per_dataset=2_500_000, num_workers=0):
    df, tap_cols, var_cols = load_and_merge_data(csv_paths, nrows_per_dataset)

    df = df[df["mem_len"] != 1].copy()

    df[tap_cols] = df[tap_cols].fillna(0.0)
    df[var_cols] = df[var_cols].fillna(0.0)
    df["threshold"] = df["threshold"].fillna(0.0)
    df["BER"] = df["BER"].fillna(0.0)
    df["N"] = df["N"].fillna(0.0)

    df = df[(df["threshold"] > 0) & (df["BER"] > 0) & (df["N"] > 0)].copy()
    df["BER"] = df["BER"].clip(lower=EPS, upper=0.5)

    print(f"Rows after cleaning/filtering: {len(df):,}")

    # ---- Extract raw arrays ----
    X_taps_raw = df[tap_cols].to_numpy(dtype=np.float32)
    X_vars_raw = df[var_cols].to_numpy(dtype=np.float32)
    num_molecules = df["N"].to_numpy(dtype=np.float32).reshape(-1, 1)
    mem_len = df["mem_len"].to_numpy(dtype=np.int64)

    X_thr_raw = df["threshold"].to_numpy(dtype=np.float32).reshape(-1, 1)
    y_raw = df["BER"].to_numpy(dtype=np.float32).reshape(-1, 1)

    X_thr = np.log10(X_thr_raw + EPS).astype(np.float32)
    y_log = np.log10(y_raw + EPS).astype(np.float32)

    # ---- Feature engineering ----
    X_taps_feat = (X_taps_raw * num_molecules).astype(np.float32)

    if np.any(X_vars_raw < 0):
        raise ValueError("Variance columns contain negative values.")

    X_vars_feat = X_vars_raw.astype(np.float32)
    abs_taps_raw = np.abs(X_taps_feat).astype(np.float32)
    snr_raw = np.log10((X_taps_feat ** 2) / (X_vars_raw + EPS) + EPS).astype(np.float32)

    L_full = X_taps_feat.shape[1]

    # ---- Validity masks ----
    mem_len_clamped = np.minimum(mem_len, L_full)
    valid_full = (np.arange(L_full)[None, :] < mem_len_clamped[:, None])  # [N, L_full]
    valid_past = valid_full[:, 1:]  # [N, L_full-1]
    valid_past_float = valid_past.astype(np.float32)
    past_lens = np.maximum(mem_len_clamped - 1, 0).astype(np.int64)

    # ---- Mask-aware global features ----
    first_mean = X_taps_feat[:, 0:1]
    past_means = X_taps_feat[:, 1:]
    first_var = X_vars_feat[:, 0:1]
    past_vars = X_vars_feat[:, 1:]

    past_means_masked = past_means * valid_past_float
    past_vars_masked = past_vars * valid_past_float

    mu0_raw = (0.5 * np.sum(past_means_masked, axis=1, keepdims=True)).astype(np.float32)
    mu1_raw = (first_mean + mu0_raw).astype(np.float32)

    var0_raw = (0.5 * np.sum(past_vars_masked, axis=1, keepdims=True)).astype(np.float32)
    var1_raw = (first_var + var0_raw).astype(np.float32)

    std0_raw = np.sqrt(np.maximum(var0_raw, EPS)).astype(np.float32)
    std1_raw = np.sqrt(np.maximum(var1_raw, EPS)).astype(np.float32)

    z0_raw = ((X_thr_raw - mu0_raw) / (std0_raw + EPS)).astype(np.float32)
    z1_raw = ((mu1_raw - X_thr_raw) / (std1_raw + EPS)).astype(np.float32)

    harmonic_side_z_raw = (
        2.0 / (1.0 / (z0_raw + EPS) + 1.0 / (z1_raw + EPS))
    ).astype(np.float32)
    abs_diff_side_z_raw = np.abs(z0_raw - z1_raw).astype(np.float32)
    harmonic_minus_gap_raw = (
        harmonic_side_z_raw - 0.25 * abs_diff_side_z_raw
    ).astype(np.float32)

    # ---- Scenario-level features (threshold-INDEPENDENT) ----
    # These capture intrinsic difficulty of the physical scenario.

    # 1. NSID: Normalized Signal-Interference Difference, range ~ [-1, +1]
    signal_raw = first_mean  # P_0 * N, shape [N, 1]
    isi_raw = np.sum(past_means_masked, axis=1, keepdims=True)  # [N, 1]
    nsid_raw = ((signal_raw - isi_raw) / (signal_raw + isi_raw + EPS)).astype(np.float32)

    # 2. Log Harmonic Discriminability (threshold-free)
    gap_raw = signal_raw  # mu1 - mu0 = P_0 * N
    d0_raw = (gap_raw / (std0_raw + EPS)).astype(np.float32)
    d1_raw = (gap_raw / (std1_raw + EPS)).astype(np.float32)
    harmonic_discrim_raw = (2.0 * d0_raw * d1_raw / (d0_raw + d1_raw + EPS)).astype(np.float32)
    log_harmonic_discrim_raw = np.log10(harmonic_discrim_raw + EPS).astype(np.float32)

    # 3. Discriminability Asymmetry
    discrim_asymmetry_raw = (np.abs(d0_raw - d1_raw) / (d0_raw + d1_raw + EPS)).astype(np.float32)

    # 4. Herfindahl Index of ISI concentration [1/n_past, 1]
    past_taps_sum = np.sum(past_means_masked, axis=1, keepdims=True)  # [N, 1]
    past_shares = past_means_masked / (past_taps_sum + EPS)           # [N, L_past]
    herfindahl_raw = np.sum(past_shares ** 2, axis=1, keepdims=True).astype(np.float32)  # [N, 1]

    # ---- Labels ----
    region_labels = raw_to_region_labels_np(y_raw)
    ordinal_targets = region_labels_to_ordinal_targets_np(region_labels)
    strat_labels = make_strat_bins(y_log, n_bins=10)

    # ---- Train / temp split ----
    split_args = dict(test_size=0.30, random_state=42)
    if strat_labels is not None:
        split_args["stratify"] = strat_labels

    (
        t_taps_feat, temp_taps_feat,
        t_vars_feat, temp_vars_feat,
        t_abs_raw, temp_abs_raw,
        t_snr_raw, temp_snr_raw,
        t_z0_raw, temp_z0_raw,
        t_z1_raw, temp_z1_raw,
        t_hmg_raw, temp_hmg_raw,
        t_nsid_raw, temp_nsid_raw,
        t_log_hd_raw, temp_log_hd_raw,
        t_da_raw, temp_da_raw,
        t_herf_raw, temp_herf_raw,
        t_thr, temp_thr,
        t_y_log, temp_y_log,
        t_region, temp_region,
        t_ord, temp_ord,
        t_past_lens, temp_past_lens,
    ) = train_test_split(
        X_taps_feat, X_vars_feat, abs_taps_raw, snr_raw,
        z0_raw, z1_raw, harmonic_minus_gap_raw,
        nsid_raw, log_harmonic_discrim_raw, discrim_asymmetry_raw, herfindahl_raw,
        X_thr,
        y_log, region_labels, ordinal_targets, past_lens,
        **split_args,
    )

    # ---- temp -> val / test split ----
    temp_strat = make_strat_bins(temp_y_log, n_bins=6)
    split_args2 = dict(test_size=0.50, random_state=42)
    if temp_strat is not None:
        split_args2["stratify"] = temp_strat

    (
        v_taps_feat, te_taps_feat,
        v_vars_feat, te_vars_feat,
        v_abs_raw, te_abs_raw,
        v_snr_raw, te_snr_raw,
        v_z0_raw, te_z0_raw,
        v_z1_raw, te_z1_raw,
        v_hmg_raw, te_hmg_raw,
        v_nsid_raw, te_nsid_raw,
        v_log_hd_raw, te_log_hd_raw,
        v_da_raw, te_da_raw,
        v_herf_raw, te_herf_raw,
        v_thr, te_thr,
        v_y_log, te_y_log,
        v_region, te_region,
        v_ord, te_ord,
        v_past_lens, te_past_lens,
    ) = train_test_split(
        temp_taps_feat, temp_vars_feat, temp_abs_raw, temp_snr_raw,
        temp_z0_raw, temp_z1_raw, temp_hmg_raw,
        temp_nsid_raw, temp_log_hd_raw, temp_da_raw, temp_herf_raw,
        temp_thr,
        temp_y_log, temp_region, temp_ord, temp_past_lens,
        **split_args2,
    )

    # =========================================================
    # Position-independent scaling
    # =========================================================
    # For each feature channel (taps, vars, abs, snr) we fit:
    #   - A SEPARATE mean/std for position 0 (first token)
    #   - A SHARED mean/std pooled across ALL valid past positions 1+
    #
    # This decouples the scaler from the number of columns,
    # so at inference any mem_len works with the same parameters.
    # =========================================================

    L_past = L_full - 1

    # Build per-split past validity masks
    t_valid_past = (np.arange(L_past)[None, :] < t_past_lens[:, None])

    # --- Fit first-token scalers (on train only) ---
    first_tap_mean, first_tap_std = float(t_taps_feat[:, 0].mean()), max(float(t_taps_feat[:, 0].std()), 1e-12)
    first_var_mean, first_var_std = float(t_vars_feat[:, 0].mean()), max(float(t_vars_feat[:, 0].std()), 1e-12)
    first_abs_mean, first_abs_std = float(t_abs_raw[:, 0].mean()), max(float(t_abs_raw[:, 0].std()), 1e-12)
    first_snr_mean, first_snr_std = float(t_snr_raw[:, 0].mean()), max(float(t_snr_raw[:, 0].std()), 1e-12)

    # --- Fit shared past scalers (pool all valid past entries from train) ---
    past_tap_mean, past_tap_std = fit_shared_scalar_scaler(t_taps_feat[:, 1:], t_valid_past)
    past_var_mean, past_var_std = fit_shared_scalar_scaler(t_vars_feat[:, 1:], t_valid_past)
    past_abs_mean, past_abs_std = fit_shared_scalar_scaler(t_abs_raw[:, 1:], t_valid_past)
    past_snr_mean, past_snr_std = fit_shared_scalar_scaler(t_snr_raw[:, 1:], t_valid_past)

    # --- Global feature scalers (standard, shape [N, 1]) ---
    z0_scaler = StandardScaler().fit(t_z0_raw)
    z1_scaler = StandardScaler().fit(t_z1_raw)
    hmg_scaler = StandardScaler().fit(t_hmg_raw)
    nsid_scaler = StandardScaler().fit(t_nsid_raw)
    log_hd_scaler = StandardScaler().fit(t_log_hd_raw)
    da_scaler = StandardScaler().fit(t_da_raw)
    herf_scaler = StandardScaler().fit(t_herf_raw)
    thr_scaler = StandardScaler().fit(t_thr)

    # ---- Helper: build first_token and past_tokens arrays ----
    def build_tokens(taps_feat, vars_feat, abs_raw, snr_raw, p_lens):
        """Returns first_token [N, 4] and past_tokens [N, L_past, 4], both scaled + re-zeroed."""
        N = taps_feat.shape[0]

        # Scale first token
        ft_tap = ((taps_feat[:, 0] - first_tap_mean) / first_tap_std).astype(np.float32)
        ft_var = ((vars_feat[:, 0] - first_var_mean) / first_var_std).astype(np.float32)
        ft_abs = ((abs_raw[:, 0] - first_abs_mean) / first_abs_std).astype(np.float32)
        ft_snr = ((snr_raw[:, 0] - first_snr_mean) / first_snr_std).astype(np.float32)
        first_token = np.stack([ft_tap, ft_var, ft_abs, ft_snr], axis=1).astype(np.float32)

        # Scale past tokens (shared scaler) + re-zero
        pt_tap = apply_shared_scale(taps_feat[:, 1:], past_tap_mean, past_tap_std,
                                    (np.arange(L_past)[None, :] < p_lens[:, None]))
        pt_var = apply_shared_scale(vars_feat[:, 1:], past_var_mean, past_var_std,
                                    (np.arange(L_past)[None, :] < p_lens[:, None]))
        pt_abs = apply_shared_scale(abs_raw[:, 1:], past_abs_mean, past_abs_std,
                                    (np.arange(L_past)[None, :] < p_lens[:, None]))
        pt_snr = apply_shared_scale(snr_raw[:, 1:], past_snr_mean, past_snr_std,
                                    (np.arange(L_past)[None, :] < p_lens[:, None]))

        past_tokens = np.stack([pt_tap, pt_var, pt_abs, pt_snr], axis=2).astype(np.float32)

        return first_token, past_tokens

    def build_global(z0, z1, hmg, nsid, log_hd, da, herf):
        return np.concatenate([
            z0_scaler.transform(z0),
            z1_scaler.transform(z1),
            hmg_scaler.transform(hmg),
            nsid_scaler.transform(nsid),
            log_hd_scaler.transform(log_hd),
            da_scaler.transform(da),
            herf_scaler.transform(herf),
        ], axis=1).astype(np.float32)

    def build_padding_mask(p_lens, L_max):
        return (np.arange(L_max)[None, :] >= p_lens[:, None])  # True = padding

    # ---- Build all splits ----
    t_first, t_past = build_tokens(t_taps_feat, t_vars_feat, t_abs_raw, t_snr_raw, t_past_lens)
    v_first, v_past = build_tokens(v_taps_feat, v_vars_feat, v_abs_raw, v_snr_raw, v_past_lens)
    te_first, te_past = build_tokens(te_taps_feat, te_vars_feat, te_abs_raw, te_snr_raw, te_past_lens)

    t_global = build_global(t_z0_raw, t_z1_raw, t_hmg_raw,
                            t_nsid_raw, t_log_hd_raw, t_da_raw, t_herf_raw)
    v_global = build_global(v_z0_raw, v_z1_raw, v_hmg_raw,
                            v_nsid_raw, v_log_hd_raw, v_da_raw, v_herf_raw)
    te_global = build_global(te_z0_raw, te_z1_raw, te_hmg_raw,
                             te_nsid_raw, te_log_hd_raw, te_da_raw, te_herf_raw)

    t_thr_s = thr_scaler.transform(t_thr).astype(np.float32)
    v_thr_s = thr_scaler.transform(v_thr).astype(np.float32)
    te_thr_s = thr_scaler.transform(te_thr).astype(np.float32)

    L_max_past = L_past
    t_mask = build_padding_mask(t_past_lens, L_max_past)
    v_mask = build_padding_mask(v_past_lens, L_max_past)
    te_mask = build_padding_mask(te_past_lens, L_max_past)

    # ---- TensorDatasets ----
    train_ds = TensorDataset(
        torch.from_numpy(t_first),
        torch.from_numpy(t_past),
        torch.from_numpy(t_global),
        torch.from_numpy(t_thr_s),
        torch.from_numpy(t_y_log.astype(np.float32)),
        torch.from_numpy(t_ord.astype(np.float32)),
        torch.from_numpy(t_region.astype(np.int64)),
        torch.from_numpy(t_mask),
    )
    val_ds = TensorDataset(
        torch.from_numpy(v_first),
        torch.from_numpy(v_past),
        torch.from_numpy(v_global),
        torch.from_numpy(v_thr_s),
        torch.from_numpy(v_y_log.astype(np.float32)),
        torch.from_numpy(v_ord.astype(np.float32)),
        torch.from_numpy(v_region.astype(np.int64)),
        torch.from_numpy(v_mask),
    )
    test_ds = TensorDataset(
        torch.from_numpy(te_first),
        torch.from_numpy(te_past),
        torch.from_numpy(te_global),
        torch.from_numpy(te_thr_s),
        torch.from_numpy(te_y_log.astype(np.float32)),
        torch.from_numpy(te_ord.astype(np.float32)),
        torch.from_numpy(te_region.astype(np.int64)),
        torch.from_numpy(te_mask),
    )

    # ---- Weighted sampler (region + SIR-aware) ----
    region_sample_weights = np.ones_like(t_region, dtype=np.float32)
    region_sample_weights[t_region == 6] = 2.5
    region_sample_weights[t_region == 7] = 3.0
    region_sample_weights[t_region == 8] = 5.0
    region_sample_weights[t_region == 9] = 5.0
    region_sample_weights[t_region == 10] = 5.0
    region_sample_weights[t_region == 11] = 5.0
    region_sample_weights[t_region == 12] = 3.0
    region_sample_weights[t_region == 13] = 2.5

    # SIR-aware weights (threshold-independent scenario difficulty)
    t_signal = t_taps_feat[:, 0]
    t_valid_past_for_sir = (np.arange(L_past)[None, :] < t_past_lens[:, None]).astype(np.float32)
    t_isi = np.sum(t_taps_feat[:, 1:] * t_valid_past_for_sir, axis=1)
    t_sir = t_signal / (t_isi + EPS)

    sir_sample_weights = np.ones_like(t_sir, dtype=np.float32)
    sir_sample_weights[t_sir < 0.26] = 3.0
    sir_sample_weights[(t_sir >= 0.26) & (t_sir < 0.36)] = 2.0

    combined_sample_weights = region_sample_weights * sir_sample_weights

    sampler = WeightedRandomSampler(
        weights=torch.from_numpy(combined_sample_weights),
        num_samples=len(combined_sample_weights),
        replacement=True,
    )

    pin_mem = torch.cuda.is_available()

    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler,
                              shuffle=False, pin_memory=pin_mem, num_workers=num_workers)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            pin_memory=pin_mem, num_workers=num_workers)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                             pin_memory=pin_mem, num_workers=num_workers)

    class_weights = compute_region_class_weights(t_region, NUM_REGION_CLASSES)
    ordinal_pos_weights = compute_ordinal_pos_weights_from_region_labels(
        t_region, num_thresholds=NUM_ORDINAL_THRESHOLDS, max_weight=20.0,
    )

    # ---- Save all scaler parameters (position-independent) ----
    scalers = {
        # First-token scalers (per feature channel)
        "first_tap_mean": first_tap_mean, "first_tap_std": first_tap_std,
        "first_var_mean": first_var_mean, "first_var_std": first_var_std,
        "first_abs_mean": first_abs_mean, "first_abs_std": first_abs_std,
        "first_snr_mean": first_snr_mean, "first_snr_std": first_snr_std,
        # Shared past scalers (single mean/std per feature, position-independent)
        "past_tap_mean": past_tap_mean, "past_tap_std": past_tap_std,
        "past_var_mean": past_var_mean, "past_var_std": past_var_std,
        "past_abs_mean": past_abs_mean, "past_abs_std": past_abs_std,
        "past_snr_mean": past_snr_mean, "past_snr_std": past_snr_std,
        # Global feature scalers (sklearn StandardScaler objects)
        "z0_scaler": z0_scaler,
        "z1_scaler": z1_scaler,
        "harmonic_minus_gap_scaler": hmg_scaler,
        "nsid_scaler": nsid_scaler,
        "log_hd_scaler": log_hd_scaler,
        "da_scaler": da_scaler,
        "herf_scaler": herf_scaler,
        "thr_scaler": thr_scaler,
        # Metadata
        "tap_cols": tap_cols,
        "var_cols": var_cols,
        "first_token_dim": 4,
        "past_token_dim": 4,
        "global_dim": 7,
        "train_max_past_seq_len": L_past,
        "scaling_strategy": "position_independent",
        "uses_positional_encoding": False,
        "permutation_invariance_post_first": True,
        "variable_length_support": True,
        "target_parameterization": "pred_log10_ber = log10(0.5) - softplus(raw_out)",
        "ordinal_thresholds": ORDINAL_THRESHOLDS,
        "num_region_classes": NUM_REGION_CLASSES,
        "class_weights": class_weights.tolist(),
        "ordinal_pos_weights": ordinal_pos_weights.tolist(),
        "data_paths": csv_paths,
        "nrows_per_dataset": nrows_per_dataset,
    }

    aux_info = {
        "class_weights": class_weights,
        "ordinal_pos_weights": ordinal_pos_weights,
        "t_region": t_region,
    }

    return train_loader, val_loader, test_loader, scalers, aux_info, L_past


# =========================
# Inference Helper
# =========================
def prepare_inference_batch(
    taps_raw_2d,
    vars_raw_2d,
    N_array,
    threshold_array,
    mem_len_array,
    scalers,
):
    """Prepare a batch for inference from raw numpy arrays.

    This handles ARBITRARY mem_len — the set can be larger than anything
    seen during training because scalers are position-independent.

    Args:
        taps_raw_2d:     [B, L] raw tap coefficients (padded to L with zeros)
        vars_raw_2d:     [B, L] raw variance values  (padded to L with zeros)
        N_array:         [B] or [B, 1] number of molecules
        threshold_array: [B] or [B, 1] raw threshold values
        mem_len_array:   [B] true memory length per sample
        scalers:         dict from joblib.load(SCALER_SAVE_PATH)

    Returns:
        first_token:      [B, 4] tensor
        past_tokens:      [B, L-1, 4] tensor
        global_feats:     [B, 7] tensor
        threshold_scaled: [B, 1] tensor
        key_padding_mask: [B, L-1] bool tensor (True = padding)
    """
    B, L = taps_raw_2d.shape
    N = N_array.reshape(-1, 1).astype(np.float32)
    thr_raw = threshold_array.reshape(-1, 1).astype(np.float32)
    mem_len = mem_len_array.reshape(-1).astype(np.int64)

    mem_len_clamped = np.minimum(mem_len, L)
    past_lens = np.maximum(mem_len_clamped - 1, 0).astype(np.int64)

    # Feature engineering (same as training)
    taps_feat = (taps_raw_2d * N).astype(np.float32)
    vars_feat = vars_raw_2d.astype(np.float32)
    abs_feat = np.abs(taps_feat).astype(np.float32)
    snr_feat = np.log10((taps_feat ** 2) / (vars_raw_2d + EPS) + EPS).astype(np.float32)

    L_past = L - 1

    # Validity masks
    valid_past = (np.arange(L_past)[None, :] < past_lens[:, None])
    valid_past_float = valid_past.astype(np.float32)

    # Global features (mask-aware)
    first_mean = taps_feat[:, 0:1]
    past_means = taps_feat[:, 1:] * valid_past_float
    first_var = vars_feat[:, 0:1]
    past_vars = vars_feat[:, 1:] * valid_past_float

    mu0 = (0.5 * past_means.sum(axis=1, keepdims=True)).astype(np.float32)
    mu1 = (first_mean + mu0).astype(np.float32)
    var0 = (0.5 * past_vars.sum(axis=1, keepdims=True)).astype(np.float32)
    var1 = (first_var + var0).astype(np.float32)

    std0 = np.sqrt(np.maximum(var0, EPS)).astype(np.float32)
    std1 = np.sqrt(np.maximum(var1, EPS)).astype(np.float32)

    z0 = ((thr_raw - mu0) / (std0 + EPS)).astype(np.float32)
    z1 = ((mu1 - thr_raw) / (std1 + EPS)).astype(np.float32)

    harmonic = (2.0 / (1.0 / (z0 + EPS) + 1.0 / (z1 + EPS))).astype(np.float32)
    abs_diff = np.abs(z0 - z1).astype(np.float32)
    hmg = (harmonic - 0.25 * abs_diff).astype(np.float32)

    # Scenario-level features (threshold-independent)
    signal = first_mean  # [B, 1]
    isi = past_means.sum(axis=1, keepdims=True)  # [B, 1]
    nsid = ((signal - isi) / (signal + isi + EPS)).astype(np.float32)

    gap = signal
    d0_inf = (gap / (std0 + EPS)).astype(np.float32)
    d1_inf = (gap / (std1 + EPS)).astype(np.float32)
    hd = (2.0 * d0_inf * d1_inf / (d0_inf + d1_inf + EPS)).astype(np.float32)
    log_hd = np.log10(hd + EPS).astype(np.float32)
    da = (np.abs(d0_inf - d1_inf) / (d0_inf + d1_inf + EPS)).astype(np.float32)

    past_taps_sum = past_means.sum(axis=1, keepdims=True)
    past_shares = past_means / (past_taps_sum + EPS)
    herf = (past_shares ** 2).sum(axis=1, keepdims=True).astype(np.float32)

    # Scale first token
    ft_tap = ((taps_feat[:, 0] - scalers["first_tap_mean"]) / scalers["first_tap_std"]).astype(np.float32)
    ft_var = ((vars_feat[:, 0] - scalers["first_var_mean"]) / scalers["first_var_std"]).astype(np.float32)
    ft_abs = ((abs_feat[:, 0] - scalers["first_abs_mean"]) / scalers["first_abs_std"]).astype(np.float32)
    ft_snr = ((snr_feat[:, 0] - scalers["first_snr_mean"]) / scalers["first_snr_std"]).astype(np.float32)
    first_token = np.stack([ft_tap, ft_var, ft_abs, ft_snr], axis=1).astype(np.float32)

    # Scale past tokens (position-independent shared scaler)
    pt_tap = apply_shared_scale(taps_feat[:, 1:], scalers["past_tap_mean"], scalers["past_tap_std"], valid_past)
    pt_var = apply_shared_scale(vars_feat[:, 1:], scalers["past_var_mean"], scalers["past_var_std"], valid_past)
    pt_abs = apply_shared_scale(abs_feat[:, 1:], scalers["past_abs_mean"], scalers["past_abs_std"], valid_past)
    pt_snr = apply_shared_scale(snr_feat[:, 1:], scalers["past_snr_mean"], scalers["past_snr_std"], valid_past)
    past_tokens = np.stack([pt_tap, pt_var, pt_abs, pt_snr], axis=2).astype(np.float32)

    # Scale globals (7 features)
    z0_s = scalers["z0_scaler"].transform(z0).astype(np.float32)
    z1_s = scalers["z1_scaler"].transform(z1).astype(np.float32)
    hmg_s = scalers["harmonic_minus_gap_scaler"].transform(hmg).astype(np.float32)
    nsid_s = scalers["nsid_scaler"].transform(nsid).astype(np.float32)
    log_hd_s = scalers["log_hd_scaler"].transform(log_hd).astype(np.float32)
    da_s = scalers["da_scaler"].transform(da).astype(np.float32)
    herf_s = scalers["herf_scaler"].transform(herf).astype(np.float32)
    global_feats = np.concatenate([z0_s, z1_s, hmg_s, nsid_s, log_hd_s, da_s, herf_s], axis=1).astype(np.float32)

    # Scale threshold
    thr_log = np.log10(thr_raw + EPS).astype(np.float32)
    thr_s = scalers["thr_scaler"].transform(thr_log).astype(np.float32)

    # Padding mask
    pad_mask = (np.arange(L_past)[None, :] >= past_lens[:, None])

    return (
        torch.from_numpy(first_token),
        torch.from_numpy(past_tokens),
        torch.from_numpy(global_feats),
        torch.from_numpy(thr_s),
        torch.from_numpy(pad_mask),
    )


def run_inference(model, taps_raw, vars_raw, N_arr, thr_arr, mem_len_arr, scalers, device):
    """End-to-end inference: raw arrays -> BER predictions.

    All inputs are numpy. Works with any mem_len, even values
    larger than max_past_seq_len seen during training.
    """
    first_tok, past_tok, glob, thr_s, pad_mask = prepare_inference_batch(
        taps_raw, vars_raw, N_arr, thr_arr, mem_len_arr, scalers,
    )

    model.eval()
    with torch.no_grad():
        first_tok = first_tok.to(device)
        past_tok = past_tok.to(device)
        glob = glob.to(device)
        thr_s = thr_s.to(device)
        pad_mask = pad_mask.to(device)

        pred_raw, ord_logits = model(first_tok, past_tok, glob, thr_s, key_padding_mask=pad_mask)

        pred_ber = model.raw_to_ber(pred_raw).cpu().numpy()
        pred_log = model.raw_to_log10ber(pred_raw).cpu().numpy()
        pred_region = ordinal_logits_to_region_labels_torch(ord_logits).cpu().numpy()

    return {
        "ber": pred_ber,
        "log10_ber": pred_log,
        "region": pred_region,
    }


# =========================
# Evaluation
# =========================
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_reg_loss = 0.0
    total_ord_loss = 0.0

    all_preds_log = []
    all_targets_log = []
    all_pred_regions = []
    all_true_regions = []

    with torch.no_grad():
        for b_first, b_past, b_global, b_thr, b_y_log, b_ord, b_region, b_mask in loader:
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_y_log = b_y_log.to(device, non_blocking=True)
            b_ord = b_ord.to(device, non_blocking=True)
            b_region = b_region.to(device, non_blocking=True)
            b_mask = b_mask.to(device, non_blocking=True)

            pred_raw_unconstrained, ord_logits = model(
                b_first, b_past, b_global, b_thr, key_padding_mask=b_mask
            )

            if has_nonfinite_tensor(pred_raw_unconstrained):
                raise RuntimeError("Non-finite prediction during evaluation.")
            if has_nonfinite_tensor(ord_logits):
                raise RuntimeError("Non-finite ordinal logits during evaluation.")

            loss, reg_loss, ord_loss = criterion(
                pred_raw_unconstrained, ord_logits, b_y_log, b_ord,
            )

            if has_nonfinite_tensor(loss):
                raise RuntimeError("Non-finite loss during evaluation.")

            pred_log = model.raw_to_log10ber(pred_raw_unconstrained)
            pred_region = ordinal_logits_to_region_labels_torch(ord_logits)

            total_loss += loss.item()
            total_reg_loss += reg_loss.item()
            total_ord_loss += ord_loss.item()

            all_preds_log.append(pred_log.cpu().numpy())
            all_targets_log.append(b_y_log.cpu().numpy())
            all_pred_regions.append(pred_region.cpu().numpy())
            all_true_regions.append(b_region.cpu().numpy())

    n_batches = max(len(loader), 1)
    avg_loss = total_loss / n_batches
    avg_reg_loss = total_reg_loss / n_batches
    avg_ord_loss = total_ord_loss / n_batches

    preds_log = np.vstack(all_preds_log)
    targets_log = np.vstack(all_targets_log)

    preds_raw = np.clip(10 ** preds_log, EPS, 0.5)
    targets_raw = np.clip(10 ** targets_log, EPS, 0.5)

    rmse_log = float(np.sqrt(np.mean((preds_log - targets_log) ** 2)))
    mae_log = float(np.mean(np.abs(preds_log - targets_log)))
    factor_error = float(10 ** rmse_log)

    rmse_raw = float(np.sqrt(np.mean((preds_raw - targets_raw) ** 2)))
    mae_raw = float(np.mean(np.abs(preds_raw - targets_raw)))

    rel_err = (preds_raw - targets_raw) / np.maximum(targets_raw, 1e-6)
    rmse_rel = float(np.sqrt(np.mean(rel_err ** 2)))
    mae_rel = float(np.mean(np.abs(rel_err)))

    pred_regions = np.concatenate(all_pred_regions).reshape(-1)
    true_regions = np.concatenate(all_true_regions).reshape(-1)

    region_acc = float(np.mean(pred_regions == true_regions))
    region_mae = float(np.mean(np.abs(pred_regions - true_regions)))

    return {
        "loss": avg_loss,
        "reg_loss": avg_reg_loss,
        "ord_loss": avg_ord_loss,
        "rmse_log": rmse_log,
        "mae_log": mae_log,
        "factor_error": factor_error,
        "rmse_raw": rmse_raw,
        "mae_raw": mae_raw,
        "rmse_rel": rmse_rel,
        "mae_rel": mae_rel,
        "region_acc": region_acc,
        "region_mae": region_mae,
    }


def evaluate_by_target_range(model, loader, device):
    model.eval()
    all_preds_log = []
    all_targets_log = []

    with torch.no_grad():
        for b_first, b_past, b_global, b_thr, b_y_log, _, _, b_mask in loader:
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_mask = b_mask.to(device, non_blocking=True)

            pred_raw_unconstrained, _ = model(
                b_first, b_past, b_global, b_thr, key_padding_mask=b_mask
            )

            if has_nonfinite_tensor(pred_raw_unconstrained):
                raise RuntimeError("Non-finite prediction during per-range evaluation.")

            pred_log = model.raw_to_log10ber(pred_raw_unconstrained)

            all_preds_log.append(pred_log.cpu().numpy())
            all_targets_log.append(b_y_log.numpy())

    preds_log = np.vstack(all_preds_log).reshape(-1)
    targets_log = np.vstack(all_targets_log).reshape(-1)

    preds_raw = np.clip(10 ** preds_log, EPS, 0.5)
    targets_raw = np.clip(10 ** targets_log, EPS, 0.5)

    ranges = {
        "low_BER(y<1e-6)": targets_raw < 1e-6,
        "mid_BER(1e-6<=y<1e-3)": (targets_raw >= 1e-6) & (targets_raw < 1e-3),
        "high_BER(1e-3<=y<0.10)": (targets_raw >= 1e-3) & (targets_raw < 0.10),
        "upper_BER(0.10<=y<0.20)": (targets_raw >= 0.10) & (targets_raw < 0.20),
        "target_BER(0.20<=y<0.40)": (targets_raw >= 0.20) & (targets_raw < 0.40),
        "very_high_BER(0.40<=y<=0.50)": targets_raw >= 0.40,
    }

    metrics = {}
    for name, mask in ranges.items():
        if np.any(mask):
            err_log = preds_log[mask] - targets_log[mask]
            err_raw = preds_raw[mask] - targets_raw[mask]
            rel_err = err_raw / np.maximum(targets_raw[mask], 1e-6)

            rmse_log = float(np.sqrt(np.mean(err_log ** 2)))
            mae_log = float(np.mean(np.abs(err_log)))
            rmse_raw = float(np.sqrt(np.mean(err_raw ** 2)))
            mae_raw = float(np.mean(np.abs(err_raw)))
            rmse_rel = float(np.sqrt(np.mean(rel_err ** 2)))
            mae_rel = float(np.mean(np.abs(rel_err)))

            metrics[name] = {
                "count": int(mask.sum()),
                "rmse_log": rmse_log,
                "mae_log": mae_log,
                "factor_error": float(10 ** rmse_log),
                "rmse_raw": rmse_raw,
                "mae_raw": mae_raw,
                "rmse_rel": rmse_rel,
                "mae_rel": mae_rel,
                "bias_raw": float(np.mean(err_raw)),
                "bias_log": float(np.mean(err_log)),
                "p90_abs_raw": float(np.percentile(np.abs(err_raw), 90)),
                "p95_abs_raw": float(np.percentile(np.abs(err_raw), 95)),
            }
        else:
            metrics[name] = None

    return metrics


# =========================
# Multi-Component Selection Scoring
# =========================
# Selection rationale:
#   BER spans 6+ orders of magnitude (1e-6 to 0.5), so multiplicative error
#   is the natural metric. We use weighted log-RMSE per region (already in log
#   space, so equivalent to weighted geometric mean of factor errors), then
#   add a bias penalty (also in log space, so multiplicative bias) and a tail
#   penalty (P95/RMSE ratio in raw space, capped, to penalize heavy tails).

# Weights for each BER region in the composite score.
# Higher weight = more important to get this region right.
SELECTION_REGION_WEIGHTS = {
    "low_BER(y<1e-6)": 0.25,
    "mid_BER(1e-6<=y<1e-3)": 0.50,
    "high_BER(1e-3<=y<0.10)": 1.00,
    "upper_BER(0.10<=y<0.20)": 2.00,
    "target_BER(0.20<=y<0.40)": 3.00,
    "very_high_BER(0.40<=y<=0.50)": 1.50,
}

# Composite weights: how much each component contributes
SELECTION_W_LOG_RMSE = 1.00   # Primary: per-region log-RMSE
SELECTION_W_BIAS = 0.50       # Secondary: bias in log space (factor bias)
SELECTION_W_TAIL = 0.05       # Tertiary: P95/RMSE ratio (heavy tails)
SELECTION_TAIL_CAP = 5.0      # Cap tail ratio so one bad region doesn't dominate


def compute_selection_score(val_range_metrics, val_metrics, min_count=50):
    """Compute a multi-component selection score, lower is better.

    Components:
      1. Weighted average of per-region RMSE(log10).
         (This is equivalent to a weighted geometric mean of factor errors.)
      2. Weighted average of |bias_log| per region (multiplicative bias).
      3. Weighted average of capped P95/RMSE ratio per region (tail penalty).

    Returns a dict with all components and the composite.
    """
    log_rmse_sum = 0.0
    bias_sum = 0.0
    tail_sum = 0.0
    total_w = 0.0

    per_region_factor = {}

    for region_name, w in SELECTION_REGION_WEIGHTS.items():
        m = val_range_metrics.get(region_name)
        if m is None or m["count"] < min_count:
            continue

        # 1. Log-RMSE component (already log-space)
        log_rmse_sum += w * m["rmse_log"]

        # 2. Bias component in log space (multiplicative bias)
        bias_sum += w * abs(m["bias_log"])

        # 3. Tail component: P95 / RMSE in raw space, capped
        if m["rmse_raw"] > 1e-9:
            tail_ratio = m["p95_abs_raw"] / (m["rmse_raw"] + 1e-9)
            tail_sum += w * min(tail_ratio, SELECTION_TAIL_CAP)
        else:
            tail_sum += w * 1.0

        total_w += w
        per_region_factor[region_name] = m["factor_error"]

    if total_w == 0:
        # Fallback to global metrics
        return {
            "composite": float(val_metrics["rmse_log"]),
            "log_rmse_weighted": float(val_metrics["rmse_log"]),
            "bias_weighted": 0.0,
            "tail_weighted": 0.0,
            "geometric_factor_error": float(val_metrics["factor_error"]),
            "fallback": True,
        }

    log_rmse_weighted = log_rmse_sum / total_w
    bias_weighted = bias_sum / total_w
    tail_weighted = tail_sum / total_w

    composite = (
        SELECTION_W_LOG_RMSE * log_rmse_weighted
        + SELECTION_W_BIAS * bias_weighted
        + SELECTION_W_TAIL * tail_weighted
    )

    # The geometric-mean factor error: 10^(weighted log_rmse)
    # Reported for human readability — "this model is off by ~Nx on average"
    geometric_factor = float(10 ** log_rmse_weighted)

    return {
        "composite": float(composite),
        "log_rmse_weighted": float(log_rmse_weighted),
        "bias_weighted": float(bias_weighted),
        "tail_weighted": float(tail_weighted),
        "geometric_factor_error": geometric_factor,
        "fallback": False,
    }


def is_acceptable_checkpoint(val_range_metrics, val_metrics):
    """Hard constraints: model must meet these to be considered for 'best'.

    These prevent pathological epochs from being selected just because
    they happen to score well on the composite (e.g., if a region has
    almost no samples and dominates by luck).
    """
    target = val_range_metrics.get("target_BER(0.20<=y<0.40)")
    if target is None or target["count"] < 100:
        return False, "target region too small"

    # No region's bias_log can exceed 0.15 (i.e., factor bias of ~1.4x)
    for region_name, m in val_range_metrics.items():
        if m is None or m["count"] < 50:
            continue
        if abs(m["bias_log"]) > 0.15:
            return False, f"{region_name} has bias_log={m['bias_log']:.3f}"

    # Overall log RMSE must be reasonable
    if val_metrics["rmse_log"] > 0.30:
        return False, f"overall rmse_log={val_metrics['rmse_log']:.3f} too high"

    return True, "ok"


# =========================
# Training
# =========================
def train_engine():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Executing on: {device}")

    train_loader, val_loader, test_loader, scalers, aux_info, max_past_seq_len = prepare_data(
        DATA_PATHS,
        batch_size=256,
        nrows_per_dataset=NROWS_PER_DATASET,
        num_workers=0,
    )

    joblib.dump(scalers, SCALER_SAVE_PATH)
    print(f"Scalers saved to: {SCALER_SAVE_PATH}")

    model = BERFirstTokenSetTransformerBoundedLogHalfMultiTask(
        max_past_seq_len=max_past_seq_len,
        token_dim=4,
        first_token_dim=4,
        threshold_dim=1,
        global_dim=7,
        d_model=128,
        num_set_layers=4,
        num_heads=4,
        mlp_ratio=4.0,
        dropout=0.10,
        num_ordinal_thresholds=NUM_ORDINAL_THRESHOLDS,
    ).to(device)

    print("Training target-focused first-token + set-transformer multitask model...")
    print(f"  Variable-length support: ENABLED (position-independent scaling)")
    print(f"  Max past sequence length (training): {max_past_seq_len}")
    print(f"  Scaling strategy: separate first-token / shared past-token scalers")

    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

    reg_loss = StableMultiObjectiveBERBoundedLogLoss(
        log_delta=0.35,
        raw_delta=0.006,
        rel_delta=0.03,
        alpha_log=0.45,
        beta_raw=0.35,
        gamma_rel=0.20,
        use_regime_weights=True,
    )

    boosted_pos = aux_info["ordinal_pos_weights"].copy()
    for i, thr in enumerate(ORDINAL_THRESHOLDS):
        if 0.20 <= thr <= 0.40:
            boosted_pos[i] *= 2.5
        elif 0.15 <= thr < 0.20:
            boosted_pos[i] *= 1.5
        elif 0.40 < thr <= 0.45:
            boosted_pos[i] *= 1.5

    ord_loss = OrdinalBCELoss(
        pos_weight=torch.tensor(boosted_pos, dtype=torch.float32, device=device),
        reduction="mean",
    )

    criterion = MultiTaskBERLoss(
        reg_loss=reg_loss,
        ord_loss=ord_loss,
        lambda_ord=0.25,
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=4, factor=0.5,
    )

    # ---- Multi-component selection state ----
    # Track multiple "best" checkpoints under different criteria.
    # We'll save all of them and the user can choose the right tradeoff at inference.
    best_checkpoints = {
        "composite": {"score": float("inf"), "state": None, "epoch": -1},   # primary: composite score
        "composite_ema": {"score": float("inf"), "state": None, "epoch": -1},  # EMA-smoothed composite
        "log_rmse": {"score": float("inf"), "state": None, "epoch": -1},    # weighted log-RMSE only
        "target_mae": {"score": float("inf"), "state": None, "epoch": -1},  # target region MAE
        "low_bias": {"score": float("inf"), "state": None, "epoch": -1},    # min weighted |bias_log|
    }

    # EMA smoothing of the composite score to reduce epoch-to-epoch noise
    EMA_ALPHA = 0.5
    ema_composite = None

    patience = 12
    wait = 0
    min_epochs_before_early_stop = 60
    max_epochs = 120

    print(f"Ordinal thresholds: {ORDINAL_THRESHOLDS}")
    print(f"Boosted ordinal pos weights: {boosted_pos}")
    print(f"Selection: weighted log-RMSE (geometric factor mean) + bias + tail penalties")
    print(f"  Region weights: {SELECTION_REGION_WEIGHTS}")
    print(f"  Composite weights: log_rmse={SELECTION_W_LOG_RMSE}, "
          f"bias={SELECTION_W_BIAS}, tail={SELECTION_W_TAIL}")
    print(f"  EMA alpha for composite smoothing: {EMA_ALPHA}")

    training_broke = False

    for epoch in range(max_epochs):
        model.train()

        for batch_idx, (b_first, b_past, b_global, b_thr, b_y_log, b_ord, _, b_mask) in enumerate(train_loader):
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_y_log = b_y_log.to(device, non_blocking=True)
            b_ord = b_ord.to(device, non_blocking=True)
            b_mask = b_mask.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            pred_raw_unconstrained, ord_logits = model(
                b_first, b_past, b_global, b_thr, key_padding_mask=b_mask,
            )

            if has_nonfinite_tensor(pred_raw_unconstrained):
                print(f"Non-finite prediction at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True
                break

            if has_nonfinite_tensor(ord_logits):
                print(f"Non-finite ordinal logits at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True
                break

            loss, reg_part, ord_part = criterion(
                pred_raw_unconstrained, ord_logits, b_y_log, b_ord,
            )

            if has_nonfinite_tensor(loss):
                print(f"Non-finite loss at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True
                break

            loss.backward()

            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            if not torch.isfinite(grad_norm):
                print(f"Non-finite gradient norm at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True
                break

            optimizer.step()

            bad_param = False
            for name, param in model.named_parameters():
                if param.requires_grad and param.data is not None and not torch.isfinite(param.data).all():
                    print(f"Non-finite parameter after optimizer step: {name}")
                    bad_param = True
                    break

            if bad_param:
                training_broke = True
                break

        if training_broke:
            print("Training stopped due to non-finite values.")
            break

        try:
            train_metrics = evaluate(model, train_loader, criterion, device)
            val_metrics = evaluate(model, val_loader, criterion, device)
            val_range_metrics = evaluate_by_target_range(model, val_loader, device)
        except RuntimeError as e:
            print(f"Evaluation failed at epoch {epoch+1}: {e}")
            break

        scheduler.step(val_metrics["loss"])
        current_lr = optimizer.param_groups[0]["lr"]

        # ---- Compute multi-component selection score ----
        sel = compute_selection_score(val_range_metrics, val_metrics)
        composite = sel["composite"]

        # EMA-smoothed composite (reduces noise from single bad/lucky epochs)
        if ema_composite is None:
            ema_composite = composite
        else:
            ema_composite = EMA_ALPHA * composite + (1.0 - EMA_ALPHA) * ema_composite

        # Components for individual best-checkpoints
        target_key = "target_BER(0.20<=y<0.40)"
        target_mae = (
            val_range_metrics[target_key]["mae_raw"]
            if val_range_metrics[target_key] is not None
            else float("inf")
        )

        # Acceptability gate
        is_acceptable, reason = is_acceptable_checkpoint(val_range_metrics, val_metrics)

        # ---- Logging ----
        print(
            f"Epoch {epoch+1:03d} | "
            f"LR: {current_lr:.2e} | "
            f"Train Loss: {train_metrics['loss']:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} | "
            f"Val RMSE(log10): {val_metrics['rmse_log']:.4f} "
            f"(~{val_metrics['factor_error']:.3f}x) | "
            f"Val Region Acc: {val_metrics['region_acc']:.4f}"
        )
        print(
            f"  SELECTION | composite={composite:.5f} | "
            f"ema_composite={ema_composite:.5f} | "
            f"weighted log-RMSE={sel['log_rmse_weighted']:.5f} "
            f"(~{sel['geometric_factor_error']:.3f}x avg factor) | "
            f"weighted |bias_log|={sel['bias_weighted']:.5f} | "
            f"tail penalty={sel['tail_weighted']:.3f} | "
            f"acceptable={is_acceptable}"
            + ("" if is_acceptable else f" ({reason})")
        )

        # Per-range tail metrics
        for rng_name, rng_stats in val_range_metrics.items():
            if rng_stats is not None:
                print(
                    f"    {rng_name}: "
                    f"n={rng_stats['count']} | "
                    f"factor~{rng_stats['factor_error']:.3f}x | "
                    f"RMSE_log={rng_stats['rmse_log']:.4f} | "
                    f"MAE_raw={rng_stats['mae_raw']:.6f} | "
                    f"P90={rng_stats['p90_abs_raw']:.6f} | "
                    f"P95={rng_stats['p95_abs_raw']:.6f} | "
                    f"bias_raw={rng_stats['bias_raw']:+.6f} | "
                    f"bias_log={rng_stats['bias_log']:+.4f}"
                )

        # ---- Update each best-checkpoint (only if acceptable) ----
        improved_any = False

        if is_acceptable:
            current_state_snapshot = None  # lazy deepcopy

            checkpoint_candidates = [
                ("composite", composite),
                ("composite_ema", ema_composite),
                ("log_rmse", sel["log_rmse_weighted"]),
                ("target_mae", target_mae),
                ("low_bias", sel["bias_weighted"]),
            ]

            for ckpt_name, score in checkpoint_candidates:
                if score < best_checkpoints[ckpt_name]["score"]:
                    if current_state_snapshot is None:
                        current_state_snapshot = copy.deepcopy(model.state_dict())
                    best_checkpoints[ckpt_name] = {
                        "score": float(score),
                        "state": current_state_snapshot,
                        "epoch": epoch + 1,
                    }
                    improved_any = True
                    print(f"  -> New best [{ckpt_name}] at epoch {epoch+1}: {score:.6f}")

        # ---- Early stopping driven by EMA composite ----
        # We use the EMA score so we don't stop on a single noisy spike
        ema_best = best_checkpoints["composite_ema"]["score"]
        if ema_composite < ema_best + 1e-9 or improved_any:
            wait = 0
        else:
            if epoch + 1 >= min_epochs_before_early_stop:
                wait += 1
                if wait >= patience:
                    print(f"Early stopping triggered after {wait} non-improving epochs (EMA basis).")
                    break

    # ---- Save last model ----
    torch.save(model.state_dict(), LAST_MODEL_SAVE_PATH)
    print(f"\nLast model saved to: {LAST_MODEL_SAVE_PATH}")

    # ---- Save all best checkpoints ----
    print("\n" + "=" * 80)
    print("BEST CHECKPOINTS SUMMARY")
    print("=" * 80)
    for ckpt_name, info in best_checkpoints.items():
        if info["state"] is None:
            print(f"  [{ckpt_name:>14s}] never updated")
            continue

        path = BEST_MODEL_SAVE_PATH.replace(".pth", f"_{ckpt_name}.pth")
        torch.save(info["state"], path)
        print(
            f"  [{ckpt_name:>14s}] epoch={info['epoch']:>3d} | "
            f"score={info['score']:.6f} | saved -> {path}"
        )

    # ---- Choose primary best for evaluation: composite_ema is most stable ----
    primary_choice = "composite_ema"
    if best_checkpoints[primary_choice]["state"] is None:
        # Fall back to composite if EMA never updated (shouldn't happen but safe)
        primary_choice = "composite"
    if best_checkpoints[primary_choice]["state"] is None:
        # Fall back to log_rmse
        primary_choice = "log_rmse"

    if best_checkpoints[primary_choice]["state"] is not None:
        model.load_state_dict(best_checkpoints[primary_choice]["state"])
        torch.save(model.state_dict(), BEST_MODEL_SAVE_PATH)
        print(
            f"\nPrimary best ({primary_choice}, epoch "
            f"{best_checkpoints[primary_choice]['epoch']}) "
            f"saved to: {BEST_MODEL_SAVE_PATH}"
        )
    else:
        print("\nWarning: no valid best checkpoint was found.")

    test_metrics = evaluate(model, test_loader, criterion, device)

    print(
        f"Test Loss: {test_metrics['loss']:.4f} | "
        f"Test Reg Loss: {test_metrics['reg_loss']:.4f} | "
        f"Test Ord Loss: {test_metrics['ord_loss']:.4f} | "
        f"Test RMSE(log10): {test_metrics['rmse_log']:.4f} | "
        f"Test MAE(log10): {test_metrics['mae_log']:.4f} | "
        f"Typical multiplicative error: ~{test_metrics['factor_error']:.2f}x | "
        f"Test RMSE(raw): {test_metrics['rmse_raw']:.6f} | "
        f"Test MAE(raw): {test_metrics['mae_raw']:.6f} | "
        f"Test RMSE(rel): {test_metrics['rmse_rel']:.6f} | "
        f"Test MAE(rel): {test_metrics['mae_rel']:.6f} | "
        f"Test Region Acc: {test_metrics['region_acc']:.4f} | "
        f"Test Region MAE: {test_metrics['region_mae']:.4f}"
    )

    range_metrics = evaluate_by_target_range(model, test_loader, device)
    print("\nPer-range test diagnostics:")
    for name, stats in range_metrics.items():
        if stats is None:
            print(f"  {name}: no samples")
        else:
            print(
                f"  {name} | count={stats['count']} | "
                f"RMSE(log10)={stats['rmse_log']:.4f} | "
                f"MAE(log10)={stats['mae_log']:.4f} | "
                f"factor~{stats['factor_error']:.2f}x | "
                f"RMSE(raw)={stats['rmse_raw']:.6f} | "
                f"MAE(raw)={stats['mae_raw']:.6f} | "
                f"RMSE(rel)={stats['rmse_rel']:.6f} | "
                f"MAE(rel)={stats['mae_rel']:.6f} | "
                f"bias_raw={stats['bias_raw']:.6f} | "
                f"bias_log={stats['bias_log']:.6f} | "
                f"P90={stats['p90_abs_raw']:.6f} | "
                f"P95={stats['p95_abs_raw']:.6f}"
            )

    return model


if __name__ == "__main__":
    os.makedirs(BASE_DIR, exist_ok=True)

    missing = [p for p in DATA_PATHS if not os.path.exists(p)]
    if missing:
        print("Critical Error: Missing dataset files:")
        for p in missing:
            print(f"  - {p}")
    else:
        print("Using training from datasets:")
        for p in DATA_PATHS:
            print(f"  - {p}")
        print(f"Row cap per dataset: {NROWS_PER_DATASET:,}")

        trained_model = train_engine()

Using training from datasets:
  - ./data_random_with_random_variances_total_v2.csv
Row cap per dataset: 5,000,000
Executing on: cuda
Loading up to 5,000,000 rows from: ./data_random_with_random_variances_total_v2.csv
Combined rows before filtering: 5,000,000
Rows after cleaning/filtering: 4,992,301
Scalers saved to: ./random_extra_multitask_scalers.pkl
Training target-focused first-token + set-transformer multitask model...
  Variable-length support: ENABLED (position-independent scaling)
  Max past sequence length (training): 13
  Scaling strategy: separate first-token / shared past-token scalers
Ordinal thresholds: [1e-06, 1e-05, 0.0001, 0.001, 0.01, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45]
Boosted ordinal pos weights: [1.        1.        1.        1.        1.        1.        1.5
 2.5       2.5       2.5       3.451725  5.8354483 6.928302 ]
Selection: weighted log-RMSE (geometric factor mean) + bias + tail penalties
  Region weights: {'low_BER(y<1e-6)': 0.25, 'mid_BER(1e-6<=y<1

Epoch 007 | LR: 1.00e-04 | Train Loss: 0.0077 | Val Loss: 0.0101 | Val RMSE(log10): 0.1665 (~1.467x) | Val Region Acc: 0.9031
  SELECTION | composite=0.29331 | ema_composite=0.34170 | weighted log-RMSE=0.18195 (~1.520x avg factor) | weighted |bias_log|=0.02764 | tail penalty=1.951 | acceptable=False (low_BER(y<1e-6) has bias_log=0.629)
    low_BER(y<1e-6): n=4670 | factor~56.195x | RMSE_log=1.7497 | MAE_raw=0.000036 | P90=0.000000 | P95=0.000004 | bias_raw=+0.000036 | bias_log=+0.6291
    mid_BER(1e-6<=y<1e-3): n=1705 | factor~26.346x | RMSE_log=1.4207 | MAE_raw=0.002382 | P90=0.003921 | P95=0.011682 | bias_raw=+0.002248 | bias_log=-0.0050
    high_BER(1e-3<=y<0.10): n=56020 | factor~1.660x | RMSE_log=0.2200 | MAE_raw=0.009312 | P90=0.019517 | P95=0.028264 | bias_raw=+0.007392 | bias_log=+0.0462
    upper_BER(0.10<=y<0.20): n=145592 | factor~1.097x | RMSE_log=0.0402 | MAE_raw=0.007627 | P90=0.017988 | P95=0.026988 | bias_raw=+0.002649 | bias_log=+0.0072
    target_BER(0.20<=y<0.40): n=

Epoch 014 | LR: 1.00e-04 | Train Loss: 0.0063 | Val Loss: 0.0106 | Val RMSE(log10): 0.1542 (~1.426x) | Val Region Acc: 0.9306
  SELECTION | composite=0.27094 | ema_composite=0.31425 | weighted log-RMSE=0.14691 (~1.403x avg factor) | weighted |bias_log|=0.07047 | tail penalty=1.776 | acceptable=False (low_BER(y<1e-6) has bias_log=1.189)
    low_BER(y<1e-6): n=4670 | factor~54.297x | RMSE_log=1.7348 | MAE_raw=0.000106 | P90=0.000000 | P95=0.000000 | bias_raw=+0.000106 | bias_log=+1.1891
    mid_BER(1e-6<=y<1e-3): n=1705 | factor~8.878x | RMSE_log=0.9483 | MAE_raw=0.000578 | P90=0.000563 | P95=0.000799 | bias_raw=+0.000335 | bias_log=-0.4831
    high_BER(1e-3<=y<0.10): n=56020 | factor~1.544x | RMSE_log=0.1885 | MAE_raw=0.006866 | P90=0.014161 | P95=0.019592 | bias_raw=+0.004870 | bias_log=+0.0168
    upper_BER(0.10<=y<0.20): n=145592 | factor~1.080x | RMSE_log=0.0336 | MAE_raw=0.005833 | P90=0.013196 | P95=0.019497 | bias_raw=+0.002920 | bias_log=+0.0079
    target_BER(0.20<=y<0.40): n=3

Epoch 021 | LR: 5.00e-05 | Train Loss: 0.0048 | Val Loss: 0.0060 | Val RMSE(log10): 0.1170 (~1.309x) | Val Region Acc: 0.9400
  SELECTION | composite=0.25643 | ema_composite=0.25539 | weighted log-RMSE=0.14600 (~1.400x avg factor) | weighted |bias_log|=0.05284 | tail penalty=1.680 | acceptable=False (mid_BER(1e-6<=y<1e-3) has bias_log=-0.707)
    low_BER(y<1e-6): n=4670 | factor~11.508x | RMSE_log=1.0610 | MAE_raw=0.000014 | P90=0.000000 | P95=0.000000 | bias_raw=+0.000014 | bias_log=+0.1442
    mid_BER(1e-6<=y<1e-3): n=1705 | factor~16.661x | RMSE_log=1.2217 | MAE_raw=0.000187 | P90=0.000469 | P95=0.000633 | bias_raw=-0.000079 | bias_log=-0.7066
    high_BER(1e-3<=y<0.10): n=56020 | factor~1.570x | RMSE_log=0.1960 | MAE_raw=0.004394 | P90=0.009265 | P95=0.013710 | bias_raw=-0.000056 | bias_log=-0.0201
    upper_BER(0.10<=y<0.20): n=145592 | factor~1.095x | RMSE_log=0.0394 | MAE_raw=0.004723 | P90=0.010376 | P95=0.016137 | bias_raw=+0.001169 | bias_log=+0.0024
    target_BER(0.20<=y<0.

Epoch 028 | LR: 5.00e-05 | Train Loss: 0.0042 | Val Loss: 0.0067 | Val RMSE(log10): 0.0817 (~1.207x) | Val Region Acc: 0.9526
  SELECTION | composite=0.19013 | ema_composite=0.20402 | weighted log-RMSE=0.09317 (~1.239x avg factor) | weighted |bias_log|=0.02625 | tail penalty=1.677 | acceptable=False (low_BER(y<1e-6) has bias_log=0.159)
    low_BER(y<1e-6): n=4670 | factor~6.998x | RMSE_log=0.8450 | MAE_raw=0.000067 | P90=0.000000 | P95=0.000000 | bias_raw=+0.000067 | bias_log=+0.1593
    mid_BER(1e-6<=y<1e-3): n=1705 | factor~5.269x | RMSE_log=0.7217 | MAE_raw=0.000205 | P90=0.000417 | P95=0.000606 | bias_raw=+0.000039 | bias_log=-0.2959
    high_BER(1e-3<=y<0.10): n=56020 | factor~1.283x | RMSE_log=0.1081 | MAE_raw=0.003973 | P90=0.008513 | P95=0.012676 | bias_raw=+0.000599 | bias_log=-0.0057
    upper_BER(0.10<=y<0.20): n=145592 | factor~1.050x | RMSE_log=0.0211 | MAE_raw=0.004350 | P90=0.009607 | P95=0.014132 | bias_raw=-0.000433 | bias_log=-0.0017
    target_BER(0.20<=y<0.40): n=31

...

Epoch 077 | LR: 1.95e-07 | Train Loss: 0.0038 | Val Loss: 0.0042 | Val RMSE(log10): 0.0408 (~1.098x) | Val Region Acc: 0.9573
  SELECTION | composite=0.13560 | ema_composite=0.13528 | weighted log-RMSE=0.05206 (~1.127x avg factor) | weighted |bias_log|=0.01453 | tail penalty=1.526 | acceptable=False (mid_BER(1e-6<=y<1e-3) has bias_log=-0.158)
    low_BER(y<1e-6): n=4670 | factor~2.319x | RMSE_log=0.3654 | MAE_raw=0.000000 | P90=0.000000 | P95=0.000000 | bias_raw=+0.000000 | bias_log=-0.0215
    mid_BER(1e-6<=y<1e-3): n=1705 | factor~2.171x | RMSE_log=0.3367 | MAE_raw=0.000080 | P90=0.000198 | P95=0.000279 | bias_raw=-0.000030 | bias_log=-0.1580
    high_BER(1e-3<=y<0.10): n=56020 | factor~1.173x | RMSE_log=0.0695 | MAE_raw=0.003147 | P90=0.006730 | P95=0.010185 | bias_raw=+0.000126 | bias_log=-0.0062
    upper_BER(0.10<=y<0.20): n=145592 | factor~1.060x | RMSE_log=0.0251 | MAE_raw=0.003587 | P90=0.007684 | P95=0.012314 | bias_raw=+0.001103 | bias_log=+0.0027
    target_BER(0.20<=y<0.40

Epoch 084 | LR: 9.77e-08 | Train Loss: 0.0038 | Val Loss: 0.0041 | Val RMSE(log10): 0.0413 (~1.100x) | Val Region Acc: 0.9575
  SELECTION | composite=0.13559 | ema_composite=0.13514 | weighted log-RMSE=0.05240 (~1.128x avg factor) | weighted |bias_log|=0.01397 | tail penalty=1.524 | acceptable=True
    low_BER(y<1e-6): n=4670 | factor~2.377x | RMSE_log=0.3761 | MAE_raw=0.000000 | P90=0.000000 | P95=0.000000 | bias_raw=+0.000000 | bias_log=-0.0223
    mid_BER(1e-6<=y<1e-3): n=1705 | factor~2.185x | RMSE_log=0.3394 | MAE_raw=0.000085 | P90=0.000210 | P95=0.000288 | bias_raw=-0.000026 | bias_log=-0.1482
    high_BER(1e-3<=y<0.10): n=56020 | factor~1.171x | RMSE_log=0.0687 | MAE_raw=0.003150 | P90=0.006719 | P95=0.010232 | bias_raw=+0.000084 | bias_log=-0.0063
    upper_BER(0.10<=y<0.20): n=145592 | factor~1.059x | RMSE_log=0.0250 | MAE_raw=0.003610 | P90=0.007710 | P95=0.012354 | bias_raw=+0.001093 | bias_log=+0.0026
    target_BER(0.20<=y<0.40): n=316217 | factor~1.028x | RMSE_log=0.0119

Epoch 091 | LR: 2.44e-08 | Train Loss: 0.0037 | Val Loss: 0.0042 | Val RMSE(log10): 0.0407 (~1.098x) | Val Region Acc: 0.9575
  SELECTION | composite=0.13329 | ema_composite=0.13336 | weighted log-RMSE=0.05076 (~1.124x avg factor) | weighted |bias_log|=0.01306 | tail penalty=1.520 | acceptable=True
    low_BER(y<1e-6): n=4670 | factor~2.390x | RMSE_log=0.3784 | MAE_raw=0.000000 | P90=0.000000 | P95=0.000000 | bias_raw=+0.000000 | bias_log=-0.0149
    mid_BER(1e-6<=y<1e-3): n=1705 | factor~2.079x | RMSE_log=0.3179 | MAE_raw=0.000082 | P90=0.000196 | P95=0.000272 | bias_raw=-0.000023 | bias_log=-0.1376
    high_BER(1e-3<=y<0.10): n=56020 | factor~1.165x | RMSE_log=0.0661 | MAE_raw=0.003137 | P90=0.006703 | P95=0.010202 | bias_raw=+0.000115 | bias_log=-0.0058
    upper_BER(0.10<=y<0.20): n=145592 | factor~1.058x | RMSE_log=0.0246 | MAE_raw=0.003591 | P90=0.007669 | P95=0.012300 | bias_raw=+0.001128 | bias_log=+0.0027
    target_BER(0.20<=y<0.40): n=316217 | factor~1.028x | RMSE_log=0.0119

Epoch 098 | LR: 1.22e-08 | Train Loss: 0.0039 | Val Loss: 0.0042 | Val RMSE(log10): 0.0405 (~1.098x) | Val Region Acc: 0.9574
  SELECTION | composite=0.13334 | ema_composite=0.13371 | weighted log-RMSE=0.05079 (~1.124x avg factor) | weighted |bias_log|=0.01312 | tail penalty=1.520 | acceptable=True
    low_BER(y<1e-6): n=4670 | factor~2.356x | RMSE_log=0.3723 | MAE_raw=0.000000 | P90=0.000000 | P95=0.000000 | bias_raw=+0.000000 | bias_log=-0.0171
    mid_BER(1e-6<=y<1e-3): n=1705 | factor~2.088x | RMSE_log=0.3197 | MAE_raw=0.000082 | P90=0.000198 | P95=0.000275 | bias_raw=-0.000023 | bias_log=-0.1372
    high_BER(1e-3<=y<0.10): n=56020 | factor~1.166x | RMSE_log=0.0669 | MAE_raw=0.003140 | P90=0.006713 | P95=0.010215 | bias_raw=+0.000138 | bias_log=-0.0058
    upper_BER(0.10<=y<0.20): n=145592 | factor~1.059x | RMSE_log=0.0247 | MAE_raw=0.003594 | P90=0.007679 | P95=0.012309 | bias_raw=+0.001146 | bias_log=+0.0028
    target_BER(0.20<=y<0.40): n=316217 | factor~1.028x | RMSE_log=0.0119

Test Loss: 0.0053 | Test Reg Loss: 0.0018 | Test Ord Loss: 0.0139 | Test RMSE(log10): 0.0416 | Test MAE(log10): 0.0107 | Typical multiplicative error: ~1.10x | Test RMSE(raw): 0.008303 | Test MAE(raw): 0.005197 | Test RMSE(rel): 138.943954 | Test MAE(rel): 0.240940 | Test Region Acc: 0.9574 | Test Region MAE: 0.0436

Per-range test diagnostics:
  low_BER(y<1e-6) | count=4536 | RMSE(log10)=0.3552 | MAE(log10)=0.1618 | factor~2.27x | RMSE(raw)=0.001687 | MAE(raw)=0.000025 | RMSE(rel)=1687.239868 | MAE(rel)=25.121969 | bias_raw=0.000025 | bias_log=-0.024813 | P90=0.000000 | P95=0.000000
  mid_BER(1e-6<=y<1e-3) | count=1759 | RMSE(log10)=0.4133 | MAE(log10)=0.2182 | factor~2.59x | RMSE(raw)=0.001305 | MAE(raw)=0.000120 | RMSE(rel)=936.833923 | MAE(rel)=28.827175 | bias_raw=0.000016 | bias_log=-0.156731 | P90=0.000194 | P95=0.000289
  high_BER(1e-3<=y<0.10) | count=56172 | RMSE(log10)=0.0661 | MAE(log10)=0.0282 | factor~1.16x | RMSE(raw)=0.005077 | MAE(raw)=0.003078 | RMSE(rel)=0.112235 | M

In [9]:
import os
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.special import erfc
from itertools import product as iproduct

EPS = 1e-12
PLOT_DIR = "./plots_v4_random"
os.makedirs(PLOT_DIR, exist_ok=True)

# =========================
# Paths
# =========================
MODEL_PATH = "random_extra_multitask_best_log_rmse.pth"
SCALER_PATH = "random_extra_multitask_scalers.pkl"

# =========================
# Config
# =========================
PHYSICS_MAX_MEM_LEN = 14
PHYSICS_MIN_MEM_LEN = 14
ARRIVAL_COVERAGE = 0.70
N_THRESHOLDS = 500
RANDOM_SEED = 60

ORDINAL_THRESHOLDS = [
    1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1,
    0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45,
]
NUM_ORDINAL_THRESHOLDS = len(ORDINAL_THRESHOLDS)
NUM_REGION_CLASSES = NUM_ORDINAL_THRESHOLDS + 1

REGION_LABELS = {
    0: "y < 1e-6",
    1: "1e-6 <= y < 1e-5",
    2: "1e-5 <= y < 1e-4",
    3: "1e-4 <= y < 1e-3",
    4: "1e-3 <= y < 1e-2",
    5: "1e-2 <= y < 1e-1",
    6: "0.10 <= y < 0.15",
    7: "0.15 <= y < 0.20",
    8: "0.20 <= y < 0.25",
    9: "0.25 <= y < 0.30",
    10: "0.30 <= y < 0.35",
    11: "0.35 <= y < 0.40",
    12: "0.40 <= y < 0.45",
    13: "0.45 <= y <= 0.50",
}


# =========================
# Physics helpers
# =========================
def Fhit_function(radius, distance, diffusionCoef, t):
    if t <= 0:
        return 0.0
    return (radius / (distance + radius)) * erfc(distance / np.sqrt(4 * diffusionCoef * t))


def calculate_hitting_probabilities(mem_len, radius, distance, diffusionCoef, Ts):
    P = np.zeros(mem_len)
    for i in range(mem_len):
        t_end = (i + 1) * Ts
        t_start = i * Ts
        P[i] = Fhit_function(radius, distance, diffusionCoef, t_end) - Fhit_function(
            radius, distance, diffusionCoef, t_start
        )
    return P


def calculate_ber_vectorized(mem_len, threshold, P_scaled, variances):
    P_arr = np.asarray(P_scaled, dtype=float)[:mem_len]
    vars_arr = np.asarray(variances, dtype=float)[:mem_len]

    seqs = np.array(list(iproduct([0, 1], repeat=mem_len)), dtype=np.float64)[:, ::-1]
    c_bit = seqs[:, 0]

    mu = (seqs * P_arr).sum(axis=1)
    var_total = (seqs * vars_arr).sum(axis=1)
    std = np.sqrt(np.maximum(var_total, 0.0))

    pe = np.empty_like(mu)
    zero_std = (std == 0)
    if np.any(zero_std):
        pe[zero_std & (c_bit == 1)] = np.where(
            mu[zero_std & (c_bit == 1)] < threshold, 1.0, 0.0)
        pe[zero_std & (c_bit == 0)] = np.where(
            mu[zero_std & (c_bit == 0)] >= threshold, 1.0, 0.0)
    nz = ~zero_std
    if np.any(nz):
        pe[nz & (c_bit == 1)] = 0.5 * erfc(
            (mu[nz & (c_bit == 1)] - threshold) / (std[nz & (c_bit == 1)] * np.sqrt(2)))
        pe[nz & (c_bit == 0)] = 0.5 * erfc(
            (threshold - mu[nz & (c_bit == 0)]) / (std[nz & (c_bit == 0)] * np.sqrt(2)))
    return float(np.mean(pe))


def raw_to_region_labels_np(y_raw):
    y = np.asarray(y_raw).reshape(-1)
    return np.searchsorted(np.asarray(ORDINAL_THRESHOLDS), y, side="right").astype(np.int64)


def ordinal_logits_to_region_labels_torch(logits):
    probs = torch.sigmoid(logits)
    passed = (probs >= 0.5).sum(dim=1)
    return passed.long()


# =========================
# Shared scaling helper
# =========================
def apply_shared_scale(data, mean, std, valid_mask=None):
    scaled = ((data - mean) / std).astype(np.float32)
    if valid_mask is not None and scaled.ndim == 2 and valid_mask.shape == scaled.shape:
        scaled = scaled * valid_mask.astype(np.float32)
    return scaled


# =========================
# Generate one physical scenario
# =========================
def generate_physical_case(rng, min_mem_len, max_mem_len, arrival_coverage, max_tries=5000):
    for _ in range(max_tries):
        radius = rng.uniform(3.0, 5.0)
        distance = rng.uniform(10.0, 15.0)
        diff = rng.uniform(50.0, 75.0)
        Ts = rng.uniform(0.5, 1.2)
        N = int(10 ** rng.uniform(3.0, 6.0))

        f_inf = radius / (radius + distance)
        target = arrival_coverage * f_inf

        cumsum, k = 0.0, 0
        while k < max_mem_len:
            pk = Fhit_function(radius, distance, diff, (k + 1) * Ts) - Fhit_function(
                radius, distance, diff, k * Ts)
            cumsum += pk
            k += 1
            if cumsum >= target:
                break

        if k < min_mem_len:
            continue

        P_ext = calculate_hitting_probabilities(k + 1, radius, distance, diff, Ts)
        P_main = P_ext[:k]
        P_extra = float(P_ext[k]) if k < len(P_ext) else 0.0

        P_scaled = P_main * N
        variances = N * P_main * (1.0 - P_main)

        return {
            "radius": radius, "distance": distance, "diffusion": diff,
            "Ts": Ts, "N": N, "mem_len": k, "P": P_main,
            "P_scaled": P_scaled, "variances": variances,
            "P_mem_len_extra": P_extra,
            "P_mem_len_extra_var": P_extra * (1.0 - P_extra),
        }

    raise RuntimeError(
        f"Could not generate a physical case with mem_len >= {min_mem_len} "
        f"after {max_tries} tries."
    )


# =========================
# Model (matches v3 training: global_dim=7, global_embed=96, cond_dim=128)
# =========================
class ConditionalFeatureModulation(nn.Module):
    def __init__(self, cond_dim, feat_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, 2 * feat_dim),
        )
        self.scale = nn.Parameter(torch.tensor(0.25, dtype=torch.float32))

    def forward(self, x, cond):
        gamma_beta = self.net(cond)
        gamma, beta = torch.chunk(gamma_beta, 2, dim=-1)
        if x.dim() == 3:
            gamma = gamma.unsqueeze(1)
            beta = beta.unsqueeze(1)
        return x * (1.0 + self.scale * gamma) + self.scale * beta


class SetSelfAttentionBlock(nn.Module):
    def __init__(self, d_model, num_heads, mlp_ratio=4.0, dropout=0.10):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=num_heads,
            dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        hidden = int(d_model * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, d_model), nn.Dropout(dropout))

    def forward(self, x, key_padding_mask=None):
        y = self.norm1(x)
        attn_out, _ = self.attn(y, y, y, need_weights=False, key_padding_mask=key_padding_mask)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x


class SetAttentionPooling(nn.Module):
    def __init__(self, d_model, hidden_dim=128):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(d_model, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, 1))

    def forward(self, x, key_padding_mask=None):
        logits = self.score(x)
        if key_padding_mask is not None:
            logits = logits.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
        weights = torch.softmax(logits, dim=1)
        weights = torch.nan_to_num(weights, nan=0.0)
        return (weights * x).sum(dim=1)


class BERFirstTokenSetTransformerBoundedLogHalfMultiTask(nn.Module):
    def __init__(self, max_past_seq_len, token_dim=4, first_token_dim=4,
                 threshold_dim=1, global_dim=7, d_model=128, num_set_layers=4,
                 num_heads=4, mlp_ratio=4.0, dropout=0.10,
                 num_ordinal_thresholds=NUM_ORDINAL_THRESHOLDS):
        super().__init__()
        self.max_past_seq_len = max_past_seq_len
        self.d_model = d_model
        self.num_ordinal_thresholds = num_ordinal_thresholds

        self.first_proj = nn.Sequential(
            nn.Linear(first_token_dim, d_model), nn.LayerNorm(d_model),
            nn.GELU(), nn.Dropout(dropout))
        self.token_proj = nn.Sequential(
            nn.Linear(token_dim, d_model), nn.LayerNorm(d_model),
            nn.GELU(), nn.Dropout(dropout))
        self.thr_embed = nn.Sequential(
            nn.Linear(threshold_dim, 32), nn.GELU(),
            nn.Linear(32, 32), nn.GELU())
        self.global_embed = nn.Sequential(
            nn.Linear(global_dim, 96), nn.GELU(),
            nn.Linear(96, 96), nn.GELU())

        cond_dim = 32 + 96
        self.first_cond_mod = ConditionalFeatureModulation(cond_dim, d_model, 256)
        self.set_cond_mod = ConditionalFeatureModulation(cond_dim, d_model, 256)

        self.set_blocks = nn.ModuleList([
            SetSelfAttentionBlock(d_model, num_heads, mlp_ratio, dropout)
            for _ in range(num_set_layers)])

        self.first_refine = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_model),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model, d_model))
        self.final_set_norm = nn.LayerNorm(d_model)
        self.set_attn_pool = SetAttentionPooling(d_model, 128)

        set_summary_dim = 3 * d_model
        head_in = d_model + set_summary_dim + cond_dim

        self.reg_head = nn.Sequential(
            nn.Linear(head_in, 256), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.15),
            nn.Linear(128, 1))
        self.ord_head = nn.Sequential(
            nn.Linear(head_in, 256), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.15),
            nn.Linear(128, num_ordinal_thresholds))

    @staticmethod
    def raw_to_log10ber(pred_raw_unconstrained):
        log10_half = torch.log10(torch.tensor(
            0.5, device=pred_raw_unconstrained.device,
            dtype=pred_raw_unconstrained.dtype))
        return log10_half - F.softplus(pred_raw_unconstrained)

    @staticmethod
    def raw_to_ber(pred_raw_unconstrained):
        pred_log = BERFirstTokenSetTransformerBoundedLogHalfMultiTask.raw_to_log10ber(
            pred_raw_unconstrained)
        return torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)

    def forward(self, first_token, past_tokens, global_feats, threshold, key_padding_mask=None):
        thr = self.thr_embed(threshold)
        g = self.global_embed(global_feats)
        cond = torch.cat([thr, g], dim=-1)

        first_x = self.first_proj(first_token)
        first_x = self.first_cond_mod(first_x, cond)
        first_x = first_x + self.first_refine(first_x)

        set_x = self.token_proj(past_tokens)
        set_x = self.set_cond_mod(set_x, cond)
        for block in self.set_blocks:
            set_x = block(set_x, key_padding_mask=key_padding_mask)
        set_x = self.final_set_norm(set_x)

        if key_padding_mask is not None:
            valid_mask = (~key_padding_mask).unsqueeze(-1).float()
        else:
            valid_mask = torch.ones(
                set_x.shape[0], set_x.shape[1], 1,
                device=set_x.device, dtype=set_x.dtype)

        pooled_attn = self.set_attn_pool(set_x, key_padding_mask=key_padding_mask)
        valid_counts = valid_mask.sum(dim=1).clamp(min=1.0)
        pooled_mean = (set_x * valid_mask).sum(dim=1) / valid_counts

        if key_padding_mask is not None:
            x_for_max = set_x.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
        else:
            x_for_max = set_x
        pooled_max = x_for_max.amax(dim=1)
        pooled_max = torch.nan_to_num(pooled_max, nan=0.0, posinf=0.0, neginf=0.0)

        set_summary = torch.cat([pooled_attn, pooled_mean, pooled_max], dim=-1)
        fused = torch.cat([first_x, set_summary, cond], dim=-1)

        pred_raw_unconstrained = self.reg_head(fused)
        ord_logits = self.ord_head(fused)
        return pred_raw_unconstrained, ord_logits


# =========================
# Build inference inputs (7 global features, position-independent scaling)
# =========================
def prepare_inference_features(case, thresholds, scalers, device):
    """Build model inputs for a threshold sweep over a single physical case.

    Returns:
        first_t, past_t, global_t, thr_t, mask_t — all torch tensors on device
        global_t has shape [B, 7]: [z0, z1, hmg, nsid, log_hd, da, herf]
    """
    B = len(thresholds)
    mem_len = case["mem_len"]
    N = float(case["N"])

    P_raw = np.asarray(case["P"], dtype=np.float32)
    var_raw = np.asarray(case["variances"], dtype=np.float32)
    thr_raw = np.asarray(thresholds, dtype=np.float32).reshape(-1, 1)

    # Feature engineering
    taps_feat = (P_raw * N).astype(np.float32)
    vars_feat = var_raw.astype(np.float32)
    abs_feat = np.abs(taps_feat).astype(np.float32)
    snr_feat = np.log10((taps_feat ** 2) / (var_raw + EPS) + EPS).astype(np.float32)

    # Broadcast to [B, mem_len]
    taps_2d = np.broadcast_to(taps_feat[None, :], (B, mem_len)).copy()
    vars_2d = np.broadcast_to(vars_feat[None, :], (B, mem_len)).copy()
    abs_2d = np.broadcast_to(abs_feat[None, :], (B, mem_len)).copy()
    snr_2d = np.broadcast_to(snr_feat[None, :], (B, mem_len)).copy()

    L_past = mem_len - 1
    valid_past = np.ones((B, L_past), dtype=bool)

    # --- Threshold-dependent global features ---
    first_mean = taps_2d[:, 0:1]
    past_means = taps_2d[:, 1:]
    first_var = vars_2d[:, 0:1]
    past_vars = vars_2d[:, 1:]

    mu0 = (0.5 * past_means.sum(axis=1, keepdims=True)).astype(np.float32)
    mu1 = (first_mean + mu0).astype(np.float32)
    var0 = (0.5 * past_vars.sum(axis=1, keepdims=True)).astype(np.float32)
    var1 = (first_var + var0).astype(np.float32)
    std0 = np.sqrt(np.maximum(var0, EPS)).astype(np.float32)
    std1 = np.sqrt(np.maximum(var1, EPS)).astype(np.float32)

    z0 = ((thr_raw - mu0) / (std0 + EPS)).astype(np.float32)
    z1 = ((mu1 - thr_raw) / (std1 + EPS)).astype(np.float32)
    harmonic = (2.0 / (1.0 / (z0 + EPS) + 1.0 / (z1 + EPS))).astype(np.float32)
    abs_diff = np.abs(z0 - z1).astype(np.float32)
    hmg = (harmonic - 0.25 * abs_diff).astype(np.float32)

    # --- Scenario-level features (threshold-independent) ---
    signal = first_mean  # [B, 1]
    isi = past_means.sum(axis=1, keepdims=True)
    nsid = ((signal - isi) / (signal + isi + EPS)).astype(np.float32)

    gap = signal
    d0_inf = (gap / (std0 + EPS)).astype(np.float32)
    d1_inf = (gap / (std1 + EPS)).astype(np.float32)
    hd = (2.0 * d0_inf * d1_inf / (d0_inf + d1_inf + EPS)).astype(np.float32)
    log_hd = np.log10(hd + EPS).astype(np.float32)
    da = (np.abs(d0_inf - d1_inf) / (d0_inf + d1_inf + EPS)).astype(np.float32)

    past_taps_sum = past_means.sum(axis=1, keepdims=True)
    past_shares = past_means / (past_taps_sum + EPS)
    herf = (past_shares ** 2).sum(axis=1, keepdims=True).astype(np.float32)

    # --- Scale first token ---
    ft_tap = ((taps_2d[:, 0] - scalers["first_tap_mean"]) / scalers["first_tap_std"]).astype(np.float32)
    ft_var = ((vars_2d[:, 0] - scalers["first_var_mean"]) / scalers["first_var_std"]).astype(np.float32)
    ft_abs = ((abs_2d[:, 0] - scalers["first_abs_mean"]) / scalers["first_abs_std"]).astype(np.float32)
    ft_snr = ((snr_2d[:, 0] - scalers["first_snr_mean"]) / scalers["first_snr_std"]).astype(np.float32)
    first_token = np.stack([ft_tap, ft_var, ft_abs, ft_snr], axis=1).astype(np.float32)

    # --- Scale past tokens ---
    pt_tap = apply_shared_scale(taps_2d[:, 1:], scalers["past_tap_mean"], scalers["past_tap_std"], valid_past)
    pt_var = apply_shared_scale(vars_2d[:, 1:], scalers["past_var_mean"], scalers["past_var_std"], valid_past)
    pt_abs = apply_shared_scale(abs_2d[:, 1:], scalers["past_abs_mean"], scalers["past_abs_std"], valid_past)
    pt_snr = apply_shared_scale(snr_2d[:, 1:], scalers["past_snr_mean"], scalers["past_snr_std"], valid_past)
    past_tokens = np.stack([pt_tap, pt_var, pt_abs, pt_snr], axis=2).astype(np.float32)

    # --- Scale globals (7 features) ---
    z0_s = scalers["z0_scaler"].transform(z0).astype(np.float32)
    z1_s = scalers["z1_scaler"].transform(z1).astype(np.float32)
    hmg_s = scalers["harmonic_minus_gap_scaler"].transform(hmg).astype(np.float32)
    nsid_s = scalers["nsid_scaler"].transform(nsid).astype(np.float32)
    log_hd_s = scalers["log_hd_scaler"].transform(log_hd).astype(np.float32)
    da_s = scalers["da_scaler"].transform(da).astype(np.float32)
    herf_s = scalers["herf_scaler"].transform(herf).astype(np.float32)
    global_feats = np.concatenate(
        [z0_s, z1_s, hmg_s, nsid_s, log_hd_s, da_s, herf_s], axis=1
    ).astype(np.float32)

    # --- Scale threshold ---
    thr_log = np.log10(thr_raw + EPS).astype(np.float32)
    thr_s = scalers["thr_scaler"].transform(thr_log).astype(np.float32)

    # --- Padding mask (no padding needed here) ---
    pad_mask = np.zeros((B, L_past), dtype=bool)

    first_t = torch.from_numpy(first_token).to(device)
    past_t = torch.from_numpy(past_tokens).to(device)
    global_t = torch.from_numpy(global_feats).to(device)
    thr_t = torch.from_numpy(thr_s).to(device)
    mask_t = torch.from_numpy(pad_mask).to(device)

    return first_t, past_t, global_t, thr_t, mask_t


# =========================
# Generate scenario
# =========================
rng = np.random.default_rng(RANDOM_SEED)
case = generate_physical_case(
    rng,
    min_mem_len=PHYSICS_MIN_MEM_LEN,
    max_mem_len=PHYSICS_MAX_MEM_LEN,
    arrival_coverage=ARRIVAL_COVERAGE,
)

print("Generated physical scenario")
print("radius    =", case["radius"])
print("distance  =", case["distance"])
print("diffusion =", case["diffusion"])
print("Ts        =", case["Ts"])
print("N         =", case["N"])
print("mem_len   =", case["mem_len"])
print("P         =", case["P"])
print("P_scaled  =", case["P_scaled"])
print("variances =", case["variances"])

# =========================
# Threshold sweep — ground truth
# =========================
thr_min = 0.0
thr_max = float(np.sum(case["P_scaled"]))
thresholds = np.linspace(thr_min, thr_max, N_THRESHOLDS)

print("Threshold search interval:", thr_min, "to", thr_max)
print("sum(P_scaled) =", np.sum(case["P_scaled"]))

real_bers = np.array([
    calculate_ber_vectorized(
        mem_len=case["mem_len"],
        threshold=thr,
        P_scaled=case["P_scaled"],
        variances=case["variances"],
    )
    for thr in thresholds
])

real_bers = np.clip(real_bers, EPS, 0.5)
real_regions = raw_to_region_labels_np(real_bers)

# =========================
# Load model + scalers
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scalers = joblib.load(SCALER_PATH)

max_past_seq_len = scalers.get("train_max_past_seq_len", scalers.get("max_past_seq_len", case["mem_len"] - 1))
ordinal_thresholds = scalers.get("ordinal_thresholds", ORDINAL_THRESHOLDS)
num_ordinal = len(ordinal_thresholds)
global_dim = scalers.get("global_dim", 7)

model = BERFirstTokenSetTransformerBoundedLogHalfMultiTask(
    max_past_seq_len=max_past_seq_len,
    token_dim=scalers.get("past_token_dim", 4),
    first_token_dim=scalers.get("first_token_dim", 4),
    threshold_dim=1,
    global_dim=global_dim,
    d_model=128,
    num_set_layers=4,
    num_heads=4,
    mlp_ratio=4.0,
    dropout=0.10,
    num_ordinal_thresholds=num_ordinal,
).to(device)

state = torch.load(MODEL_PATH, map_location=device, weights_only=True)
model.load_state_dict(state)
model.eval()

print(f"\nModel loaded: max_past_seq_len={max_past_seq_len}, "
      f"global_dim={global_dim}, num_ordinal={num_ordinal}")
print(f"Scaling strategy: {scalers.get('scaling_strategy', 'unknown')}")

# =========================
# Predict across thresholds
# =========================
first_t, past_t, global_t, thr_t, mask_t = prepare_inference_features(
    case, thresholds, scalers, device,
)

with torch.no_grad():
    pred_raw_out, ord_logits = model(
        first_t, past_t, global_t, thr_t, key_padding_mask=mask_t,
    )
    pred_bers = model.raw_to_ber(pred_raw_out).cpu().numpy().reshape(-1)
    pred_log = model.raw_to_log10ber(pred_raw_out).cpu().numpy().reshape(-1)
    pred_regions = ordinal_logits_to_region_labels_torch(ord_logits).cpu().numpy().reshape(-1)
    pred_region_probs = torch.sigmoid(ord_logits).cpu().numpy()

pred_bers = np.clip(pred_bers, EPS, 0.5)

# =========================
# Comparison table
# =========================
results = pd.DataFrame({
    "threshold": thresholds,
    "real_BER": real_bers,
    "estimated_BER": pred_bers,
    "real_region": real_regions,
    "predicted_region": pred_regions,
    "abs_error": np.abs(pred_bers - real_bers),
    "abs_log10_error": np.abs(
        np.log10(np.clip(pred_bers, EPS, 0.5)) -
        np.log10(np.clip(real_bers, EPS, 0.5))
    ),
    "region_abs_error": np.abs(pred_regions.astype(np.int64) - real_regions.astype(np.int64)),
})

print(results.head(15))

print("\nSummary")
print("Mean abs raw error        :", results["abs_error"].mean())
print("Mean abs log10 error      :", results["abs_log10_error"].mean())
print("Max  abs log10 error      :", results["abs_log10_error"].max())
print("Mean region abs error     :", results["region_abs_error"].mean())
print("Exact region accuracy     :", np.mean(results["real_region"] == results["predicted_region"]))

best_real_idx = np.argmin(real_bers)
best_est_idx = np.argmin(pred_bers)

print("\nBest threshold from real BER      :", thresholds[best_real_idx])
print("Minimum real BER                  :", real_bers[best_real_idx])
print("Real BER region there             :", REGION_LABELS.get(int(real_regions[best_real_idx]), "?"))

print("Best threshold from estimated BER :", thresholds[best_est_idx])
print("Estimated BER at that threshold   :", pred_bers[best_est_idx])
print("Predicted BER region there        :", REGION_LABELS.get(int(pred_regions[best_est_idx]), "?"))

# =========================
# Plot 1: log-scale BER
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["estimated_BER"], label="Estimated BER")
plt.plot(results["threshold"], results["real_BER"], label="Real BER")
plt.yscale("log")
plt.xlabel("Threshold")
plt.ylabel("BER")
plt.title(f"Threshold vs BER (log scale) — mem_len={case['mem_len']}")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "01_ber_log_scale.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/01_ber_log_scale.png")

# =========================
# Plot 2: linear-scale BER
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["estimated_BER"], label="Estimated BER")
plt.plot(results["threshold"], results["real_BER"], label="Real BER")
plt.yscale("linear")
plt.xlabel("Threshold")
plt.ylabel("BER")
plt.title(f"Threshold vs BER (linear scale) — mem_len={case['mem_len']}")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "02_ber_linear_scale.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/02_ber_linear_scale.png")

# =========================
# Plot 3: predicted vs true region
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["predicted_region"], label="Predicted region")
plt.plot(results["threshold"], results["real_region"], label="True region")
plt.xlabel("Threshold")
plt.ylabel("BER Region Class")
plt.title(f"Threshold vs BER Region — mem_len={case['mem_len']}")
plt.yticks(list(REGION_LABELS.keys()),
           [REGION_LABELS[k] for k in sorted(REGION_LABELS.keys())],
           fontsize=7)
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "03_region_comparison.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/03_region_comparison.png")

# =========================
# Plot 4: ordinal threshold probabilities
# =========================
plt.figure(figsize=(12, 7))
for i, thr_val in enumerate(ordinal_thresholds):
    plt.plot(thresholds, pred_region_probs[:, i], label=f"P(y >= {thr_val:g})")
plt.xlabel("Threshold")
plt.ylabel("Ordinal Probability")
plt.title(f"Ordinal Head Outputs Across Threshold Sweep — mem_len={case['mem_len']}")
plt.legend(fontsize=7, ncol=2)
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "04_ordinal_probabilities.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/04_ordinal_probabilities.png")

# =========================
# Plot 5: absolute error by threshold
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["abs_error"], label="Abs raw error", alpha=0.8)
plt.plot(results["threshold"], results["abs_log10_error"], label="Abs log10 error", alpha=0.8)
plt.yscale("log")
plt.xlabel("Threshold")
plt.ylabel("Error")
plt.title(f"Prediction Error Across Threshold Sweep — mem_len={case['mem_len']}")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "05_error_by_threshold.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/05_error_by_threshold.png")

Generated physical scenario
radius    = 4.999903475776163
distance  = 13.063595613360427
diffusion = 63.15707394395149
Ts        = 0.5834416162676823
N         = 2026
mem_len   = 14
P         = [0.03545102 0.042582   0.02704795 0.01857409 0.01368091 0.01059455
 0.00850978 0.00702648 0.00592783 0.00508774 0.00442855 0.00390015
 0.00346895 0.00311165]
P_scaled  = [71.82377081 86.27113641 54.79914164 37.63111579 27.71752004 21.46456524
 17.24082032 14.23564842 12.00979138 10.30776824  8.9722469   7.90171201
  7.0280834   6.304198  ]
variances = [69.27754472 82.59753869 53.31693734 36.93215188 27.33831919 21.23715776
 17.09410468 14.13562193 11.93859933 10.25532496  8.93251283  7.87089412
  7.00370336  6.28458156]
Threshold search interval: 0.0 to 383.7075186062634
sum(P_scaled) = 383.7075186062634

Model loaded: max_past_seq_len=13, global_dim=7, num_ordinal=13
Scaling strategy: position_independent
    threshold  real_BER  estimated_BER  real_region  predicted_region  \
0    0.000000  0.

In [2]:
import os
import re
import copy
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.multiprocessing as mp
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

mp.set_sharing_strategy("file_system")

# =========================
# Configuration
# =========================
BASE_DIR = "./"

DATA_PATHS = [
    os.path.join(BASE_DIR, "data_random_with_random_variances_total_v2.csv"),
    os.path.join(BASE_DIR, "data_physics_with_variances_total.csv"),
]

NROWS_PER_DATASET = 5_000_000

BEST_MODEL_SAVE_PATH = os.path.join(
    BASE_DIR,
    "random_extra_multitask_best2.pth",
)
LAST_MODEL_SAVE_PATH = os.path.join(
    BASE_DIR,
    "random_extra_multitask_last2.pth",
)
SCALER_SAVE_PATH = os.path.join(
    BASE_DIR,
    "random_extra_multitask_scalers2.pkl",
)

EPS = 1e-12
LOG10_HALF = float(np.log10(0.5))

ORDINAL_THRESHOLDS = [
    1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1,
    0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45,
]
NUM_ORDINAL_THRESHOLDS = len(ORDINAL_THRESHOLDS)
NUM_REGION_CLASSES = NUM_ORDINAL_THRESHOLDS + 1


# =========================
# Utilities
# =========================
def get_sorted_seq_cols(columns, prefix):
    pattern = re.compile(rf"^{re.escape(prefix)}_(\d+)$")
    matched = []
    for col in columns:
        m = pattern.match(col)
        if m:
            matched.append((int(m.group(1)), col))
    matched.sort(key=lambda x: x[0])
    return [col for _, col in matched]


def make_strat_bins(y_log, n_bins=10):
    y_flat = y_log.reshape(-1)
    quantiles = np.linspace(0, 1, n_bins + 1)
    edges = np.quantile(y_flat, quantiles)
    edges = np.unique(edges)

    if len(edges) < 3:
        return None

    bins = np.digitize(y_flat, edges[1:-1], right=True)
    counts = np.bincount(bins)
    if np.any(counts < 2):
        return None
    return bins


def has_nonfinite_tensor(x):
    return not torch.isfinite(x).all().item()


def validate_required_columns(df, file_path):
    if "mem_len" not in df.columns:
        raise ValueError(f"Required column 'mem_len' not found in {file_path}")
    if "N" not in df.columns:
        raise ValueError(f"Required column 'N' not found in {file_path}")

    tap_cols = get_sorted_seq_cols(df.columns, "tap")
    var_cols = get_sorted_seq_cols(df.columns, "var")

    if not tap_cols:
        raise ValueError(f"No tap_* columns found in {file_path}")
    if not var_cols:
        raise ValueError(f"No var_* columns found in {file_path}")
    if len(tap_cols) != len(var_cols):
        raise ValueError(
            f"tap/var length mismatch in {file_path}: "
            f"{len(tap_cols)} tap cols vs {len(var_cols)} var cols"
        )

    required_cols = tap_cols + var_cols + ["threshold", "BER", "N"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in {file_path}: {missing}")

    return tap_cols, var_cols


def load_and_merge_data(csv_paths, nrows_per_dataset):
    dfs = []
    reference_tap_cols = None
    reference_var_cols = None

    for path in csv_paths:
        if not os.path.exists(path):
            raise FileNotFoundError(f"Dataset not found: {path}")

        print(f"Loading up to {nrows_per_dataset:,} rows from: {path}")
        df = pd.read_csv(path, nrows=nrows_per_dataset)

        tap_cols, var_cols = validate_required_columns(df, path)

        if reference_tap_cols is None:
            reference_tap_cols = tap_cols
            reference_var_cols = var_cols
        else:
            if tap_cols != reference_tap_cols:
                raise ValueError("tap columns do not match across files.")
            if var_cols != reference_var_cols:
                raise ValueError("var columns do not match across files.")

        df["source_dataset"] = os.path.basename(path)
        dfs.append(df)

    merged = pd.concat(dfs, ignore_index=True)
    print(f"Combined rows before filtering: {len(merged):,}")

    return merged, reference_tap_cols, reference_var_cols


def y_log_to_raw_np(y_log):
    return np.clip(10 ** y_log, EPS, 0.5)


def raw_to_region_labels_np(y_raw):
    y = np.asarray(y_raw).reshape(-1)
    labels = np.searchsorted(np.asarray(ORDINAL_THRESHOLDS), y, side="right")
    return labels.astype(np.int64)


def region_labels_to_ordinal_targets_np(labels, num_thresholds=NUM_ORDINAL_THRESHOLDS):
    labels = np.asarray(labels).reshape(-1)
    thresholds = np.arange(1, num_thresholds + 1, dtype=np.int64)
    ordinal = (labels[:, None] >= thresholds[None, :]).astype(np.float32)
    return ordinal


def ordinal_logits_to_region_labels_torch(logits):
    probs = torch.sigmoid(logits)
    passed = (probs >= 0.5).sum(dim=1)
    return passed.long()


def compute_region_class_weights(labels_np, num_classes=NUM_REGION_CLASSES, max_weight=8.0):
    counts = np.bincount(labels_np.reshape(-1), minlength=num_classes).astype(np.float64)
    counts = np.maximum(counts, 1.0)
    weights = counts.sum() / (num_classes * counts)
    weights = np.sqrt(weights)
    weights = np.clip(weights, 1.0, max_weight)
    return weights.astype(np.float32)


def compute_ordinal_pos_weights_from_region_labels(
    region_labels_np,
    num_thresholds=NUM_ORDINAL_THRESHOLDS,
    max_weight=20.0,
):
    ordinal_targets = region_labels_to_ordinal_targets_np(region_labels_np, num_thresholds)
    pos_counts = ordinal_targets.sum(axis=0)
    neg_counts = ordinal_targets.shape[0] - pos_counts
    pos_counts = np.maximum(pos_counts, 1.0)
    pos_weight = neg_counts / pos_counts
    pos_weight = np.clip(pos_weight, 1.0, max_weight)
    return pos_weight.astype(np.float32)


# =========================
# Position-Independent Scaling
# =========================
class SharedFeatureScaler:
    """A scaler that stores a single (mean, std) per feature channel,
    shared across all sequence positions.

    For the first token (always position 0) we keep a separate scaler,
    since the current-symbol tap has a genuinely different distribution
    than the ISI taps at positions 1+.

    For all past positions we pool every valid entry into one distribution
    and fit a single mean/std.  This makes the scaler independent of
    sequence length — at inference time, any number of past taps can be
    scaled with the same parameters.
    """

    def __init__(self):
        self.first_mean = None  # shape [n_features]
        self.first_std = None
        self.past_mean = None   # shape [n_features]
        self.past_std = None

    def fit(self, first_data_list, past_data_list, past_valid_list):
        """
        Args:
            first_data_list: list of arrays, each [N, n_features] for the
                             first-token features (taps[:,0], vars[:,0], ...).
                             Concatenated across splits if desired, or just train.
            past_data_list:  list of arrays, each [N, L_past, n_features] or
                             [N, L_past] for a single feature channel.
            past_valid_list: list of bool arrays, each [N, L_past], True = valid.
        """
        # --- First token ---
        first_all = np.concatenate(first_data_list, axis=0)  # [N_total, F]
        self.first_mean = first_all.mean(axis=0).astype(np.float64)
        self.first_std = first_all.std(axis=0).astype(np.float64)
        self.first_std = np.maximum(self.first_std, 1e-12)

        # --- Past tokens (pool all valid entries per feature) ---
        valid_entries = []
        for data, valid in zip(past_data_list, past_valid_list):
            if data.ndim == 2:
                # single feature: [N, L] -> expand to [N, L, 1]
                data = data[:, :, None]
            # data: [N, L, F], valid: [N, L]
            valid_expanded = valid[:, :, None]  # [N, L, 1]
            # Gather valid entries: [?, F]
            valid_entries.append(data[np.broadcast_to(valid_expanded, data.shape)].reshape(-1, data.shape[-1]))

        pooled = np.concatenate(valid_entries, axis=0)  # [total_valid, F]
        self.past_mean = pooled.mean(axis=0).astype(np.float64)
        self.past_std = pooled.std(axis=0).astype(np.float64)
        self.past_std = np.maximum(self.past_std, 1e-12)

    def transform_first(self, first_data):
        """first_data: [N, F] -> scaled [N, F]"""
        return ((first_data - self.first_mean) / self.first_std).astype(np.float32)

    def transform_past(self, past_data, past_lens):
        """past_data: [N, L, F] or [N, L] -> scaled + re-zeroed [N, L, ...].
        past_lens: [N] int, number of valid past positions per row."""
        shape = past_data.shape
        if past_data.ndim == 2:
            scaled = ((past_data - self.past_mean[0]) / self.past_std[0]).astype(np.float32)
            L = shape[1]
        else:
            scaled = ((past_data - self.past_mean) / self.past_std).astype(np.float32)
            L = shape[1]

        # Re-zero padding positions
        valid = (np.arange(L)[None, :] < past_lens[:, None])  # [N, L]
        if scaled.ndim == 3:
            valid = valid[:, :, None]
        scaled = scaled * valid.astype(np.float32)
        return scaled


def fit_shared_scalar_scaler(train_data, train_valid_mask):
    """Fit a single-feature StandardScaler on pooled valid entries.
    train_data: [N, L], train_valid_mask: [N, L] bool.
    Returns (mean, std) as floats."""
    entries = train_data[train_valid_mask].reshape(-1)
    if len(entries) == 0:
        entries = train_data.reshape(-1)
    mean = float(entries.mean())
    std = float(max(entries.std(), 1e-12))
    return mean, std


def apply_shared_scale(data, mean, std, valid_mask=None):
    """Scale data with a single (mean, std), optionally re-zero invalid positions.
    data: [N, L] or [N, 1], valid_mask: [N, L] bool or None."""
    scaled = ((data - mean) / std).astype(np.float32)
    if valid_mask is not None and scaled.ndim == 2 and valid_mask.shape == scaled.shape:
        scaled = scaled * valid_mask.astype(np.float32)
    return scaled


# =========================
# Targeted Regression Loss
# =========================
class StableMultiObjectiveBERBoundedLogLoss(nn.Module):
    def __init__(
        self,
        log_delta=0.35,
        raw_delta=0.006,
        rel_delta=0.03,
        alpha_log=0.45,
        beta_raw=0.35,
        gamma_rel=0.20,
        use_regime_weights=True,
    ):
        super().__init__()
        self.log_delta = log_delta
        self.raw_delta = raw_delta
        self.rel_delta = rel_delta
        self.alpha_log = alpha_log
        self.beta_raw = beta_raw
        self.gamma_rel = gamma_rel
        self.use_regime_weights = use_regime_weights

    @staticmethod
    def huber_elementwise(pred, target, delta):
        err = pred - target
        abs_err = err.abs()
        return torch.where(
            abs_err < delta,
            0.5 * err * err,
            delta * (abs_err - 0.5 * delta),
        )

    @staticmethod
    def raw_to_pred_log(pred_raw_unconstrained):
        log10_half = torch.log10(
            torch.tensor(
                0.5,
                device=pred_raw_unconstrained.device,
                dtype=pred_raw_unconstrained.dtype,
            )
        )
        return log10_half - F.softplus(pred_raw_unconstrained)

    @staticmethod
    def raw_to_pred_ber(pred_raw_unconstrained):
        pred_log = StableMultiObjectiveBERBoundedLogLoss.raw_to_pred_log(
            pred_raw_unconstrained
        )
        return torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)

    def forward(self, pred_raw_unconstrained, target_log):
        pred_log = self.raw_to_pred_log(pred_raw_unconstrained)
        pred_raw = torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)

        target_raw = torch.pow(10.0, target_log).clamp(min=EPS, max=0.5)
        target_log_for_loss = torch.log10(target_raw)

        log_loss = self.huber_elementwise(pred_log, target_log_for_loss, self.log_delta)
        raw_loss = self.huber_elementwise(pred_raw, target_raw, self.raw_delta)
        rel_err = (pred_raw - target_raw) / torch.clamp(target_raw, min=1e-6)
        rel_loss = self.huber_elementwise(rel_err, torch.zeros_like(rel_err), self.rel_delta)

        total = (
            self.alpha_log * log_loss
            + self.beta_raw * raw_loss
            + self.gamma_rel * rel_loss
        )

        if self.use_regime_weights:
            weights = torch.ones_like(target_raw)
            weights = torch.where(
                (target_raw >= 1e-2) & (target_raw < 0.1),
                torch.full_like(weights, 1.5), weights,
            )
            weights = torch.where(
                (target_raw >= 0.1) & (target_raw < 0.15),
                torch.full_like(weights, 2.5), weights,
            )
            weights = torch.where(
                (target_raw >= 0.15) & (target_raw < 0.2),
                torch.full_like(weights, 3.0), weights,
            )
            weights = torch.where(
                (target_raw >= 0.2) & (target_raw < 0.3),
                torch.full_like(weights, 4.0), weights,
            )
            weights = torch.where(
                (target_raw >= 0.3) & (target_raw < 0.4),
                torch.full_like(weights, 4.5), weights,
            )
            weights = torch.where(
                (target_raw >= 0.4) & (target_raw < 0.45),
                torch.full_like(weights, 3.0), weights,
            )
            weights = torch.where(
                target_raw >= 0.45,
                torch.full_like(weights, 2.5), weights,
            )
            total = total * weights

        return total.mean()


class OrdinalBCELoss(nn.Module):
    def __init__(self, pos_weight=None, reduction="mean"):
        super().__init__()
        if pos_weight is not None and not isinstance(pos_weight, torch.Tensor):
            pos_weight = torch.tensor(pos_weight, dtype=torch.float32)
        self.register_buffer(
            "pos_weight", pos_weight if pos_weight is not None else None
        )
        self.reduction = reduction

    def forward(self, logits, ordinal_targets):
        return F.binary_cross_entropy_with_logits(
            logits,
            ordinal_targets,
            pos_weight=self.pos_weight,
            reduction=self.reduction,
        )


class MultiTaskBERLoss(nn.Module):
    def __init__(self, reg_loss, ord_loss, lambda_ord=0.25):
        super().__init__()
        self.reg_loss = reg_loss
        self.ord_loss = ord_loss
        self.lambda_ord = lambda_ord

    def forward(self, pred_raw_unconstrained, ord_logits, target_log, target_ord):
        reg = self.reg_loss(pred_raw_unconstrained, target_log)
        ordl = self.ord_loss(ord_logits, target_ord)
        total = reg + self.lambda_ord * ordl
        return total, reg.detach(), ordl.detach()


# =========================
# Set Transformer Blocks
# =========================
class SetSelfAttentionBlock(nn.Module):
    def __init__(self, d_model, num_heads, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norm2 = nn.LayerNorm(d_model)

        hidden = int(d_model * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x, key_padding_mask=None):
        """
        Args:
            x: [B, L, D]
            key_padding_mask: [B, L] bool, True = padding (ignore)
        """
        y = self.norm1(x)
        attn_out, _ = self.attn(
            y, y, y,
            need_weights=False,
            key_padding_mask=key_padding_mask,
        )
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x


class ConditionalFeatureModulation(nn.Module):
    def __init__(self, cond_dim, feat_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, 2 * feat_dim),
        )
        self.scale = nn.Parameter(torch.tensor(0.25, dtype=torch.float32))

    def forward(self, x, cond):
        gamma_beta = self.net(cond)
        gamma, beta = torch.chunk(gamma_beta, 2, dim=-1)

        if x.dim() == 3:
            gamma = gamma.unsqueeze(1)
            beta = beta.unsqueeze(1)

        return x * (1.0 + self.scale * gamma) + self.scale * beta


class SetAttentionPooling(nn.Module):
    def __init__(self, d_model, hidden_dim=128):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x, key_padding_mask=None):
        """
        Args:
            x: [B, L, D]
            key_padding_mask: [B, L] bool, True = padding
        Returns:
            pooled: [B, D]
        """
        logits = self.score(x)  # [B, L, 1]

        if key_padding_mask is not None:
            logits = logits.masked_fill(
                key_padding_mask.unsqueeze(-1), float("-inf")
            )

        weights = torch.softmax(logits, dim=1)  # [B, L, 1]
        # Safety: all-masked rows produce NaN from softmax(-inf); replace with 0
        weights = torch.nan_to_num(weights, nan=0.0)

        pooled = (weights * x).sum(dim=1)  # [B, D]
        return pooled


# =========================
# Model
# =========================
class BERFirstTokenSetTransformerBoundedLogHalfMultiTask(nn.Module):
    def __init__(
        self,
        max_past_seq_len,
        token_dim=4,
        first_token_dim=4,
        threshold_dim=1,
        global_dim=7,
        d_model=128,
        num_set_layers=4,
        num_heads=4,
        mlp_ratio=4.0,
        dropout=0.10,
        num_ordinal_thresholds=NUM_ORDINAL_THRESHOLDS,
    ):
        super().__init__()

        self.max_past_seq_len = max_past_seq_len
        self.token_dim = token_dim
        self.first_token_dim = first_token_dim
        self.d_model = d_model
        self.num_ordinal_thresholds = num_ordinal_thresholds

        self.first_proj = nn.Sequential(
            nn.Linear(first_token_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.token_proj = nn.Sequential(
            nn.Linear(token_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.thr_embed = nn.Sequential(
            nn.Linear(threshold_dim, 32),
            nn.GELU(),
            nn.Linear(32, 32),
            nn.GELU(),
        )

        self.global_embed = nn.Sequential(
            nn.Linear(global_dim, 96),
            nn.GELU(),
            nn.Linear(96, 96),
            nn.GELU(),
        )

        cond_dim = 32 + 96

        self.first_cond_mod = ConditionalFeatureModulation(
            cond_dim=cond_dim, feat_dim=d_model, hidden_dim=256,
        )
        self.set_cond_mod = ConditionalFeatureModulation(
            cond_dim=cond_dim, feat_dim=d_model, hidden_dim=256,
        )

        self.set_blocks = nn.ModuleList(
            [
                SetSelfAttentionBlock(
                    d_model=d_model,
                    num_heads=num_heads,
                    mlp_ratio=mlp_ratio,
                    dropout=dropout,
                )
                for _ in range(num_set_layers)
            ]
        )

        self.first_refine = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
        )

        self.final_set_norm = nn.LayerNorm(d_model)
        self.set_attn_pool = SetAttentionPooling(d_model=d_model, hidden_dim=128)

        set_summary_dim = 3 * d_model
        head_in = d_model + set_summary_dim + cond_dim

        self.reg_head = nn.Sequential(
            nn.Linear(head_in, 256),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(128, 1),
        )

        self.ord_head = nn.Sequential(
            nn.Linear(head_in, 256),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(128, num_ordinal_thresholds),
        )

    @staticmethod
    def raw_to_log10ber(pred_raw_unconstrained):
        log10_half = torch.log10(
            torch.tensor(
                0.5,
                device=pred_raw_unconstrained.device,
                dtype=pred_raw_unconstrained.dtype,
            )
        )
        return log10_half - F.softplus(pred_raw_unconstrained)

    @staticmethod
    def raw_to_ber(pred_raw_unconstrained):
        pred_log = BERFirstTokenSetTransformerBoundedLogHalfMultiTask.raw_to_log10ber(
            pred_raw_unconstrained
        )
        return torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)

    def forward(self, first_token, past_tokens, global_feats, threshold, key_padding_mask=None):
        """
        Args:
            first_token:       [B, first_token_dim]
            past_tokens:       [B, L, token_dim]  (L can vary between training and inference)
            global_feats:      [B, global_dim]
            threshold:         [B, 1]
            key_padding_mask:  [B, L] bool, True = padding position
        """
        thr = self.thr_embed(threshold)
        g = self.global_embed(global_feats)
        cond = torch.cat([thr, g], dim=-1)

        # First token path
        first_x = self.first_proj(first_token)
        first_x = self.first_cond_mod(first_x, cond)
        first_x = first_x + self.first_refine(first_x)

        # Past tokens path with mask
        set_x = self.token_proj(past_tokens)
        set_x = self.set_cond_mod(set_x, cond)

        for block in self.set_blocks:
            set_x = block(set_x, key_padding_mask=key_padding_mask)

        set_x = self.final_set_norm(set_x)

        # --- Masked pooling ---
        if key_padding_mask is not None:
            valid_mask = (~key_padding_mask).unsqueeze(-1).float()  # [B, L, 1]
        else:
            valid_mask = torch.ones(
                set_x.shape[0], set_x.shape[1], 1,
                device=set_x.device, dtype=set_x.dtype,
            )

        # Attention pooling
        pooled_attn = self.set_attn_pool(set_x, key_padding_mask=key_padding_mask)

        # Masked mean pooling
        valid_counts = valid_mask.sum(dim=1).clamp(min=1.0)  # [B, 1]
        pooled_mean = (set_x * valid_mask).sum(dim=1) / valid_counts

        # Masked max pooling
        if key_padding_mask is not None:
            x_for_max = set_x.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
        else:
            x_for_max = set_x
        pooled_max = x_for_max.amax(dim=1)
        pooled_max = torch.nan_to_num(pooled_max, nan=0.0, posinf=0.0, neginf=0.0)

        set_summary = torch.cat([pooled_attn, pooled_mean, pooled_max], dim=-1)
        fused = torch.cat([first_x, set_summary, cond], dim=-1)

        pred_raw_unconstrained = self.reg_head(fused)
        ord_logits = self.ord_head(fused)

        return pred_raw_unconstrained, ord_logits


# =========================
# Data
# =========================
def prepare_data(csv_paths, batch_size=256, nrows_per_dataset=2_500_000, num_workers=0):
    df, tap_cols, var_cols = load_and_merge_data(csv_paths, nrows_per_dataset)

    df = df[df["mem_len"] != 1].copy()

    df[tap_cols] = df[tap_cols].fillna(0.0)
    df[var_cols] = df[var_cols].fillna(0.0)
    df["threshold"] = df["threshold"].fillna(0.0)
    df["BER"] = df["BER"].fillna(0.0)
    df["N"] = df["N"].fillna(0.0)

    df = df[(df["threshold"] > 0) & (df["BER"] > 0) & (df["N"] > 0)].copy()
    df["BER"] = df["BER"].clip(lower=EPS, upper=0.5)

    print(f"Rows after cleaning/filtering: {len(df):,}")

    # ---- Extract raw arrays ----
    X_taps_raw = df[tap_cols].to_numpy(dtype=np.float32)
    X_vars_raw = df[var_cols].to_numpy(dtype=np.float32)
    num_molecules = df["N"].to_numpy(dtype=np.float32).reshape(-1, 1)
    mem_len = df["mem_len"].to_numpy(dtype=np.int64)

    X_thr_raw = df["threshold"].to_numpy(dtype=np.float32).reshape(-1, 1)
    y_raw = df["BER"].to_numpy(dtype=np.float32).reshape(-1, 1)

    X_thr = np.log10(X_thr_raw + EPS).astype(np.float32)
    y_log = np.log10(y_raw + EPS).astype(np.float32)

    # ---- Feature engineering ----
    X_taps_feat = (X_taps_raw * num_molecules).astype(np.float32)

    if np.any(X_vars_raw < 0):
        raise ValueError("Variance columns contain negative values.")

    X_vars_feat = X_vars_raw.astype(np.float32)
    abs_taps_raw = np.abs(X_taps_feat).astype(np.float32)
    snr_raw = np.log10((X_taps_feat ** 2) / (X_vars_raw + EPS) + EPS).astype(np.float32)

    L_full = X_taps_feat.shape[1]

    # ---- Validity masks ----
    mem_len_clamped = np.minimum(mem_len, L_full)
    valid_full = (np.arange(L_full)[None, :] < mem_len_clamped[:, None])  # [N, L_full]
    valid_past = valid_full[:, 1:]  # [N, L_full-1]
    valid_past_float = valid_past.astype(np.float32)
    past_lens = np.maximum(mem_len_clamped - 1, 0).astype(np.int64)

    # ---- Mask-aware global features ----
    first_mean = X_taps_feat[:, 0:1]
    past_means = X_taps_feat[:, 1:]
    first_var = X_vars_feat[:, 0:1]
    past_vars = X_vars_feat[:, 1:]

    past_means_masked = past_means * valid_past_float
    past_vars_masked = past_vars * valid_past_float

    mu0_raw = (0.5 * np.sum(past_means_masked, axis=1, keepdims=True)).astype(np.float32)
    mu1_raw = (first_mean + mu0_raw).astype(np.float32)

    var0_raw = (0.5 * np.sum(past_vars_masked, axis=1, keepdims=True)).astype(np.float32)
    var1_raw = (first_var + var0_raw).astype(np.float32)

    std0_raw = np.sqrt(np.maximum(var0_raw, EPS)).astype(np.float32)
    std1_raw = np.sqrt(np.maximum(var1_raw, EPS)).astype(np.float32)

    z0_raw = ((X_thr_raw - mu0_raw) / (std0_raw + EPS)).astype(np.float32)
    z1_raw = ((mu1_raw - X_thr_raw) / (std1_raw + EPS)).astype(np.float32)

    harmonic_side_z_raw = (
        2.0 / (1.0 / (z0_raw + EPS) + 1.0 / (z1_raw + EPS))
    ).astype(np.float32)
    abs_diff_side_z_raw = np.abs(z0_raw - z1_raw).astype(np.float32)
    harmonic_minus_gap_raw = (
        harmonic_side_z_raw - 0.25 * abs_diff_side_z_raw
    ).astype(np.float32)

    # ---- Scenario-level features (threshold-INDEPENDENT) ----
    # These capture intrinsic difficulty of the physical scenario.

    # 1. NSID: Normalized Signal-Interference Difference, range ~ [-1, +1]
    signal_raw = first_mean  # P_0 * N, shape [N, 1]
    isi_raw = np.sum(past_means_masked, axis=1, keepdims=True)  # [N, 1]
    nsid_raw = ((signal_raw - isi_raw) / (signal_raw + isi_raw + EPS)).astype(np.float32)

    # 2. Log Harmonic Discriminability (threshold-free)
    gap_raw = signal_raw  # mu1 - mu0 = P_0 * N
    d0_raw = (gap_raw / (std0_raw + EPS)).astype(np.float32)
    d1_raw = (gap_raw / (std1_raw + EPS)).astype(np.float32)
    harmonic_discrim_raw = (2.0 * d0_raw * d1_raw / (d0_raw + d1_raw + EPS)).astype(np.float32)
    log_harmonic_discrim_raw = np.log10(harmonic_discrim_raw + EPS).astype(np.float32)

    # 3. Discriminability Asymmetry
    discrim_asymmetry_raw = (np.abs(d0_raw - d1_raw) / (d0_raw + d1_raw + EPS)).astype(np.float32)

    # 4. Herfindahl Index of ISI concentration [1/n_past, 1]
    past_taps_sum = np.sum(past_means_masked, axis=1, keepdims=True)  # [N, 1]
    past_shares = past_means_masked / (past_taps_sum + EPS)           # [N, L_past]
    herfindahl_raw = np.sum(past_shares ** 2, axis=1, keepdims=True).astype(np.float32)  # [N, 1]

    # ---- Labels ----
    region_labels = raw_to_region_labels_np(y_raw)
    ordinal_targets = region_labels_to_ordinal_targets_np(region_labels)
    strat_labels = make_strat_bins(y_log, n_bins=10)

    # ---- Train / temp split ----
    split_args = dict(test_size=0.30, random_state=42)
    if strat_labels is not None:
        split_args["stratify"] = strat_labels

    (
        t_taps_feat, temp_taps_feat,
        t_vars_feat, temp_vars_feat,
        t_abs_raw, temp_abs_raw,
        t_snr_raw, temp_snr_raw,
        t_z0_raw, temp_z0_raw,
        t_z1_raw, temp_z1_raw,
        t_hmg_raw, temp_hmg_raw,
        t_nsid_raw, temp_nsid_raw,
        t_log_hd_raw, temp_log_hd_raw,
        t_da_raw, temp_da_raw,
        t_herf_raw, temp_herf_raw,
        t_thr, temp_thr,
        t_y_log, temp_y_log,
        t_region, temp_region,
        t_ord, temp_ord,
        t_past_lens, temp_past_lens,
    ) = train_test_split(
        X_taps_feat, X_vars_feat, abs_taps_raw, snr_raw,
        z0_raw, z1_raw, harmonic_minus_gap_raw,
        nsid_raw, log_harmonic_discrim_raw, discrim_asymmetry_raw, herfindahl_raw,
        X_thr,
        y_log, region_labels, ordinal_targets, past_lens,
        **split_args,
    )

    # ---- temp -> val / test split ----
    temp_strat = make_strat_bins(temp_y_log, n_bins=6)
    split_args2 = dict(test_size=0.50, random_state=42)
    if temp_strat is not None:
        split_args2["stratify"] = temp_strat

    (
        v_taps_feat, te_taps_feat,
        v_vars_feat, te_vars_feat,
        v_abs_raw, te_abs_raw,
        v_snr_raw, te_snr_raw,
        v_z0_raw, te_z0_raw,
        v_z1_raw, te_z1_raw,
        v_hmg_raw, te_hmg_raw,
        v_nsid_raw, te_nsid_raw,
        v_log_hd_raw, te_log_hd_raw,
        v_da_raw, te_da_raw,
        v_herf_raw, te_herf_raw,
        v_thr, te_thr,
        v_y_log, te_y_log,
        v_region, te_region,
        v_ord, te_ord,
        v_past_lens, te_past_lens,
    ) = train_test_split(
        temp_taps_feat, temp_vars_feat, temp_abs_raw, temp_snr_raw,
        temp_z0_raw, temp_z1_raw, temp_hmg_raw,
        temp_nsid_raw, temp_log_hd_raw, temp_da_raw, temp_herf_raw,
        temp_thr,
        temp_y_log, temp_region, temp_ord, temp_past_lens,
        **split_args2,
    )

    # =========================================================
    # Position-independent scaling
    # =========================================================
    # For each feature channel (taps, vars, abs, snr) we fit:
    #   - A SEPARATE mean/std for position 0 (first token)
    #   - A SHARED mean/std pooled across ALL valid past positions 1+
    #
    # This decouples the scaler from the number of columns,
    # so at inference any mem_len works with the same parameters.
    # =========================================================

    L_past = L_full - 1

    # Build per-split past validity masks
    t_valid_past = (np.arange(L_past)[None, :] < t_past_lens[:, None])

    # --- Fit first-token scalers (on train only) ---
    first_tap_mean, first_tap_std = float(t_taps_feat[:, 0].mean()), max(float(t_taps_feat[:, 0].std()), 1e-12)
    first_var_mean, first_var_std = float(t_vars_feat[:, 0].mean()), max(float(t_vars_feat[:, 0].std()), 1e-12)
    first_abs_mean, first_abs_std = float(t_abs_raw[:, 0].mean()), max(float(t_abs_raw[:, 0].std()), 1e-12)
    first_snr_mean, first_snr_std = float(t_snr_raw[:, 0].mean()), max(float(t_snr_raw[:, 0].std()), 1e-12)

    # --- Fit shared past scalers (pool all valid past entries from train) ---
    past_tap_mean, past_tap_std = fit_shared_scalar_scaler(t_taps_feat[:, 1:], t_valid_past)
    past_var_mean, past_var_std = fit_shared_scalar_scaler(t_vars_feat[:, 1:], t_valid_past)
    past_abs_mean, past_abs_std = fit_shared_scalar_scaler(t_abs_raw[:, 1:], t_valid_past)
    past_snr_mean, past_snr_std = fit_shared_scalar_scaler(t_snr_raw[:, 1:], t_valid_past)

    # --- Global feature scalers (standard, shape [N, 1]) ---
    z0_scaler = StandardScaler().fit(t_z0_raw)
    z1_scaler = StandardScaler().fit(t_z1_raw)
    hmg_scaler = StandardScaler().fit(t_hmg_raw)
    nsid_scaler = StandardScaler().fit(t_nsid_raw)
    log_hd_scaler = StandardScaler().fit(t_log_hd_raw)
    da_scaler = StandardScaler().fit(t_da_raw)
    herf_scaler = StandardScaler().fit(t_herf_raw)
    thr_scaler = StandardScaler().fit(t_thr)

    # ---- Helper: build first_token and past_tokens arrays ----
    def build_tokens(taps_feat, vars_feat, abs_raw, snr_raw, p_lens):
        """Returns first_token [N, 4] and past_tokens [N, L_past, 4], both scaled + re-zeroed."""
        N = taps_feat.shape[0]

        # Scale first token
        ft_tap = ((taps_feat[:, 0] - first_tap_mean) / first_tap_std).astype(np.float32)
        ft_var = ((vars_feat[:, 0] - first_var_mean) / first_var_std).astype(np.float32)
        ft_abs = ((abs_raw[:, 0] - first_abs_mean) / first_abs_std).astype(np.float32)
        ft_snr = ((snr_raw[:, 0] - first_snr_mean) / first_snr_std).astype(np.float32)
        first_token = np.stack([ft_tap, ft_var, ft_abs, ft_snr], axis=1).astype(np.float32)

        # Scale past tokens (shared scaler) + re-zero
        pt_tap = apply_shared_scale(taps_feat[:, 1:], past_tap_mean, past_tap_std,
                                    (np.arange(L_past)[None, :] < p_lens[:, None]))
        pt_var = apply_shared_scale(vars_feat[:, 1:], past_var_mean, past_var_std,
                                    (np.arange(L_past)[None, :] < p_lens[:, None]))
        pt_abs = apply_shared_scale(abs_raw[:, 1:], past_abs_mean, past_abs_std,
                                    (np.arange(L_past)[None, :] < p_lens[:, None]))
        pt_snr = apply_shared_scale(snr_raw[:, 1:], past_snr_mean, past_snr_std,
                                    (np.arange(L_past)[None, :] < p_lens[:, None]))

        past_tokens = np.stack([pt_tap, pt_var, pt_abs, pt_snr], axis=2).astype(np.float32)

        return first_token, past_tokens

    def build_global(z0, z1, hmg, nsid, log_hd, da, herf):
        return np.concatenate([
            z0_scaler.transform(z0),
            z1_scaler.transform(z1),
            hmg_scaler.transform(hmg),
            nsid_scaler.transform(nsid),
            log_hd_scaler.transform(log_hd),
            da_scaler.transform(da),
            herf_scaler.transform(herf),
        ], axis=1).astype(np.float32)

    def build_padding_mask(p_lens, L_max):
        return (np.arange(L_max)[None, :] >= p_lens[:, None])  # True = padding

    # ---- Build all splits ----
    t_first, t_past = build_tokens(t_taps_feat, t_vars_feat, t_abs_raw, t_snr_raw, t_past_lens)
    v_first, v_past = build_tokens(v_taps_feat, v_vars_feat, v_abs_raw, v_snr_raw, v_past_lens)
    te_first, te_past = build_tokens(te_taps_feat, te_vars_feat, te_abs_raw, te_snr_raw, te_past_lens)

    t_global = build_global(t_z0_raw, t_z1_raw, t_hmg_raw,
                            t_nsid_raw, t_log_hd_raw, t_da_raw, t_herf_raw)
    v_global = build_global(v_z0_raw, v_z1_raw, v_hmg_raw,
                            v_nsid_raw, v_log_hd_raw, v_da_raw, v_herf_raw)
    te_global = build_global(te_z0_raw, te_z1_raw, te_hmg_raw,
                             te_nsid_raw, te_log_hd_raw, te_da_raw, te_herf_raw)

    t_thr_s = thr_scaler.transform(t_thr).astype(np.float32)
    v_thr_s = thr_scaler.transform(v_thr).astype(np.float32)
    te_thr_s = thr_scaler.transform(te_thr).astype(np.float32)

    L_max_past = L_past
    t_mask = build_padding_mask(t_past_lens, L_max_past)
    v_mask = build_padding_mask(v_past_lens, L_max_past)
    te_mask = build_padding_mask(te_past_lens, L_max_past)

    # ---- TensorDatasets ----
    train_ds = TensorDataset(
        torch.from_numpy(t_first),
        torch.from_numpy(t_past),
        torch.from_numpy(t_global),
        torch.from_numpy(t_thr_s),
        torch.from_numpy(t_y_log.astype(np.float32)),
        torch.from_numpy(t_ord.astype(np.float32)),
        torch.from_numpy(t_region.astype(np.int64)),
        torch.from_numpy(t_mask),
    )
    val_ds = TensorDataset(
        torch.from_numpy(v_first),
        torch.from_numpy(v_past),
        torch.from_numpy(v_global),
        torch.from_numpy(v_thr_s),
        torch.from_numpy(v_y_log.astype(np.float32)),
        torch.from_numpy(v_ord.astype(np.float32)),
        torch.from_numpy(v_region.astype(np.int64)),
        torch.from_numpy(v_mask),
    )
    test_ds = TensorDataset(
        torch.from_numpy(te_first),
        torch.from_numpy(te_past),
        torch.from_numpy(te_global),
        torch.from_numpy(te_thr_s),
        torch.from_numpy(te_y_log.astype(np.float32)),
        torch.from_numpy(te_ord.astype(np.float32)),
        torch.from_numpy(te_region.astype(np.int64)),
        torch.from_numpy(te_mask),
    )

    # ---- Weighted sampler (region + SIR-aware) ----
    region_sample_weights = np.ones_like(t_region, dtype=np.float32)
    region_sample_weights[t_region == 6] = 2.5
    region_sample_weights[t_region == 7] = 3.0
    region_sample_weights[t_region == 8] = 5.0
    region_sample_weights[t_region == 9] = 5.0
    region_sample_weights[t_region == 10] = 5.0
    region_sample_weights[t_region == 11] = 5.0
    region_sample_weights[t_region == 12] = 3.0
    region_sample_weights[t_region == 13] = 2.5

    # SIR-aware weights (threshold-independent scenario difficulty)
    t_signal = t_taps_feat[:, 0]
    t_valid_past_for_sir = (np.arange(L_past)[None, :] < t_past_lens[:, None]).astype(np.float32)
    t_isi = np.sum(t_taps_feat[:, 1:] * t_valid_past_for_sir, axis=1)
    t_sir = t_signal / (t_isi + EPS)

    sir_sample_weights = np.ones_like(t_sir, dtype=np.float32)
    sir_sample_weights[t_sir < 0.26] = 3.0
    sir_sample_weights[(t_sir >= 0.26) & (t_sir < 0.36)] = 2.0

    combined_sample_weights = region_sample_weights * sir_sample_weights

    sampler = WeightedRandomSampler(
        weights=torch.from_numpy(combined_sample_weights),
        num_samples=len(combined_sample_weights),
        replacement=True,
    )

    pin_mem = torch.cuda.is_available()

    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler,
                              shuffle=False, pin_memory=pin_mem, num_workers=num_workers)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            pin_memory=pin_mem, num_workers=num_workers)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                             pin_memory=pin_mem, num_workers=num_workers)

    class_weights = compute_region_class_weights(t_region, NUM_REGION_CLASSES)
    ordinal_pos_weights = compute_ordinal_pos_weights_from_region_labels(
        t_region, num_thresholds=NUM_ORDINAL_THRESHOLDS, max_weight=20.0,
    )

    # ---- Save all scaler parameters (position-independent) ----
    scalers = {
        # First-token scalers (per feature channel)
        "first_tap_mean": first_tap_mean, "first_tap_std": first_tap_std,
        "first_var_mean": first_var_mean, "first_var_std": first_var_std,
        "first_abs_mean": first_abs_mean, "first_abs_std": first_abs_std,
        "first_snr_mean": first_snr_mean, "first_snr_std": first_snr_std,
        # Shared past scalers (single mean/std per feature, position-independent)
        "past_tap_mean": past_tap_mean, "past_tap_std": past_tap_std,
        "past_var_mean": past_var_mean, "past_var_std": past_var_std,
        "past_abs_mean": past_abs_mean, "past_abs_std": past_abs_std,
        "past_snr_mean": past_snr_mean, "past_snr_std": past_snr_std,
        # Global feature scalers (sklearn StandardScaler objects)
        "z0_scaler": z0_scaler,
        "z1_scaler": z1_scaler,
        "harmonic_minus_gap_scaler": hmg_scaler,
        "nsid_scaler": nsid_scaler,
        "log_hd_scaler": log_hd_scaler,
        "da_scaler": da_scaler,
        "herf_scaler": herf_scaler,
        "thr_scaler": thr_scaler,
        # Metadata
        "tap_cols": tap_cols,
        "var_cols": var_cols,
        "first_token_dim": 4,
        "past_token_dim": 4,
        "global_dim": 7,
        "train_max_past_seq_len": L_past,
        "scaling_strategy": "position_independent",
        "uses_positional_encoding": False,
        "permutation_invariance_post_first": True,
        "variable_length_support": True,
        "target_parameterization": "pred_log10_ber = log10(0.5) - softplus(raw_out)",
        "ordinal_thresholds": ORDINAL_THRESHOLDS,
        "num_region_classes": NUM_REGION_CLASSES,
        "class_weights": class_weights.tolist(),
        "ordinal_pos_weights": ordinal_pos_weights.tolist(),
        "data_paths": csv_paths,
        "nrows_per_dataset": nrows_per_dataset,
    }

    aux_info = {
        "class_weights": class_weights,
        "ordinal_pos_weights": ordinal_pos_weights,
        "t_region": t_region,
    }

    return train_loader, val_loader, test_loader, scalers, aux_info, L_past


# =========================
# Inference Helper
# =========================
def prepare_inference_batch(
    taps_raw_2d,
    vars_raw_2d,
    N_array,
    threshold_array,
    mem_len_array,
    scalers,
):
    """Prepare a batch for inference from raw numpy arrays.

    This handles ARBITRARY mem_len — the set can be larger than anything
    seen during training because scalers are position-independent.

    Args:
        taps_raw_2d:     [B, L] raw tap coefficients (padded to L with zeros)
        vars_raw_2d:     [B, L] raw variance values  (padded to L with zeros)
        N_array:         [B] or [B, 1] number of molecules
        threshold_array: [B] or [B, 1] raw threshold values
        mem_len_array:   [B] true memory length per sample
        scalers:         dict from joblib.load(SCALER_SAVE_PATH)

    Returns:
        first_token:      [B, 4] tensor
        past_tokens:      [B, L-1, 4] tensor
        global_feats:     [B, 7] tensor
        threshold_scaled: [B, 1] tensor
        key_padding_mask: [B, L-1] bool tensor (True = padding)
    """
    B, L = taps_raw_2d.shape
    N = N_array.reshape(-1, 1).astype(np.float32)
    thr_raw = threshold_array.reshape(-1, 1).astype(np.float32)
    mem_len = mem_len_array.reshape(-1).astype(np.int64)

    mem_len_clamped = np.minimum(mem_len, L)
    past_lens = np.maximum(mem_len_clamped - 1, 0).astype(np.int64)

    # Feature engineering (same as training)
    taps_feat = (taps_raw_2d * N).astype(np.float32)
    vars_feat = vars_raw_2d.astype(np.float32)
    abs_feat = np.abs(taps_feat).astype(np.float32)
    snr_feat = np.log10((taps_feat ** 2) / (vars_raw_2d + EPS) + EPS).astype(np.float32)

    L_past = L - 1

    # Validity masks
    valid_past = (np.arange(L_past)[None, :] < past_lens[:, None])
    valid_past_float = valid_past.astype(np.float32)

    # Global features (mask-aware)
    first_mean = taps_feat[:, 0:1]
    past_means = taps_feat[:, 1:] * valid_past_float
    first_var = vars_feat[:, 0:1]
    past_vars = vars_feat[:, 1:] * valid_past_float

    mu0 = (0.5 * past_means.sum(axis=1, keepdims=True)).astype(np.float32)
    mu1 = (first_mean + mu0).astype(np.float32)
    var0 = (0.5 * past_vars.sum(axis=1, keepdims=True)).astype(np.float32)
    var1 = (first_var + var0).astype(np.float32)

    std0 = np.sqrt(np.maximum(var0, EPS)).astype(np.float32)
    std1 = np.sqrt(np.maximum(var1, EPS)).astype(np.float32)

    z0 = ((thr_raw - mu0) / (std0 + EPS)).astype(np.float32)
    z1 = ((mu1 - thr_raw) / (std1 + EPS)).astype(np.float32)

    harmonic = (2.0 / (1.0 / (z0 + EPS) + 1.0 / (z1 + EPS))).astype(np.float32)
    abs_diff = np.abs(z0 - z1).astype(np.float32)
    hmg = (harmonic - 0.25 * abs_diff).astype(np.float32)

    # Scenario-level features (threshold-independent)
    signal = first_mean  # [B, 1]
    isi = past_means.sum(axis=1, keepdims=True)  # [B, 1]
    nsid = ((signal - isi) / (signal + isi + EPS)).astype(np.float32)

    gap = signal
    d0_inf = (gap / (std0 + EPS)).astype(np.float32)
    d1_inf = (gap / (std1 + EPS)).astype(np.float32)
    hd = (2.0 * d0_inf * d1_inf / (d0_inf + d1_inf + EPS)).astype(np.float32)
    log_hd = np.log10(hd + EPS).astype(np.float32)
    da = (np.abs(d0_inf - d1_inf) / (d0_inf + d1_inf + EPS)).astype(np.float32)

    past_taps_sum = past_means.sum(axis=1, keepdims=True)
    past_shares = past_means / (past_taps_sum + EPS)
    herf = (past_shares ** 2).sum(axis=1, keepdims=True).astype(np.float32)

    # Scale first token
    ft_tap = ((taps_feat[:, 0] - scalers["first_tap_mean"]) / scalers["first_tap_std"]).astype(np.float32)
    ft_var = ((vars_feat[:, 0] - scalers["first_var_mean"]) / scalers["first_var_std"]).astype(np.float32)
    ft_abs = ((abs_feat[:, 0] - scalers["first_abs_mean"]) / scalers["first_abs_std"]).astype(np.float32)
    ft_snr = ((snr_feat[:, 0] - scalers["first_snr_mean"]) / scalers["first_snr_std"]).astype(np.float32)
    first_token = np.stack([ft_tap, ft_var, ft_abs, ft_snr], axis=1).astype(np.float32)

    # Scale past tokens (position-independent shared scaler)
    pt_tap = apply_shared_scale(taps_feat[:, 1:], scalers["past_tap_mean"], scalers["past_tap_std"], valid_past)
    pt_var = apply_shared_scale(vars_feat[:, 1:], scalers["past_var_mean"], scalers["past_var_std"], valid_past)
    pt_abs = apply_shared_scale(abs_feat[:, 1:], scalers["past_abs_mean"], scalers["past_abs_std"], valid_past)
    pt_snr = apply_shared_scale(snr_feat[:, 1:], scalers["past_snr_mean"], scalers["past_snr_std"], valid_past)
    past_tokens = np.stack([pt_tap, pt_var, pt_abs, pt_snr], axis=2).astype(np.float32)

    # Scale globals (7 features)
    z0_s = scalers["z0_scaler"].transform(z0).astype(np.float32)
    z1_s = scalers["z1_scaler"].transform(z1).astype(np.float32)
    hmg_s = scalers["harmonic_minus_gap_scaler"].transform(hmg).astype(np.float32)
    nsid_s = scalers["nsid_scaler"].transform(nsid).astype(np.float32)
    log_hd_s = scalers["log_hd_scaler"].transform(log_hd).astype(np.float32)
    da_s = scalers["da_scaler"].transform(da).astype(np.float32)
    herf_s = scalers["herf_scaler"].transform(herf).astype(np.float32)
    global_feats = np.concatenate([z0_s, z1_s, hmg_s, nsid_s, log_hd_s, da_s, herf_s], axis=1).astype(np.float32)

    # Scale threshold
    thr_log = np.log10(thr_raw + EPS).astype(np.float32)
    thr_s = scalers["thr_scaler"].transform(thr_log).astype(np.float32)

    # Padding mask
    pad_mask = (np.arange(L_past)[None, :] >= past_lens[:, None])

    return (
        torch.from_numpy(first_token),
        torch.from_numpy(past_tokens),
        torch.from_numpy(global_feats),
        torch.from_numpy(thr_s),
        torch.from_numpy(pad_mask),
    )


def run_inference(model, taps_raw, vars_raw, N_arr, thr_arr, mem_len_arr, scalers, device):
    """End-to-end inference: raw arrays -> BER predictions.

    All inputs are numpy. Works with any mem_len, even values
    larger than max_past_seq_len seen during training.
    """
    first_tok, past_tok, glob, thr_s, pad_mask = prepare_inference_batch(
        taps_raw, vars_raw, N_arr, thr_arr, mem_len_arr, scalers,
    )

    model.eval()
    with torch.no_grad():
        first_tok = first_tok.to(device)
        past_tok = past_tok.to(device)
        glob = glob.to(device)
        thr_s = thr_s.to(device)
        pad_mask = pad_mask.to(device)

        pred_raw, ord_logits = model(first_tok, past_tok, glob, thr_s, key_padding_mask=pad_mask)

        pred_ber = model.raw_to_ber(pred_raw).cpu().numpy()
        pred_log = model.raw_to_log10ber(pred_raw).cpu().numpy()
        pred_region = ordinal_logits_to_region_labels_torch(ord_logits).cpu().numpy()

    return {
        "ber": pred_ber,
        "log10_ber": pred_log,
        "region": pred_region,
    }


# =========================
# Evaluation
# =========================
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_reg_loss = 0.0
    total_ord_loss = 0.0

    all_preds_log = []
    all_targets_log = []
    all_pred_regions = []
    all_true_regions = []

    with torch.no_grad():
        for b_first, b_past, b_global, b_thr, b_y_log, b_ord, b_region, b_mask in loader:
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_y_log = b_y_log.to(device, non_blocking=True)
            b_ord = b_ord.to(device, non_blocking=True)
            b_region = b_region.to(device, non_blocking=True)
            b_mask = b_mask.to(device, non_blocking=True)

            pred_raw_unconstrained, ord_logits = model(
                b_first, b_past, b_global, b_thr, key_padding_mask=b_mask
            )

            if has_nonfinite_tensor(pred_raw_unconstrained):
                raise RuntimeError("Non-finite prediction during evaluation.")
            if has_nonfinite_tensor(ord_logits):
                raise RuntimeError("Non-finite ordinal logits during evaluation.")

            loss, reg_loss, ord_loss = criterion(
                pred_raw_unconstrained, ord_logits, b_y_log, b_ord,
            )

            if has_nonfinite_tensor(loss):
                raise RuntimeError("Non-finite loss during evaluation.")

            pred_log = model.raw_to_log10ber(pred_raw_unconstrained)
            pred_region = ordinal_logits_to_region_labels_torch(ord_logits)

            total_loss += loss.item()
            total_reg_loss += reg_loss.item()
            total_ord_loss += ord_loss.item()

            all_preds_log.append(pred_log.cpu().numpy())
            all_targets_log.append(b_y_log.cpu().numpy())
            all_pred_regions.append(pred_region.cpu().numpy())
            all_true_regions.append(b_region.cpu().numpy())

    n_batches = max(len(loader), 1)
    avg_loss = total_loss / n_batches
    avg_reg_loss = total_reg_loss / n_batches
    avg_ord_loss = total_ord_loss / n_batches

    preds_log = np.vstack(all_preds_log)
    targets_log = np.vstack(all_targets_log)

    preds_raw = np.clip(10 ** preds_log, EPS, 0.5)
    targets_raw = np.clip(10 ** targets_log, EPS, 0.5)

    rmse_log = float(np.sqrt(np.mean((preds_log - targets_log) ** 2)))
    mae_log = float(np.mean(np.abs(preds_log - targets_log)))
    factor_error = float(10 ** rmse_log)

    rmse_raw = float(np.sqrt(np.mean((preds_raw - targets_raw) ** 2)))
    mae_raw = float(np.mean(np.abs(preds_raw - targets_raw)))

    rel_err = (preds_raw - targets_raw) / np.maximum(targets_raw, 1e-6)
    rmse_rel = float(np.sqrt(np.mean(rel_err ** 2)))
    mae_rel = float(np.mean(np.abs(rel_err)))

    pred_regions = np.concatenate(all_pred_regions).reshape(-1)
    true_regions = np.concatenate(all_true_regions).reshape(-1)

    region_acc = float(np.mean(pred_regions == true_regions))
    region_mae = float(np.mean(np.abs(pred_regions - true_regions)))

    return {
        "loss": avg_loss,
        "reg_loss": avg_reg_loss,
        "ord_loss": avg_ord_loss,
        "rmse_log": rmse_log,
        "mae_log": mae_log,
        "factor_error": factor_error,
        "rmse_raw": rmse_raw,
        "mae_raw": mae_raw,
        "rmse_rel": rmse_rel,
        "mae_rel": mae_rel,
        "region_acc": region_acc,
        "region_mae": region_mae,
    }


def evaluate_by_target_range(model, loader, device):
    model.eval()
    all_preds_log = []
    all_targets_log = []

    with torch.no_grad():
        for b_first, b_past, b_global, b_thr, b_y_log, _, _, b_mask in loader:
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_mask = b_mask.to(device, non_blocking=True)

            pred_raw_unconstrained, _ = model(
                b_first, b_past, b_global, b_thr, key_padding_mask=b_mask
            )

            if has_nonfinite_tensor(pred_raw_unconstrained):
                raise RuntimeError("Non-finite prediction during per-range evaluation.")

            pred_log = model.raw_to_log10ber(pred_raw_unconstrained)

            all_preds_log.append(pred_log.cpu().numpy())
            all_targets_log.append(b_y_log.numpy())

    preds_log = np.vstack(all_preds_log).reshape(-1)
    targets_log = np.vstack(all_targets_log).reshape(-1)

    preds_raw = np.clip(10 ** preds_log, EPS, 0.5)
    targets_raw = np.clip(10 ** targets_log, EPS, 0.5)

    ranges = {
        "low_BER(y<1e-6)": targets_raw < 1e-6,
        "mid_BER(1e-6<=y<1e-3)": (targets_raw >= 1e-6) & (targets_raw < 1e-3),
        "high_BER(1e-3<=y<0.10)": (targets_raw >= 1e-3) & (targets_raw < 0.10),
        "upper_BER(0.10<=y<0.20)": (targets_raw >= 0.10) & (targets_raw < 0.20),
        "target_BER(0.20<=y<0.40)": (targets_raw >= 0.20) & (targets_raw < 0.40),
        "very_high_BER(0.40<=y<=0.50)": targets_raw >= 0.40,
    }

    metrics = {}
    for name, mask in ranges.items():
        if np.any(mask):
            err_log = preds_log[mask] - targets_log[mask]
            err_raw = preds_raw[mask] - targets_raw[mask]
            rel_err = err_raw / np.maximum(targets_raw[mask], 1e-6)

            rmse_log = float(np.sqrt(np.mean(err_log ** 2)))
            mae_log = float(np.mean(np.abs(err_log)))
            rmse_raw = float(np.sqrt(np.mean(err_raw ** 2)))
            mae_raw = float(np.mean(np.abs(err_raw)))
            rmse_rel = float(np.sqrt(np.mean(rel_err ** 2)))
            mae_rel = float(np.mean(np.abs(rel_err)))

            metrics[name] = {
                "count": int(mask.sum()),
                "rmse_log": rmse_log,
                "mae_log": mae_log,
                "factor_error": float(10 ** rmse_log),
                "rmse_raw": rmse_raw,
                "mae_raw": mae_raw,
                "rmse_rel": rmse_rel,
                "mae_rel": mae_rel,
                "bias_raw": float(np.mean(err_raw)),
                "bias_log": float(np.mean(err_log)),
                "p90_abs_raw": float(np.percentile(np.abs(err_raw), 90)),
                "p95_abs_raw": float(np.percentile(np.abs(err_raw), 95)),
            }
        else:
            metrics[name] = None

    return metrics


# =========================
# Multi-Component Selection Scoring
# =========================
# Selection rationale:
#   BER spans 6+ orders of magnitude (1e-6 to 0.5), so multiplicative error
#   is the natural metric. We use weighted log-RMSE per region (already in log
#   space, so equivalent to weighted geometric mean of factor errors), then
#   add a bias penalty (also in log space, so multiplicative bias) and a tail
#   penalty (P95/RMSE ratio in raw space, capped, to penalize heavy tails).

# Weights for each BER region in the composite score.
# Higher weight = more important to get this region right.
SELECTION_REGION_WEIGHTS = {
    "low_BER(y<1e-6)": 0.25,
    "mid_BER(1e-6<=y<1e-3)": 0.50,
    "high_BER(1e-3<=y<0.10)": 1.00,
    "upper_BER(0.10<=y<0.20)": 2.00,
    "target_BER(0.20<=y<0.40)": 3.00,
    "very_high_BER(0.40<=y<=0.50)": 1.50,
}

# Composite weights: how much each component contributes
SELECTION_W_LOG_RMSE = 1.00   # Primary: per-region log-RMSE
SELECTION_W_BIAS = 0.50       # Secondary: bias in log space (factor bias)
SELECTION_W_TAIL = 0.05       # Tertiary: P95/RMSE ratio (heavy tails)
SELECTION_TAIL_CAP = 5.0      # Cap tail ratio so one bad region doesn't dominate


def compute_selection_score(val_range_metrics, val_metrics, min_count=50):
    """Compute a multi-component selection score, lower is better.

    Components:
      1. Weighted average of per-region RMSE(log10).
         (This is equivalent to a weighted geometric mean of factor errors.)
      2. Weighted average of |bias_log| per region (multiplicative bias).
      3. Weighted average of capped P95/RMSE ratio per region (tail penalty).

    Returns a dict with all components and the composite.
    """
    log_rmse_sum = 0.0
    bias_sum = 0.0
    tail_sum = 0.0
    total_w = 0.0

    per_region_factor = {}

    for region_name, w in SELECTION_REGION_WEIGHTS.items():
        m = val_range_metrics.get(region_name)
        if m is None or m["count"] < min_count:
            continue

        # 1. Log-RMSE component (already log-space)
        log_rmse_sum += w * m["rmse_log"]

        # 2. Bias component in log space (multiplicative bias)
        bias_sum += w * abs(m["bias_log"])

        # 3. Tail component: P95 / RMSE in raw space, capped
        if m["rmse_raw"] > 1e-9:
            tail_ratio = m["p95_abs_raw"] / (m["rmse_raw"] + 1e-9)
            tail_sum += w * min(tail_ratio, SELECTION_TAIL_CAP)
        else:
            tail_sum += w * 1.0

        total_w += w
        per_region_factor[region_name] = m["factor_error"]

    if total_w == 0:
        # Fallback to global metrics
        return {
            "composite": float(val_metrics["rmse_log"]),
            "log_rmse_weighted": float(val_metrics["rmse_log"]),
            "bias_weighted": 0.0,
            "tail_weighted": 0.0,
            "geometric_factor_error": float(val_metrics["factor_error"]),
            "fallback": True,
        }

    log_rmse_weighted = log_rmse_sum / total_w
    bias_weighted = bias_sum / total_w
    tail_weighted = tail_sum / total_w

    composite = (
        SELECTION_W_LOG_RMSE * log_rmse_weighted
        + SELECTION_W_BIAS * bias_weighted
        + SELECTION_W_TAIL * tail_weighted
    )

    # The geometric-mean factor error: 10^(weighted log_rmse)
    # Reported for human readability — "this model is off by ~Nx on average"
    geometric_factor = float(10 ** log_rmse_weighted)

    return {
        "composite": float(composite),
        "log_rmse_weighted": float(log_rmse_weighted),
        "bias_weighted": float(bias_weighted),
        "tail_weighted": float(tail_weighted),
        "geometric_factor_error": geometric_factor,
        "fallback": False,
    }


def is_acceptable_checkpoint(val_range_metrics, val_metrics):
    """Hard constraints: model must meet these to be considered for 'best'.

    These prevent pathological epochs from being selected just because
    they happen to score well on the composite (e.g., if a region has
    almost no samples and dominates by luck).
    """
    target = val_range_metrics.get("target_BER(0.20<=y<0.40)")
    if target is None or target["count"] < 100:
        return False, "target region too small"

    # No region's bias_log can exceed 0.15 (i.e., factor bias of ~1.4x)
    for region_name, m in val_range_metrics.items():
        if m is None or m["count"] < 50:
            continue
        if abs(m["bias_log"]) > 0.15:
            return False, f"{region_name} has bias_log={m['bias_log']:.3f}"

    # Overall log RMSE must be reasonable
    if val_metrics["rmse_log"] > 0.30:
        return False, f"overall rmse_log={val_metrics['rmse_log']:.3f} too high"

    return True, "ok"


# =========================
# Training
# =========================
def train_engine():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Executing on: {device}")

    train_loader, val_loader, test_loader, scalers, aux_info, max_past_seq_len = prepare_data(
        DATA_PATHS,
        batch_size=256,
        nrows_per_dataset=NROWS_PER_DATASET,
        num_workers=0,
    )

    joblib.dump(scalers, SCALER_SAVE_PATH)
    print(f"Scalers saved to: {SCALER_SAVE_PATH}")

    model = BERFirstTokenSetTransformerBoundedLogHalfMultiTask(
        max_past_seq_len=max_past_seq_len,
        token_dim=4,
        first_token_dim=4,
        threshold_dim=1,
        global_dim=7,
        d_model=128,
        num_set_layers=4,
        num_heads=4,
        mlp_ratio=4.0,
        dropout=0.10,
        num_ordinal_thresholds=NUM_ORDINAL_THRESHOLDS,
    ).to(device)

    print("Training target-focused first-token + set-transformer multitask model...")
    print(f"  Variable-length support: ENABLED (position-independent scaling)")
    print(f"  Max past sequence length (training): {max_past_seq_len}")
    print(f"  Scaling strategy: separate first-token / shared past-token scalers")

    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

    reg_loss = StableMultiObjectiveBERBoundedLogLoss(
        log_delta=0.35,
        raw_delta=0.006,
        rel_delta=0.03,
        alpha_log=0.45,
        beta_raw=0.35,
        gamma_rel=0.20,
        use_regime_weights=True,
    )

    boosted_pos = aux_info["ordinal_pos_weights"].copy()
    for i, thr in enumerate(ORDINAL_THRESHOLDS):
        if 0.20 <= thr <= 0.40:
            boosted_pos[i] *= 2.5
        elif 0.15 <= thr < 0.20:
            boosted_pos[i] *= 1.5
        elif 0.40 < thr <= 0.45:
            boosted_pos[i] *= 1.5

    ord_loss = OrdinalBCELoss(
        pos_weight=torch.tensor(boosted_pos, dtype=torch.float32, device=device),
        reduction="mean",
    )

    criterion = MultiTaskBERLoss(
        reg_loss=reg_loss,
        ord_loss=ord_loss,
        lambda_ord=0.25,
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=4, factor=0.5,
    )

    # ---- Multi-component selection state ----
    # Track multiple "best" checkpoints under different criteria.
    # We'll save all of them and the user can choose the right tradeoff at inference.
    best_checkpoints = {
        "composite": {"score": float("inf"), "state": None, "epoch": -1},   # primary: composite score
        "composite_ema": {"score": float("inf"), "state": None, "epoch": -1},  # EMA-smoothed composite
        "log_rmse": {"score": float("inf"), "state": None, "epoch": -1},    # weighted log-RMSE only
        "target_mae": {"score": float("inf"), "state": None, "epoch": -1},  # target region MAE
        "low_bias": {"score": float("inf"), "state": None, "epoch": -1},    # min weighted |bias_log|
    }

    # EMA smoothing of the composite score to reduce epoch-to-epoch noise
    EMA_ALPHA = 0.5
    ema_composite = None

    patience = 12
    wait = 0
    min_epochs_before_early_stop = 60
    max_epochs = 120

    print(f"Ordinal thresholds: {ORDINAL_THRESHOLDS}")
    print(f"Boosted ordinal pos weights: {boosted_pos}")
    print(f"Selection: weighted log-RMSE (geometric factor mean) + bias + tail penalties")
    print(f"  Region weights: {SELECTION_REGION_WEIGHTS}")
    print(f"  Composite weights: log_rmse={SELECTION_W_LOG_RMSE}, "
          f"bias={SELECTION_W_BIAS}, tail={SELECTION_W_TAIL}")
    print(f"  EMA alpha for composite smoothing: {EMA_ALPHA}")

    training_broke = False

    for epoch in range(max_epochs):
        model.train()

        for batch_idx, (b_first, b_past, b_global, b_thr, b_y_log, b_ord, _, b_mask) in enumerate(train_loader):
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_y_log = b_y_log.to(device, non_blocking=True)
            b_ord = b_ord.to(device, non_blocking=True)
            b_mask = b_mask.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            pred_raw_unconstrained, ord_logits = model(
                b_first, b_past, b_global, b_thr, key_padding_mask=b_mask,
            )

            if has_nonfinite_tensor(pred_raw_unconstrained):
                print(f"Non-finite prediction at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True
                break

            if has_nonfinite_tensor(ord_logits):
                print(f"Non-finite ordinal logits at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True
                break

            loss, reg_part, ord_part = criterion(
                pred_raw_unconstrained, ord_logits, b_y_log, b_ord,
            )

            if has_nonfinite_tensor(loss):
                print(f"Non-finite loss at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True
                break

            loss.backward()

            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            if not torch.isfinite(grad_norm):
                print(f"Non-finite gradient norm at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True
                break

            optimizer.step()

            bad_param = False
            for name, param in model.named_parameters():
                if param.requires_grad and param.data is not None and not torch.isfinite(param.data).all():
                    print(f"Non-finite parameter after optimizer step: {name}")
                    bad_param = True
                    break

            if bad_param:
                training_broke = True
                break

        if training_broke:
            print("Training stopped due to non-finite values.")
            break

        try:
            train_metrics = evaluate(model, train_loader, criterion, device)
            val_metrics = evaluate(model, val_loader, criterion, device)
            val_range_metrics = evaluate_by_target_range(model, val_loader, device)
        except RuntimeError as e:
            print(f"Evaluation failed at epoch {epoch+1}: {e}")
            break

        scheduler.step(val_metrics["loss"])
        current_lr = optimizer.param_groups[0]["lr"]

        # ---- Compute multi-component selection score ----
        sel = compute_selection_score(val_range_metrics, val_metrics)
        composite = sel["composite"]

        # EMA-smoothed composite (reduces noise from single bad/lucky epochs)
        if ema_composite is None:
            ema_composite = composite
        else:
            ema_composite = EMA_ALPHA * composite + (1.0 - EMA_ALPHA) * ema_composite

        # Components for individual best-checkpoints
        target_key = "target_BER(0.20<=y<0.40)"
        target_mae = (
            val_range_metrics[target_key]["mae_raw"]
            if val_range_metrics[target_key] is not None
            else float("inf")
        )

        # Acceptability gate
        is_acceptable, reason = is_acceptable_checkpoint(val_range_metrics, val_metrics)

        # ---- Logging ----
        print(
            f"Epoch {epoch+1:03d} | "
            f"LR: {current_lr:.2e} | "
            f"Train Loss: {train_metrics['loss']:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} | "
            f"Val RMSE(log10): {val_metrics['rmse_log']:.4f} "
            f"(~{val_metrics['factor_error']:.3f}x) | "
            f"Val Region Acc: {val_metrics['region_acc']:.4f}"
        )
        print(
            f"  SELECTION | composite={composite:.5f} | "
            f"ema_composite={ema_composite:.5f} | "
            f"weighted log-RMSE={sel['log_rmse_weighted']:.5f} "
            f"(~{sel['geometric_factor_error']:.3f}x avg factor) | "
            f"weighted |bias_log|={sel['bias_weighted']:.5f} | "
            f"tail penalty={sel['tail_weighted']:.3f} | "
            f"acceptable={is_acceptable}"
            + ("" if is_acceptable else f" ({reason})")
        )

        # Per-range tail metrics
        for rng_name, rng_stats in val_range_metrics.items():
            if rng_stats is not None:
                print(
                    f"    {rng_name}: "
                    f"n={rng_stats['count']} | "
                    f"factor~{rng_stats['factor_error']:.3f}x | "
                    f"RMSE_log={rng_stats['rmse_log']:.4f} | "
                    f"MAE_raw={rng_stats['mae_raw']:.6f} | "
                    f"P90={rng_stats['p90_abs_raw']:.6f} | "
                    f"P95={rng_stats['p95_abs_raw']:.6f} | "
                    f"bias_raw={rng_stats['bias_raw']:+.6f} | "
                    f"bias_log={rng_stats['bias_log']:+.4f}"
                )

        # ---- Update each best-checkpoint (only if acceptable) ----
        improved_any = False

        if is_acceptable:
            current_state_snapshot = None  # lazy deepcopy

            checkpoint_candidates = [
                ("composite", composite),
                ("composite_ema", ema_composite),
                ("log_rmse", sel["log_rmse_weighted"]),
                ("target_mae", target_mae),
                ("low_bias", sel["bias_weighted"]),
            ]

            for ckpt_name, score in checkpoint_candidates:
                if score < best_checkpoints[ckpt_name]["score"]:
                    if current_state_snapshot is None:
                        current_state_snapshot = copy.deepcopy(model.state_dict())
                    best_checkpoints[ckpt_name] = {
                        "score": float(score),
                        "state": current_state_snapshot,
                        "epoch": epoch + 1,
                    }
                    improved_any = True
                    print(f"  -> New best [{ckpt_name}] at epoch {epoch+1}: {score:.6f}")

        # ---- Early stopping driven by EMA composite ----
        # We use the EMA score so we don't stop on a single noisy spike
        ema_best = best_checkpoints["composite_ema"]["score"]
        if ema_composite < ema_best + 1e-9 or improved_any:
            wait = 0
        else:
            if epoch + 1 >= min_epochs_before_early_stop:
                wait += 1
                if wait >= patience:
                    print(f"Early stopping triggered after {wait} non-improving epochs (EMA basis).")
                    break

    # ---- Save last model ----
    torch.save(model.state_dict(), LAST_MODEL_SAVE_PATH)
    print(f"\nLast model saved to: {LAST_MODEL_SAVE_PATH}")

    # ---- Save all best checkpoints ----
    print("\n" + "=" * 80)
    print("BEST CHECKPOINTS SUMMARY")
    print("=" * 80)
    for ckpt_name, info in best_checkpoints.items():
        if info["state"] is None:
            print(f"  [{ckpt_name:>14s}] never updated")
            continue

        path = BEST_MODEL_SAVE_PATH.replace(".pth", f"_{ckpt_name}.pth")
        torch.save(info["state"], path)
        print(
            f"  [{ckpt_name:>14s}] epoch={info['epoch']:>3d} | "
            f"score={info['score']:.6f} | saved -> {path}"
        )

    # ---- Choose primary best for evaluation: composite_ema is most stable ----
    primary_choice = "composite_ema"
    if best_checkpoints[primary_choice]["state"] is None:
        # Fall back to composite if EMA never updated (shouldn't happen but safe)
        primary_choice = "composite"
    if best_checkpoints[primary_choice]["state"] is None:
        # Fall back to log_rmse
        primary_choice = "log_rmse"

    if best_checkpoints[primary_choice]["state"] is not None:
        model.load_state_dict(best_checkpoints[primary_choice]["state"])
        torch.save(model.state_dict(), BEST_MODEL_SAVE_PATH)
        print(
            f"\nPrimary best ({primary_choice}, epoch "
            f"{best_checkpoints[primary_choice]['epoch']}) "
            f"saved to: {BEST_MODEL_SAVE_PATH}"
        )
    else:
        print("\nWarning: no valid best checkpoint was found.")

    test_metrics = evaluate(model, test_loader, criterion, device)

    print(
        f"Test Loss: {test_metrics['loss']:.4f} | "
        f"Test Reg Loss: {test_metrics['reg_loss']:.4f} | "
        f"Test Ord Loss: {test_metrics['ord_loss']:.4f} | "
        f"Test RMSE(log10): {test_metrics['rmse_log']:.4f} | "
        f"Test MAE(log10): {test_metrics['mae_log']:.4f} | "
        f"Typical multiplicative error: ~{test_metrics['factor_error']:.2f}x | "
        f"Test RMSE(raw): {test_metrics['rmse_raw']:.6f} | "
        f"Test MAE(raw): {test_metrics['mae_raw']:.6f} | "
        f"Test RMSE(rel): {test_metrics['rmse_rel']:.6f} | "
        f"Test MAE(rel): {test_metrics['mae_rel']:.6f} | "
        f"Test Region Acc: {test_metrics['region_acc']:.4f} | "
        f"Test Region MAE: {test_metrics['region_mae']:.4f}"
    )

    range_metrics = evaluate_by_target_range(model, test_loader, device)
    print("\nPer-range test diagnostics:")
    for name, stats in range_metrics.items():
        if stats is None:
            print(f"  {name}: no samples")
        else:
            print(
                f"  {name} | count={stats['count']} | "
                f"RMSE(log10)={stats['rmse_log']:.4f} | "
                f"MAE(log10)={stats['mae_log']:.4f} | "
                f"factor~{stats['factor_error']:.2f}x | "
                f"RMSE(raw)={stats['rmse_raw']:.6f} | "
                f"MAE(raw)={stats['mae_raw']:.6f} | "
                f"RMSE(rel)={stats['rmse_rel']:.6f} | "
                f"MAE(rel)={stats['mae_rel']:.6f} | "
                f"bias_raw={stats['bias_raw']:.6f} | "
                f"bias_log={stats['bias_log']:.6f} | "
                f"P90={stats['p90_abs_raw']:.6f} | "
                f"P95={stats['p95_abs_raw']:.6f}"
            )

    return model


if __name__ == "__main__":
    os.makedirs(BASE_DIR, exist_ok=True)

    missing = [p for p in DATA_PATHS if not os.path.exists(p)]
    if missing:
        print("Critical Error: Missing dataset files:")
        for p in missing:
            print(f"  - {p}")
    else:
        print("Using training from datasets:")
        for p in DATA_PATHS:
            print(f"  - {p}")
        print(f"Row cap per dataset: {NROWS_PER_DATASET:,}")

        trained_model = train_engine()

Using training from datasets:
  - ./../ber_data_generation/data/data_physics_total.csv
  - ./../ber_data_generation/data/data_random_total.csv
Row cap per dataset: 5,000,000
Executing on: cuda
Loading up to 5,000,000 rows from: ./../ber_data_generation/data/data_physics_total.csv


ParserError: Error tokenizing data. C error: Expected 25 fields in line 3630711, saw 36


In [13]:
import os
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.special import erfc
from itertools import product as iproduct

EPS = 1e-12
PLOT_DIR = "./plots_mixed_v1"
os.makedirs(PLOT_DIR, exist_ok=True)

# =========================
# Paths
# =========================
MODEL_PATH = "random_extra_multitask_best2_log_rmse.pth"
SCALER_PATH = "random_extra_multitask_scalers2.pkl"

# =========================
# Config
# =========================
PHYSICS_MAX_MEM_LEN = 14
PHYSICS_MIN_MEM_LEN = 14
ARRIVAL_COVERAGE = 0.70
N_THRESHOLDS = 500
RANDOM_SEED = 60

ORDINAL_THRESHOLDS = [
    1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1,
    0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45,
]
NUM_ORDINAL_THRESHOLDS = len(ORDINAL_THRESHOLDS)
NUM_REGION_CLASSES = NUM_ORDINAL_THRESHOLDS + 1

REGION_LABELS = {
    0: "y < 1e-6",
    1: "1e-6 <= y < 1e-5",
    2: "1e-5 <= y < 1e-4",
    3: "1e-4 <= y < 1e-3",
    4: "1e-3 <= y < 1e-2",
    5: "1e-2 <= y < 1e-1",
    6: "0.10 <= y < 0.15",
    7: "0.15 <= y < 0.20",
    8: "0.20 <= y < 0.25",
    9: "0.25 <= y < 0.30",
    10: "0.30 <= y < 0.35",
    11: "0.35 <= y < 0.40",
    12: "0.40 <= y < 0.45",
    13: "0.45 <= y <= 0.50",
}


# =========================
# Physics helpers
# =========================
def Fhit_function(radius, distance, diffusionCoef, t):
    if t <= 0:
        return 0.0
    return (radius / (distance + radius)) * erfc(distance / np.sqrt(4 * diffusionCoef * t))


def calculate_hitting_probabilities(mem_len, radius, distance, diffusionCoef, Ts):
    P = np.zeros(mem_len)
    for i in range(mem_len):
        t_end = (i + 1) * Ts
        t_start = i * Ts
        P[i] = Fhit_function(radius, distance, diffusionCoef, t_end) - Fhit_function(
            radius, distance, diffusionCoef, t_start
        )
    return P


def calculate_ber_vectorized(mem_len, threshold, P_scaled, variances):
    P_arr = np.asarray(P_scaled, dtype=float)[:mem_len]
    vars_arr = np.asarray(variances, dtype=float)[:mem_len]

    seqs = np.array(list(iproduct([0, 1], repeat=mem_len)), dtype=np.float64)[:, ::-1]
    c_bit = seqs[:, 0]

    mu = (seqs * P_arr).sum(axis=1)
    var_total = (seqs * vars_arr).sum(axis=1)
    std = np.sqrt(np.maximum(var_total, 0.0))

    pe = np.empty_like(mu)
    zero_std = (std == 0)
    if np.any(zero_std):
        pe[zero_std & (c_bit == 1)] = np.where(
            mu[zero_std & (c_bit == 1)] < threshold, 1.0, 0.0)
        pe[zero_std & (c_bit == 0)] = np.where(
            mu[zero_std & (c_bit == 0)] >= threshold, 1.0, 0.0)
    nz = ~zero_std
    if np.any(nz):
        pe[nz & (c_bit == 1)] = 0.5 * erfc(
            (mu[nz & (c_bit == 1)] - threshold) / (std[nz & (c_bit == 1)] * np.sqrt(2)))
        pe[nz & (c_bit == 0)] = 0.5 * erfc(
            (threshold - mu[nz & (c_bit == 0)]) / (std[nz & (c_bit == 0)] * np.sqrt(2)))
    return float(np.mean(pe))


def raw_to_region_labels_np(y_raw):
    y = np.asarray(y_raw).reshape(-1)
    return np.searchsorted(np.asarray(ORDINAL_THRESHOLDS), y, side="right").astype(np.int64)


def ordinal_logits_to_region_labels_torch(logits):
    probs = torch.sigmoid(logits)
    passed = (probs >= 0.5).sum(dim=1)
    return passed.long()


# =========================
# Shared scaling helper
# =========================
def apply_shared_scale(data, mean, std, valid_mask=None):
    scaled = ((data - mean) / std).astype(np.float32)
    if valid_mask is not None and scaled.ndim == 2 and valid_mask.shape == scaled.shape:
        scaled = scaled * valid_mask.astype(np.float32)
    return scaled


# =========================
# Generate one physical scenario
# =========================
def generate_physical_case(rng, min_mem_len, max_mem_len, arrival_coverage, max_tries=5000):
    for _ in range(max_tries):
        radius = rng.uniform(3.0, 5.0)
        distance = rng.uniform(10.0, 15.0)
        diff = rng.uniform(50.0, 75.0)
        Ts = rng.uniform(0.5, 1.2)
        N = int(10 ** rng.uniform(3.0, 6.0))

        f_inf = radius / (radius + distance)
        target = arrival_coverage * f_inf

        cumsum, k = 0.0, 0
        while k < max_mem_len:
            pk = Fhit_function(radius, distance, diff, (k + 1) * Ts) - Fhit_function(
                radius, distance, diff, k * Ts)
            cumsum += pk
            k += 1
            if cumsum >= target:
                break

        if k < min_mem_len:
            continue

        P_ext = calculate_hitting_probabilities(k + 1, radius, distance, diff, Ts)
        P_main = P_ext[:k]
        P_extra = float(P_ext[k]) if k < len(P_ext) else 0.0

        P_scaled = P_main * N
        variances = N * P_main * (1.0 - P_main)

        return {
            "radius": radius, "distance": distance, "diffusion": diff,
            "Ts": Ts, "N": N, "mem_len": k, "P": P_main,
            "P_scaled": P_scaled, "variances": variances,
            "P_mem_len_extra": P_extra,
            "P_mem_len_extra_var": P_extra * (1.0 - P_extra),
        }

    raise RuntimeError(
        f"Could not generate a physical case with mem_len >= {min_mem_len} "
        f"after {max_tries} tries."
    )


# =========================
# Model (matches v3 training: global_dim=7, global_embed=96, cond_dim=128)
# =========================
class ConditionalFeatureModulation(nn.Module):
    def __init__(self, cond_dim, feat_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, 2 * feat_dim),
        )
        self.scale = nn.Parameter(torch.tensor(0.25, dtype=torch.float32))

    def forward(self, x, cond):
        gamma_beta = self.net(cond)
        gamma, beta = torch.chunk(gamma_beta, 2, dim=-1)
        if x.dim() == 3:
            gamma = gamma.unsqueeze(1)
            beta = beta.unsqueeze(1)
        return x * (1.0 + self.scale * gamma) + self.scale * beta


class SetSelfAttentionBlock(nn.Module):
    def __init__(self, d_model, num_heads, mlp_ratio=4.0, dropout=0.10):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=num_heads,
            dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        hidden = int(d_model * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, d_model), nn.Dropout(dropout))

    def forward(self, x, key_padding_mask=None):
        y = self.norm1(x)
        attn_out, _ = self.attn(y, y, y, need_weights=False, key_padding_mask=key_padding_mask)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x


class SetAttentionPooling(nn.Module):
    def __init__(self, d_model, hidden_dim=128):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(d_model, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, 1))

    def forward(self, x, key_padding_mask=None):
        logits = self.score(x)
        if key_padding_mask is not None:
            logits = logits.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
        weights = torch.softmax(logits, dim=1)
        weights = torch.nan_to_num(weights, nan=0.0)
        return (weights * x).sum(dim=1)


class BERFirstTokenSetTransformerBoundedLogHalfMultiTask(nn.Module):
    def __init__(self, max_past_seq_len, token_dim=4, first_token_dim=4,
                 threshold_dim=1, global_dim=7, d_model=128, num_set_layers=4,
                 num_heads=4, mlp_ratio=4.0, dropout=0.10,
                 num_ordinal_thresholds=NUM_ORDINAL_THRESHOLDS):
        super().__init__()
        self.max_past_seq_len = max_past_seq_len
        self.d_model = d_model
        self.num_ordinal_thresholds = num_ordinal_thresholds

        self.first_proj = nn.Sequential(
            nn.Linear(first_token_dim, d_model), nn.LayerNorm(d_model),
            nn.GELU(), nn.Dropout(dropout))
        self.token_proj = nn.Sequential(
            nn.Linear(token_dim, d_model), nn.LayerNorm(d_model),
            nn.GELU(), nn.Dropout(dropout))
        self.thr_embed = nn.Sequential(
            nn.Linear(threshold_dim, 32), nn.GELU(),
            nn.Linear(32, 32), nn.GELU())
        self.global_embed = nn.Sequential(
            nn.Linear(global_dim, 96), nn.GELU(),
            nn.Linear(96, 96), nn.GELU())

        cond_dim = 32 + 96
        self.first_cond_mod = ConditionalFeatureModulation(cond_dim, d_model, 256)
        self.set_cond_mod = ConditionalFeatureModulation(cond_dim, d_model, 256)

        self.set_blocks = nn.ModuleList([
            SetSelfAttentionBlock(d_model, num_heads, mlp_ratio, dropout)
            for _ in range(num_set_layers)])

        self.first_refine = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_model),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model, d_model))
        self.final_set_norm = nn.LayerNorm(d_model)
        self.set_attn_pool = SetAttentionPooling(d_model, 128)

        set_summary_dim = 3 * d_model
        head_in = d_model + set_summary_dim + cond_dim

        self.reg_head = nn.Sequential(
            nn.Linear(head_in, 256), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.15),
            nn.Linear(128, 1))
        self.ord_head = nn.Sequential(
            nn.Linear(head_in, 256), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.15),
            nn.Linear(128, num_ordinal_thresholds))

    @staticmethod
    def raw_to_log10ber(pred_raw_unconstrained):
        log10_half = torch.log10(torch.tensor(
            0.5, device=pred_raw_unconstrained.device,
            dtype=pred_raw_unconstrained.dtype))
        return log10_half - F.softplus(pred_raw_unconstrained)

    @staticmethod
    def raw_to_ber(pred_raw_unconstrained):
        pred_log = BERFirstTokenSetTransformerBoundedLogHalfMultiTask.raw_to_log10ber(
            pred_raw_unconstrained)
        return torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)

    def forward(self, first_token, past_tokens, global_feats, threshold, key_padding_mask=None):
        thr = self.thr_embed(threshold)
        g = self.global_embed(global_feats)
        cond = torch.cat([thr, g], dim=-1)

        first_x = self.first_proj(first_token)
        first_x = self.first_cond_mod(first_x, cond)
        first_x = first_x + self.first_refine(first_x)

        set_x = self.token_proj(past_tokens)
        set_x = self.set_cond_mod(set_x, cond)
        for block in self.set_blocks:
            set_x = block(set_x, key_padding_mask=key_padding_mask)
        set_x = self.final_set_norm(set_x)

        if key_padding_mask is not None:
            valid_mask = (~key_padding_mask).unsqueeze(-1).float()
        else:
            valid_mask = torch.ones(
                set_x.shape[0], set_x.shape[1], 1,
                device=set_x.device, dtype=set_x.dtype)

        pooled_attn = self.set_attn_pool(set_x, key_padding_mask=key_padding_mask)
        valid_counts = valid_mask.sum(dim=1).clamp(min=1.0)
        pooled_mean = (set_x * valid_mask).sum(dim=1) / valid_counts

        if key_padding_mask is not None:
            x_for_max = set_x.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
        else:
            x_for_max = set_x
        pooled_max = x_for_max.amax(dim=1)
        pooled_max = torch.nan_to_num(pooled_max, nan=0.0, posinf=0.0, neginf=0.0)

        set_summary = torch.cat([pooled_attn, pooled_mean, pooled_max], dim=-1)
        fused = torch.cat([first_x, set_summary, cond], dim=-1)

        pred_raw_unconstrained = self.reg_head(fused)
        ord_logits = self.ord_head(fused)
        return pred_raw_unconstrained, ord_logits


# =========================
# Build inference inputs (7 global features, position-independent scaling)
# =========================
def prepare_inference_features(case, thresholds, scalers, device):
    """Build model inputs for a threshold sweep over a single physical case.

    Returns:
        first_t, past_t, global_t, thr_t, mask_t — all torch tensors on device
        global_t has shape [B, 7]: [z0, z1, hmg, nsid, log_hd, da, herf]
    """
    B = len(thresholds)
    mem_len = case["mem_len"]
    N = float(case["N"])

    P_raw = np.asarray(case["P"], dtype=np.float32)
    var_raw = np.asarray(case["variances"], dtype=np.float32)
    thr_raw = np.asarray(thresholds, dtype=np.float32).reshape(-1, 1)

    # Feature engineering
    taps_feat = (P_raw * N).astype(np.float32)
    vars_feat = var_raw.astype(np.float32)
    abs_feat = np.abs(taps_feat).astype(np.float32)
    snr_feat = np.log10((taps_feat ** 2) / (var_raw + EPS) + EPS).astype(np.float32)

    # Broadcast to [B, mem_len]
    taps_2d = np.broadcast_to(taps_feat[None, :], (B, mem_len)).copy()
    vars_2d = np.broadcast_to(vars_feat[None, :], (B, mem_len)).copy()
    abs_2d = np.broadcast_to(abs_feat[None, :], (B, mem_len)).copy()
    snr_2d = np.broadcast_to(snr_feat[None, :], (B, mem_len)).copy()

    L_past = mem_len - 1
    valid_past = np.ones((B, L_past), dtype=bool)

    # --- Threshold-dependent global features ---
    first_mean = taps_2d[:, 0:1]
    past_means = taps_2d[:, 1:]
    first_var = vars_2d[:, 0:1]
    past_vars = vars_2d[:, 1:]

    mu0 = (0.5 * past_means.sum(axis=1, keepdims=True)).astype(np.float32)
    mu1 = (first_mean + mu0).astype(np.float32)
    var0 = (0.5 * past_vars.sum(axis=1, keepdims=True)).astype(np.float32)
    var1 = (first_var + var0).astype(np.float32)
    std0 = np.sqrt(np.maximum(var0, EPS)).astype(np.float32)
    std1 = np.sqrt(np.maximum(var1, EPS)).astype(np.float32)

    z0 = ((thr_raw - mu0) / (std0 + EPS)).astype(np.float32)
    z1 = ((mu1 - thr_raw) / (std1 + EPS)).astype(np.float32)
    harmonic = (2.0 / (1.0 / (z0 + EPS) + 1.0 / (z1 + EPS))).astype(np.float32)
    abs_diff = np.abs(z0 - z1).astype(np.float32)
    hmg = (harmonic - 0.25 * abs_diff).astype(np.float32)

    # --- Scenario-level features (threshold-independent) ---
    signal = first_mean  # [B, 1]
    isi = past_means.sum(axis=1, keepdims=True)
    nsid = ((signal - isi) / (signal + isi + EPS)).astype(np.float32)

    gap = signal
    d0_inf = (gap / (std0 + EPS)).astype(np.float32)
    d1_inf = (gap / (std1 + EPS)).astype(np.float32)
    hd = (2.0 * d0_inf * d1_inf / (d0_inf + d1_inf + EPS)).astype(np.float32)
    log_hd = np.log10(hd + EPS).astype(np.float32)
    da = (np.abs(d0_inf - d1_inf) / (d0_inf + d1_inf + EPS)).astype(np.float32)

    past_taps_sum = past_means.sum(axis=1, keepdims=True)
    past_shares = past_means / (past_taps_sum + EPS)
    herf = (past_shares ** 2).sum(axis=1, keepdims=True).astype(np.float32)

    # --- Scale first token ---
    ft_tap = ((taps_2d[:, 0] - scalers["first_tap_mean"]) / scalers["first_tap_std"]).astype(np.float32)
    ft_var = ((vars_2d[:, 0] - scalers["first_var_mean"]) / scalers["first_var_std"]).astype(np.float32)
    ft_abs = ((abs_2d[:, 0] - scalers["first_abs_mean"]) / scalers["first_abs_std"]).astype(np.float32)
    ft_snr = ((snr_2d[:, 0] - scalers["first_snr_mean"]) / scalers["first_snr_std"]).astype(np.float32)
    first_token = np.stack([ft_tap, ft_var, ft_abs, ft_snr], axis=1).astype(np.float32)

    # --- Scale past tokens ---
    pt_tap = apply_shared_scale(taps_2d[:, 1:], scalers["past_tap_mean"], scalers["past_tap_std"], valid_past)
    pt_var = apply_shared_scale(vars_2d[:, 1:], scalers["past_var_mean"], scalers["past_var_std"], valid_past)
    pt_abs = apply_shared_scale(abs_2d[:, 1:], scalers["past_abs_mean"], scalers["past_abs_std"], valid_past)
    pt_snr = apply_shared_scale(snr_2d[:, 1:], scalers["past_snr_mean"], scalers["past_snr_std"], valid_past)
    past_tokens = np.stack([pt_tap, pt_var, pt_abs, pt_snr], axis=2).astype(np.float32)

    # --- Scale globals (7 features) ---
    z0_s = scalers["z0_scaler"].transform(z0).astype(np.float32)
    z1_s = scalers["z1_scaler"].transform(z1).astype(np.float32)
    hmg_s = scalers["harmonic_minus_gap_scaler"].transform(hmg).astype(np.float32)
    nsid_s = scalers["nsid_scaler"].transform(nsid).astype(np.float32)
    log_hd_s = scalers["log_hd_scaler"].transform(log_hd).astype(np.float32)
    da_s = scalers["da_scaler"].transform(da).astype(np.float32)
    herf_s = scalers["herf_scaler"].transform(herf).astype(np.float32)
    global_feats = np.concatenate(
        [z0_s, z1_s, hmg_s, nsid_s, log_hd_s, da_s, herf_s], axis=1
    ).astype(np.float32)

    # --- Scale threshold ---
    thr_log = np.log10(thr_raw + EPS).astype(np.float32)
    thr_s = scalers["thr_scaler"].transform(thr_log).astype(np.float32)

    # --- Padding mask (no padding needed here) ---
    pad_mask = np.zeros((B, L_past), dtype=bool)

    first_t = torch.from_numpy(first_token).to(device)
    past_t = torch.from_numpy(past_tokens).to(device)
    global_t = torch.from_numpy(global_feats).to(device)
    thr_t = torch.from_numpy(thr_s).to(device)
    mask_t = torch.from_numpy(pad_mask).to(device)

    return first_t, past_t, global_t, thr_t, mask_t


# =========================
# Generate scenario
# =========================
rng = np.random.default_rng(RANDOM_SEED)
case = generate_physical_case(
    rng,
    min_mem_len=PHYSICS_MIN_MEM_LEN,
    max_mem_len=PHYSICS_MAX_MEM_LEN,
    arrival_coverage=ARRIVAL_COVERAGE,
)

print("Generated physical scenario")
print("radius    =", case["radius"])
print("distance  =", case["distance"])
print("diffusion =", case["diffusion"])
print("Ts        =", case["Ts"])
print("N         =", case["N"])
print("mem_len   =", case["mem_len"])
print("P         =", case["P"])
print("P_scaled  =", case["P_scaled"])
print("variances =", case["variances"])

# =========================
# Threshold sweep — ground truth
# =========================
thr_min = 0.0
thr_max = float(np.sum(case["P_scaled"]))
thresholds = np.linspace(thr_min, thr_max, N_THRESHOLDS)

print("Threshold search interval:", thr_min, "to", thr_max)
print("sum(P_scaled) =", np.sum(case["P_scaled"]))

real_bers = np.array([
    calculate_ber_vectorized(
        mem_len=case["mem_len"],
        threshold=thr,
        P_scaled=case["P_scaled"],
        variances=case["variances"],
    )
    for thr in thresholds
])

real_bers = np.clip(real_bers, EPS, 0.5)
real_regions = raw_to_region_labels_np(real_bers)

# =========================
# Load model + scalers
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scalers = joblib.load(SCALER_PATH)

max_past_seq_len = scalers.get("train_max_past_seq_len", scalers.get("max_past_seq_len", case["mem_len"] - 1))
ordinal_thresholds = scalers.get("ordinal_thresholds", ORDINAL_THRESHOLDS)
num_ordinal = len(ordinal_thresholds)
global_dim = scalers.get("global_dim", 7)

model = BERFirstTokenSetTransformerBoundedLogHalfMultiTask(
    max_past_seq_len=max_past_seq_len,
    token_dim=scalers.get("past_token_dim", 4),
    first_token_dim=scalers.get("first_token_dim", 4),
    threshold_dim=1,
    global_dim=global_dim,
    d_model=128,
    num_set_layers=4,
    num_heads=4,
    mlp_ratio=4.0,
    dropout=0.10,
    num_ordinal_thresholds=num_ordinal,
).to(device)

state = torch.load(MODEL_PATH, map_location=device, weights_only=True)
model.load_state_dict(state)
model.eval()

print(f"\nModel loaded: max_past_seq_len={max_past_seq_len}, "
      f"global_dim={global_dim}, num_ordinal={num_ordinal}")
print(f"Scaling strategy: {scalers.get('scaling_strategy', 'unknown')}")

# =========================
# Predict across thresholds
# =========================
first_t, past_t, global_t, thr_t, mask_t = prepare_inference_features(
    case, thresholds, scalers, device,
)

with torch.no_grad():
    pred_raw_out, ord_logits = model(
        first_t, past_t, global_t, thr_t, key_padding_mask=mask_t,
    )
    pred_bers = model.raw_to_ber(pred_raw_out).cpu().numpy().reshape(-1)
    pred_log = model.raw_to_log10ber(pred_raw_out).cpu().numpy().reshape(-1)
    pred_regions = ordinal_logits_to_region_labels_torch(ord_logits).cpu().numpy().reshape(-1)
    pred_region_probs = torch.sigmoid(ord_logits).cpu().numpy()

pred_bers = np.clip(pred_bers, EPS, 0.5)

# =========================
# Comparison table
# =========================
results = pd.DataFrame({
    "threshold": thresholds,
    "real_BER": real_bers,
    "estimated_BER": pred_bers,
    "real_region": real_regions,
    "predicted_region": pred_regions,
    "abs_error": np.abs(pred_bers - real_bers),
    "abs_log10_error": np.abs(
        np.log10(np.clip(pred_bers, EPS, 0.5)) -
        np.log10(np.clip(real_bers, EPS, 0.5))
    ),
    "region_abs_error": np.abs(pred_regions.astype(np.int64) - real_regions.astype(np.int64)),
})

print(results.head(15))

print("\nSummary")
print("Mean abs raw error        :", results["abs_error"].mean())
print("Mean abs log10 error      :", results["abs_log10_error"].mean())
print("Max  abs log10 error      :", results["abs_log10_error"].max())
print("Mean region abs error     :", results["region_abs_error"].mean())
print("Exact region accuracy     :", np.mean(results["real_region"] == results["predicted_region"]))

best_real_idx = np.argmin(real_bers)
best_est_idx = np.argmin(pred_bers)

print("\nBest threshold from real BER      :", thresholds[best_real_idx])
print("Minimum real BER                  :", real_bers[best_real_idx])
print("Real BER region there             :", REGION_LABELS.get(int(real_regions[best_real_idx]), "?"))

print("Best threshold from estimated BER :", thresholds[best_est_idx])
print("Estimated BER at that threshold   :", pred_bers[best_est_idx])
print("Predicted BER region there        :", REGION_LABELS.get(int(pred_regions[best_est_idx]), "?"))

# =========================
# Plot 1: log-scale BER
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["estimated_BER"], label="Estimated BER")
plt.plot(results["threshold"], results["real_BER"], label="Real BER")
plt.yscale("log")
plt.xlabel("Threshold")
plt.ylabel("BER")
plt.title(f"Threshold vs BER (log scale) — mem_len={case['mem_len']}")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "01_ber_log_scale.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/01_ber_log_scale.png")

# =========================
# Plot 2: linear-scale BER
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["estimated_BER"], label="Estimated BER")
plt.plot(results["threshold"], results["real_BER"], label="Real BER")
plt.yscale("linear")
plt.xlabel("Threshold")
plt.ylabel("BER")
plt.title(f"Threshold vs BER (linear scale) — mem_len={case['mem_len']}")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "02_ber_linear_scale.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/02_ber_linear_scale.png")

# =========================
# Plot 3: predicted vs true region
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["predicted_region"], label="Predicted region")
plt.plot(results["threshold"], results["real_region"], label="True region")
plt.xlabel("Threshold")
plt.ylabel("BER Region Class")
plt.title(f"Threshold vs BER Region — mem_len={case['mem_len']}")
plt.yticks(list(REGION_LABELS.keys()),
           [REGION_LABELS[k] for k in sorted(REGION_LABELS.keys())],
           fontsize=7)
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "03_region_comparison.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/03_region_comparison.png")

# =========================
# Plot 4: ordinal threshold probabilities
# =========================
plt.figure(figsize=(12, 7))
for i, thr_val in enumerate(ordinal_thresholds):
    plt.plot(thresholds, pred_region_probs[:, i], label=f"P(y >= {thr_val:g})")
plt.xlabel("Threshold")
plt.ylabel("Ordinal Probability")
plt.title(f"Ordinal Head Outputs Across Threshold Sweep — mem_len={case['mem_len']}")
plt.legend(fontsize=7, ncol=2)
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "04_ordinal_probabilities.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/04_ordinal_probabilities.png")

# =========================
# Plot 5: absolute error by threshold
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["abs_error"], label="Abs raw error", alpha=0.8)
plt.plot(results["threshold"], results["abs_log10_error"], label="Abs log10 error", alpha=0.8)
plt.yscale("log")
plt.xlabel("Threshold")
plt.ylabel("Error")
plt.title(f"Prediction Error Across Threshold Sweep — mem_len={case['mem_len']}")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "05_error_by_threshold.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/05_error_by_threshold.png")

Generated physical scenario
radius    = 4.999903475776163
distance  = 13.063595613360427
diffusion = 63.15707394395149
Ts        = 0.5834416162676823
N         = 2026
mem_len   = 14
P         = [0.03545102 0.042582   0.02704795 0.01857409 0.01368091 0.01059455
 0.00850978 0.00702648 0.00592783 0.00508774 0.00442855 0.00390015
 0.00346895 0.00311165]
P_scaled  = [71.82377081 86.27113641 54.79914164 37.63111579 27.71752004 21.46456524
 17.24082032 14.23564842 12.00979138 10.30776824  8.9722469   7.90171201
  7.0280834   6.304198  ]
variances = [69.27754472 82.59753869 53.31693734 36.93215188 27.33831919 21.23715776
 17.09410468 14.13562193 11.93859933 10.25532496  8.93251283  7.87089412
  7.00370336  6.28458156]
Threshold search interval: 0.0 to 383.7075186062634
sum(P_scaled) = 383.7075186062634

Model loaded: max_past_seq_len=13, global_dim=7, num_ordinal=13
Scaling strategy: position_independent
    threshold  real_BER  estimated_BER  real_region  predicted_region  \
0    0.000000  0.

In [4]:
import os
import re
import copy
import warnings
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.multiprocessing as mp
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

mp.set_sharing_strategy("file_system")

# =========================
# Configuration
# =========================
BASE_DIR = "./"
DATA_DIR = "./../ber_data_generation/data/"

DATA_PATHS = [
    os.path.join(DATA_DIR, "data_physics_total.csv"),
    os.path.join(DATA_DIR, "data_random_total.csv"),
]

NROWS_PER_DATASET = 5_000_000

BEST_MODEL_SAVE_PATH = os.path.join(
    BASE_DIR, "combined_variablemem_masked_robust_best.pth"
)
LAST_MODEL_SAVE_PATH = os.path.join(
    BASE_DIR, "combined_variablemem_masked_robust_last.pth"
)
SCALER_SAVE_PATH = os.path.join(
    BASE_DIR, "combined_variablemem_masked_robust_scalers.pkl"
)

EPS = 1e-12
LOG10_HALF = float(np.log10(0.5))

# If True, a post-first token with both tap ~= 0 and variance ~= 0 is treated as
# non-contributing even when mem_len includes it. This enforces invariance such as
# [0.3, 0.2, 0.1] == [0.3, 0.2, 0.1, 0.0] when the added term has zero variance.
# Keep False for apples-to-apples comparison with the original script.
# Padding is masked by mem_len; numerically zero real tokens are still kept.
MASK_ZERO_PAST_TOKENS = False
ZERO_TAP_EPS = 0.0
ZERO_VAR_EPS = 0.0

# Match the original successful baseline unless you intentionally want otherwise.
DROP_MEM_LEN_ONE = True

# For data_physics_total.csv / data_random_total.csv, mem_len is a count:
# mem_len=4 means source tap_1...tap_4 are valid. After normalization, those
# become internal tap_0...tap_3. Set True only for older datasets where
# mem_len=k meant internal tap_0...tap_k were valid.
MEM_LEN_IS_LAST_VALID_INDEX = False

# Match the original head structure by default: attention + mean + max.
# Masked sum can be useful, but it changes the model family relative to your baseline.
INCLUDE_MASKED_SUM_POOL = False

# Numerical guards for engineered physical features.
# These prevent rare margin cancellations from producing inf before StandardScaler.
HARMONIC_DENOM_EPS = 1e-9
FEATURE_CLIP_ABS = 1e6

# CSV/data compatibility controls.
# The new total CSVs may contain occasional rows with more fields than the header.
# "truncate" keeps the row and discards undeclared extra fields; "skip" drops it.
CSV_BAD_LINE_POLICY = "truncate"

# If the dataset has tap_1...tap_K and no tap_0, normalize internally so tap_1
# becomes tap_0. If tap_0 exists, no shift is applied.
SEQUENCE_INDEX_BASE = "auto"  # "auto", 0, or 1

# If var_i columns are absent, synthesize variance channels from taps.
# "binomial" assumes taps are probabilities and counts follow N*p*(1-p).
# Other options: "poisson" -> N*p, "zeros" -> 0.
MISSING_VARIANCE_POLICY = "binomial"

# 10 ordered BER regions -> 9 ordinal thresholds
ORDINAL_THRESHOLDS = [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 0.2, 0.3, 0.4]
NUM_ORDINAL_THRESHOLDS = len(ORDINAL_THRESHOLDS)
NUM_REGION_CLASSES = NUM_ORDINAL_THRESHOLDS + 1


# =========================
# Utilities
# =========================
def get_sorted_seq_cols(columns, prefix):
    pattern = re.compile(rf"^{re.escape(prefix)}_(\d+)$")
    matched = []
    for col in columns:
        m = pattern.match(col)
        if m:
            matched.append((int(m.group(1)), col))
    matched.sort(key=lambda x: x[0])
    return [col for _, col in matched]


def get_seq_indices(columns, prefix):
    pattern = re.compile(rf"^{re.escape(prefix)}_(\d+)$")
    indices = []
    for col in columns:
        m = pattern.match(col)
        if m:
            indices.append(int(m.group(1)))
    return sorted(indices)


def make_strat_bins(y_log, n_bins=10):
    y_flat = y_log.reshape(-1)
    quantiles = np.linspace(0, 1, n_bins + 1)
    edges = np.quantile(y_flat, quantiles)
    edges = np.unique(edges)

    if len(edges) < 3:
        return None

    bins = np.digitize(y_flat, edges[1:-1], right=True)
    counts = np.bincount(bins)
    if np.any(counts < 2):
        return None
    return bins


def has_nonfinite_tensor(x):
    return not torch.isfinite(x).all().item()


def sanitize_np_feature(x, name, clip_abs=FEATURE_CLIP_ABS):
    """Return a finite float32 feature array safe for sklearn scalers."""
    arr = np.asarray(x, dtype=np.float32)
    bad = ~np.isfinite(arr)
    if np.any(bad):
        warnings.warn(
            f"{name} contained {int(bad.sum())} non-finite values; replacing them with 0.",
            RuntimeWarning,
        )
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    arr = np.clip(arr, -clip_abs, clip_abs).astype(np.float32)
    return arr

def robust_read_csv(path, nrows=None):
    """Read a CSV defensively.

    Some generated CSVs have rare lines with more comma-separated fields than
    the header declares. The default pandas C parser aborts on those rows. This
    reader first tries the fast parser, then falls back to the Python parser and
    either truncates or skips malformed over-wide rows.
    """
    try:
        return pd.read_csv(path, nrows=nrows, low_memory=False)
    except pd.errors.ParserError as exc:
        warnings.warn(
            f"Fast CSV parser failed for {path}: {exc}. Falling back to "
            f"engine='python' with CSV_BAD_LINE_POLICY={CSV_BAD_LINE_POLICY!r}.",
            RuntimeWarning,
        )

    header = list(pd.read_csv(path, nrows=0).columns)
    n_cols = len(header)

    if CSV_BAD_LINE_POLICY == "skip":
        bad_line_handler = lambda fields: None
    elif CSV_BAD_LINE_POLICY == "truncate":
        def bad_line_handler(fields):
            # Keep the declared schema; discard fields that have no header.
            if len(fields) > n_cols:
                return fields[:n_cols]
            return fields
    else:
        raise ValueError(
            "CSV_BAD_LINE_POLICY must be either 'truncate' or 'skip', "
            f"got {CSV_BAD_LINE_POLICY!r}."
        )

    return pd.read_csv(
        path,
        nrows=nrows,
        engine="python",
        on_bad_lines=bad_line_handler,
    )


def coerce_numeric_columns(df, numeric_cols, context="dataframe"):
    """Coerce declared numeric columns to float-compatible values.

    Malformed generator tokens such as '0.4.951805764502404' become NaN here
    instead of crashing later in .to_numpy(dtype=np.float32).  Required scalar
    fields with NaN are dropped.  Malformed tap cells are dropped only if they
    fall inside the valid mem_len region; malformed padding cells are ignored.
    """
    df = df.copy()
    total_bad = 0
    bad_by_col = {}
    for col in numeric_cols:
        if col not in df.columns:
            continue
        original = df[col]
        was_missing = original.isna()
        converted = pd.to_numeric(original, errors="coerce")
        newly_bad = converted.isna() & ~was_missing
        n_bad = int(newly_bad.sum())
        if n_bad:
            total_bad += n_bad
            bad_by_col[col] = n_bad
        df[col] = converted
    if total_bad:
        preview = dict(list(bad_by_col.items())[:12])
        warnings.warn(
            f"{context}: coerced {total_bad:,} malformed numeric cells to NaN. "
            f"First affected columns: {preview}",
            RuntimeWarning,
        )
    return df, bad_by_col


def drop_bad_required_numeric_rows(df, required_cols, context="dataframe"):
    bad = df[required_cols].isna().any(axis=1)
    n_bad = int(bad.sum())
    if n_bad:
        warnings.warn(
            f"{context}: dropping {n_bad:,} rows with malformed/missing required "
            f"numeric fields among {required_cols}.",
            RuntimeWarning,
        )
        df = df.loc[~bad].copy()
    return df


def validate_required_columns(df, file_path):
    required_base = ["mem_len", "N", "threshold", "BER"]
    missing_base = [c for c in required_base if c not in df.columns]
    if missing_base:
        raise ValueError(f"Missing required columns in {file_path}: {missing_base}")

    tap_indices = get_seq_indices(df.columns, "tap")
    var_indices = get_seq_indices(df.columns, "var")

    if not tap_indices:
        raise ValueError(f"No tap_* columns found in {file_path}")

    if not var_indices:
        warnings.warn(
            f"No var_* columns found in {file_path}; variance channels will be "
            f"synthesized using MISSING_VARIANCE_POLICY={MISSING_VARIANCE_POLICY!r}.",
            RuntimeWarning,
        )
    else:
        missing_vars_for_taps = sorted(set(tap_indices) - set(var_indices))
        if missing_vars_for_taps:
            warnings.warn(
                f"{file_path} is missing var_i columns for some tap_i indices; "
                "those variance entries will be synthesized. First missing indices: "
                f"{missing_vars_for_taps[:10]}",
                RuntimeWarning,
            )

    return tap_indices, var_indices


def determine_sequence_index_offset(all_tap_indices):
    if SEQUENCE_INDEX_BASE == "auto":
        # New total CSVs use tap_1...tap_15; previous variance CSVs used tap_0...tap_14.
        # Internally the model always uses tap_0 as the first/current token.
        return 1 if (0 not in all_tap_indices and min(all_tap_indices) == 1) else 0
    if SEQUENCE_INDEX_BASE in (0, 1):
        return int(SEQUENCE_INDEX_BASE)
    raise ValueError("SEQUENCE_INDEX_BASE must be 'auto', 0, or 1.")


def load_and_merge_data(csv_paths, nrows_per_dataset):
    """Load one or more CSVs and align sequence columns to internal tap_0...tap_K.

    Supports:
      * tap columns starting at tap_0 or tap_1;
      * files with different maximum tap lengths;
      * files with no var_i columns;
      * rare malformed rows with more fields than the header.

    Missing tap entries remain NaN until prepare_data(), where they are converted
    to zero storage padding and masked by mem_len. Missing variance entries remain
    NaN here and are synthesized in prepare_data().
    """
    dfs = []
    all_tap_indices = set()
    all_var_indices = set()

    for path in csv_paths:
        if not os.path.exists(path):
            raise FileNotFoundError(f"Dataset not found: {path}")

        print(f"Loading up to {nrows_per_dataset:,} rows from: {path}")
        df = robust_read_csv(path, nrows=nrows_per_dataset)
        tap_indices, var_indices = validate_required_columns(df, path)
        all_tap_indices.update(tap_indices)
        all_var_indices.update(var_indices)
        df["source_dataset"] = os.path.basename(path)
        dfs.append(df)

    if not all_tap_indices:
        raise ValueError("No tap columns found across datasets.")

    index_offset = determine_sequence_index_offset(all_tap_indices)
    usable_source_tap_indices = sorted(i for i in all_tap_indices if i >= index_offset)
    if not usable_source_tap_indices:
        raise ValueError(
            f"No usable tap indices after applying SEQUENCE_INDEX_BASE={SEQUENCE_INDEX_BASE!r}."
        )

    max_internal_idx = max(i - index_offset for i in usable_source_tap_indices)
    internal_indices = list(range(max_internal_idx + 1))
    tap_cols = [f"tap_{i}" for i in internal_indices]
    var_cols = [f"var_{i}" for i in internal_indices]

    aligned = []
    for df in dfs:
        out = pd.DataFrame(index=df.index)
        for c in ["mem_len", "threshold", "BER", "N", "source_dataset"]:
            out[c] = df[c]

        # Copy source tap_{i+offset} into internal tap_i.
        for internal_i in internal_indices:
            source_i = internal_i + index_offset
            src_tap = f"tap_{source_i}"
            src_var = f"var_{source_i}"
            out[f"tap_{internal_i}"] = df[src_tap] if src_tap in df.columns else np.nan
            out[f"var_{internal_i}"] = df[src_var] if src_var in df.columns else np.nan

        aligned.append(out[tap_cols + var_cols + ["mem_len", "threshold", "BER", "N", "source_dataset"]])

    merged = pd.concat(aligned, ignore_index=True)

    numeric_cols = tap_cols + var_cols + ["mem_len", "threshold", "BER", "N"]
    merged, numeric_bad_by_col = coerce_numeric_columns(
        merged,
        numeric_cols,
        context="aligned merged CSV data",
    )
    if numeric_bad_by_col:
        print(
            "Malformed numeric fields were found after CSV alignment; invalid "
            "required rows will be dropped and invalid valid-tap rows will be "
            "dropped after mem_len masking."
        )

    print(f"Combined rows before filtering: {len(merged):,}")
    print(
        f"Aligned total sequence length = {len(tap_cols)} internal columns: "
        f"tap_0 ... tap_{max_internal_idx} "
        f"(source index offset={index_offset})"
    )
    if not all_var_indices:
        print(
            "No source var_i columns found; using synthesized variance channels "
            f"with MISSING_VARIANCE_POLICY={MISSING_VARIANCE_POLICY!r}."
        )

    return merged, tap_cols, var_cols


def synthesize_variances_from_taps(taps_raw, num_molecules):
    """Create variance proxy for datasets that only contain tap probabilities.

    taps_raw is expected to be probability-like before multiplication by N.
    Returns count-domain variance values with shape [N, L].
    """
    p = np.nan_to_num(taps_raw, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    p_nonneg = np.maximum(p, 0.0)

    if MISSING_VARIANCE_POLICY == "binomial":
        p_clip = np.clip(p_nonneg, 0.0, 1.0)
        return (num_molecules * p_clip * (1.0 - p_clip)).astype(np.float32)
    if MISSING_VARIANCE_POLICY == "poisson":
        return (num_molecules * p_nonneg).astype(np.float32)
    if MISSING_VARIANCE_POLICY == "zeros":
        return np.zeros_like(p_nonneg, dtype=np.float32)

    raise ValueError(
        "MISSING_VARIANCE_POLICY must be 'binomial', 'poisson', or 'zeros', "
        f"got {MISSING_VARIANCE_POLICY!r}."
    )

def y_log_to_raw_np(y_log):
    return np.clip(10 ** y_log, EPS, 0.5)


def raw_to_region_labels_np(y_raw):
    y = np.asarray(y_raw).reshape(-1)
    labels = np.searchsorted(np.asarray(ORDINAL_THRESHOLDS), y, side="right")
    return labels.astype(np.int64)


def region_labels_to_ordinal_targets_np(labels, num_thresholds=NUM_ORDINAL_THRESHOLDS):
    labels = np.asarray(labels).reshape(-1)
    thresholds = np.arange(1, num_thresholds + 1, dtype=np.int64)
    ordinal = (labels[:, None] >= thresholds[None, :]).astype(np.float32)
    return ordinal


def ordinal_logits_to_region_labels_torch(logits):
    probs = torch.sigmoid(logits)
    passed = (probs >= 0.5).sum(dim=1)
    return passed.long()


def compute_region_class_weights(labels_np, num_classes=NUM_REGION_CLASSES, max_weight=8.0):
    counts = np.bincount(labels_np.reshape(-1), minlength=num_classes).astype(np.float64)
    counts = np.maximum(counts, 1.0)
    weights = counts.sum() / (num_classes * counts)
    weights = np.sqrt(weights)
    weights = np.clip(weights, 1.0, max_weight)
    return weights.astype(np.float32)


def compute_ordinal_pos_weights_from_region_labels(
    region_labels_np,
    num_thresholds=NUM_ORDINAL_THRESHOLDS,
    max_weight=20.0,
):
    ordinal_targets = region_labels_to_ordinal_targets_np(region_labels_np, num_thresholds)
    pos_counts = ordinal_targets.sum(axis=0)
    neg_counts = ordinal_targets.shape[0] - pos_counts
    pos_counts = np.maximum(pos_counts, 1.0)
    pos_weight = neg_counts / pos_counts
    pos_weight = np.clip(pos_weight, 1.0, max_weight)
    return pos_weight.astype(np.float32)


class MaskedColumnStandardScaler:
    """Per-column StandardScaler that ignores padded/invalid sequence positions.

    This preserves the original script's useful behavior of scaling tap_0, tap_1,
    ..., tap_L independently, while preventing padded values from contributing to
    each column's fitted mean/std. Transformed invalid positions are forced to 0.
    """

    def __init__(self):
        self.mean_ = None
        self.var_ = None
        self.scale_ = None
        self.n_features_in_ = None
        self.n_samples_seen_ = None

    def fit(self, x, valid_mask):
        x = np.asarray(x, dtype=np.float32)
        valid_mask = np.asarray(valid_mask, dtype=bool)

        if x.ndim != 2:
            raise ValueError(f"Expected x to be 2D, got shape {x.shape}")
        if valid_mask.shape != x.shape:
            raise ValueError(
                f"valid_mask shape {valid_mask.shape} does not match x shape {x.shape}"
            )

        n_cols = x.shape[1]
        self.n_features_in_ = n_cols
        self.mean_ = np.zeros(n_cols, dtype=np.float32)
        self.var_ = np.ones(n_cols, dtype=np.float32)
        self.scale_ = np.ones(n_cols, dtype=np.float32)
        self.n_samples_seen_ = np.zeros(n_cols, dtype=np.int64)

        for j in range(n_cols):
            m = valid_mask[:, j] & np.isfinite(x[:, j])
            values = x[m, j]
            self.n_samples_seen_[j] = int(values.size)
            if values.size > 0:
                mean = float(values.mean())
                var = float(values.var())
                self.mean_[j] = mean
                self.var_[j] = max(var, EPS)
                self.scale_[j] = float(np.sqrt(self.var_[j]))

        return self

    def transform(self, x, valid_mask):
        if self.mean_ is None or self.scale_ is None:
            raise RuntimeError("Scaler has not been fitted.")

        x = np.asarray(x, dtype=np.float32)
        valid_mask = np.asarray(valid_mask, dtype=bool)

        if x.ndim != 2:
            raise ValueError(f"Expected x to be 2D, got shape {x.shape}")
        if x.shape[1] != self.n_features_in_:
            raise ValueError(
                f"Expected {self.n_features_in_} features, got {x.shape[1]}"
            )
        if valid_mask.shape != x.shape:
            raise ValueError(
                f"valid_mask shape {valid_mask.shape} does not match x shape {x.shape}"
            )

        out = ((x - self.mean_[None, :]) / self.scale_[None, :]).astype(np.float32)
        out[~valid_mask] = 0.0
        out[~np.isfinite(out)] = 0.0
        return out

    def fit_transform(self, x, valid_mask):
        self.fit(x, valid_mask)
        return self.transform(x, valid_mask)


# =========================
# Stable Multi-objective Regression Loss
# =========================
class StableMultiObjectiveBERBoundedLogLoss(nn.Module):
    def __init__(
        self,
        log_delta=0.35,
        raw_delta=0.003,
        alpha_log=0.85,
        beta_raw=0.15,
        use_regime_weights=False,
        low_thr=1e-4,
        mid_thr=1e-2,
        w_low=3.0,
        w_mid=4.0,
        w_high=1.0,
    ):
        super().__init__()
        self.log_delta = log_delta
        self.raw_delta = raw_delta
        self.alpha_log = alpha_log
        self.beta_raw = beta_raw

        self.use_regime_weights = use_regime_weights
        self.low_thr = low_thr
        self.mid_thr = mid_thr
        self.w_low = w_low
        self.w_mid = w_mid
        self.w_high = w_high

    @staticmethod
    def huber_elementwise(pred, target, delta):
        err = pred - target
        abs_err = err.abs()
        return torch.where(
            abs_err < delta,
            0.5 * err * err,
            delta * (abs_err - 0.5 * delta),
        )

    @staticmethod
    def raw_to_pred_log(pred_raw_unconstrained):
        log10_half = torch.log10(
            torch.tensor(
                0.5,
                device=pred_raw_unconstrained.device,
                dtype=pred_raw_unconstrained.dtype,
            )
        )
        return log10_half - F.softplus(pred_raw_unconstrained)

    @staticmethod
    def raw_to_pred_ber(pred_raw_unconstrained):
        pred_log = StableMultiObjectiveBERBoundedLogLoss.raw_to_pred_log(
            pred_raw_unconstrained
        )
        return torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)

    def forward(self, pred_raw_unconstrained, target_log):
        pred_log = self.raw_to_pred_log(pred_raw_unconstrained)
        pred_raw = torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)

        target_raw = torch.pow(10.0, target_log).clamp(min=EPS, max=0.5)
        target_log_for_loss = torch.log10(target_raw)

        log_loss = self.huber_elementwise(pred_log, target_log_for_loss, self.log_delta)
        raw_loss = self.huber_elementwise(pred_raw, target_raw, self.raw_delta)

        total = self.alpha_log * log_loss + self.beta_raw * raw_loss

        if self.use_regime_weights:
            weights = torch.full_like(target_raw, self.w_high)
            weights = torch.where(
                target_raw < self.mid_thr,
                torch.full_like(weights, self.w_mid),
                weights,
            )
            weights = torch.where(
                target_raw < self.low_thr,
                torch.full_like(weights, self.w_low),
                weights,
            )
            total = total * weights

        return total.mean()


class OrdinalBCELoss(nn.Module):
    def __init__(self, pos_weight=None, reduction="mean"):
        super().__init__()
        if pos_weight is not None and not isinstance(pos_weight, torch.Tensor):
            pos_weight = torch.tensor(pos_weight, dtype=torch.float32)
        self.register_buffer("pos_weight", pos_weight if pos_weight is not None else None)
        self.reduction = reduction

    def forward(self, logits, ordinal_targets):
        return F.binary_cross_entropy_with_logits(
            logits,
            ordinal_targets,
            pos_weight=self.pos_weight,
            reduction=self.reduction,
        )


class MultiTaskBERLoss(nn.Module):
    def __init__(self, reg_loss, ord_loss, lambda_ord=0.15):
        super().__init__()
        self.reg_loss = reg_loss
        self.ord_loss = ord_loss
        self.lambda_ord = lambda_ord

    def forward(self, pred_raw_unconstrained, ord_logits, target_log, target_ord):
        reg = self.reg_loss(pred_raw_unconstrained, target_log)
        ordl = self.ord_loss(ord_logits, target_ord)
        total = reg + self.lambda_ord * ordl
        return total, reg.detach(), ordl.detach()


# =========================
# Masked Set Transformer Blocks
# =========================
def safe_valid_mask(valid_mask):
    """Ensure every row has at least one unmasked key for MultiheadAttention/softmax.

    The returned mask is only for avoiding all-masked softmax. Outputs are still
    zeroed using the original valid_mask, so dummy unmasked positions do not
    contribute to predictions.
    """
    if valid_mask is None:
        return None
    safe = valid_mask.bool().clone()
    empty = ~safe.any(dim=1)
    if empty.any():
        safe[empty, 0] = True
    return safe


class SetSelfAttentionBlock(nn.Module):
    def __init__(self, d_model, num_heads, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norm2 = nn.LayerNorm(d_model)

        hidden = int(d_model * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x, valid_mask=None):
        # valid_mask: (B, S), True = real/contributing token, False = padding
        safe_mask = safe_valid_mask(valid_mask)
        key_padding_mask = None if safe_mask is None else ~safe_mask

        y = self.norm1(x)
        if valid_mask is not None:
            # Keep invalid queries/keys numerically harmless. safe_mask may expose
            # one dummy key for empty rows, but that dummy key remains zero-valued.
            y = y.masked_fill(~valid_mask.unsqueeze(-1), 0.0)

        attn_out, _ = self.attn(
            y,
            y,
            y,
            key_padding_mask=key_padding_mask,
            need_weights=False,
        )
        if valid_mask is not None:
            attn_out = attn_out.masked_fill(~valid_mask.unsqueeze(-1), 0.0)

        x = x + attn_out

        mlp_in = self.norm2(x)
        if valid_mask is not None:
            mlp_in = mlp_in.masked_fill(~valid_mask.unsqueeze(-1), 0.0)
        mlp_out = self.mlp(mlp_in)
        if valid_mask is not None:
            mlp_out = mlp_out.masked_fill(~valid_mask.unsqueeze(-1), 0.0)

        x = x + mlp_out
        if valid_mask is not None:
            x = x.masked_fill(~valid_mask.unsqueeze(-1), 0.0)
        return x


class ConditionalFeatureModulation(nn.Module):
    def __init__(self, cond_dim, feat_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, 2 * feat_dim),
        )
        self.scale = nn.Parameter(torch.tensor(0.25, dtype=torch.float32))

    def forward(self, x, cond):
        gamma_beta = self.net(cond)
        gamma, beta = torch.chunk(gamma_beta, 2, dim=-1)

        if x.dim() == 3:
            gamma = gamma.unsqueeze(1)
            beta = beta.unsqueeze(1)

        return x * (1.0 + self.scale * gamma) + self.scale * beta


class SetAttentionPooling(nn.Module):
    def __init__(self, d_model, hidden_dim=128):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x, valid_mask=None):
        logits = self.score(x).squeeze(-1)  # (B, S)
        if valid_mask is None:
            weights = torch.softmax(logits, dim=1).unsqueeze(-1)
            return (weights * x).sum(dim=1)

        safe_mask = safe_valid_mask(valid_mask)
        logits = logits.masked_fill(~safe_mask, torch.finfo(logits.dtype).min)
        weights = torch.softmax(logits, dim=1)
        weights = weights.masked_fill(~valid_mask.bool(), 0.0).unsqueeze(-1)
        return (weights * x).sum(dim=1)


def masked_mean(x, valid_mask):
    mask = valid_mask.unsqueeze(-1).to(dtype=x.dtype)
    denom = mask.sum(dim=1).clamp_min(1.0)
    return (x * mask).sum(dim=1) / denom


def masked_sum(x, valid_mask):
    mask = valid_mask.unsqueeze(-1).to(dtype=x.dtype)
    return (x * mask).sum(dim=1)


def masked_max(x, valid_mask):
    # Avoid in-place edits to the output of amax; changing it in-place breaks
    # autograd because AmaxBackward needs the original tensor version.
    bool_mask = valid_mask.bool()
    x_masked = x.masked_fill(~bool_mask.unsqueeze(-1), torch.finfo(x.dtype).min)
    out = x_masked.amax(dim=1)
    empty = ~bool_mask.any(dim=1, keepdim=True)
    zeros = torch.zeros_like(out)
    return torch.where(empty, zeros, out)


# =========================
# Model
# =========================
class BERFirstTokenSetTransformerBoundedLogHalfMultiTask(nn.Module):
    def __init__(
        self,
        past_seq_len,
        token_dim=4,       # tap, var, abs, snr
        first_token_dim=4, # same feature tuple for index 0
        threshold_dim=1,
        global_dim=3,
        d_model=128,
        num_set_layers=4,
        num_heads=4,
        mlp_ratio=4.0,
        dropout=0.10,
        num_ordinal_thresholds=NUM_ORDINAL_THRESHOLDS,
        include_masked_sum_pool=INCLUDE_MASKED_SUM_POOL,
    ):
        super().__init__()

        self.past_seq_len = past_seq_len
        self.token_dim = token_dim
        self.first_token_dim = first_token_dim
        self.d_model = d_model
        self.num_ordinal_thresholds = num_ordinal_thresholds
        self.include_masked_sum_pool = include_masked_sum_pool

        self.first_proj = nn.Sequential(
            nn.Linear(first_token_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.token_proj = nn.Sequential(
            nn.Linear(token_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.thr_embed = nn.Sequential(
            nn.Linear(threshold_dim, 32),
            nn.GELU(),
            nn.Linear(32, 32),
            nn.GELU(),
        )

        self.global_embed = nn.Sequential(
            nn.Linear(global_dim, 64),
            nn.GELU(),
            nn.Linear(64, 64),
            nn.GELU(),
        )

        cond_dim = 32 + 64

        self.first_cond_mod = ConditionalFeatureModulation(
            cond_dim=cond_dim,
            feat_dim=d_model,
            hidden_dim=256,
        )
        self.set_cond_mod = ConditionalFeatureModulation(
            cond_dim=cond_dim,
            feat_dim=d_model,
            hidden_dim=256,
        )

        self.set_blocks = nn.ModuleList([
            SetSelfAttentionBlock(
                d_model=d_model,
                num_heads=num_heads,
                mlp_ratio=mlp_ratio,
                dropout=dropout,
            )
            for _ in range(num_set_layers)
        ])

        self.first_refine = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
        )

        self.final_set_norm = nn.LayerNorm(d_model)
        self.set_attn_pool = SetAttentionPooling(d_model=d_model, hidden_dim=128)

        # set summary: masked attention + masked mean + masked max + masked sum
        # The sum term is zero-padding invariant and helps preserve aggregate ISI.
        set_summary_dim = (4 if self.include_masked_sum_pool else 3) * d_model
        head_in = d_model + set_summary_dim + cond_dim

        self.reg_head = nn.Sequential(
            nn.Linear(head_in, 256),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(128, 1),
        )

        self.ord_head = nn.Sequential(
            nn.Linear(head_in, 256),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(128, num_ordinal_thresholds),
        )

    @staticmethod
    def raw_to_log10ber(pred_raw_unconstrained):
        log10_half = torch.log10(
            torch.tensor(
                0.5,
                device=pred_raw_unconstrained.device,
                dtype=pred_raw_unconstrained.dtype,
            )
        )
        return log10_half - F.softplus(pred_raw_unconstrained)

    @staticmethod
    def raw_to_ber(pred_raw_unconstrained):
        pred_log = BERFirstTokenSetTransformerBoundedLogHalfMultiTask.raw_to_log10ber(
            pred_raw_unconstrained
        )
        return torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)

    def forward(self, first_token, past_tokens, global_feats, threshold, past_mask=None):
        # first_token: (B, 4)
        # past_tokens: (B, S, 4), padded to S=max_past_len
        # past_mask: (B, S), True = contributing real token, False = padding/non-contributing zero
        # no positional encoding: permutation-equivariant over post-first tokens

        thr = self.thr_embed(threshold)
        g = self.global_embed(global_feats)
        cond = torch.cat([thr, g], dim=-1)

        first_x = self.first_proj(first_token)
        first_x = self.first_cond_mod(first_x, cond)
        first_x = first_x + self.first_refine(first_x)

        set_x = self.token_proj(past_tokens)
        set_x = self.set_cond_mod(set_x, cond)
        if past_mask is not None:
            set_x = set_x.masked_fill(~past_mask.unsqueeze(-1), 0.0)

        for block in self.set_blocks:
            set_x = block(set_x, valid_mask=past_mask)

        set_x = self.final_set_norm(set_x)
        if past_mask is not None:
            set_x = set_x.masked_fill(~past_mask.unsqueeze(-1), 0.0)

        if past_mask is None:
            pooled_attn = self.set_attn_pool(set_x)
            pooled_mean = set_x.mean(dim=1)
            pooled_max = set_x.amax(dim=1)
            parts = [pooled_attn, pooled_mean, pooled_max]
            if self.include_masked_sum_pool:
                parts.append(set_x.sum(dim=1))
        else:
            pooled_attn = self.set_attn_pool(set_x, past_mask)
            pooled_mean = masked_mean(set_x, past_mask)
            pooled_max = masked_max(set_x, past_mask)
            parts = [pooled_attn, pooled_mean, pooled_max]
            if self.include_masked_sum_pool:
                parts.append(masked_sum(set_x, past_mask))

        set_summary = torch.cat(parts, dim=-1)
        fused = torch.cat([first_x, set_summary, cond], dim=-1)

        pred_raw_unconstrained = self.reg_head(fused)
        ord_logits = self.ord_head(fused)

        return pred_raw_unconstrained, ord_logits


# =========================
# Data
# =========================
def prepare_data(csv_paths, batch_size=256, nrows_per_dataset=2_500_000, num_workers=0):
    df, tap_cols, var_cols = load_and_merge_data(csv_paths, nrows_per_dataset)

    df = drop_bad_required_numeric_rows(
        df,
        ["mem_len", "threshold", "BER", "N"],
        context="before training filters",
    )
    df["mem_len"] = df["mem_len"].astype(np.int64)

    # Match the original baseline: remove mem_len == 1 rows and remove zero/no-error
    # BER rows before clipping. Do not convert BER=0 into EPS for this comparison.
    if DROP_MEM_LEN_ONE:
        df = df[df["mem_len"] != 1].copy()

    df = df[(df["mem_len"] >= 1) & (df["threshold"] > 0) & (df["BER"] > 0) & (df["N"] > 0)].copy()
    df["BER"] = df["BER"].clip(lower=EPS, upper=0.5)

    L = len(tap_cols)
    if L < 2:
        raise ValueError("Need at least 2 tap columns so index 0 and post-first set both exist.")

    max_mem_len_in_data = int(df["mem_len"].max())
    if MEM_LEN_IS_LAST_VALID_INDEX:
        if max_mem_len_in_data >= L:
            raise ValueError(
                f"Data contains mem_len={max_mem_len_in_data}, interpreted as last valid index, "
                f"but only {L} aligned tap/var columns exist with max index {L - 1}."
            )
    else:
        if max_mem_len_in_data > L:
            raise ValueError(
                f"Data contains mem_len={max_mem_len_in_data}, interpreted as total valid count, "
                f"but only {L} aligned tap/var columns exist."
            )

    print(f"Rows after cleaning/filtering: {len(df):,}")
    print(f"Memory length field range after filtering: {int(df['mem_len'].min())} ... {int(df['mem_len'].max())}")
    print(f"MEM_LEN_IS_LAST_VALID_INDEX = {MEM_LEN_IS_LAST_VALID_INDEX}")

    X_taps_raw = df[tap_cols].to_numpy(dtype=np.float32)
    X_vars_raw = df[var_cols].to_numpy(dtype=np.float32)

    num_molecules = df["N"].to_numpy(dtype=np.float32).reshape(-1, 1)
    X_thr_raw = df["threshold"].to_numpy(dtype=np.float32).reshape(-1, 1)
    y_raw = df["BER"].to_numpy(dtype=np.float32).reshape(-1, 1)

    mem_len = df["mem_len"].to_numpy(dtype=np.int64)
    if MEM_LEN_IS_LAST_VALID_INDEX:
        # mem_len=k means tap_0 ... tap_k are valid. Example: mem_len=14 with
        # columns tap_0 ... tap_14 gives 15 valid total tokens and 14 past tokens.
        mem_len = np.clip(mem_len, 0, L - 1)
        valid_total_mask = np.arange(L)[None, :] <= mem_len[:, None]
    else:
        # mem_len=k means there are k valid total tap columns: tap_0 ... tap_{k-1}.
        mem_len = np.clip(mem_len, 1, L)
        valid_total_mask = np.arange(L)[None, :] < mem_len[:, None]

    # Drop rows where parser/generator corruption produced a non-numeric tap
    # inside the valid memory region.  NaNs outside valid_total_mask are just
    # storage padding and are converted to zero below.
    bad_valid_tap = (~np.isfinite(X_taps_raw)) & valid_total_mask
    bad_valid_tap_rows = bad_valid_tap.any(axis=1)
    n_bad_valid_tap_rows = int(bad_valid_tap_rows.sum())
    if n_bad_valid_tap_rows:
        warnings.warn(
            f"Dropping {n_bad_valid_tap_rows:,} rows with malformed/non-numeric "
            "tap values inside the valid mem_len region.",
            RuntimeWarning,
        )
        keep = ~bad_valid_tap_rows
        X_taps_raw = X_taps_raw[keep]
        X_vars_raw = X_vars_raw[keep]
        num_molecules = num_molecules[keep]
        X_thr_raw = X_thr_raw[keep]
        y_raw = y_raw[keep]
        mem_len = mem_len[keep]
        valid_total_mask = valid_total_mask[keep]
        print(f"Rows after dropping malformed valid taps: {X_taps_raw.shape[0]:,}")

    # Missing tap values are storage padding. Missing variance values mean either
    # padding or a dataset without per-tap variances. For valid positions, synthesize
    # variance proxies from the raw tap probabilities and N.
    X_taps_raw = np.nan_to_num(X_taps_raw, nan=0.0, posinf=0.0, neginf=0.0)
    missing_var_mask = ~np.isfinite(X_vars_raw)
    if np.any(missing_var_mask):
        synthetic_vars = synthesize_variances_from_taps(X_taps_raw, num_molecules)
        X_vars_raw = np.where(missing_var_mask, synthetic_vars, X_vars_raw).astype(np.float32)
    X_vars_raw = np.nan_to_num(X_vars_raw, nan=0.0, posinf=0.0, neginf=0.0)

    if np.any(X_vars_raw[valid_total_mask] < 0):
        raise ValueError("Valid variance columns contain negative values; cannot use them safely.")

    X_thr = np.log10(X_thr_raw + EPS).astype(np.float32)
    y_log = np.log10(y_raw + EPS).astype(np.float32)

    X_taps_feat = (X_taps_raw * num_molecules).astype(np.float32)
    X_vars_feat = X_vars_raw.astype(np.float32)

    # Invalid positions are storage padding, never physics.
    X_taps_feat[~valid_total_mask] = 0.0
    X_vars_feat[~valid_total_mask] = 0.0

    token_total_mask = valid_total_mask.copy()
    if MASK_ZERO_PAST_TOKENS:
        zero_contrib = (np.abs(X_taps_feat) <= ZERO_TAP_EPS) & (X_vars_feat <= ZERO_VAR_EPS)
        token_total_mask[:, 1:] = token_total_mask[:, 1:] & (~zero_contrib[:, 1:])
        # Preserve the first-token branch even when the first tap happens to be zero.
        token_total_mask[:, 0] = valid_total_mask[:, 0]

    past_mask = token_total_mask[:, 1:]

    abs_taps_raw = np.abs(X_taps_feat).astype(np.float32)
    snr_raw = np.log10((X_taps_feat ** 2) / (X_vars_feat + EPS) + EPS).astype(np.float32)
    snr_raw[~token_total_mask] = 0.0

    first_mean = X_taps_feat[:, 0:1]
    past_means = X_taps_feat[:, 1:]
    first_var = X_vars_feat[:, 0:1]
    past_vars = X_vars_feat[:, 1:]

    # Physical globals use masked/zeroed invalid terms. Extra padded zeros therefore
    # do not change mu0, mu1, var0, var1, or any derived z margin.
    mu0_raw = (0.5 * np.sum(past_means, axis=1, keepdims=True)).astype(np.float32)
    mu1_raw = (first_mean + mu0_raw).astype(np.float32)

    var0_raw = (0.5 * np.sum(past_vars, axis=1, keepdims=True)).astype(np.float32)
    var1_raw = (first_var + var0_raw).astype(np.float32)

    std0_raw = np.sqrt(np.maximum(var0_raw, EPS)).astype(np.float32)
    std1_raw = np.sqrt(np.maximum(var1_raw, EPS)).astype(np.float32)

    z0_raw = ((X_thr_raw - mu0_raw) / (std0_raw + EPS)).astype(np.float32)
    z1_raw = ((mu1_raw - X_thr_raw) / (std1_raw + EPS)).astype(np.float32)

    # Finite-safe harmonic mean. The algebraically equivalent form
    #     2 / (1/z0 + 1/z1) = 2*z0*z1/(z0+z1)
    # avoids explicit reciprocals. If z0 + z1 nearly cancels, use 0
    # instead of producing +/-inf; this keeps StandardScaler and training finite.
    harmonic_num = (2.0 * z0_raw * z1_raw).astype(np.float32)
    harmonic_denom = (z0_raw + z1_raw).astype(np.float32)
    harmonic_side_z_raw = np.divide(
        harmonic_num,
        harmonic_denom,
        out=np.zeros_like(harmonic_num, dtype=np.float32),
        where=np.abs(harmonic_denom) > HARMONIC_DENOM_EPS,
    ).astype(np.float32)

    abs_diff_side_z_raw = np.abs(z0_raw - z1_raw).astype(np.float32)
    harmonic_minus_gap_raw = (
        harmonic_side_z_raw - 0.25 * abs_diff_side_z_raw
    ).astype(np.float32)

    # Defensive cleanup before splitting/scaling.
    snr_raw = sanitize_np_feature(snr_raw, "snr_raw")
    z0_raw = sanitize_np_feature(z0_raw, "z0_raw")
    z1_raw = sanitize_np_feature(z1_raw, "z1_raw")
    harmonic_minus_gap_raw = sanitize_np_feature(
        harmonic_minus_gap_raw,
        "harmonic_minus_gap_raw",
    )

    region_labels = raw_to_region_labels_np(y_raw)
    ordinal_targets = region_labels_to_ordinal_targets_np(region_labels)

    strat_labels = make_strat_bins(y_log, n_bins=10)

    split_args = dict(test_size=0.30, random_state=42)
    if strat_labels is not None:
        split_args["stratify"] = strat_labels

    (
        t_taps_raw, temp_taps_raw,
        t_vars_raw, temp_vars_raw,
        t_abs_raw, temp_abs_raw,
        t_snr_raw, temp_snr_raw,
        t_token_mask, temp_token_mask,
        t_past_mask, temp_past_mask,
        t_z0_raw, temp_z0_raw,
        t_z1_raw, temp_z1_raw,
        t_hmg_raw, temp_hmg_raw,
        t_thr, temp_thr,
        t_y_log, temp_y_log,
        t_region, temp_region,
        t_ord, temp_ord,
        t_mem_len, temp_mem_len,
    ) = train_test_split(
        X_taps_feat,
        X_vars_feat,
        abs_taps_raw,
        snr_raw,
        token_total_mask,
        past_mask,
        z0_raw,
        z1_raw,
        harmonic_minus_gap_raw,
        X_thr,
        y_log,
        region_labels,
        ordinal_targets,
        mem_len,
        **split_args,
    )

    temp_strat = make_strat_bins(temp_y_log, n_bins=6)

    split_args2 = dict(test_size=0.50, random_state=42)
    if temp_strat is not None:
        split_args2["stratify"] = temp_strat

    (
        v_taps_raw, te_taps_raw,
        v_vars_raw, te_vars_raw,
        v_abs_raw, te_abs_raw,
        v_snr_raw, te_snr_raw,
        v_token_mask, te_token_mask,
        v_past_mask, te_past_mask,
        v_z0_raw, te_z0_raw,
        v_z1_raw, te_z1_raw,
        v_hmg_raw, te_hmg_raw,
        v_thr, te_thr,
        v_y_log, te_y_log,
        v_region, te_region,
        v_ord, te_ord,
        v_mem_len, te_mem_len,
    ) = train_test_split(
        temp_taps_raw,
        temp_vars_raw,
        temp_abs_raw,
        temp_snr_raw,
        temp_token_mask,
        temp_past_mask,
        temp_z0_raw,
        temp_z1_raw,
        temp_hmg_raw,
        temp_thr,
        temp_y_log,
        temp_region,
        temp_ord,
        temp_mem_len,
        **split_args2,
    )

    # Match the original baseline's per-position scaling, but fit each position only
    # on valid tokens. Padded/non-contributing positions become exact zeros.
    tap_scaler = MaskedColumnStandardScaler()
    var_scaler = MaskedColumnStandardScaler()
    abs_scaler = MaskedColumnStandardScaler()
    snr_scaler = MaskedColumnStandardScaler()

    z0_scaler = StandardScaler()
    z1_scaler = StandardScaler()
    harmonic_minus_gap_scaler = StandardScaler()
    thr_scaler = StandardScaler()

    t_taps = tap_scaler.fit_transform(t_taps_raw, t_token_mask).astype(np.float32)
    v_taps = tap_scaler.transform(v_taps_raw, v_token_mask).astype(np.float32)
    te_taps = tap_scaler.transform(te_taps_raw, te_token_mask).astype(np.float32)

    t_vars = var_scaler.fit_transform(t_vars_raw, t_token_mask).astype(np.float32)
    v_vars = var_scaler.transform(v_vars_raw, v_token_mask).astype(np.float32)
    te_vars = var_scaler.transform(te_vars_raw, te_token_mask).astype(np.float32)

    t_abs = abs_scaler.fit_transform(t_abs_raw, t_token_mask).astype(np.float32)
    v_abs = abs_scaler.transform(v_abs_raw, v_token_mask).astype(np.float32)
    te_abs = abs_scaler.transform(te_abs_raw, te_token_mask).astype(np.float32)

    t_snr = snr_scaler.fit_transform(t_snr_raw, t_token_mask).astype(np.float32)
    v_snr = snr_scaler.transform(v_snr_raw, v_token_mask).astype(np.float32)
    te_snr = snr_scaler.transform(te_snr_raw, te_token_mask).astype(np.float32)

    t_z0 = z0_scaler.fit_transform(t_z0_raw).astype(np.float32)
    v_z0 = z0_scaler.transform(v_z0_raw).astype(np.float32)
    te_z0 = z0_scaler.transform(te_z0_raw).astype(np.float32)

    t_z1 = z1_scaler.fit_transform(t_z1_raw).astype(np.float32)
    v_z1 = z1_scaler.transform(v_z1_raw).astype(np.float32)
    te_z1 = z1_scaler.transform(te_z1_raw).astype(np.float32)

    t_hmg = harmonic_minus_gap_scaler.fit_transform(t_hmg_raw).astype(np.float32)
    v_hmg = harmonic_minus_gap_scaler.transform(v_hmg_raw).astype(np.float32)
    te_hmg = harmonic_minus_gap_scaler.transform(te_hmg_raw).astype(np.float32)

    t_thr = thr_scaler.fit_transform(t_thr).astype(np.float32)
    v_thr = thr_scaler.transform(v_thr).astype(np.float32)
    te_thr = thr_scaler.transform(te_thr).astype(np.float32)

    def build_first_and_past_tokens(taps_scaled, vars_scaled, abs_scaled, snr_scaled, past_valid_mask):
        first_token = np.stack(
            [
                taps_scaled[:, 0],
                vars_scaled[:, 0],
                abs_scaled[:, 0],
                snr_scaled[:, 0],
            ],
            axis=1,
        ).astype(np.float32)

        past_tokens = np.stack(
            [
                taps_scaled[:, 1:],
                vars_scaled[:, 1:],
                abs_scaled[:, 1:],
                snr_scaled[:, 1:],
            ],
            axis=2,
        ).astype(np.float32)

        past_tokens[~past_valid_mask.astype(bool)] = 0.0
        return first_token, past_tokens

    def build_global_features(z0_scaled, z1_scaled, hmg_scaled):
        # No mem_len feature is included. mem_len is used only for masking/diagnostics.
        return np.concatenate([z0_scaled, z1_scaled, hmg_scaled], axis=1).astype(np.float32)

    t_first, t_past = build_first_and_past_tokens(t_taps, t_vars, t_abs, t_snr, t_past_mask)
    v_first, v_past = build_first_and_past_tokens(v_taps, v_vars, v_abs, v_snr, v_past_mask)
    te_first, te_past = build_first_and_past_tokens(te_taps, te_vars, te_abs, te_snr, te_past_mask)

    t_global = build_global_features(t_z0, t_z1, t_hmg)
    v_global = build_global_features(v_z0, v_z1, v_hmg)
    te_global = build_global_features(te_z0, te_z1, te_hmg)

    train_ds = TensorDataset(
        torch.from_numpy(t_first),
        torch.from_numpy(t_past),
        torch.from_numpy(t_past_mask.astype(np.bool_)),
        torch.from_numpy(t_global),
        torch.from_numpy(t_thr),
        torch.from_numpy(t_y_log.astype(np.float32)),
        torch.from_numpy(t_ord.astype(np.float32)),
        torch.from_numpy(t_region.astype(np.int64)),
        torch.from_numpy(t_mem_len.astype(np.int64)),
    )
    val_ds = TensorDataset(
        torch.from_numpy(v_first),
        torch.from_numpy(v_past),
        torch.from_numpy(v_past_mask.astype(np.bool_)),
        torch.from_numpy(v_global),
        torch.from_numpy(v_thr),
        torch.from_numpy(v_y_log.astype(np.float32)),
        torch.from_numpy(v_ord.astype(np.float32)),
        torch.from_numpy(v_region.astype(np.int64)),
        torch.from_numpy(v_mem_len.astype(np.int64)),
    )
    test_ds = TensorDataset(
        torch.from_numpy(te_first),
        torch.from_numpy(te_past),
        torch.from_numpy(te_past_mask.astype(np.bool_)),
        torch.from_numpy(te_global),
        torch.from_numpy(te_thr),
        torch.from_numpy(te_y_log.astype(np.float32)),
        torch.from_numpy(te_ord.astype(np.float32)),
        torch.from_numpy(te_region.astype(np.int64)),
        torch.from_numpy(te_mem_len.astype(np.int64)),
    )

    pin_mem = torch.cuda.is_available()

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        pin_memory=pin_mem,
        num_workers=num_workers,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        pin_memory=pin_mem,
        num_workers=num_workers,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        pin_memory=pin_mem,
        num_workers=num_workers,
    )

    class_weights = compute_region_class_weights(t_region, NUM_REGION_CLASSES)
    ordinal_pos_weights = compute_ordinal_pos_weights_from_region_labels(
        t_region,
        num_thresholds=NUM_ORDINAL_THRESHOLDS,
        max_weight=20.0,
    )

    scalers = {
        "tap_scaler": tap_scaler,
        "var_scaler": var_scaler,
        "abs_scaler": abs_scaler,
        "snr_scaler": snr_scaler,
        "z0_scaler": z0_scaler,
        "z1_scaler": z1_scaler,
        "harmonic_minus_gap_scaler": harmonic_minus_gap_scaler,
        "thr_scaler": thr_scaler,
        "tap_cols": tap_cols,
        "var_cols": var_cols,
        "first_token_dim": 4,
        "past_token_dim": 4,
        "global_dim": 3,
        "first_token_feature_order": [
            "tap0_scaled_mean",
            "var0",
            "abs_tap0_scaled_mean",
            "snr0_proxy_log",
        ],
        "past_token_feature_order": [
            "tap_i_scaled_mean",
            "var_i",
            "abs_tap_i_scaled_mean",
            "snr_i_proxy_log",
        ],
        "global_feature_order": [
            "z0_margin",
            "z1_margin",
            "harmonic_minus_gap",
        ],
        "seq_len_total": L,
        "past_seq_len": L - 1,
        "uses_positional_encoding": False,
        "permutation_invariance_post_first": True,
        "padding_is_masked": True,
        "mem_len_used_as_predictive_feature": False,
        "mask_zero_past_tokens": MASK_ZERO_PAST_TOKENS,
        "zero_tap_eps": ZERO_TAP_EPS,
        "zero_var_eps": ZERO_VAR_EPS,
        "drop_mem_len_one": DROP_MEM_LEN_ONE,
        "mem_len_is_last_valid_index": MEM_LEN_IS_LAST_VALID_INDEX,
        "include_masked_sum_pool": INCLUDE_MASKED_SUM_POOL,
        "set_summary": "masked attention + masked mean + masked max" + (" + masked sum" if INCLUDE_MASKED_SUM_POOL else ""),
        "sequence_scaling": "per-column scaler fitted only on valid/contributing tokens",
        "target_parameterization": "pred_log10_ber = log10(0.5) - softplus(raw_out)",
        "ordinal_thresholds": ORDINAL_THRESHOLDS,
        "num_region_classes": NUM_REGION_CLASSES,
        "class_weights": class_weights.tolist(),
        "ordinal_pos_weights": ordinal_pos_weights.tolist(),
        "preprocessing": {
            "tap_transform": "tap_mean * num_molecules",
            "var_transform": "use raw variance",
            "padding_rule": "if MEM_LEN_IS_LAST_VALID_INDEX: positions i > mem_len are masked; else positions i >= mem_len are masked",
            "zero_past_rule": "if enabled, post-first tokens with tap=0 and var=0 are masked",
            "z0_definition": "z0 = (threshold_raw - mu0) / sqrt(var0 + eps)",
            "z1_definition": "z1 = (mu1 - threshold_raw) / sqrt(var1 + eps)",
            "harmonic_side_z_definition": "finite-safe 2*z0*z1/(z0+z1), zero when denominator nearly cancels",
            "harmonic_minus_gap_definition": "harmonic_side_z - 0.25 * abs(z0 - z1)",
            "mu0_definition": "0.5 * sum(valid past scaled means)",
            "mu1_definition": "first_scaled_mean + mu0",
            "var0_definition": "0.5 * sum(valid past variances)",
            "var1_definition": "first_variance + var0",
        },
        "data_paths": csv_paths,
        "nrows_per_dataset": nrows_per_dataset,
        "csv_bad_line_policy": CSV_BAD_LINE_POLICY,
        "sequence_index_base": SEQUENCE_INDEX_BASE,
        "missing_variance_policy": MISSING_VARIANCE_POLICY,
    }

    aux_info = {
        "class_weights": class_weights,
        "ordinal_pos_weights": ordinal_pos_weights,
    }

    return train_loader, val_loader, test_loader, scalers, aux_info, (L - 1)


# =========================
# Evaluation
# =========================
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_reg_loss = 0.0
    total_ord_loss = 0.0

    all_preds_log = []
    all_targets_log = []
    all_pred_regions = []
    all_true_regions = []

    with torch.no_grad():
        for b_first, b_past, b_past_mask, b_global, b_thr, b_y_log, b_ord, b_region, _ in loader:
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_past_mask = b_past_mask.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_y_log = b_y_log.to(device, non_blocking=True)
            b_ord = b_ord.to(device, non_blocking=True)
            b_region = b_region.to(device, non_blocking=True)

            pred_raw_unconstrained, ord_logits = model(
                b_first, b_past, b_global, b_thr, past_mask=b_past_mask
            )

            if has_nonfinite_tensor(pred_raw_unconstrained):
                raise RuntimeError("Non-finite prediction detected during evaluation.")
            if has_nonfinite_tensor(ord_logits):
                raise RuntimeError("Non-finite ordinal logits detected during evaluation.")

            loss, reg_loss, ord_loss = criterion(
                pred_raw_unconstrained, ord_logits, b_y_log, b_ord
            )

            if has_nonfinite_tensor(loss):
                raise RuntimeError("Non-finite loss detected during evaluation.")

            pred_log = model.raw_to_log10ber(pred_raw_unconstrained)
            pred_region = ordinal_logits_to_region_labels_torch(ord_logits)

            total_loss += loss.item()
            total_reg_loss += reg_loss.item()
            total_ord_loss += ord_loss.item()

            all_preds_log.append(pred_log.cpu().numpy())
            all_targets_log.append(b_y_log.cpu().numpy())
            all_pred_regions.append(pred_region.cpu().numpy())
            all_true_regions.append(b_region.cpu().numpy())

    avg_loss = total_loss / max(len(loader), 1)
    avg_reg_loss = total_reg_loss / max(len(loader), 1)
    avg_ord_loss = total_ord_loss / max(len(loader), 1)

    preds_log = np.vstack(all_preds_log)
    targets_log = np.vstack(all_targets_log)

    preds_raw = np.clip(10 ** preds_log, EPS, 0.5)
    targets_raw = np.clip(10 ** targets_log, EPS, 0.5)

    rmse_log = float(np.sqrt(np.mean((preds_log - targets_log) ** 2)))
    mae_log = float(np.mean(np.abs(preds_log - targets_log)))
    factor_error = float(10 ** rmse_log)

    rmse_raw = float(np.sqrt(np.mean((preds_raw - targets_raw) ** 2)))
    mae_raw = float(np.mean(np.abs(preds_raw - targets_raw)))

    pred_regions = np.concatenate(all_pred_regions).reshape(-1)
    true_regions = np.concatenate(all_true_regions).reshape(-1)

    region_acc = float(np.mean(pred_regions == true_regions))
    region_mae = float(np.mean(np.abs(pred_regions - true_regions)))

    return {
        "loss": avg_loss,
        "reg_loss": avg_reg_loss,
        "ord_loss": avg_ord_loss,
        "rmse_log": rmse_log,
        "mae_log": mae_log,
        "factor_error": factor_error,
        "rmse_raw": rmse_raw,
        "mae_raw": mae_raw,
        "region_acc": region_acc,
        "region_mae": region_mae,
    }


def evaluate_by_target_range(model, loader, device):
    model.eval()
    all_preds_log = []
    all_targets_log = []

    with torch.no_grad():
        for b_first, b_past, b_past_mask, b_global, b_thr, b_y_log, _, _, _ in loader:
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_past_mask = b_past_mask.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)

            pred_raw_unconstrained, _ = model(
                b_first, b_past, b_global, b_thr, past_mask=b_past_mask
            )

            if has_nonfinite_tensor(pred_raw_unconstrained):
                raise RuntimeError("Non-finite prediction detected during per-range evaluation.")

            pred_log = model.raw_to_log10ber(pred_raw_unconstrained)

            all_preds_log.append(pred_log.cpu().numpy())
            all_targets_log.append(b_y_log.numpy())

    preds_log = np.vstack(all_preds_log).reshape(-1)
    targets_log = np.vstack(all_targets_log).reshape(-1)

    preds_raw = np.clip(10 ** preds_log, EPS, 0.5)
    targets_raw = np.clip(10 ** targets_log, EPS, 0.5)

    ranges = {
        "low_BER(y<1e-6)": targets_raw < 1e-6,
        "mid_BER(1e-6<=y<1e-3)": (targets_raw >= 1e-6) & (targets_raw < 1e-3),
        "high_BER(1e-3<=y<0.10)": (targets_raw >= 1e-3) & (targets_raw < 0.10),
        "extreme_BER(y>=0.10)": targets_raw >= 0.10,
    }

    metrics = {}
    for name, mask in ranges.items():
        if np.any(mask):
            rmse_log = float(np.sqrt(np.mean((preds_log[mask] - targets_log[mask]) ** 2)))
            mae_log = float(np.mean(np.abs(preds_log[mask] - targets_log[mask])))
            rmse_raw = float(np.sqrt(np.mean((preds_raw[mask] - targets_raw[mask]) ** 2)))
            mae_raw = float(np.mean(np.abs(preds_raw[mask] - targets_raw[mask])))

            metrics[name] = {
                "count": int(mask.sum()),
                "rmse_log": rmse_log,
                "mae_log": mae_log,
                "factor_error": float(10 ** rmse_log),
                "rmse_raw": rmse_raw,
                "mae_raw": mae_raw,
            }
        else:
            metrics[name] = None

    return metrics


def evaluate_by_region_class(model, loader, device):
    model.eval()
    all_pred_regions = []
    all_true_regions = []

    with torch.no_grad():
        for b_first, b_past, b_past_mask, b_global, b_thr, _, _, b_region, _ in loader:
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_past_mask = b_past_mask.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)

            _, ord_logits = model(b_first, b_past, b_global, b_thr, past_mask=b_past_mask)
            pred_region = ordinal_logits_to_region_labels_torch(ord_logits)

            all_pred_regions.append(pred_region.cpu().numpy())
            all_true_regions.append(b_region.numpy())

    pred_regions = np.concatenate(all_pred_regions).reshape(-1)
    true_regions = np.concatenate(all_true_regions).reshape(-1)

    results = {}
    for cls in range(NUM_REGION_CLASSES):
        mask = true_regions == cls
        if np.any(mask):
            acc = float(np.mean(pred_regions[mask] == true_regions[mask]))
            mae = float(np.mean(np.abs(pred_regions[mask] - true_regions[mask])))
            results[cls] = {
                "count": int(mask.sum()),
                "acc": acc,
                "ordinal_abs_error": mae,
            }
        else:
            results[cls] = None

    return results


def evaluate_by_memory_length(model, loader, device, max_groups=40):
    """Diagnostics only. mem_len is not a model input."""
    model.eval()
    all_preds_log = []
    all_targets_log = []
    all_mem_len = []

    with torch.no_grad():
        for b_first, b_past, b_past_mask, b_global, b_thr, b_y_log, _, _, b_mem_len in loader:
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_past_mask = b_past_mask.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)

            pred_raw_unconstrained, _ = model(
                b_first, b_past, b_global, b_thr, past_mask=b_past_mask
            )
            pred_log = model.raw_to_log10ber(pred_raw_unconstrained)

            all_preds_log.append(pred_log.cpu().numpy())
            all_targets_log.append(b_y_log.numpy())
            all_mem_len.append(b_mem_len.numpy())

    preds_log = np.vstack(all_preds_log).reshape(-1)
    targets_log = np.vstack(all_targets_log).reshape(-1)
    mem_len = np.concatenate(all_mem_len).reshape(-1)

    preds_raw = np.clip(10 ** preds_log, EPS, 0.5)
    targets_raw = np.clip(10 ** targets_log, EPS, 0.5)

    unique_lengths = np.unique(mem_len)
    if unique_lengths.size > max_groups:
        # Avoid unreadable output for huge length diversity; use quantile bins.
        edges = np.unique(np.quantile(mem_len, np.linspace(0, 1, max_groups + 1)).astype(np.int64))
        groups = []
        for lo, hi in zip(edges[:-1], edges[1:]):
            groups.append((f"{lo}-{hi}", (mem_len >= lo) & (mem_len <= hi)))
    else:
        groups = [(str(int(L)), mem_len == L) for L in unique_lengths]

    results = {}
    for name, mask in groups:
        if not np.any(mask):
            continue
        rmse_log = float(np.sqrt(np.mean((preds_log[mask] - targets_log[mask]) ** 2)))
        mae_log = float(np.mean(np.abs(preds_log[mask] - targets_log[mask])))
        rmse_raw = float(np.sqrt(np.mean((preds_raw[mask] - targets_raw[mask]) ** 2)))
        mae_raw = float(np.mean(np.abs(preds_raw[mask] - targets_raw[mask])))
        results[name] = {
            "count": int(mask.sum()),
            "rmse_log": rmse_log,
            "mae_log": mae_log,
            "factor_error": float(10 ** rmse_log),
            "rmse_raw": rmse_raw,
            "mae_raw": mae_raw,
        }
    return results


# =========================
# Training
# =========================
def train_engine():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Executing on: {device}")

    train_loader, val_loader, test_loader, scalers, aux_info, past_seq_len = prepare_data(
        DATA_PATHS,
        batch_size=256,
        nrows_per_dataset=NROWS_PER_DATASET,
        num_workers=0,
    )

    joblib.dump(scalers, SCALER_SAVE_PATH)
    print(f"Scalers saved to: {SCALER_SAVE_PATH}")

    model = BERFirstTokenSetTransformerBoundedLogHalfMultiTask(
        past_seq_len=past_seq_len,
        token_dim=4,
        first_token_dim=4,
        threshold_dim=1,
        global_dim=3,
        d_model=128,
        num_set_layers=4,
        num_heads=4,
        mlp_ratio=4.0,
        dropout=0.10,
        num_ordinal_thresholds=NUM_ORDINAL_THRESHOLDS,
        include_masked_sum_pool=INCLUDE_MASKED_SUM_POOL,
    ).to(device)

    print("Training first-token + masked set-transformer model from scratch...")
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

    reg_loss = StableMultiObjectiveBERBoundedLogLoss(
        log_delta=0.35,
        raw_delta=0.003,
        alpha_log=0.85,
        beta_raw=0.15,
        use_regime_weights=True,
        low_thr=1e-4,
        mid_thr=1e-2,
        w_low=3.0,
        w_mid=4.0,
        w_high=1.0,
    )

    ord_loss = OrdinalBCELoss(
        pos_weight=torch.tensor(
            aux_info["ordinal_pos_weights"],
            dtype=torch.float32,
            device=device,
        ),
        reduction="mean",
    )

    criterion = MultiTaskBERLoss(
        reg_loss=reg_loss,
        ord_loss=ord_loss,
        lambda_ord=0.15,
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        patience=4,
        factor=0.5,
    )

    best_val_loss = float("inf")
    best_state = None
    patience = 12
    wait = 0
    min_epochs_before_early_stop = 60
    max_epochs = 120

    print(f"Detected max post-first set size = {past_seq_len}")
    print(f"Ordinal thresholds: {ORDINAL_THRESHOLDS}")
    print(f"Ordinal pos weights: {aux_info['ordinal_pos_weights']}")
    print("mem_len is used for masks/diagnostics only, not as a model feature.")
    print(f"DROP_MEM_LEN_ONE = {DROP_MEM_LEN_ONE}")
    print(f"MASK_ZERO_PAST_TOKENS = {MASK_ZERO_PAST_TOKENS}")
    print(f"MEM_LEN_IS_LAST_VALID_INDEX = {MEM_LEN_IS_LAST_VALID_INDEX}")
    print(f"INCLUDE_MASKED_SUM_POOL = {INCLUDE_MASKED_SUM_POOL}")

    training_broke = False

    for epoch in range(max_epochs):
        model.train()

        for batch_idx, (b_first, b_past, b_past_mask, b_global, b_thr, b_y_log, b_ord, _, _) in enumerate(train_loader):
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_past_mask = b_past_mask.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_y_log = b_y_log.to(device, non_blocking=True)
            b_ord = b_ord.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            pred_raw_unconstrained, ord_logits = model(
                b_first, b_past, b_global, b_thr, past_mask=b_past_mask
            )

            if has_nonfinite_tensor(pred_raw_unconstrained):
                print(f"Non-finite prediction detected at epoch {epoch + 1}, batch {batch_idx + 1}.")
                training_broke = True
                break

            if has_nonfinite_tensor(ord_logits):
                print(f"Non-finite ordinal logits detected at epoch {epoch + 1}, batch {batch_idx + 1}.")
                training_broke = True
                break

            loss, reg_part, ord_part = criterion(
                pred_raw_unconstrained, ord_logits, b_y_log, b_ord
            )

            if has_nonfinite_tensor(loss):
                print(f"Non-finite loss detected at epoch {epoch + 1}, batch {batch_idx + 1}.")
                training_broke = True
                break

            loss.backward()

            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            if not torch.isfinite(grad_norm):
                print(f"Non-finite gradient norm detected at epoch {epoch + 1}, batch {batch_idx + 1}.")
                training_broke = True
                break

            optimizer.step()

            bad_param = False
            for name, param in model.named_parameters():
                if param.requires_grad and param.data is not None and not torch.isfinite(param.data).all():
                    print(f"Non-finite parameter detected after optimizer step: {name}")
                    bad_param = True
                    break

            if bad_param:
                training_broke = True
                break

        if training_broke:
            print("Training stopped because non-finite values were detected.")
            break

        try:
            train_metrics = evaluate(model, train_loader, criterion, device)
            val_metrics = evaluate(model, val_loader, criterion, device)
        except RuntimeError as e:
            print(f"Evaluation failed at epoch {epoch + 1}: {e}")
            break

        scheduler.step(val_metrics["loss"])
        current_lr = optimizer.param_groups[0]["lr"]

        print(
            f"Epoch {epoch + 1:03d} | "
            f"LR: {current_lr:.2e} | "
            f"Train Loss: {train_metrics['loss']:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} | "
            f"Train Reg: {train_metrics['reg_loss']:.4f} | "
            f"Val Reg: {val_metrics['reg_loss']:.4f} | "
            f"Train Ord: {train_metrics['ord_loss']:.4f} | "
            f"Val Ord: {val_metrics['ord_loss']:.4f} | "
            f"Train RMSE(log10): {train_metrics['rmse_log']:.4f} (~{train_metrics['factor_error']:.2f}x) | "
            f"Val RMSE(log10): {val_metrics['rmse_log']:.4f} (~{val_metrics['factor_error']:.2f}x) | "
            f"Train Region Acc: {train_metrics['region_acc']:.4f} | "
            f"Val Region Acc: {val_metrics['region_acc']:.4f} | "
            f"Train Region MAE: {train_metrics['region_mae']:.4f} | "
            f"Val Region MAE: {val_metrics['region_mae']:.4f}"
        )

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
            print(
                f"  -> New best model saved at epoch {epoch + 1} "
                f"with Val Loss: {val_metrics['loss']:.6f}"
            )
        else:
            if epoch + 1 >= min_epochs_before_early_stop:
                wait += 1
                if wait >= patience:
                    print("Early stopping triggered.")
                    break

    torch.save(model.state_dict(), LAST_MODEL_SAVE_PATH)
    print(f"Last model saved to: {LAST_MODEL_SAVE_PATH}")

    if best_state is not None:
        model.load_state_dict(best_state)
        torch.save(model.state_dict(), BEST_MODEL_SAVE_PATH)
        print(f"Best model saved to: {BEST_MODEL_SAVE_PATH}")
    else:
        print("Warning: no valid best checkpoint was found.")

    test_metrics = evaluate(model, test_loader, criterion, device)

    print(
        f"Test Loss: {test_metrics['loss']:.4f} | "
        f"Test Reg Loss: {test_metrics['reg_loss']:.4f} | "
        f"Test Ord Loss: {test_metrics['ord_loss']:.4f} | "
        f"Test RMSE(log10): {test_metrics['rmse_log']:.4f} | "
        f"Test MAE(log10): {test_metrics['mae_log']:.4f} | "
        f"Typical multiplicative error: ~{test_metrics['factor_error']:.2f}x | "
        f"Test RMSE(raw): {test_metrics['rmse_raw']:.6f} | "
        f"Test MAE(raw): {test_metrics['mae_raw']:.6f} | "
        f"Test Region Acc: {test_metrics['region_acc']:.4f} | "
        f"Test Region MAE: {test_metrics['region_mae']:.4f}"
    )

    range_metrics = evaluate_by_target_range(model, test_loader, device)
    print("\nPer-range test diagnostics:")
    for name, stats in range_metrics.items():
        if stats is None:
            print(f"{name}: no samples")
        else:
            print(
                f"{name} | count={stats['count']} | "
                f"RMSE(log10)={stats['rmse_log']:.4f} | "
                f"MAE(log10)={stats['mae_log']:.4f} | "
                f"factor~{stats['factor_error']:.2f}x | "
                f"RMSE(raw)={stats['rmse_raw']:.6f} | "
                f"MAE(raw)={stats['mae_raw']:.6f}"
            )

    region_metrics = evaluate_by_region_class(model, test_loader, device)
    print("\nPer-region ordinal diagnostics:")
    region_names = {
        0: "y < 1e-6",
        1: "1e-6 <= y < 1e-5",
        2: "1e-5 <= y < 1e-4",
        3: "1e-4 <= y < 1e-3",
        4: "1e-3 <= y < 1e-2",
        5: "1e-2 <= y < 1e-1",
        6: "0.1 <= y < 0.2",
        7: "0.2 <= y < 0.3",
        8: "0.3 <= y < 0.4",
        9: "0.4 <= y <= 0.5",
    }
    for cls, stats in region_metrics.items():
        if stats is None:
            print(f"class={cls} ({region_names[cls]}): no samples")
        else:
            print(
                f"class={cls} ({region_names[cls]}) | "
                f"count={stats['count']} | "
                f"acc={stats['acc']:.4f} | "
                f"ordinal_abs_error={stats['ordinal_abs_error']:.4f}"
            )

    mem_metrics = evaluate_by_memory_length(model, test_loader, device)
    print("\nPer-memory-length test diagnostics:")
    for mem_group, stats in mem_metrics.items():
        print(
            f"mem_len={mem_group} | count={stats['count']} | "
            f"RMSE(log10)={stats['rmse_log']:.4f} | "
            f"MAE(log10)={stats['mae_log']:.4f} | "
            f"factor~{stats['factor_error']:.2f}x | "
            f"RMSE(raw)={stats['rmse_raw']:.6f} | "
            f"MAE(raw)={stats['mae_raw']:.6f}"
        )

    return model


if __name__ == "__main__":
    os.makedirs(BASE_DIR, exist_ok=True)

    missing = [p for p in DATA_PATHS if not os.path.exists(p)]
    if missing:
        print("Critical Error: Missing dataset files:")
        for p in missing:
            print(f"  - {p}")
    else:
        print("Using training from datasets:")
        for p in DATA_PATHS:
            print(f"  - {p}")
        print(f"Row cap per dataset: {NROWS_PER_DATASET:,}")

        trained_model = train_engine()

Using training from datasets:
  - ./../ber_data_generation/data/data_physics_total.csv
  - ./../ber_data_generation/data/data_random_total.csv
Row cap per dataset: 5,000,000
Executing on: cuda
Loading up to 5,000,000 rows from: ./../ber_data_generation/data/data_physics_total.csv


/tmp/ipykernel_267758/3716556319.py:160: RuntimeWarning: Fast CSV parser failed for ./../ber_data_generation/data/data_physics_total.csv: Error tokenizing data. C error: Expected 25 fields in line 3630711, saw 36
. Falling back to engine='python' with CSV_BAD_LINE_POLICY='truncate'.
  warnings.warn(
/tmp/ipykernel_267758/3716556319.py:250: RuntimeWarning: No var_* columns found in ./../ber_data_generation/data/data_physics_total.csv; variance channels will be synthesized using MISSING_VARIANCE_POLICY='binomial'.
  warnings.warn(


Loading up to 5,000,000 rows from: ./../ber_data_generation/data/data_random_total.csv


/tmp/ipykernel_267758/3716556319.py:250: RuntimeWarning: No var_* columns found in ./../ber_data_generation/data/data_random_total.csv; variance channels will be synthesized using MISSING_VARIANCE_POLICY='binomial'.
  warnings.warn(
/tmp/ipykernel_267758/3716556319.py:216: RuntimeWarning: aligned merged CSV data: coerced 1 malformed numeric cells to NaN. First affected columns: {'tap_1': 1}
  warnings.warn(


Malformed numeric fields were found after CSV alignment; invalid required rows will be dropped and invalid valid-tap rows will be dropped after mem_len masking.
Combined rows before filtering: 10,000,000
Aligned total sequence length = 15 internal columns: tap_0 ... tap_14 (source index offset=1)
No source var_i columns found; using synthesized variance channels with MISSING_VARIANCE_POLICY='binomial'.
Rows after cleaning/filtering: 7,990,695
Memory length field range after filtering: 2 ... 15
MEM_LEN_IS_LAST_VALID_INDEX = False


/tmp/ipykernel_267758/3716556319.py:1020: RuntimeWarning: Dropping 1 rows with malformed/non-numeric tap values inside the valid mem_len region.
  warnings.warn(


Rows after dropping malformed valid taps: 7,990,694
Scalers saved to: ./combined_variablemem_masked_robust_scalers.pkl
Training first-token + masked set-transformer model from scratch...
Detected max post-first set size = 14
Ordinal thresholds: [1e-06, 1e-05, 0.0001, 0.001, 0.01, 0.1, 0.2, 0.3, 0.4]
Ordinal pos weights: [1. 1. 1. 1. 1. 1. 1. 1. 1.]
mem_len is used for masks/diagnostics only, not as a model feature.
DROP_MEM_LEN_ONE = True
MASK_ZERO_PAST_TOKENS = False
MEM_LEN_IS_LAST_VALID_INDEX = False
INCLUDE_MASKED_SUM_POOL = False
Epoch 001 | LR: 1.00e-04 | Train Loss: 0.1553 | Val Loss: 0.1553 | Train Reg: 0.1391 | Val Reg: 0.1391 | Train Ord: 0.1079 | Val Ord: 0.1079 | Train RMSE(log10): 1.4729 (~29.71x) | Val RMSE(log10): 1.4719 (~29.64x) | Train Region Acc: 0.8375 | Val Region Acc: 0.8377 | Train Region MAE: 0.2745 | Val Region MAE: 0.2746
  -> New best model saved at epoch 1 with Val Loss: 0.155273
Epoch 002 | LR: 1.00e-04 | Train Loss: 0.0796 | Val Loss: 0.0794 | Train Reg: 0

KeyboardInterrupt: 

In [8]:
"""
Fine-Tuning Script
==================
Continues training from the last saved checkpoint on new datasets.

Handles two CSV formats:
  1. Full physics format with var_i columns (existing)
  2. New format without var_i columns — variance is computed from
     binomial physics: var_i = N * tap_i * (1 - tap_i)

The new format also lacks `mem_len` in some cases — derived from the
number of non-NaN tap columns.

Fine-tuning differences vs from-scratch training:
  - Loads model & scalers from the previous run
  - Lower learning rate (1/5 of base)
  - Shorter schedule (60 epochs max)
  - Smaller patience for early stopping
  - Reuses scalers from previous run by default (controlled by REUSE_SCALERS)
  - Saves to a NEW set of files so previous checkpoints stay intact
"""
import os
import re
import copy
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.multiprocessing as mp
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

mp.set_sharing_strategy("file_system")

# =========================
# Configuration
# =========================
BASE_DIR = "./"
DATA_DIR = "./../ber_data_generation/data/"

# ---- Source: previous training run to resume from ----
PRETRAINED_MODEL_PATH = os.path.join(
    BASE_DIR, "random_extra_multitask_last2.pth"
)
PRETRAINED_SCALER_PATH = os.path.join(
    BASE_DIR, "random_extra_multitask_scalers2.pkl"
)
# If True, keep using the pretrained scalers (recommended — keeps the model
# and scalers consistent). If False, refit scalers on the fine-tune data.
REUSE_SCALERS = True

# ---- Destination: NEW files for fine-tuned model ----
BEST_MODEL_SAVE_PATH = os.path.join(
    BASE_DIR, "random_extra_multitask_finetune_best.pth"
)
LAST_MODEL_SAVE_PATH = os.path.join(
    BASE_DIR, "random_extra_multitask_finetune_last.pth"
)
SCALER_SAVE_PATH = os.path.join(
    BASE_DIR, "random_extra_multitask_finetune_scalers.pkl"
)

# ---- Fine-tune data (no var_* columns; will be computed) ----
DATA_PATHS = [
    os.path.join(DATA_DIR, "data_physics_total.csv"),
    os.path.join(DATA_DIR, "data_random_total.csv"),
]

NROWS_PER_DATASET = 5_000_000

# ---- Fine-tune hyperparameters (gentler than from-scratch) ----
FINETUNE_LR = 2e-5         # ~1/5 of base 1e-4 — preserve pretrained features
FINETUNE_WEIGHT_DECAY = 1e-5
FINETUNE_MAX_EPOCHS = 60
FINETUNE_MIN_EPOCHS = 25
FINETUNE_PATIENCE = 8
SCHEDULER_PATIENCE = 3
SCHEDULER_FACTOR = 0.5
GRAD_CLIP = 1.0

EPS = 1e-12
LOG10_HALF = float(np.log10(0.5))

ORDINAL_THRESHOLDS = [
    1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1,
    0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45,
]
NUM_ORDINAL_THRESHOLDS = len(ORDINAL_THRESHOLDS)
NUM_REGION_CLASSES = NUM_ORDINAL_THRESHOLDS + 1


# =========================
# Utilities
# =========================
def get_sorted_seq_cols(columns, prefix):
    pattern = re.compile(rf"^{re.escape(prefix)}_(\d+)$")
    matched = []
    for col in columns:
        m = pattern.match(col)
        if m:
            matched.append((int(m.group(1)), col))
    matched.sort(key=lambda x: x[0])
    return [col for _, col in matched]


def make_strat_bins(y_log, n_bins=10):
    y_flat = y_log.reshape(-1)
    quantiles = np.linspace(0, 1, n_bins + 1)
    edges = np.quantile(y_flat, quantiles)
    edges = np.unique(edges)
    if len(edges) < 3:
        return None
    bins = np.digitize(y_flat, edges[1:-1], right=True)
    counts = np.bincount(bins)
    if np.any(counts < 2):
        return None
    return bins


def has_nonfinite_tensor(x):
    return not torch.isfinite(x).all().item()


def validate_required_columns(df, file_path, allow_missing_vars=True):
    """Validate columns. With allow_missing_vars=True, var_* columns are
    optional and will be computed from taps if absent."""
    if "N" not in df.columns:
        raise ValueError(f"Required column 'N' not found in {file_path}")

    tap_cols = get_sorted_seq_cols(df.columns, "tap")
    var_cols = get_sorted_seq_cols(df.columns, "var")

    if not tap_cols:
        raise ValueError(f"No tap_* columns found in {file_path}")

    if var_cols:
        if len(tap_cols) != len(var_cols):
            raise ValueError(
                f"tap/var length mismatch in {file_path}: "
                f"{len(tap_cols)} tap cols vs {len(var_cols)} var cols"
            )
        has_vars = True
    else:
        if not allow_missing_vars:
            raise ValueError(f"No var_* columns found in {file_path}")
        has_vars = False

    base_required = tap_cols + ["threshold", "BER", "N"]
    if has_vars:
        base_required = base_required + var_cols
    missing = [c for c in base_required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in {file_path}: {missing}")

    return tap_cols, var_cols, has_vars


def derive_mem_len_from_taps(df, tap_cols):
    """If mem_len column is missing, derive it from the number of leading
    non-null tap columns. Assumes tap_1..tap_K are filled and tap_(K+1)+ are NaN."""
    if "mem_len" in df.columns:
        return df["mem_len"].to_numpy(dtype=np.int64)

    print(f"  mem_len missing; deriving from non-null tap counts.")
    non_null = df[tap_cols].notna().to_numpy()
    # mem_len = number of non-null taps
    derived = non_null.sum(axis=1).astype(np.int64)
    return derived


def compute_variances_from_taps(taps_raw, N_array, valid_mask):
    """Compute binomial variance var_i = N * tap_i * (1 - tap_i) for each
    valid position. Padding positions (where valid_mask==False) get 0.

    Args:
        taps_raw: [N, L] raw tap probabilities (may have NaN/0 in pad positions)
        N_array:  [N] number of molecules
        valid_mask: [N, L] bool, True = real tap

    Returns:
        vars_raw: [N, L] variance values (raw, not multiplied by N already)
                   matches the convention of the var_* columns in original CSVs
    """
    N = N_array.reshape(-1, 1).astype(np.float64)
    taps = np.where(np.isnan(taps_raw), 0.0, taps_raw).astype(np.float64)
    taps = np.clip(taps, 0.0, 1.0)
    vars_full = N * taps * (1.0 - taps)
    # Re-zero invalid positions
    vars_full = vars_full * valid_mask.astype(np.float64)
    return vars_full.astype(np.float32)


def _read_csv_robust(path, nrows):
    """Read a CSV, skipping any rows that have MORE fields than the header
    declares (treated as corrupted). The C engine with on_bad_lines='skip'
    handles this efficiently — pandas drops over-long rows and keeps only
    the well-formed ones.
    """
    try:
        df = pd.read_csv(path, nrows=nrows, on_bad_lines="skip", engine="c")
    except (ValueError, pd.errors.ParserError) as e:
        # Fallback: python engine is more lenient
        print(f"  C engine failed ({e}); using python engine")
        df = pd.read_csv(path, nrows=nrows, on_bad_lines="skip", engine="python")
    return df


def load_and_merge_data(csv_paths, nrows_per_dataset):
    dfs = []
    reference_tap_cols = None
    union_var_cols = None
    has_vars_per_file = []

    for path in csv_paths:
        if not os.path.exists(path):
            raise FileNotFoundError(f"Dataset not found: {path}")

        print(f"Loading up to {nrows_per_dataset:,} rows from: {path}")
        df = _read_csv_robust(path, nrows=nrows_per_dataset)

        tap_cols, var_cols, has_vars = validate_required_columns(
            df, path, allow_missing_vars=True
        )

        if reference_tap_cols is None:
            reference_tap_cols = tap_cols
            union_var_cols = var_cols if has_vars else []
        else:
            # Reconcile tap columns across files: use the UNION sorted by index.
            # If one file has more tap columns than another, missing columns
            # will be filled with NaN (treated as padding downstream).
            if tap_cols != reference_tap_cols:
                ref_set = set(reference_tap_cols)
                this_set = set(tap_cols)
                only_here = sorted(this_set - ref_set,
                                   key=lambda c: int(c.split("_")[1]))
                only_ref = sorted(ref_set - this_set,
                                  key=lambda c: int(c.split("_")[1]))
                if only_here:
                    print(f"  File adds {len(only_here)} tap cols not in "
                          f"reference (e.g., {only_here[:3]}...). Reference will be extended.")
                if only_ref:
                    print(f"  File missing {len(only_ref)} reference tap cols "
                          f"(e.g., {only_ref[:3]}...); will be NaN-filled.")
                # Extend the reference with any new columns
                reference_tap_cols = sorted(
                    ref_set | this_set,
                    key=lambda c: int(c.split("_")[1]),
                )
                # Add NaN columns to existing dfs that lack the new cols
                for prev in dfs:
                    for c in only_here:
                        if c not in prev.columns:
                            prev[c] = np.nan
                # Add NaN columns to current df for missing ref cols
                for c in only_ref:
                    if c not in df.columns:
                        df[c] = np.nan

            if has_vars and not union_var_cols:
                union_var_cols = var_cols

        df["source_dataset"] = os.path.basename(path)
        df["_has_vars"] = has_vars
        has_vars_per_file.append(has_vars)
        dfs.append(df)

    merged = pd.concat(dfs, ignore_index=True)
    print(f"Combined rows before filtering: {len(merged):,}")
    print(f"  Files with var_* columns: {sum(has_vars_per_file)}/{len(has_vars_per_file)}")
    print(f"  Final tap column count: {len(reference_tap_cols)}")

    return merged, reference_tap_cols, union_var_cols


def y_log_to_raw_np(y_log):
    return np.clip(10 ** y_log, EPS, 0.5)


def raw_to_region_labels_np(y_raw):
    y = np.asarray(y_raw).reshape(-1)
    labels = np.searchsorted(np.asarray(ORDINAL_THRESHOLDS), y, side="right")
    return labels.astype(np.int64)


def region_labels_to_ordinal_targets_np(labels, num_thresholds=NUM_ORDINAL_THRESHOLDS):
    labels = np.asarray(labels).reshape(-1)
    thresholds = np.arange(1, num_thresholds + 1, dtype=np.int64)
    ordinal = (labels[:, None] >= thresholds[None, :]).astype(np.float32)
    return ordinal


def ordinal_logits_to_region_labels_torch(logits):
    probs = torch.sigmoid(logits)
    passed = (probs >= 0.5).sum(dim=1)
    return passed.long()


def compute_region_class_weights(labels_np, num_classes=NUM_REGION_CLASSES, max_weight=8.0):
    counts = np.bincount(labels_np.reshape(-1), minlength=num_classes).astype(np.float64)
    counts = np.maximum(counts, 1.0)
    weights = counts.sum() / (num_classes * counts)
    weights = np.sqrt(weights)
    weights = np.clip(weights, 1.0, max_weight)
    return weights.astype(np.float32)


def compute_ordinal_pos_weights_from_region_labels(
    region_labels_np,
    num_thresholds=NUM_ORDINAL_THRESHOLDS,
    max_weight=20.0,
):
    ordinal_targets = region_labels_to_ordinal_targets_np(region_labels_np, num_thresholds)
    pos_counts = ordinal_targets.sum(axis=0)
    neg_counts = ordinal_targets.shape[0] - pos_counts
    pos_counts = np.maximum(pos_counts, 1.0)
    pos_weight = neg_counts / pos_counts
    pos_weight = np.clip(pos_weight, 1.0, max_weight)
    return pos_weight.astype(np.float32)


# =========================
# Position-Independent Scaling helpers
# =========================
def fit_shared_scalar_scaler(train_data, train_valid_mask):
    entries = train_data[train_valid_mask].reshape(-1)
    if len(entries) == 0:
        entries = train_data.reshape(-1)
    mean = float(entries.mean())
    std = float(max(entries.std(), 1e-12))
    return mean, std


def apply_shared_scale(data, mean, std, valid_mask=None):
    scaled = ((data - mean) / std).astype(np.float32)
    if valid_mask is not None and scaled.ndim == 2 and valid_mask.shape == scaled.shape:
        scaled = scaled * valid_mask.astype(np.float32)
    return scaled


# =========================
# Loss
# =========================
class StableMultiObjectiveBERBoundedLogLoss(nn.Module):
    def __init__(self, log_delta=0.35, raw_delta=0.006, rel_delta=0.03,
                 alpha_log=0.45, beta_raw=0.35, gamma_rel=0.20,
                 use_regime_weights=True):
        super().__init__()
        self.log_delta = log_delta
        self.raw_delta = raw_delta
        self.rel_delta = rel_delta
        self.alpha_log = alpha_log
        self.beta_raw = beta_raw
        self.gamma_rel = gamma_rel
        self.use_regime_weights = use_regime_weights

    @staticmethod
    def huber_elementwise(pred, target, delta):
        err = pred - target
        abs_err = err.abs()
        return torch.where(abs_err < delta,
                           0.5 * err * err,
                           delta * (abs_err - 0.5 * delta))

    @staticmethod
    def raw_to_pred_log(pred_raw_unconstrained):
        log10_half = torch.log10(torch.tensor(
            0.5, device=pred_raw_unconstrained.device,
            dtype=pred_raw_unconstrained.dtype))
        return log10_half - F.softplus(pred_raw_unconstrained)

    @staticmethod
    def raw_to_pred_ber(pred_raw_unconstrained):
        pred_log = StableMultiObjectiveBERBoundedLogLoss.raw_to_pred_log(pred_raw_unconstrained)
        return torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)

    def forward(self, pred_raw_unconstrained, target_log):
        pred_log = self.raw_to_pred_log(pred_raw_unconstrained)
        pred_raw = torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)
        target_raw = torch.pow(10.0, target_log).clamp(min=EPS, max=0.5)
        target_log_for_loss = torch.log10(target_raw)

        log_loss = self.huber_elementwise(pred_log, target_log_for_loss, self.log_delta)
        raw_loss = self.huber_elementwise(pred_raw, target_raw, self.raw_delta)
        rel_err = (pred_raw - target_raw) / torch.clamp(target_raw, min=1e-6)
        rel_loss = self.huber_elementwise(rel_err, torch.zeros_like(rel_err), self.rel_delta)

        total = (self.alpha_log * log_loss
                 + self.beta_raw * raw_loss
                 + self.gamma_rel * rel_loss)

        if self.use_regime_weights:
            weights = torch.ones_like(target_raw)
            weights = torch.where((target_raw >= 1e-2) & (target_raw < 0.1),
                                  torch.full_like(weights, 1.5), weights)
            weights = torch.where((target_raw >= 0.1) & (target_raw < 0.15),
                                  torch.full_like(weights, 2.5), weights)
            weights = torch.where((target_raw >= 0.15) & (target_raw < 0.2),
                                  torch.full_like(weights, 3.0), weights)
            weights = torch.where((target_raw >= 0.2) & (target_raw < 0.3),
                                  torch.full_like(weights, 4.0), weights)
            weights = torch.where((target_raw >= 0.3) & (target_raw < 0.4),
                                  torch.full_like(weights, 4.5), weights)
            weights = torch.where((target_raw >= 0.4) & (target_raw < 0.45),
                                  torch.full_like(weights, 3.0), weights)
            weights = torch.where(target_raw >= 0.45,
                                  torch.full_like(weights, 2.5), weights)
            total = total * weights

        return total.mean()


class OrdinalBCELoss(nn.Module):
    def __init__(self, pos_weight=None, reduction="mean"):
        super().__init__()
        if pos_weight is not None and not isinstance(pos_weight, torch.Tensor):
            pos_weight = torch.tensor(pos_weight, dtype=torch.float32)
        self.register_buffer("pos_weight", pos_weight if pos_weight is not None else None)
        self.reduction = reduction

    def forward(self, logits, ordinal_targets):
        return F.binary_cross_entropy_with_logits(
            logits, ordinal_targets, pos_weight=self.pos_weight, reduction=self.reduction)


class MultiTaskBERLoss(nn.Module):
    def __init__(self, reg_loss, ord_loss, lambda_ord=0.25):
        super().__init__()
        self.reg_loss = reg_loss
        self.ord_loss = ord_loss
        self.lambda_ord = lambda_ord

    def forward(self, pred_raw_unconstrained, ord_logits, target_log, target_ord):
        reg = self.reg_loss(pred_raw_unconstrained, target_log)
        ordl = self.ord_loss(ord_logits, target_ord)
        total = reg + self.lambda_ord * ordl
        return total, reg.detach(), ordl.detach()


# =========================
# Set Transformer Model (must match v4 architecture exactly)
# =========================
class SetSelfAttentionBlock(nn.Module):
    def __init__(self, d_model, num_heads, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=num_heads,
            dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        hidden = int(d_model * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, d_model), nn.Dropout(dropout))

    def forward(self, x, key_padding_mask=None):
        y = self.norm1(x)
        attn_out, _ = self.attn(y, y, y, need_weights=False, key_padding_mask=key_padding_mask)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x


class ConditionalFeatureModulation(nn.Module):
    def __init__(self, cond_dim, feat_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, 2 * feat_dim))
        self.scale = nn.Parameter(torch.tensor(0.25, dtype=torch.float32))

    def forward(self, x, cond):
        gamma_beta = self.net(cond)
        gamma, beta = torch.chunk(gamma_beta, 2, dim=-1)
        if x.dim() == 3:
            gamma = gamma.unsqueeze(1)
            beta = beta.unsqueeze(1)
        return x * (1.0 + self.scale * gamma) + self.scale * beta


class SetAttentionPooling(nn.Module):
    def __init__(self, d_model, hidden_dim=128):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(d_model, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, 1))

    def forward(self, x, key_padding_mask=None):
        logits = self.score(x)
        if key_padding_mask is not None:
            logits = logits.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
        weights = torch.softmax(logits, dim=1)
        weights = torch.nan_to_num(weights, nan=0.0)
        return (weights * x).sum(dim=1)


class BERFirstTokenSetTransformerBoundedLogHalfMultiTask(nn.Module):
    def __init__(self, max_past_seq_len, token_dim=4, first_token_dim=4,
                 threshold_dim=1, global_dim=7, d_model=128, num_set_layers=4,
                 num_heads=4, mlp_ratio=4.0, dropout=0.10,
                 num_ordinal_thresholds=NUM_ORDINAL_THRESHOLDS):
        super().__init__()
        self.max_past_seq_len = max_past_seq_len
        self.d_model = d_model
        self.num_ordinal_thresholds = num_ordinal_thresholds

        self.first_proj = nn.Sequential(
            nn.Linear(first_token_dim, d_model), nn.LayerNorm(d_model),
            nn.GELU(), nn.Dropout(dropout))
        self.token_proj = nn.Sequential(
            nn.Linear(token_dim, d_model), nn.LayerNorm(d_model),
            nn.GELU(), nn.Dropout(dropout))
        self.thr_embed = nn.Sequential(
            nn.Linear(threshold_dim, 32), nn.GELU(),
            nn.Linear(32, 32), nn.GELU())
        self.global_embed = nn.Sequential(
            nn.Linear(global_dim, 96), nn.GELU(),
            nn.Linear(96, 96), nn.GELU())

        cond_dim = 32 + 96
        self.first_cond_mod = ConditionalFeatureModulation(cond_dim, d_model, 256)
        self.set_cond_mod = ConditionalFeatureModulation(cond_dim, d_model, 256)

        self.set_blocks = nn.ModuleList([
            SetSelfAttentionBlock(d_model, num_heads, mlp_ratio, dropout)
            for _ in range(num_set_layers)])

        self.first_refine = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_model),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model, d_model))
        self.final_set_norm = nn.LayerNorm(d_model)
        self.set_attn_pool = SetAttentionPooling(d_model, 128)

        head_in = d_model + 3 * d_model + cond_dim
        self.reg_head = nn.Sequential(
            nn.Linear(head_in, 256), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.15), nn.Linear(128, 1))
        self.ord_head = nn.Sequential(
            nn.Linear(head_in, 256), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.15),
            nn.Linear(128, num_ordinal_thresholds))

    @staticmethod
    def raw_to_log10ber(x):
        log10_half = torch.log10(torch.tensor(0.5, device=x.device, dtype=x.dtype))
        return log10_half - F.softplus(x)

    @staticmethod
    def raw_to_ber(x):
        return torch.pow(10.0, BERFirstTokenSetTransformerBoundedLogHalfMultiTask.raw_to_log10ber(x)).clamp(EPS, 0.5)

    def forward(self, first_token, past_tokens, global_feats, threshold, key_padding_mask=None):
        thr = self.thr_embed(threshold)
        g = self.global_embed(global_feats)
        cond = torch.cat([thr, g], dim=-1)

        first_x = self.first_proj(first_token)
        first_x = self.first_cond_mod(first_x, cond)
        first_x = first_x + self.first_refine(first_x)

        set_x = self.token_proj(past_tokens)
        set_x = self.set_cond_mod(set_x, cond)
        for block in self.set_blocks:
            set_x = block(set_x, key_padding_mask=key_padding_mask)
        set_x = self.final_set_norm(set_x)

        if key_padding_mask is not None:
            valid_mask = (~key_padding_mask).unsqueeze(-1).float()
        else:
            valid_mask = torch.ones(set_x.shape[0], set_x.shape[1], 1,
                                    device=set_x.device, dtype=set_x.dtype)

        pooled_attn = self.set_attn_pool(set_x, key_padding_mask=key_padding_mask)
        valid_counts = valid_mask.sum(dim=1).clamp(min=1.0)
        pooled_mean = (set_x * valid_mask).sum(dim=1) / valid_counts

        if key_padding_mask is not None:
            x_for_max = set_x.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
        else:
            x_for_max = set_x
        pooled_max = torch.nan_to_num(x_for_max.amax(dim=1), nan=0.0, posinf=0.0, neginf=0.0)

        fused = torch.cat([first_x, pooled_attn, pooled_mean, pooled_max, cond], dim=-1)
        return self.reg_head(fused), self.ord_head(fused)


# =========================
# Data preparation (handles missing var_* columns)
# =========================
def prepare_data_for_finetune(
    csv_paths, batch_size=256, nrows_per_dataset=2_500_000, num_workers=0,
    pretrained_scalers=None, reuse_scalers=True,
):
    """Same pipeline as v4's prepare_data, plus:
       - var_* columns are computed from taps if missing
       - mem_len column is derived from non-null taps if missing
       - if reuse_scalers=True and pretrained_scalers is provided, those
         scalers are reused (recommended for fine-tuning)
    """
    df, tap_cols, var_cols = load_and_merge_data(csv_paths, nrows_per_dataset)

    # ---- Derive mem_len if missing ----
    if "mem_len" not in df.columns:
        df["mem_len"] = derive_mem_len_from_taps(df, tap_cols)

    df = df[df["mem_len"] != 1].copy()

    # ---- Fill NaNs in tap columns with 0 (they represent padding) ----
    df[tap_cols] = df[tap_cols].fillna(0.0)
    df["threshold"] = df["threshold"].fillna(0.0)
    df["BER"] = df["BER"].fillna(0.0)
    df["N"] = df["N"].fillna(0.0)

    df = df[(df["BER"] > 0) & (df["N"] > 0)].copy()
    df["BER"] = df["BER"].clip(lower=EPS, upper=0.5)

    # Allow threshold==0 rows? In original training threshold>0 was required.
    # The new random data has rows with threshold=0; filter them out
    # because they don't correspond to a real decision threshold.
    n_before = len(df)
    df = df[df["threshold"] > 0].copy()
    if len(df) < n_before:
        print(f"  Dropped {n_before - len(df):,} rows with threshold=0")

    print(f"Rows after cleaning/filtering: {len(df):,}")

    # ---- Extract raw arrays ----
    X_taps_raw = df[tap_cols].to_numpy(dtype=np.float32)
    num_molecules = df["N"].to_numpy(dtype=np.float32).reshape(-1, 1)
    mem_len = df["mem_len"].to_numpy(dtype=np.int64)

    X_thr_raw = df["threshold"].to_numpy(dtype=np.float32).reshape(-1, 1)
    y_raw = df["BER"].to_numpy(dtype=np.float32).reshape(-1, 1)

    X_thr = np.log10(X_thr_raw + EPS).astype(np.float32)
    y_log = np.log10(y_raw + EPS).astype(np.float32)

    L_full = X_taps_raw.shape[1]
    mem_len_clamped = np.minimum(mem_len, L_full)
    valid_full = (np.arange(L_full)[None, :] < mem_len_clamped[:, None])
    valid_past = valid_full[:, 1:]
    valid_past_float = valid_past.astype(np.float32)
    past_lens = np.maximum(mem_len_clamped - 1, 0).astype(np.int64)

    print(f"  Tap column count L_full = {L_full}, "
          f"mem_len range = [{int(mem_len.min())}, {int(mem_len.max())}], "
          f"past_lens range = [{int(past_lens.min())}, {int(past_lens.max())}]")
    if mem_len.max() > L_full:
        print(f"    NOTE: {(mem_len > L_full).sum():,} rows have mem_len > L_full; "
              f"will be clamped to {L_full}")

    # ---- Variances: load from CSV if present, else compute from taps ----
    if var_cols and all(c in df.columns for c in var_cols):
        # File has var columns — use them, fill missing with computed values per row
        X_vars_raw_csv = df[var_cols].to_numpy(dtype=np.float32)

        # Identify rows where _has_vars indicates the file had var columns
        if "_has_vars" in df.columns:
            has_vars_mask = df["_has_vars"].to_numpy(dtype=bool)
        else:
            has_vars_mask = np.ones(len(df), dtype=bool)

        # Compute variance for ALL rows, then overlay CSV values where available
        X_vars_computed = compute_variances_from_taps(X_taps_raw, num_molecules.reshape(-1), valid_full)

        X_vars_raw = X_vars_computed.copy()
        # For rows that DO have var columns in CSV, use those values where non-NaN
        has_vars_idx = np.where(has_vars_mask)[0]
        if len(has_vars_idx) > 0:
            csv_vars = np.where(np.isnan(X_vars_raw_csv[has_vars_idx]),
                                X_vars_computed[has_vars_idx],
                                X_vars_raw_csv[has_vars_idx])
            X_vars_raw[has_vars_idx] = csv_vars

        n_computed = (~has_vars_mask).sum()
        print(f"  Variances: {has_vars_mask.sum():,} loaded from CSV, "
              f"{n_computed:,} computed from N*tap*(1-tap)")
    else:
        # No var columns at all — compute everything
        X_vars_raw = compute_variances_from_taps(
            X_taps_raw, num_molecules.reshape(-1), valid_full)
        print(f"  Variances: all {len(df):,} computed from N*tap*(1-tap)")

    # Sanity: variances should be non-negative and zero on padding
    if np.any(X_vars_raw < 0):
        # Numerical garbage; clamp
        X_vars_raw = np.maximum(X_vars_raw, 0.0)

    # ---- Feature engineering (matches v4) ----
    X_taps_feat = (X_taps_raw * num_molecules).astype(np.float32)
    X_vars_feat = X_vars_raw.astype(np.float32)
    abs_taps_raw = np.abs(X_taps_feat).astype(np.float32)
    snr_raw = np.log10((X_taps_feat ** 2) / (X_vars_raw + EPS) + EPS).astype(np.float32)

    # ---- Mask-aware sums ----
    first_mean = X_taps_feat[:, 0:1]
    past_means = X_taps_feat[:, 1:]
    first_var = X_vars_feat[:, 0:1]
    past_vars = X_vars_feat[:, 1:]

    past_means_masked = past_means * valid_past_float
    past_vars_masked = past_vars * valid_past_float

    mu0_raw = (0.5 * np.sum(past_means_masked, axis=1, keepdims=True)).astype(np.float32)
    mu1_raw = (first_mean + mu0_raw).astype(np.float32)
    var0_raw = (0.5 * np.sum(past_vars_masked, axis=1, keepdims=True)).astype(np.float32)
    var1_raw = (first_var + var0_raw).astype(np.float32)

    std0_raw = np.sqrt(np.maximum(var0_raw, EPS)).astype(np.float32)
    std1_raw = np.sqrt(np.maximum(var1_raw, EPS)).astype(np.float32)

    z0_raw = ((X_thr_raw - mu0_raw) / (std0_raw + EPS)).astype(np.float32)
    z1_raw = ((mu1_raw - X_thr_raw) / (std1_raw + EPS)).astype(np.float32)

    harmonic_side_z_raw = (
        2.0 / (1.0 / (z0_raw + EPS) + 1.0 / (z1_raw + EPS))
    ).astype(np.float32)
    abs_diff_side_z_raw = np.abs(z0_raw - z1_raw).astype(np.float32)
    harmonic_minus_gap_raw = (
        harmonic_side_z_raw - 0.25 * abs_diff_side_z_raw
    ).astype(np.float32)

    # ---- Scenario-level features ----
    signal_raw = first_mean
    isi_raw = np.sum(past_means_masked, axis=1, keepdims=True)
    nsid_raw = ((signal_raw - isi_raw) / (signal_raw + isi_raw + EPS)).astype(np.float32)

    gap_raw = signal_raw
    d0_raw = (gap_raw / (std0_raw + EPS)).astype(np.float32)
    d1_raw = (gap_raw / (std1_raw + EPS)).astype(np.float32)
    harmonic_discrim_raw = (2.0 * d0_raw * d1_raw / (d0_raw + d1_raw + EPS)).astype(np.float32)
    log_harmonic_discrim_raw = np.log10(harmonic_discrim_raw + EPS).astype(np.float32)
    discrim_asymmetry_raw = (np.abs(d0_raw - d1_raw) / (d0_raw + d1_raw + EPS)).astype(np.float32)

    past_taps_sum = np.sum(past_means_masked, axis=1, keepdims=True)
    past_shares = past_means_masked / (past_taps_sum + EPS)
    herfindahl_raw = np.sum(past_shares ** 2, axis=1, keepdims=True).astype(np.float32)

    # ---- Labels ----
    region_labels = raw_to_region_labels_np(y_raw)
    ordinal_targets = region_labels_to_ordinal_targets_np(region_labels)
    strat_labels = make_strat_bins(y_log, n_bins=10)

    # ---- Train / temp split ----
    split_args = dict(test_size=0.30, random_state=42)
    if strat_labels is not None:
        split_args["stratify"] = strat_labels

    (
        t_taps_feat, temp_taps_feat,
        t_vars_feat, temp_vars_feat,
        t_abs_raw, temp_abs_raw,
        t_snr_raw, temp_snr_raw,
        t_z0_raw, temp_z0_raw,
        t_z1_raw, temp_z1_raw,
        t_hmg_raw, temp_hmg_raw,
        t_nsid_raw, temp_nsid_raw,
        t_log_hd_raw, temp_log_hd_raw,
        t_da_raw, temp_da_raw,
        t_herf_raw, temp_herf_raw,
        t_thr, temp_thr,
        t_y_log, temp_y_log,
        t_region, temp_region,
        t_ord, temp_ord,
        t_past_lens, temp_past_lens,
    ) = train_test_split(
        X_taps_feat, X_vars_feat, abs_taps_raw, snr_raw,
        z0_raw, z1_raw, harmonic_minus_gap_raw,
        nsid_raw, log_harmonic_discrim_raw, discrim_asymmetry_raw, herfindahl_raw,
        X_thr,
        y_log, region_labels, ordinal_targets, past_lens,
        **split_args,
    )

    temp_strat = make_strat_bins(temp_y_log, n_bins=6)
    split_args2 = dict(test_size=0.50, random_state=42)
    if temp_strat is not None:
        split_args2["stratify"] = temp_strat

    (
        v_taps_feat, te_taps_feat,
        v_vars_feat, te_vars_feat,
        v_abs_raw, te_abs_raw,
        v_snr_raw, te_snr_raw,
        v_z0_raw, te_z0_raw,
        v_z1_raw, te_z1_raw,
        v_hmg_raw, te_hmg_raw,
        v_nsid_raw, te_nsid_raw,
        v_log_hd_raw, te_log_hd_raw,
        v_da_raw, te_da_raw,
        v_herf_raw, te_herf_raw,
        v_thr, te_thr,
        v_y_log, te_y_log,
        v_region, te_region,
        v_ord, te_ord,
        v_past_lens, te_past_lens,
    ) = train_test_split(
        temp_taps_feat, temp_vars_feat, temp_abs_raw, temp_snr_raw,
        temp_z0_raw, temp_z1_raw, temp_hmg_raw,
        temp_nsid_raw, temp_log_hd_raw, temp_da_raw, temp_herf_raw,
        temp_thr,
        temp_y_log, temp_region, temp_ord, temp_past_lens,
        **split_args2,
    )

    L_past = L_full - 1
    t_valid_past = (np.arange(L_past)[None, :] < t_past_lens[:, None])

    # =========================================================
    # Scalers: reuse pretrained or fit from scratch
    # =========================================================
    if reuse_scalers and pretrained_scalers is not None:
        print("  Reusing pretrained scalers (recommended for fine-tuning).")

        first_tap_mean = pretrained_scalers["first_tap_mean"]
        first_tap_std = pretrained_scalers["first_tap_std"]
        first_var_mean = pretrained_scalers["first_var_mean"]
        first_var_std = pretrained_scalers["first_var_std"]
        first_abs_mean = pretrained_scalers["first_abs_mean"]
        first_abs_std = pretrained_scalers["first_abs_std"]
        first_snr_mean = pretrained_scalers["first_snr_mean"]
        first_snr_std = pretrained_scalers["first_snr_std"]

        past_tap_mean = pretrained_scalers["past_tap_mean"]
        past_tap_std = pretrained_scalers["past_tap_std"]
        past_var_mean = pretrained_scalers["past_var_mean"]
        past_var_std = pretrained_scalers["past_var_std"]
        past_abs_mean = pretrained_scalers["past_abs_mean"]
        past_abs_std = pretrained_scalers["past_abs_std"]
        past_snr_mean = pretrained_scalers["past_snr_mean"]
        past_snr_std = pretrained_scalers["past_snr_std"]

        z0_scaler = pretrained_scalers["z0_scaler"]
        z1_scaler = pretrained_scalers["z1_scaler"]
        hmg_scaler = pretrained_scalers["harmonic_minus_gap_scaler"]
        nsid_scaler = pretrained_scalers["nsid_scaler"]
        log_hd_scaler = pretrained_scalers["log_hd_scaler"]
        da_scaler = pretrained_scalers["da_scaler"]
        herf_scaler = pretrained_scalers["herf_scaler"]
        thr_scaler = pretrained_scalers["thr_scaler"]
    else:
        print("  Refitting scalers on fine-tune data.")
        first_tap_mean, first_tap_std = float(t_taps_feat[:, 0].mean()), max(float(t_taps_feat[:, 0].std()), 1e-12)
        first_var_mean, first_var_std = float(t_vars_feat[:, 0].mean()), max(float(t_vars_feat[:, 0].std()), 1e-12)
        first_abs_mean, first_abs_std = float(t_abs_raw[:, 0].mean()), max(float(t_abs_raw[:, 0].std()), 1e-12)
        first_snr_mean, first_snr_std = float(t_snr_raw[:, 0].mean()), max(float(t_snr_raw[:, 0].std()), 1e-12)

        past_tap_mean, past_tap_std = fit_shared_scalar_scaler(t_taps_feat[:, 1:], t_valid_past)
        past_var_mean, past_var_std = fit_shared_scalar_scaler(t_vars_feat[:, 1:], t_valid_past)
        past_abs_mean, past_abs_std = fit_shared_scalar_scaler(t_abs_raw[:, 1:], t_valid_past)
        past_snr_mean, past_snr_std = fit_shared_scalar_scaler(t_snr_raw[:, 1:], t_valid_past)

        z0_scaler = StandardScaler().fit(t_z0_raw)
        z1_scaler = StandardScaler().fit(t_z1_raw)
        hmg_scaler = StandardScaler().fit(t_hmg_raw)
        nsid_scaler = StandardScaler().fit(t_nsid_raw)
        log_hd_scaler = StandardScaler().fit(t_log_hd_raw)
        da_scaler = StandardScaler().fit(t_da_raw)
        herf_scaler = StandardScaler().fit(t_herf_raw)
        thr_scaler = StandardScaler().fit(t_thr)

    def build_tokens(taps_feat, vars_feat, abs_raw, snr_raw, p_lens):
        ft_tap = ((taps_feat[:, 0] - first_tap_mean) / first_tap_std).astype(np.float32)
        ft_var = ((vars_feat[:, 0] - first_var_mean) / first_var_std).astype(np.float32)
        ft_abs = ((abs_raw[:, 0] - first_abs_mean) / first_abs_std).astype(np.float32)
        ft_snr = ((snr_raw[:, 0] - first_snr_mean) / first_snr_std).astype(np.float32)
        first_token = np.stack([ft_tap, ft_var, ft_abs, ft_snr], axis=1).astype(np.float32)

        pt_valid = (np.arange(L_past)[None, :] < p_lens[:, None])
        pt_tap = apply_shared_scale(taps_feat[:, 1:], past_tap_mean, past_tap_std, pt_valid)
        pt_var = apply_shared_scale(vars_feat[:, 1:], past_var_mean, past_var_std, pt_valid)
        pt_abs = apply_shared_scale(abs_raw[:, 1:], past_abs_mean, past_abs_std, pt_valid)
        pt_snr = apply_shared_scale(snr_raw[:, 1:], past_snr_mean, past_snr_std, pt_valid)
        past_tokens = np.stack([pt_tap, pt_var, pt_abs, pt_snr], axis=2).astype(np.float32)

        return first_token, past_tokens

    def build_global(z0, z1, hmg, nsid, log_hd, da, herf):
        return np.concatenate([
            z0_scaler.transform(z0),
            z1_scaler.transform(z1),
            hmg_scaler.transform(hmg),
            nsid_scaler.transform(nsid),
            log_hd_scaler.transform(log_hd),
            da_scaler.transform(da),
            herf_scaler.transform(herf),
        ], axis=1).astype(np.float32)

    def build_padding_mask(p_lens, L_max):
        return (np.arange(L_max)[None, :] >= p_lens[:, None])

    t_first, t_past = build_tokens(t_taps_feat, t_vars_feat, t_abs_raw, t_snr_raw, t_past_lens)
    v_first, v_past = build_tokens(v_taps_feat, v_vars_feat, v_abs_raw, v_snr_raw, v_past_lens)
    te_first, te_past = build_tokens(te_taps_feat, te_vars_feat, te_abs_raw, te_snr_raw, te_past_lens)

    t_global = build_global(t_z0_raw, t_z1_raw, t_hmg_raw,
                            t_nsid_raw, t_log_hd_raw, t_da_raw, t_herf_raw)
    v_global = build_global(v_z0_raw, v_z1_raw, v_hmg_raw,
                            v_nsid_raw, v_log_hd_raw, v_da_raw, v_herf_raw)
    te_global = build_global(te_z0_raw, te_z1_raw, te_hmg_raw,
                             te_nsid_raw, te_log_hd_raw, te_da_raw, te_herf_raw)

    t_thr_s = thr_scaler.transform(t_thr).astype(np.float32)
    v_thr_s = thr_scaler.transform(v_thr).astype(np.float32)
    te_thr_s = thr_scaler.transform(te_thr).astype(np.float32)

    L_max_past = L_past
    t_mask = build_padding_mask(t_past_lens, L_max_past)
    v_mask = build_padding_mask(v_past_lens, L_max_past)
    te_mask = build_padding_mask(te_past_lens, L_max_past)

    train_ds = TensorDataset(
        torch.from_numpy(t_first), torch.from_numpy(t_past),
        torch.from_numpy(t_global), torch.from_numpy(t_thr_s),
        torch.from_numpy(t_y_log.astype(np.float32)),
        torch.from_numpy(t_ord.astype(np.float32)),
        torch.from_numpy(t_region.astype(np.int64)),
        torch.from_numpy(t_mask))
    val_ds = TensorDataset(
        torch.from_numpy(v_first), torch.from_numpy(v_past),
        torch.from_numpy(v_global), torch.from_numpy(v_thr_s),
        torch.from_numpy(v_y_log.astype(np.float32)),
        torch.from_numpy(v_ord.astype(np.float32)),
        torch.from_numpy(v_region.astype(np.int64)),
        torch.from_numpy(v_mask))
    test_ds = TensorDataset(
        torch.from_numpy(te_first), torch.from_numpy(te_past),
        torch.from_numpy(te_global), torch.from_numpy(te_thr_s),
        torch.from_numpy(te_y_log.astype(np.float32)),
        torch.from_numpy(te_ord.astype(np.float32)),
        torch.from_numpy(te_region.astype(np.int64)),
        torch.from_numpy(te_mask))

    # ---- Sampler (region + SIR-aware) ----
    region_sample_weights = np.ones_like(t_region, dtype=np.float32)
    region_sample_weights[t_region == 6] = 2.5
    region_sample_weights[t_region == 7] = 3.0
    region_sample_weights[t_region == 8] = 5.0
    region_sample_weights[t_region == 9] = 5.0
    region_sample_weights[t_region == 10] = 5.0
    region_sample_weights[t_region == 11] = 5.0
    region_sample_weights[t_region == 12] = 3.0
    region_sample_weights[t_region == 13] = 2.5

    t_signal = t_taps_feat[:, 0]
    t_valid_past_for_sir = (np.arange(L_past)[None, :] < t_past_lens[:, None]).astype(np.float32)
    t_isi = np.sum(t_taps_feat[:, 1:] * t_valid_past_for_sir, axis=1)
    t_sir = t_signal / (t_isi + EPS)

    sir_sample_weights = np.ones_like(t_sir, dtype=np.float32)
    sir_sample_weights[t_sir < 0.26] = 3.0
    sir_sample_weights[(t_sir >= 0.26) & (t_sir < 0.36)] = 2.0

    combined_sample_weights = region_sample_weights * sir_sample_weights

    sampler = WeightedRandomSampler(
        weights=torch.from_numpy(combined_sample_weights),
        num_samples=len(combined_sample_weights),
        replacement=True)

    pin_mem = torch.cuda.is_available()
    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler,
                              shuffle=False, pin_memory=pin_mem, num_workers=num_workers)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                            pin_memory=pin_mem, num_workers=num_workers)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                             pin_memory=pin_mem, num_workers=num_workers)

    class_weights = compute_region_class_weights(t_region, NUM_REGION_CLASSES)
    ordinal_pos_weights = compute_ordinal_pos_weights_from_region_labels(
        t_region, num_thresholds=NUM_ORDINAL_THRESHOLDS, max_weight=20.0)

    scalers = {
        "first_tap_mean": first_tap_mean, "first_tap_std": first_tap_std,
        "first_var_mean": first_var_mean, "first_var_std": first_var_std,
        "first_abs_mean": first_abs_mean, "first_abs_std": first_abs_std,
        "first_snr_mean": first_snr_mean, "first_snr_std": first_snr_std,
        "past_tap_mean": past_tap_mean, "past_tap_std": past_tap_std,
        "past_var_mean": past_var_mean, "past_var_std": past_var_std,
        "past_abs_mean": past_abs_mean, "past_abs_std": past_abs_std,
        "past_snr_mean": past_snr_mean, "past_snr_std": past_snr_std,
        "z0_scaler": z0_scaler, "z1_scaler": z1_scaler,
        "harmonic_minus_gap_scaler": hmg_scaler,
        "nsid_scaler": nsid_scaler, "log_hd_scaler": log_hd_scaler,
        "da_scaler": da_scaler, "herf_scaler": herf_scaler,
        "thr_scaler": thr_scaler,
        "tap_cols": tap_cols, "var_cols": var_cols,
        "first_token_dim": 4, "past_token_dim": 4,
        "global_dim": 7,
        "train_max_past_seq_len": L_past,
        "scaling_strategy": "position_independent",
        "uses_positional_encoding": False,
        "permutation_invariance_post_first": True,
        "variable_length_support": True,
        "target_parameterization": "pred_log10_ber = log10(0.5) - softplus(raw_out)",
        "ordinal_thresholds": ORDINAL_THRESHOLDS,
        "num_region_classes": NUM_REGION_CLASSES,
        "class_weights": class_weights.tolist(),
        "ordinal_pos_weights": ordinal_pos_weights.tolist(),
        "data_paths": csv_paths,
        "nrows_per_dataset": nrows_per_dataset,
        "finetuned_from": PRETRAINED_MODEL_PATH,
        "scalers_reused": reuse_scalers,
        "variances_computed_from_taps": True,
    }

    aux_info = {
        "class_weights": class_weights,
        "ordinal_pos_weights": ordinal_pos_weights,
        "t_region": t_region,
    }

    return train_loader, val_loader, test_loader, scalers, aux_info, L_past


# =========================
# Evaluation (same as v4)
# =========================
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_reg_loss = 0.0
    total_ord_loss = 0.0
    all_preds_log = []
    all_targets_log = []
    all_pred_regions = []
    all_true_regions = []

    with torch.no_grad():
        for b_first, b_past, b_global, b_thr, b_y_log, b_ord, b_region, b_mask in loader:
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_y_log = b_y_log.to(device, non_blocking=True)
            b_ord = b_ord.to(device, non_blocking=True)
            b_region = b_region.to(device, non_blocking=True)
            b_mask = b_mask.to(device, non_blocking=True)

            pred_raw_unconstrained, ord_logits = model(
                b_first, b_past, b_global, b_thr, key_padding_mask=b_mask)

            if has_nonfinite_tensor(pred_raw_unconstrained):
                raise RuntimeError("Non-finite prediction during evaluation.")
            if has_nonfinite_tensor(ord_logits):
                raise RuntimeError("Non-finite ordinal logits during evaluation.")

            loss, reg_loss, ord_loss = criterion(
                pred_raw_unconstrained, ord_logits, b_y_log, b_ord)

            if has_nonfinite_tensor(loss):
                raise RuntimeError("Non-finite loss during evaluation.")

            pred_log = model.raw_to_log10ber(pred_raw_unconstrained)
            pred_region = ordinal_logits_to_region_labels_torch(ord_logits)

            total_loss += loss.item()
            total_reg_loss += reg_loss.item()
            total_ord_loss += ord_loss.item()

            all_preds_log.append(pred_log.cpu().numpy())
            all_targets_log.append(b_y_log.cpu().numpy())
            all_pred_regions.append(pred_region.cpu().numpy())
            all_true_regions.append(b_region.cpu().numpy())

    n_batches = max(len(loader), 1)
    avg_loss = total_loss / n_batches
    avg_reg_loss = total_reg_loss / n_batches
    avg_ord_loss = total_ord_loss / n_batches

    preds_log = np.vstack(all_preds_log)
    targets_log = np.vstack(all_targets_log)
    preds_raw = np.clip(10 ** preds_log, EPS, 0.5)
    targets_raw = np.clip(10 ** targets_log, EPS, 0.5)

    rmse_log = float(np.sqrt(np.mean((preds_log - targets_log) ** 2)))
    mae_log = float(np.mean(np.abs(preds_log - targets_log)))
    factor_error = float(10 ** rmse_log)
    rmse_raw = float(np.sqrt(np.mean((preds_raw - targets_raw) ** 2)))
    mae_raw = float(np.mean(np.abs(preds_raw - targets_raw)))
    rel_err = (preds_raw - targets_raw) / np.maximum(targets_raw, 1e-6)
    rmse_rel = float(np.sqrt(np.mean(rel_err ** 2)))
    mae_rel = float(np.mean(np.abs(rel_err)))

    pred_regions = np.concatenate(all_pred_regions).reshape(-1)
    true_regions = np.concatenate(all_true_regions).reshape(-1)
    region_acc = float(np.mean(pred_regions == true_regions))
    region_mae = float(np.mean(np.abs(pred_regions - true_regions)))

    return {
        "loss": avg_loss, "reg_loss": avg_reg_loss, "ord_loss": avg_ord_loss,
        "rmse_log": rmse_log, "mae_log": mae_log, "factor_error": factor_error,
        "rmse_raw": rmse_raw, "mae_raw": mae_raw,
        "rmse_rel": rmse_rel, "mae_rel": mae_rel,
        "region_acc": region_acc, "region_mae": region_mae,
    }


def evaluate_by_target_range(model, loader, device):
    model.eval()
    all_preds_log = []
    all_targets_log = []

    with torch.no_grad():
        for b_first, b_past, b_global, b_thr, b_y_log, _, _, b_mask in loader:
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_mask = b_mask.to(device, non_blocking=True)

            pred_raw_unconstrained, _ = model(
                b_first, b_past, b_global, b_thr, key_padding_mask=b_mask)
            if has_nonfinite_tensor(pred_raw_unconstrained):
                raise RuntimeError("Non-finite prediction during per-range evaluation.")
            pred_log = model.raw_to_log10ber(pred_raw_unconstrained)
            all_preds_log.append(pred_log.cpu().numpy())
            all_targets_log.append(b_y_log.numpy())

    preds_log = np.vstack(all_preds_log).reshape(-1)
    targets_log = np.vstack(all_targets_log).reshape(-1)
    preds_raw = np.clip(10 ** preds_log, EPS, 0.5)
    targets_raw = np.clip(10 ** targets_log, EPS, 0.5)

    ranges = {
        "low_BER(y<1e-6)": targets_raw < 1e-6,
        "mid_BER(1e-6<=y<1e-3)": (targets_raw >= 1e-6) & (targets_raw < 1e-3),
        "high_BER(1e-3<=y<0.10)": (targets_raw >= 1e-3) & (targets_raw < 0.10),
        "upper_BER(0.10<=y<0.20)": (targets_raw >= 0.10) & (targets_raw < 0.20),
        "target_BER(0.20<=y<0.40)": (targets_raw >= 0.20) & (targets_raw < 0.40),
        "very_high_BER(0.40<=y<=0.50)": targets_raw >= 0.40,
    }

    metrics = {}
    for name, mask in ranges.items():
        if np.any(mask):
            err_log = preds_log[mask] - targets_log[mask]
            err_raw = preds_raw[mask] - targets_raw[mask]
            rel_err = err_raw / np.maximum(targets_raw[mask], 1e-6)
            metrics[name] = {
                "count": int(mask.sum()),
                "rmse_log": float(np.sqrt(np.mean(err_log ** 2))),
                "mae_log": float(np.mean(np.abs(err_log))),
                "factor_error": float(10 ** float(np.sqrt(np.mean(err_log ** 2)))),
                "rmse_raw": float(np.sqrt(np.mean(err_raw ** 2))),
                "mae_raw": float(np.mean(np.abs(err_raw))),
                "rmse_rel": float(np.sqrt(np.mean(rel_err ** 2))),
                "mae_rel": float(np.mean(np.abs(rel_err))),
                "bias_raw": float(np.mean(err_raw)),
                "bias_log": float(np.mean(err_log)),
                "p90_abs_raw": float(np.percentile(np.abs(err_raw), 90)),
                "p95_abs_raw": float(np.percentile(np.abs(err_raw), 95)),
            }
        else:
            metrics[name] = None
    return metrics


# =========================
# Selection scoring (same as v4)
# =========================
SELECTION_REGION_WEIGHTS = {
    "low_BER(y<1e-6)": 0.25,
    "mid_BER(1e-6<=y<1e-3)": 0.50,
    "high_BER(1e-3<=y<0.10)": 1.00,
    "upper_BER(0.10<=y<0.20)": 2.00,
    "target_BER(0.20<=y<0.40)": 3.00,
    "very_high_BER(0.40<=y<=0.50)": 1.50,
}
SELECTION_W_LOG_RMSE = 1.00
SELECTION_W_BIAS = 0.50
SELECTION_W_TAIL = 0.05
SELECTION_TAIL_CAP = 5.0


def compute_selection_score(val_range_metrics, val_metrics, min_count=50):
    log_rmse_sum = 0.0
    bias_sum = 0.0
    tail_sum = 0.0
    total_w = 0.0

    for region_name, w in SELECTION_REGION_WEIGHTS.items():
        m = val_range_metrics.get(region_name)
        if m is None or m["count"] < min_count:
            continue
        log_rmse_sum += w * m["rmse_log"]
        bias_sum += w * abs(m["bias_log"])
        if m["rmse_raw"] > 1e-9:
            tail_ratio = m["p95_abs_raw"] / (m["rmse_raw"] + 1e-9)
            tail_sum += w * min(tail_ratio, SELECTION_TAIL_CAP)
        else:
            tail_sum += w * 1.0
        total_w += w

    if total_w == 0:
        return {
            "composite": float(val_metrics["rmse_log"]),
            "log_rmse_weighted": float(val_metrics["rmse_log"]),
            "bias_weighted": 0.0, "tail_weighted": 0.0,
            "geometric_factor_error": float(val_metrics["factor_error"]),
            "fallback": True,
        }

    log_rmse_weighted = log_rmse_sum / total_w
    bias_weighted = bias_sum / total_w
    tail_weighted = tail_sum / total_w
    composite = (SELECTION_W_LOG_RMSE * log_rmse_weighted
                 + SELECTION_W_BIAS * bias_weighted
                 + SELECTION_W_TAIL * tail_weighted)
    return {
        "composite": float(composite),
        "log_rmse_weighted": float(log_rmse_weighted),
        "bias_weighted": float(bias_weighted),
        "tail_weighted": float(tail_weighted),
        "geometric_factor_error": float(10 ** log_rmse_weighted),
        "fallback": False,
    }


def is_acceptable_checkpoint(val_range_metrics, val_metrics):
    target = val_range_metrics.get("target_BER(0.20<=y<0.40)")
    if target is None or target["count"] < 100:
        return False, "target region too small"
    for region_name, m in val_range_metrics.items():
        if m is None or m["count"] < 50:
            continue
        if abs(m["bias_log"]) > 0.15:
            return False, f"{region_name} has bias_log={m['bias_log']:.3f}"
    if val_metrics["rmse_log"] > 0.30:
        return False, f"overall rmse_log={val_metrics['rmse_log']:.3f} too high"
    return True, "ok"


# =========================
# Fine-tune main
# =========================
def finetune():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Executing on: {device}")
    print(f"\nFINE-TUNING from: {PRETRAINED_MODEL_PATH}")
    print(f"  Pretrained scalers: {PRETRAINED_SCALER_PATH}")
    print(f"  Reuse scalers: {REUSE_SCALERS}")

    if not os.path.exists(PRETRAINED_MODEL_PATH):
        raise FileNotFoundError(f"Pretrained model not found: {PRETRAINED_MODEL_PATH}")
    if not os.path.exists(PRETRAINED_SCALER_PATH):
        raise FileNotFoundError(f"Pretrained scalers not found: {PRETRAINED_SCALER_PATH}")

    pretrained_scalers = joblib.load(PRETRAINED_SCALER_PATH)
    print(f"  Pretrained global_dim: {pretrained_scalers.get('global_dim', '?')}")

    # ---- Prepare data ----
    train_loader, val_loader, test_loader, scalers, aux_info, max_past_seq_len = (
        prepare_data_for_finetune(
            DATA_PATHS, batch_size=256,
            nrows_per_dataset=NROWS_PER_DATASET, num_workers=0,
            pretrained_scalers=pretrained_scalers, reuse_scalers=REUSE_SCALERS,
        )
    )

    joblib.dump(scalers, SCALER_SAVE_PATH)
    print(f"\nFine-tune scalers saved to: {SCALER_SAVE_PATH}")

    # ---- Build model & load pretrained weights ----
    model = BERFirstTokenSetTransformerBoundedLogHalfMultiTask(
        max_past_seq_len=max_past_seq_len,
        token_dim=4, first_token_dim=4, threshold_dim=1,
        global_dim=7, d_model=128, num_set_layers=4,
        num_heads=4, mlp_ratio=4.0, dropout=0.10,
        num_ordinal_thresholds=NUM_ORDINAL_THRESHOLDS,
    ).to(device)

    pretrained_state = torch.load(PRETRAINED_MODEL_PATH, map_location=device, weights_only=True)
    missing, unexpected = model.load_state_dict(pretrained_state, strict=False)
    if missing:
        print(f"WARNING: missing keys when loading pretrained: {missing}")
    if unexpected:
        print(f"WARNING: unexpected keys when loading pretrained: {unexpected}")
    print(f"Pretrained weights loaded.")

    # ---- Pre-fine-tune evaluation (sanity check) ----
    print("\n--- Pre-fine-tune evaluation (pretrained model on new data) ---")
    pre_criterion = MultiTaskBERLoss(
        StableMultiObjectiveBERBoundedLogLoss(),
        OrdinalBCELoss(),
        lambda_ord=0.25,
    )
    try:
        pre_val = evaluate(model, val_loader, pre_criterion, device)
        pre_range = evaluate_by_target_range(model, val_loader, device)
        pre_sel = compute_selection_score(pre_range, pre_val)
        print(f"  Pretrained on new val: composite={pre_sel['composite']:.5f} | "
              f"factor~{pre_sel['geometric_factor_error']:.3f}x | "
              f"RMSE(log10)={pre_val['rmse_log']:.4f} | "
              f"region_acc={pre_val['region_acc']:.4f}")
        for rng, m in pre_range.items():
            if m is not None:
                print(f"    {rng}: factor~{m['factor_error']:.3f}x, "
                      f"bias_log={m['bias_log']:+.4f}, count={m['count']}")
    except RuntimeError as e:
        print(f"  Pre-eval failed: {e}")

    # ---- Optimizer & loss for fine-tuning ----
    print(f"\nFine-tune hyperparameters:")
    print(f"  LR={FINETUNE_LR}, weight_decay={FINETUNE_WEIGHT_DECAY}")
    print(f"  max_epochs={FINETUNE_MAX_EPOCHS}, min_epochs={FINETUNE_MIN_EPOCHS}, "
          f"patience={FINETUNE_PATIENCE}")

    optimizer = optim.AdamW(model.parameters(), lr=FINETUNE_LR,
                            weight_decay=FINETUNE_WEIGHT_DECAY)

    reg_loss = StableMultiObjectiveBERBoundedLogLoss(
        log_delta=0.35, raw_delta=0.006, rel_delta=0.03,
        alpha_log=0.45, beta_raw=0.35, gamma_rel=0.20,
        use_regime_weights=True)

    boosted_pos = aux_info["ordinal_pos_weights"].copy()
    for i, thr in enumerate(ORDINAL_THRESHOLDS):
        if 0.20 <= thr <= 0.40:
            boosted_pos[i] *= 2.5
        elif 0.15 <= thr < 0.20:
            boosted_pos[i] *= 1.5
        elif 0.40 < thr <= 0.45:
            boosted_pos[i] *= 1.5

    ord_loss = OrdinalBCELoss(
        pos_weight=torch.tensor(boosted_pos, dtype=torch.float32, device=device),
        reduction="mean")
    criterion = MultiTaskBERLoss(reg_loss=reg_loss, ord_loss=ord_loss, lambda_ord=0.25)

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=SCHEDULER_PATIENCE, factor=SCHEDULER_FACTOR)

    # ---- Multi-component selection state ----
    best_checkpoints = {
        "composite": {"score": float("inf"), "state": None, "epoch": -1},
        "composite_ema": {"score": float("inf"), "state": None, "epoch": -1},
        "log_rmse": {"score": float("inf"), "state": None, "epoch": -1},
        "target_mae": {"score": float("inf"), "state": None, "epoch": -1},
        "low_bias": {"score": float("inf"), "state": None, "epoch": -1},
    }
    EMA_ALPHA = 0.5
    ema_composite = None
    wait = 0

    # Seed best with pre-fine-tune score so fine-tuning must IMPROVE on the
    # pretrained baseline before being marked as best.
    try:
        base_state = copy.deepcopy(model.state_dict())
        pre_acceptable, _ = is_acceptable_checkpoint(pre_range, pre_val)
        if pre_acceptable:
            for ckpt_name, score_value in [
                ("composite", pre_sel["composite"]),
                ("composite_ema", pre_sel["composite"]),
                ("log_rmse", pre_sel["log_rmse_weighted"]),
                ("target_mae", pre_range["target_BER(0.20<=y<0.40)"]["mae_raw"]
                    if pre_range.get("target_BER(0.20<=y<0.40)") else float("inf")),
                ("low_bias", pre_sel["bias_weighted"]),
            ]:
                best_checkpoints[ckpt_name] = {
                    "score": float(score_value),
                    "state": base_state,
                    "epoch": 0,  # 0 = pretrained baseline
                }
            print("\nSeeded best checkpoints with pretrained baseline scores. "
                  "Fine-tune must improve to take over.")
    except Exception as e:
        print(f"  Could not seed pretrained baseline as best: {e}")

    print(f"\nSelection: weighted log-RMSE + bias + tail penalties")
    print(f"  Region weights: {SELECTION_REGION_WEIGHTS}")
    print(f"  Composite weights: log_rmse={SELECTION_W_LOG_RMSE}, "
          f"bias={SELECTION_W_BIAS}, tail={SELECTION_W_TAIL}")
    print(f"  EMA alpha: {EMA_ALPHA}\n")

    training_broke = False

    for epoch in range(FINETUNE_MAX_EPOCHS):
        model.train()

        for batch_idx, (b_first, b_past, b_global, b_thr, b_y_log, b_ord, _, b_mask) in enumerate(train_loader):
            b_first = b_first.to(device, non_blocking=True)
            b_past = b_past.to(device, non_blocking=True)
            b_global = b_global.to(device, non_blocking=True)
            b_thr = b_thr.to(device, non_blocking=True)
            b_y_log = b_y_log.to(device, non_blocking=True)
            b_ord = b_ord.to(device, non_blocking=True)
            b_mask = b_mask.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            pred_raw_unconstrained, ord_logits = model(
                b_first, b_past, b_global, b_thr, key_padding_mask=b_mask)

            if has_nonfinite_tensor(pred_raw_unconstrained):
                print(f"Non-finite prediction at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True; break
            if has_nonfinite_tensor(ord_logits):
                print(f"Non-finite ordinal logits at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True; break

            loss, reg_part, ord_part = criterion(
                pred_raw_unconstrained, ord_logits, b_y_log, b_ord)
            if has_nonfinite_tensor(loss):
                print(f"Non-finite loss at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True; break

            loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
            if not torch.isfinite(grad_norm):
                print(f"Non-finite gradient norm at epoch {epoch+1}, batch {batch_idx+1}.")
                training_broke = True; break

            optimizer.step()

            bad_param = False
            for nm, param in model.named_parameters():
                if param.requires_grad and param.data is not None and not torch.isfinite(param.data).all():
                    print(f"Non-finite parameter after optimizer step: {nm}")
                    bad_param = True; break
            if bad_param:
                training_broke = True; break

        if training_broke:
            print("Training stopped due to non-finite values.")
            break

        try:
            train_metrics = evaluate(model, train_loader, criterion, device)
            val_metrics = evaluate(model, val_loader, criterion, device)
            val_range_metrics = evaluate_by_target_range(model, val_loader, device)
        except RuntimeError as e:
            print(f"Evaluation failed at epoch {epoch+1}: {e}")
            break

        scheduler.step(val_metrics["loss"])
        current_lr = optimizer.param_groups[0]["lr"]

        sel = compute_selection_score(val_range_metrics, val_metrics)
        composite = sel["composite"]
        if ema_composite is None:
            ema_composite = composite
        else:
            ema_composite = EMA_ALPHA * composite + (1.0 - EMA_ALPHA) * ema_composite

        target_key = "target_BER(0.20<=y<0.40)"
        target_mae = (val_range_metrics[target_key]["mae_raw"]
                      if val_range_metrics[target_key] is not None else float("inf"))

        is_acceptable, reason = is_acceptable_checkpoint(val_range_metrics, val_metrics)

        print(
            f"FT-Epoch {epoch+1:03d} | LR: {current_lr:.2e} | "
            f"Train Loss: {train_metrics['loss']:.4f} | "
            f"Val Loss: {val_metrics['loss']:.4f} | "
            f"Val RMSE(log10): {val_metrics['rmse_log']:.4f} "
            f"(~{val_metrics['factor_error']:.3f}x) | "
            f"Val Region Acc: {val_metrics['region_acc']:.4f}"
        )
        print(
            f"  SELECTION | composite={composite:.5f} | ema={ema_composite:.5f} | "
            f"weighted log-RMSE={sel['log_rmse_weighted']:.5f} "
            f"(~{sel['geometric_factor_error']:.3f}x) | "
            f"|bias_log|={sel['bias_weighted']:.5f} | "
            f"tail={sel['tail_weighted']:.3f} | acceptable={is_acceptable}"
            + ("" if is_acceptable else f" ({reason})")
        )
        for rng_name, rng_stats in val_range_metrics.items():
            if rng_stats is not None:
                print(
                    f"    {rng_name}: n={rng_stats['count']} | "
                    f"factor~{rng_stats['factor_error']:.3f}x | "
                    f"MAE_raw={rng_stats['mae_raw']:.6f} | "
                    f"P90={rng_stats['p90_abs_raw']:.6f} | "
                    f"bias_log={rng_stats['bias_log']:+.4f}"
                )

        improved_any = False
        if is_acceptable:
            current_state_snapshot = None
            checkpoint_candidates = [
                ("composite", composite),
                ("composite_ema", ema_composite),
                ("log_rmse", sel["log_rmse_weighted"]),
                ("target_mae", target_mae),
                ("low_bias", sel["bias_weighted"]),
            ]
            for ckpt_name, score in checkpoint_candidates:
                if score < best_checkpoints[ckpt_name]["score"]:
                    if current_state_snapshot is None:
                        current_state_snapshot = copy.deepcopy(model.state_dict())
                    best_checkpoints[ckpt_name] = {
                        "score": float(score),
                        "state": current_state_snapshot,
                        "epoch": epoch + 1,
                    }
                    improved_any = True
                    print(f"  -> New best [{ckpt_name}] at FT-epoch {epoch+1}: {score:.6f}")

        ema_best = best_checkpoints["composite_ema"]["score"]
        if ema_composite < ema_best + 1e-9 or improved_any:
            wait = 0
        else:
            if epoch + 1 >= FINETUNE_MIN_EPOCHS:
                wait += 1
                if wait >= FINETUNE_PATIENCE:
                    print(f"Early stopping triggered after {wait} non-improving epochs.")
                    break

    # ---- Save last ----
    torch.save(model.state_dict(), LAST_MODEL_SAVE_PATH)
    print(f"\nLast fine-tuned model saved to: {LAST_MODEL_SAVE_PATH}")

    # ---- Save all best checkpoints ----
    print("\n" + "=" * 80)
    print("FINE-TUNE BEST CHECKPOINTS SUMMARY")
    print("=" * 80)
    for ckpt_name, info in best_checkpoints.items():
        if info["state"] is None:
            print(f"  [{ckpt_name:>14s}] never updated")
            continue
        path = BEST_MODEL_SAVE_PATH.replace(".pth", f"_{ckpt_name}.pth")
        torch.save(info["state"], path)
        print(f"  [{ckpt_name:>14s}] epoch={info['epoch']:>3d} | "
              f"score={info['score']:.6f} | saved -> {path}")

    primary_choice = "composite_ema"
    if best_checkpoints[primary_choice]["state"] is None:
        primary_choice = "composite"
    if best_checkpoints[primary_choice]["state"] is None:
        primary_choice = "log_rmse"

    if best_checkpoints[primary_choice]["state"] is not None:
        model.load_state_dict(best_checkpoints[primary_choice]["state"])
        torch.save(model.state_dict(), BEST_MODEL_SAVE_PATH)
        print(f"\nPrimary best ({primary_choice}, "
              f"FT-epoch {best_checkpoints[primary_choice]['epoch']}) "
              f"saved to: {BEST_MODEL_SAVE_PATH}")
    else:
        print("\nWarning: no valid best checkpoint was found.")

    # ---- Test ----
    test_metrics = evaluate(model, test_loader, criterion, device)
    print(
        f"\nTest Loss: {test_metrics['loss']:.4f} | "
        f"Test RMSE(log10): {test_metrics['rmse_log']:.4f} | "
        f"~{test_metrics['factor_error']:.2f}x | "
        f"Test Region Acc: {test_metrics['region_acc']:.4f}"
    )

    range_metrics = evaluate_by_target_range(model, test_loader, device)
    print("\nPer-range test diagnostics:")
    for name, stats in range_metrics.items():
        if stats is None:
            print(f"  {name}: no samples")
        else:
            print(
                f"  {name} | count={stats['count']} | "
                f"factor~{stats['factor_error']:.2f}x | "
                f"RMSE(log10)={stats['rmse_log']:.4f} | "
                f"MAE_raw={stats['mae_raw']:.6f} | "
                f"bias_log={stats['bias_log']:+.4f} | "
                f"P90={stats['p90_abs_raw']:.6f} | "
                f"P95={stats['p95_abs_raw']:.6f}"
            )

    return model


if __name__ == "__main__":
    os.makedirs(BASE_DIR, exist_ok=True)
    missing = [p for p in DATA_PATHS if not os.path.exists(p)]
    if missing:
        print("Critical Error: Missing dataset files:")
        for p in missing:
            print(f"  - {p}")
    else:
        print("Fine-tuning from datasets:")
        for p in DATA_PATHS:
            print(f"  - {p}")
        print(f"Row cap per dataset: {NROWS_PER_DATASET:,}")
        finetune()

Fine-tuning from datasets:
  - ./../ber_data_generation/data/data_physics_total.csv
  - ./../ber_data_generation/data/data_random_total.csv
Row cap per dataset: 5,000,000
Executing on: cuda

FINE-TUNING from: ./random_extra_multitask_last2.pth
  Pretrained scalers: ./random_extra_multitask_scalers2.pkl
  Reuse scalers: True
  Pretrained global_dim: 7
Loading up to 5,000,000 rows from: ./../ber_data_generation/data/data_physics_total.csv
Loading up to 5,000,000 rows from: ./../ber_data_generation/data/data_random_total.csv
Combined rows before filtering: 10,000,000
  Files with var_* columns: 0/2
  Final tap column count: 15
  Dropped 342,676 rows with threshold=0
Rows after cleaning/filtering: 7,990,695
  Tap column count L_full = 15, mem_len range = [2, 15], past_lens range = [1, 14]
  Variances: all 7,990,695 computed from N*tap*(1-tap)
  Reusing pretrained scalers (recommended for fine-tuning).

Fine-tune scalers saved to: ./random_extra_multitask_finetune_scalers.pkl
Pretrained wei

FT-Epoch 008 | LR: 2.00e-05 | Train Loss: 0.0019 | Val Loss: 0.0017 | Val RMSE(log10): 0.0495 (~1.121x) | Val Region Acc: 0.9793
  SELECTION | composite=0.13357 | ema=0.13452 | weighted log-RMSE=0.04719 (~1.115x) | |bias_log|=0.01066 | tail=1.621 | acceptable=True
    low_BER(y<1e-6): n=54827 | factor~1.473x | MAE_raw=0.000000 | P90=0.000000 | bias_log=+0.0651
    mid_BER(1e-6<=y<1e-3): n=8285 | factor~1.998x | MAE_raw=0.000045 | P90=0.000121 | bias_log=-0.0429
    high_BER(1e-3<=y<0.10): n=46295 | factor~1.283x | MAE_raw=0.003581 | P90=0.007134 | bias_log=+0.0173
    upper_BER(0.10<=y<0.20): n=82146 | factor~1.067x | MAE_raw=0.006223 | P90=0.011265 | bias_log=+0.0150
    target_BER(0.20<=y<0.40): n=246451 | factor~1.022x | MAE_raw=0.003723 | P90=0.007694 | bias_log=+0.0010
    very_high_BER(0.40<=y<=0.50): n=760600 | factor~1.005x | MAE_raw=0.000581 | P90=0.001776 | bias_log=+0.0001
FT-Epoch 009 | LR: 2.00e-05 | Train Loss: 0.0019 | Val Loss: 0.0018 | Val RMSE(log10): 0.0425 (~1.103x)

FT-Epoch 017 | LR: 1.00e-05 | Train Loss: 0.0018 | Val Loss: 0.0017 | Val RMSE(log10): 0.0350 (~1.084x) | Val Region Acc: 0.9796
  SELECTION | composite=0.12269 | ema=0.12472 | weighted log-RMSE=0.03419 (~1.082x) | |bias_log|=0.01649 | tail=1.605 | acceptable=True
    low_BER(y<1e-6): n=54827 | factor~1.332x | MAE_raw=0.000000 | P90=0.000000 | bias_log=-0.0076
    mid_BER(1e-6<=y<1e-3): n=8285 | factor~1.522x | MAE_raw=0.000073 | P90=0.000207 | bias_log=+0.0826
    high_BER(1e-3<=y<0.10): n=46295 | factor~1.185x | MAE_raw=0.005626 | P90=0.010106 | bias_log=+0.0561
    upper_BER(0.10<=y<0.20): n=82146 | factor~1.067x | MAE_raw=0.006588 | P90=0.011933 | bias_log=+0.0171
    target_BER(0.20<=y<0.40): n=246451 | factor~1.020x | MAE_raw=0.003334 | P90=0.007224 | bias_log=+0.0008
    very_high_BER(0.40<=y<=0.50): n=760600 | factor~1.005x | MAE_raw=0.000534 | P90=0.001642 | bias_log=+0.0000
  -> New best [composite_ema] at FT-epoch 17: 0.124718
FT-Epoch 018 | LR: 5.00e-06 | Train Loss: 0.0019

FT-Epoch 026 | LR: 2.50e-06 | Train Loss: 0.0018 | Val Loss: 0.0015 | Val RMSE(log10): 0.0220 (~1.052x) | Val Region Acc: 0.9808
  SELECTION | composite=0.10900 | ema=0.11110 | weighted log-RMSE=0.02546 (~1.060x) | |bias_log|=0.00976 | tail=1.573 | acceptable=True
    low_BER(y<1e-6): n=54827 | factor~1.174x | MAE_raw=0.000000 | P90=0.000000 | bias_log=+0.0029
    mid_BER(1e-6<=y<1e-3): n=8285 | factor~1.276x | MAE_raw=0.000018 | P90=0.000051 | bias_log=-0.0347
    high_BER(1e-3<=y<0.10): n=46295 | factor~1.129x | MAE_raw=0.003896 | P90=0.007865 | bias_log=+0.0276
    upper_BER(0.10<=y<0.20): n=82146 | factor~1.067x | MAE_raw=0.006596 | P90=0.011960 | bias_log=+0.0167
    target_BER(0.20<=y<0.40): n=246451 | factor~1.021x | MAE_raw=0.003357 | P90=0.007439 | bias_log=+0.0005
    very_high_BER(0.40<=y<=0.50): n=760600 | factor~1.005x | MAE_raw=0.000517 | P90=0.001570 | bias_log=+0.0000
  -> New best [composite_ema] at FT-epoch 26: 0.111102
FT-Epoch 027 | LR: 2.50e-06 | Train Loss: 0.0018

FT-Epoch 035 | LR: 6.25e-07 | Train Loss: 0.0018 | Val Loss: 0.0015 | Val RMSE(log10): 0.0189 (~1.044x) | Val Region Acc: 0.9806
  SELECTION | composite=0.10527 | ema=0.10611 | weighted log-RMSE=0.02363 (~1.056x) | |bias_log|=0.00823 | tail=1.550 | acceptable=True
    low_BER(y<1e-6): n=54827 | factor~1.133x | MAE_raw=0.000000 | P90=0.000000 | bias_log=-0.0064
    mid_BER(1e-6<=y<1e-3): n=8285 | factor~1.255x | MAE_raw=0.000016 | P90=0.000040 | bias_log=-0.0131
    high_BER(1e-3<=y<0.10): n=46295 | factor~1.111x | MAE_raw=0.003506 | P90=0.007137 | bias_log=+0.0253
    upper_BER(0.10<=y<0.20): n=82146 | factor~1.066x | MAE_raw=0.006390 | P90=0.011653 | bias_log=+0.0159
    target_BER(0.20<=y<0.40): n=246451 | factor~1.021x | MAE_raw=0.003392 | P90=0.007450 | bias_log=+0.0008
    very_high_BER(0.40<=y<=0.50): n=760600 | factor~1.005x | MAE_raw=0.000518 | P90=0.001538 | bias_log=+0.0000
FT-Epoch 036 | LR: 6.25e-07 | Train Loss: 0.0018 | Val Loss: 0.0015 | Val RMSE(log10): 0.0179 (~1.042x)

FT-Epoch 044 | LR: 1.56e-07 | Train Loss: 0.0018 | Val Loss: 0.0015 | Val RMSE(log10): 0.0169 (~1.040x) | Val Region Acc: 0.9812
  SELECTION | composite=0.10381 | ema=0.10357 | weighted log-RMSE=0.02242 (~1.053x) | |bias_log|=0.00842 | tail=1.544 | acceptable=True
    low_BER(y<1e-6): n=54827 | factor~1.102x | MAE_raw=0.000000 | P90=0.000000 | bias_log=+0.0053
    mid_BER(1e-6<=y<1e-3): n=8285 | factor~1.217x | MAE_raw=0.000018 | P90=0.000053 | bias_log=+0.0084
    high_BER(1e-3<=y<0.10): n=46295 | factor~1.110x | MAE_raw=0.003705 | P90=0.007418 | bias_log=+0.0289
    upper_BER(0.10<=y<0.20): n=82146 | factor~1.067x | MAE_raw=0.006498 | P90=0.011706 | bias_log=+0.0164
    target_BER(0.20<=y<0.40): n=246451 | factor~1.021x | MAE_raw=0.003353 | P90=0.007405 | bias_log=+0.0007
    very_high_BER(0.40<=y<=0.50): n=760600 | factor~1.005x | MAE_raw=0.000505 | P90=0.001485 | bias_log=+0.0000
FT-Epoch 045 | LR: 1.56e-07 | Train Loss: 0.0018 | Val Loss: 0.0015 | Val RMSE(log10): 0.0167 (~1.039x)

FT-Epoch 053 | LR: 3.91e-08 | Train Loss: 0.0018 | Val Loss: 0.0015 | Val RMSE(log10): 0.0165 (~1.039x) | Val Region Acc: 0.9811
  SELECTION | composite=0.10313 | ema=0.10320 | weighted log-RMSE=0.02222 (~1.053x) | |bias_log|=0.00762 | tail=1.542 | acceptable=True
    low_BER(y<1e-6): n=54827 | factor~1.097x | MAE_raw=0.000000 | P90=0.000000 | bias_log=+0.0036
    mid_BER(1e-6<=y<1e-3): n=8285 | factor~1.212x | MAE_raw=0.000015 | P90=0.000041 | bias_log=-0.0027
    high_BER(1e-3<=y<0.10): n=46295 | factor~1.108x | MAE_raw=0.003560 | P90=0.007277 | bias_log=+0.0263
    upper_BER(0.10<=y<0.20): n=82146 | factor~1.067x | MAE_raw=0.006460 | P90=0.011724 | bias_log=+0.0162
    target_BER(0.20<=y<0.40): n=246451 | factor~1.021x | MAE_raw=0.003350 | P90=0.007469 | bias_log=+0.0006
    very_high_BER(0.40<=y<=0.50): n=760600 | factor~1.005x | MAE_raw=0.000504 | P90=0.001475 | bias_log=+0.0000
FT-Epoch 054 | LR: 1.95e-08 | Train Loss: 0.0018 | Val Loss: 0.0015 | Val RMSE(log10): 0.0166 (~1.039x)

In [9]:
import os
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.special import erfc
from itertools import product as iproduct

EPS = 1e-12
PLOT_DIR = "./plots_mixed_newdata"
os.makedirs(PLOT_DIR, exist_ok=True)

# =========================
# Paths
# =========================
MODEL_PATH = "random_extra_multitask_finetune_best.pth"
SCALER_PATH = "random_extra_multitask_scalers2.pkl"

# =========================
# Config
# =========================
PHYSICS_MAX_MEM_LEN = 14
PHYSICS_MIN_MEM_LEN = 14
ARRIVAL_COVERAGE = 0.70
N_THRESHOLDS = 500
RANDOM_SEED = 60

ORDINAL_THRESHOLDS = [
    1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1,
    0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45,
]
NUM_ORDINAL_THRESHOLDS = len(ORDINAL_THRESHOLDS)
NUM_REGION_CLASSES = NUM_ORDINAL_THRESHOLDS + 1

REGION_LABELS = {
    0: "y < 1e-6",
    1: "1e-6 <= y < 1e-5",
    2: "1e-5 <= y < 1e-4",
    3: "1e-4 <= y < 1e-3",
    4: "1e-3 <= y < 1e-2",
    5: "1e-2 <= y < 1e-1",
    6: "0.10 <= y < 0.15",
    7: "0.15 <= y < 0.20",
    8: "0.20 <= y < 0.25",
    9: "0.25 <= y < 0.30",
    10: "0.30 <= y < 0.35",
    11: "0.35 <= y < 0.40",
    12: "0.40 <= y < 0.45",
    13: "0.45 <= y <= 0.50",
}


# =========================
# Physics helpers
# =========================
def Fhit_function(radius, distance, diffusionCoef, t):
    if t <= 0:
        return 0.0
    return (radius / (distance + radius)) * erfc(distance / np.sqrt(4 * diffusionCoef * t))


def calculate_hitting_probabilities(mem_len, radius, distance, diffusionCoef, Ts):
    P = np.zeros(mem_len)
    for i in range(mem_len):
        t_end = (i + 1) * Ts
        t_start = i * Ts
        P[i] = Fhit_function(radius, distance, diffusionCoef, t_end) - Fhit_function(
            radius, distance, diffusionCoef, t_start
        )
    return P


def calculate_ber_vectorized(mem_len, threshold, P_scaled, variances):
    P_arr = np.asarray(P_scaled, dtype=float)[:mem_len]
    vars_arr = np.asarray(variances, dtype=float)[:mem_len]

    seqs = np.array(list(iproduct([0, 1], repeat=mem_len)), dtype=np.float64)[:, ::-1]
    c_bit = seqs[:, 0]

    mu = (seqs * P_arr).sum(axis=1)
    var_total = (seqs * vars_arr).sum(axis=1)
    std = np.sqrt(np.maximum(var_total, 0.0))

    pe = np.empty_like(mu)
    zero_std = (std == 0)
    if np.any(zero_std):
        pe[zero_std & (c_bit == 1)] = np.where(
            mu[zero_std & (c_bit == 1)] < threshold, 1.0, 0.0)
        pe[zero_std & (c_bit == 0)] = np.where(
            mu[zero_std & (c_bit == 0)] >= threshold, 1.0, 0.0)
    nz = ~zero_std
    if np.any(nz):
        pe[nz & (c_bit == 1)] = 0.5 * erfc(
            (mu[nz & (c_bit == 1)] - threshold) / (std[nz & (c_bit == 1)] * np.sqrt(2)))
        pe[nz & (c_bit == 0)] = 0.5 * erfc(
            (threshold - mu[nz & (c_bit == 0)]) / (std[nz & (c_bit == 0)] * np.sqrt(2)))
    return float(np.mean(pe))


def raw_to_region_labels_np(y_raw):
    y = np.asarray(y_raw).reshape(-1)
    return np.searchsorted(np.asarray(ORDINAL_THRESHOLDS), y, side="right").astype(np.int64)


def ordinal_logits_to_region_labels_torch(logits):
    probs = torch.sigmoid(logits)
    passed = (probs >= 0.5).sum(dim=1)
    return passed.long()


# =========================
# Shared scaling helper
# =========================
def apply_shared_scale(data, mean, std, valid_mask=None):
    scaled = ((data - mean) / std).astype(np.float32)
    if valid_mask is not None and scaled.ndim == 2 and valid_mask.shape == scaled.shape:
        scaled = scaled * valid_mask.astype(np.float32)
    return scaled


# =========================
# Generate one physical scenario
# =========================
def generate_physical_case(rng, min_mem_len, max_mem_len, arrival_coverage, max_tries=5000):
    for _ in range(max_tries):
        radius = rng.uniform(3.0, 5.0)
        distance = rng.uniform(10.0, 15.0)
        diff = rng.uniform(50.0, 75.0)
        Ts = rng.uniform(0.5, 1.2)
        N = int(10 ** rng.uniform(3.0, 6.0))

        f_inf = radius / (radius + distance)
        target = arrival_coverage * f_inf

        cumsum, k = 0.0, 0
        while k < max_mem_len:
            pk = Fhit_function(radius, distance, diff, (k + 1) * Ts) - Fhit_function(
                radius, distance, diff, k * Ts)
            cumsum += pk
            k += 1
            if cumsum >= target:
                break

        if k < min_mem_len:
            continue

        P_ext = calculate_hitting_probabilities(k + 1, radius, distance, diff, Ts)
        P_main = P_ext[:k]
        P_extra = float(P_ext[k]) if k < len(P_ext) else 0.0

        P_scaled = P_main * N
        variances = N * P_main * (1.0 - P_main)

        return {
            "radius": radius, "distance": distance, "diffusion": diff,
            "Ts": Ts, "N": N, "mem_len": k, "P": P_main,
            "P_scaled": P_scaled, "variances": variances,
            "P_mem_len_extra": P_extra,
            "P_mem_len_extra_var": P_extra * (1.0 - P_extra),
        }

    raise RuntimeError(
        f"Could not generate a physical case with mem_len >= {min_mem_len} "
        f"after {max_tries} tries."
    )


# =========================
# Model (matches v3 training: global_dim=7, global_embed=96, cond_dim=128)
# =========================
class ConditionalFeatureModulation(nn.Module):
    def __init__(self, cond_dim, feat_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, 2 * feat_dim),
        )
        self.scale = nn.Parameter(torch.tensor(0.25, dtype=torch.float32))

    def forward(self, x, cond):
        gamma_beta = self.net(cond)
        gamma, beta = torch.chunk(gamma_beta, 2, dim=-1)
        if x.dim() == 3:
            gamma = gamma.unsqueeze(1)
            beta = beta.unsqueeze(1)
        return x * (1.0 + self.scale * gamma) + self.scale * beta


class SetSelfAttentionBlock(nn.Module):
    def __init__(self, d_model, num_heads, mlp_ratio=4.0, dropout=0.10):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=num_heads,
            dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        hidden = int(d_model * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, d_model), nn.Dropout(dropout))

    def forward(self, x, key_padding_mask=None):
        y = self.norm1(x)
        attn_out, _ = self.attn(y, y, y, need_weights=False, key_padding_mask=key_padding_mask)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x


class SetAttentionPooling(nn.Module):
    def __init__(self, d_model, hidden_dim=128):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(d_model, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, 1))

    def forward(self, x, key_padding_mask=None):
        logits = self.score(x)
        if key_padding_mask is not None:
            logits = logits.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
        weights = torch.softmax(logits, dim=1)
        weights = torch.nan_to_num(weights, nan=0.0)
        return (weights * x).sum(dim=1)


class BERFirstTokenSetTransformerBoundedLogHalfMultiTask(nn.Module):
    def __init__(self, max_past_seq_len, token_dim=4, first_token_dim=4,
                 threshold_dim=1, global_dim=7, d_model=128, num_set_layers=4,
                 num_heads=4, mlp_ratio=4.0, dropout=0.10,
                 num_ordinal_thresholds=NUM_ORDINAL_THRESHOLDS):
        super().__init__()
        self.max_past_seq_len = max_past_seq_len
        self.d_model = d_model
        self.num_ordinal_thresholds = num_ordinal_thresholds

        self.first_proj = nn.Sequential(
            nn.Linear(first_token_dim, d_model), nn.LayerNorm(d_model),
            nn.GELU(), nn.Dropout(dropout))
        self.token_proj = nn.Sequential(
            nn.Linear(token_dim, d_model), nn.LayerNorm(d_model),
            nn.GELU(), nn.Dropout(dropout))
        self.thr_embed = nn.Sequential(
            nn.Linear(threshold_dim, 32), nn.GELU(),
            nn.Linear(32, 32), nn.GELU())
        self.global_embed = nn.Sequential(
            nn.Linear(global_dim, 96), nn.GELU(),
            nn.Linear(96, 96), nn.GELU())

        cond_dim = 32 + 96
        self.first_cond_mod = ConditionalFeatureModulation(cond_dim, d_model, 256)
        self.set_cond_mod = ConditionalFeatureModulation(cond_dim, d_model, 256)

        self.set_blocks = nn.ModuleList([
            SetSelfAttentionBlock(d_model, num_heads, mlp_ratio, dropout)
            for _ in range(num_set_layers)])

        self.first_refine = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_model),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model, d_model))
        self.final_set_norm = nn.LayerNorm(d_model)
        self.set_attn_pool = SetAttentionPooling(d_model, 128)

        set_summary_dim = 3 * d_model
        head_in = d_model + set_summary_dim + cond_dim

        self.reg_head = nn.Sequential(
            nn.Linear(head_in, 256), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.15),
            nn.Linear(128, 1))
        self.ord_head = nn.Sequential(
            nn.Linear(head_in, 256), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.15),
            nn.Linear(128, num_ordinal_thresholds))

    @staticmethod
    def raw_to_log10ber(pred_raw_unconstrained):
        log10_half = torch.log10(torch.tensor(
            0.5, device=pred_raw_unconstrained.device,
            dtype=pred_raw_unconstrained.dtype))
        return log10_half - F.softplus(pred_raw_unconstrained)

    @staticmethod
    def raw_to_ber(pred_raw_unconstrained):
        pred_log = BERFirstTokenSetTransformerBoundedLogHalfMultiTask.raw_to_log10ber(
            pred_raw_unconstrained)
        return torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)

    def forward(self, first_token, past_tokens, global_feats, threshold, key_padding_mask=None):
        thr = self.thr_embed(threshold)
        g = self.global_embed(global_feats)
        cond = torch.cat([thr, g], dim=-1)

        first_x = self.first_proj(first_token)
        first_x = self.first_cond_mod(first_x, cond)
        first_x = first_x + self.first_refine(first_x)

        set_x = self.token_proj(past_tokens)
        set_x = self.set_cond_mod(set_x, cond)
        for block in self.set_blocks:
            set_x = block(set_x, key_padding_mask=key_padding_mask)
        set_x = self.final_set_norm(set_x)

        if key_padding_mask is not None:
            valid_mask = (~key_padding_mask).unsqueeze(-1).float()
        else:
            valid_mask = torch.ones(
                set_x.shape[0], set_x.shape[1], 1,
                device=set_x.device, dtype=set_x.dtype)

        pooled_attn = self.set_attn_pool(set_x, key_padding_mask=key_padding_mask)
        valid_counts = valid_mask.sum(dim=1).clamp(min=1.0)
        pooled_mean = (set_x * valid_mask).sum(dim=1) / valid_counts

        if key_padding_mask is not None:
            x_for_max = set_x.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
        else:
            x_for_max = set_x
        pooled_max = x_for_max.amax(dim=1)
        pooled_max = torch.nan_to_num(pooled_max, nan=0.0, posinf=0.0, neginf=0.0)

        set_summary = torch.cat([pooled_attn, pooled_mean, pooled_max], dim=-1)
        fused = torch.cat([first_x, set_summary, cond], dim=-1)

        pred_raw_unconstrained = self.reg_head(fused)
        ord_logits = self.ord_head(fused)
        return pred_raw_unconstrained, ord_logits


# =========================
# Build inference inputs (7 global features, position-independent scaling)
# =========================
def prepare_inference_features(case, thresholds, scalers, device):
    """Build model inputs for a threshold sweep over a single physical case.

    Returns:
        first_t, past_t, global_t, thr_t, mask_t — all torch tensors on device
        global_t has shape [B, 7]: [z0, z1, hmg, nsid, log_hd, da, herf]
    """
    B = len(thresholds)
    mem_len = case["mem_len"]
    N = float(case["N"])

    P_raw = np.asarray(case["P"], dtype=np.float32)
    var_raw = np.asarray(case["variances"], dtype=np.float32)
    thr_raw = np.asarray(thresholds, dtype=np.float32).reshape(-1, 1)

    # Feature engineering
    taps_feat = (P_raw * N).astype(np.float32)
    vars_feat = var_raw.astype(np.float32)
    abs_feat = np.abs(taps_feat).astype(np.float32)
    snr_feat = np.log10((taps_feat ** 2) / (var_raw + EPS) + EPS).astype(np.float32)

    # Broadcast to [B, mem_len]
    taps_2d = np.broadcast_to(taps_feat[None, :], (B, mem_len)).copy()
    vars_2d = np.broadcast_to(vars_feat[None, :], (B, mem_len)).copy()
    abs_2d = np.broadcast_to(abs_feat[None, :], (B, mem_len)).copy()
    snr_2d = np.broadcast_to(snr_feat[None, :], (B, mem_len)).copy()

    L_past = mem_len - 1
    valid_past = np.ones((B, L_past), dtype=bool)

    # --- Threshold-dependent global features ---
    first_mean = taps_2d[:, 0:1]
    past_means = taps_2d[:, 1:]
    first_var = vars_2d[:, 0:1]
    past_vars = vars_2d[:, 1:]

    mu0 = (0.5 * past_means.sum(axis=1, keepdims=True)).astype(np.float32)
    mu1 = (first_mean + mu0).astype(np.float32)
    var0 = (0.5 * past_vars.sum(axis=1, keepdims=True)).astype(np.float32)
    var1 = (first_var + var0).astype(np.float32)
    std0 = np.sqrt(np.maximum(var0, EPS)).astype(np.float32)
    std1 = np.sqrt(np.maximum(var1, EPS)).astype(np.float32)

    z0 = ((thr_raw - mu0) / (std0 + EPS)).astype(np.float32)
    z1 = ((mu1 - thr_raw) / (std1 + EPS)).astype(np.float32)
    harmonic = (2.0 / (1.0 / (z0 + EPS) + 1.0 / (z1 + EPS))).astype(np.float32)
    abs_diff = np.abs(z0 - z1).astype(np.float32)
    hmg = (harmonic - 0.25 * abs_diff).astype(np.float32)

    # --- Scenario-level features (threshold-independent) ---
    signal = first_mean  # [B, 1]
    isi = past_means.sum(axis=1, keepdims=True)
    nsid = ((signal - isi) / (signal + isi + EPS)).astype(np.float32)

    gap = signal
    d0_inf = (gap / (std0 + EPS)).astype(np.float32)
    d1_inf = (gap / (std1 + EPS)).astype(np.float32)
    hd = (2.0 * d0_inf * d1_inf / (d0_inf + d1_inf + EPS)).astype(np.float32)
    log_hd = np.log10(hd + EPS).astype(np.float32)
    da = (np.abs(d0_inf - d1_inf) / (d0_inf + d1_inf + EPS)).astype(np.float32)

    past_taps_sum = past_means.sum(axis=1, keepdims=True)
    past_shares = past_means / (past_taps_sum + EPS)
    herf = (past_shares ** 2).sum(axis=1, keepdims=True).astype(np.float32)

    # --- Scale first token ---
    ft_tap = ((taps_2d[:, 0] - scalers["first_tap_mean"]) / scalers["first_tap_std"]).astype(np.float32)
    ft_var = ((vars_2d[:, 0] - scalers["first_var_mean"]) / scalers["first_var_std"]).astype(np.float32)
    ft_abs = ((abs_2d[:, 0] - scalers["first_abs_mean"]) / scalers["first_abs_std"]).astype(np.float32)
    ft_snr = ((snr_2d[:, 0] - scalers["first_snr_mean"]) / scalers["first_snr_std"]).astype(np.float32)
    first_token = np.stack([ft_tap, ft_var, ft_abs, ft_snr], axis=1).astype(np.float32)

    # --- Scale past tokens ---
    pt_tap = apply_shared_scale(taps_2d[:, 1:], scalers["past_tap_mean"], scalers["past_tap_std"], valid_past)
    pt_var = apply_shared_scale(vars_2d[:, 1:], scalers["past_var_mean"], scalers["past_var_std"], valid_past)
    pt_abs = apply_shared_scale(abs_2d[:, 1:], scalers["past_abs_mean"], scalers["past_abs_std"], valid_past)
    pt_snr = apply_shared_scale(snr_2d[:, 1:], scalers["past_snr_mean"], scalers["past_snr_std"], valid_past)
    past_tokens = np.stack([pt_tap, pt_var, pt_abs, pt_snr], axis=2).astype(np.float32)

    # --- Scale globals (7 features) ---
    z0_s = scalers["z0_scaler"].transform(z0).astype(np.float32)
    z1_s = scalers["z1_scaler"].transform(z1).astype(np.float32)
    hmg_s = scalers["harmonic_minus_gap_scaler"].transform(hmg).astype(np.float32)
    nsid_s = scalers["nsid_scaler"].transform(nsid).astype(np.float32)
    log_hd_s = scalers["log_hd_scaler"].transform(log_hd).astype(np.float32)
    da_s = scalers["da_scaler"].transform(da).astype(np.float32)
    herf_s = scalers["herf_scaler"].transform(herf).astype(np.float32)
    global_feats = np.concatenate(
        [z0_s, z1_s, hmg_s, nsid_s, log_hd_s, da_s, herf_s], axis=1
    ).astype(np.float32)

    # --- Scale threshold ---
    thr_log = np.log10(thr_raw + EPS).astype(np.float32)
    thr_s = scalers["thr_scaler"].transform(thr_log).astype(np.float32)

    # --- Padding mask (no padding needed here) ---
    pad_mask = np.zeros((B, L_past), dtype=bool)

    first_t = torch.from_numpy(first_token).to(device)
    past_t = torch.from_numpy(past_tokens).to(device)
    global_t = torch.from_numpy(global_feats).to(device)
    thr_t = torch.from_numpy(thr_s).to(device)
    mask_t = torch.from_numpy(pad_mask).to(device)

    return first_t, past_t, global_t, thr_t, mask_t


# =========================
# Generate scenario
# =========================
rng = np.random.default_rng(RANDOM_SEED)
case = generate_physical_case(
    rng,
    min_mem_len=PHYSICS_MIN_MEM_LEN,
    max_mem_len=PHYSICS_MAX_MEM_LEN,
    arrival_coverage=ARRIVAL_COVERAGE,
)

print("Generated physical scenario")
print("radius    =", case["radius"])
print("distance  =", case["distance"])
print("diffusion =", case["diffusion"])
print("Ts        =", case["Ts"])
print("N         =", case["N"])
print("mem_len   =", case["mem_len"])
print("P         =", case["P"])
print("P_scaled  =", case["P_scaled"])
print("variances =", case["variances"])

# =========================
# Threshold sweep — ground truth
# =========================
thr_min = 0.0
thr_max = float(np.sum(case["P_scaled"]))
thresholds = np.linspace(thr_min, thr_max, N_THRESHOLDS)

print("Threshold search interval:", thr_min, "to", thr_max)
print("sum(P_scaled) =", np.sum(case["P_scaled"]))

real_bers = np.array([
    calculate_ber_vectorized(
        mem_len=case["mem_len"],
        threshold=thr,
        P_scaled=case["P_scaled"],
        variances=case["variances"],
    )
    for thr in thresholds
])

real_bers = np.clip(real_bers, EPS, 0.5)
real_regions = raw_to_region_labels_np(real_bers)

# =========================
# Load model + scalers
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scalers = joblib.load(SCALER_PATH)

max_past_seq_len = scalers.get("train_max_past_seq_len", scalers.get("max_past_seq_len", case["mem_len"] - 1))
ordinal_thresholds = scalers.get("ordinal_thresholds", ORDINAL_THRESHOLDS)
num_ordinal = len(ordinal_thresholds)
global_dim = scalers.get("global_dim", 7)

model = BERFirstTokenSetTransformerBoundedLogHalfMultiTask(
    max_past_seq_len=max_past_seq_len,
    token_dim=scalers.get("past_token_dim", 4),
    first_token_dim=scalers.get("first_token_dim", 4),
    threshold_dim=1,
    global_dim=global_dim,
    d_model=128,
    num_set_layers=4,
    num_heads=4,
    mlp_ratio=4.0,
    dropout=0.10,
    num_ordinal_thresholds=num_ordinal,
).to(device)

state = torch.load(MODEL_PATH, map_location=device, weights_only=True)
model.load_state_dict(state)
model.eval()

print(f"\nModel loaded: max_past_seq_len={max_past_seq_len}, "
      f"global_dim={global_dim}, num_ordinal={num_ordinal}")
print(f"Scaling strategy: {scalers.get('scaling_strategy', 'unknown')}")

# =========================
# Predict across thresholds
# =========================
first_t, past_t, global_t, thr_t, mask_t = prepare_inference_features(
    case, thresholds, scalers, device,
)

with torch.no_grad():
    pred_raw_out, ord_logits = model(
        first_t, past_t, global_t, thr_t, key_padding_mask=mask_t,
    )
    pred_bers = model.raw_to_ber(pred_raw_out).cpu().numpy().reshape(-1)
    pred_log = model.raw_to_log10ber(pred_raw_out).cpu().numpy().reshape(-1)
    pred_regions = ordinal_logits_to_region_labels_torch(ord_logits).cpu().numpy().reshape(-1)
    pred_region_probs = torch.sigmoid(ord_logits).cpu().numpy()

pred_bers = np.clip(pred_bers, EPS, 0.5)

# =========================
# Comparison table
# =========================
results = pd.DataFrame({
    "threshold": thresholds,
    "real_BER": real_bers,
    "estimated_BER": pred_bers,
    "real_region": real_regions,
    "predicted_region": pred_regions,
    "abs_error": np.abs(pred_bers - real_bers),
    "abs_log10_error": np.abs(
        np.log10(np.clip(pred_bers, EPS, 0.5)) -
        np.log10(np.clip(real_bers, EPS, 0.5))
    ),
    "region_abs_error": np.abs(pred_regions.astype(np.int64) - real_regions.astype(np.int64)),
})

print(results.head(15))

print("\nSummary")
print("Mean abs raw error        :", results["abs_error"].mean())
print("Mean abs log10 error      :", results["abs_log10_error"].mean())
print("Max  abs log10 error      :", results["abs_log10_error"].max())
print("Mean region abs error     :", results["region_abs_error"].mean())
print("Exact region accuracy     :", np.mean(results["real_region"] == results["predicted_region"]))

best_real_idx = np.argmin(real_bers)
best_est_idx = np.argmin(pred_bers)

print("\nBest threshold from real BER      :", thresholds[best_real_idx])
print("Minimum real BER                  :", real_bers[best_real_idx])
print("Real BER region there             :", REGION_LABELS.get(int(real_regions[best_real_idx]), "?"))

print("Best threshold from estimated BER :", thresholds[best_est_idx])
print("Estimated BER at that threshold   :", pred_bers[best_est_idx])
print("Predicted BER region there        :", REGION_LABELS.get(int(pred_regions[best_est_idx]), "?"))

# =========================
# Plot 1: log-scale BER
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["estimated_BER"], label="Estimated BER")
plt.plot(results["threshold"], results["real_BER"], label="Real BER")
plt.yscale("log")
plt.xlabel("Threshold")
plt.ylabel("BER")
plt.title(f"Threshold vs BER (log scale) — mem_len={case['mem_len']}")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "01_ber_log_scale.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/01_ber_log_scale.png")

# =========================
# Plot 2: linear-scale BER
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["estimated_BER"], label="Estimated BER")
plt.plot(results["threshold"], results["real_BER"], label="Real BER")
plt.yscale("linear")
plt.xlabel("Threshold")
plt.ylabel("BER")
plt.title(f"Threshold vs BER (linear scale) — mem_len={case['mem_len']}")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "02_ber_linear_scale.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/02_ber_linear_scale.png")

# =========================
# Plot 3: predicted vs true region
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["predicted_region"], label="Predicted region")
plt.plot(results["threshold"], results["real_region"], label="True region")
plt.xlabel("Threshold")
plt.ylabel("BER Region Class")
plt.title(f"Threshold vs BER Region — mem_len={case['mem_len']}")
plt.yticks(list(REGION_LABELS.keys()),
           [REGION_LABELS[k] for k in sorted(REGION_LABELS.keys())],
           fontsize=7)
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "03_region_comparison.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/03_region_comparison.png")

# =========================
# Plot 4: ordinal threshold probabilities
# =========================
plt.figure(figsize=(12, 7))
for i, thr_val in enumerate(ordinal_thresholds):
    plt.plot(thresholds, pred_region_probs[:, i], label=f"P(y >= {thr_val:g})")
plt.xlabel("Threshold")
plt.ylabel("Ordinal Probability")
plt.title(f"Ordinal Head Outputs Across Threshold Sweep — mem_len={case['mem_len']}")
plt.legend(fontsize=7, ncol=2)
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "04_ordinal_probabilities.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/04_ordinal_probabilities.png")

# =========================
# Plot 5: absolute error by threshold
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["abs_error"], label="Abs raw error", alpha=0.8)
plt.plot(results["threshold"], results["abs_log10_error"], label="Abs log10 error", alpha=0.8)
plt.yscale("log")
plt.xlabel("Threshold")
plt.ylabel("Error")
plt.title(f"Prediction Error Across Threshold Sweep — mem_len={case['mem_len']}")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "05_error_by_threshold.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/05_error_by_threshold.png")

Generated physical scenario
radius    = 4.999903475776163
distance  = 13.063595613360427
diffusion = 63.15707394395149
Ts        = 0.5834416162676823
N         = 2026
mem_len   = 14
P         = [0.03545102 0.042582   0.02704795 0.01857409 0.01368091 0.01059455
 0.00850978 0.00702648 0.00592783 0.00508774 0.00442855 0.00390015
 0.00346895 0.00311165]
P_scaled  = [71.82377081 86.27113641 54.79914164 37.63111579 27.71752004 21.46456524
 17.24082032 14.23564842 12.00979138 10.30776824  8.9722469   7.90171201
  7.0280834   6.304198  ]
variances = [69.27754472 82.59753869 53.31693734 36.93215188 27.33831919 21.23715776
 17.09410468 14.13562193 11.93859933 10.25532496  8.93251283  7.87089412
  7.00370336  6.28458156]
Threshold search interval: 0.0 to 383.7075186062634
sum(P_scaled) = 383.7075186062634

Model loaded: max_past_seq_len=13, global_dim=7, num_ordinal=13
Scaling strategy: position_independent
    threshold  real_BER  estimated_BER  real_region  predicted_region  \
0    0.000000  0.

In [2]:
import os
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.special import erfc
from itertools import product as iproduct

EPS = 1e-12
PLOT_DIR = "./plots_mixed_newdata_log_threshold"
os.makedirs(PLOT_DIR, exist_ok=True)

# =========================
# Paths
# =========================
MODEL_PATH = "random_extra_multitask_finetune_best.pth"
SCALER_PATH = "random_extra_multitask_finetune_scalers.pkl"

# =========================
# Config
# =========================
PHYSICS_MAX_MEM_LEN = 14
PHYSICS_MIN_MEM_LEN = 14
ARRIVAL_COVERAGE = 0.70
N_THRESHOLDS = 500
RANDOM_SEED = 60

# =========================
# Test-time smoothing config
# =========================
# The smoother recomputes ALL threshold-dependent features for thr-eps, thr, thr+eps.
# This is important because threshold enters both thr_t and global features such as z0/z1/HMG.
USE_TEST_TIME_SMOOTHING = True
SMOOTHING_EPS_FRAC = 0.005     # eps = SMOOTHING_EPS_FRAC * threshold sweep span
SMOOTHING_SPACE = "log"         # "log" is usually best globally; "raw" can look better at high BER
SMOOTHING_WEIGHTS = (0.25, 0.50, 0.25)


ORDINAL_THRESHOLDS = [
    1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1,
    0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45,
]
NUM_ORDINAL_THRESHOLDS = len(ORDINAL_THRESHOLDS)
NUM_REGION_CLASSES = NUM_ORDINAL_THRESHOLDS + 1

REGION_LABELS = {
    0: "y < 1e-6",
    1: "1e-6 <= y < 1e-5",
    2: "1e-5 <= y < 1e-4",
    3: "1e-4 <= y < 1e-3",
    4: "1e-3 <= y < 1e-2",
    5: "1e-2 <= y < 1e-1",
    6: "0.10 <= y < 0.15",
    7: "0.15 <= y < 0.20",
    8: "0.20 <= y < 0.25",
    9: "0.25 <= y < 0.30",
    10: "0.30 <= y < 0.35",
    11: "0.35 <= y < 0.40",
    12: "0.40 <= y < 0.45",
    13: "0.45 <= y <= 0.50",
}


# =========================
# Physics helpers
# =========================
def Fhit_function(radius, distance, diffusionCoef, t):
    if t <= 0:
        return 0.0
    return (radius / (distance + radius)) * erfc(distance / np.sqrt(4 * diffusionCoef * t))


def calculate_hitting_probabilities(mem_len, radius, distance, diffusionCoef, Ts):
    P = np.zeros(mem_len)
    for i in range(mem_len):
        t_end = (i + 1) * Ts
        t_start = i * Ts
        P[i] = Fhit_function(radius, distance, diffusionCoef, t_end) - Fhit_function(
            radius, distance, diffusionCoef, t_start
        )
    return P


def calculate_ber_vectorized(mem_len, threshold, P_scaled, variances):
    P_arr = np.asarray(P_scaled, dtype=float)[:mem_len]
    vars_arr = np.asarray(variances, dtype=float)[:mem_len]

    seqs = np.array(list(iproduct([0, 1], repeat=mem_len)), dtype=np.float64)[:, ::-1]
    c_bit = seqs[:, 0]

    mu = (seqs * P_arr).sum(axis=1)
    var_total = (seqs * vars_arr).sum(axis=1)
    std = np.sqrt(np.maximum(var_total, 0.0))

    pe = np.empty_like(mu)
    zero_std = (std == 0)
    if np.any(zero_std):
        pe[zero_std & (c_bit == 1)] = np.where(
            mu[zero_std & (c_bit == 1)] < threshold, 1.0, 0.0)
        pe[zero_std & (c_bit == 0)] = np.where(
            mu[zero_std & (c_bit == 0)] >= threshold, 1.0, 0.0)
    nz = ~zero_std
    if np.any(nz):
        pe[nz & (c_bit == 1)] = 0.5 * erfc(
            (mu[nz & (c_bit == 1)] - threshold) / (std[nz & (c_bit == 1)] * np.sqrt(2)))
        pe[nz & (c_bit == 0)] = 0.5 * erfc(
            (threshold - mu[nz & (c_bit == 0)]) / (std[nz & (c_bit == 0)] * np.sqrt(2)))
    return float(np.mean(pe))


def raw_to_region_labels_np(y_raw):
    y = np.asarray(y_raw).reshape(-1)
    return np.searchsorted(np.asarray(ORDINAL_THRESHOLDS), y, side="right").astype(np.int64)


def ordinal_logits_to_region_labels_torch(logits):
    probs = torch.sigmoid(logits)
    passed = (probs >= 0.5).sum(dim=1)
    return passed.long()


# =========================
# Shared scaling helper
# =========================
def apply_shared_scale(data, mean, std, valid_mask=None):
    scaled = ((data - mean) / std).astype(np.float32)
    if valid_mask is not None and scaled.ndim == 2 and valid_mask.shape == scaled.shape:
        scaled = scaled * valid_mask.astype(np.float32)
    return scaled


def safe_harmonic_z(z0, z1, denom_eps=1e-9):
    """Finite-safe version of 2/(1/z0 + 1/z1) = 2*z0*z1/(z0+z1).

    The direct reciprocal form can create +/-inf when z0 or z1 is near zero.
    This helper uses np.divide(..., where=...) and sanitizes non-finite outputs.
    """
    z0 = np.asarray(z0, dtype=np.float32)
    z1 = np.asarray(z1, dtype=np.float32)
    num = (2.0 * z0 * z1).astype(np.float32)
    den = (z0 + z1).astype(np.float32)
    out = np.divide(
        num,
        den,
        out=np.zeros_like(num, dtype=np.float32),
        where=np.abs(den) > denom_eps,
    ).astype(np.float32)
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


# =========================
# Generate one physical scenario
# =========================
def generate_physical_case(rng, min_mem_len, max_mem_len, arrival_coverage, max_tries=5000):
    for _ in range(max_tries):
        radius = rng.uniform(3.0, 5.0)
        distance = rng.uniform(10.0, 15.0)
        diff = rng.uniform(50.0, 75.0)
        Ts = rng.uniform(0.5, 1.2)
        N = int(10 ** rng.uniform(3.0, 6.0))

        f_inf = radius / (radius + distance)
        target = arrival_coverage * f_inf

        cumsum, k = 0.0, 0
        while k < max_mem_len:
            pk = Fhit_function(radius, distance, diff, (k + 1) * Ts) - Fhit_function(
                radius, distance, diff, k * Ts)
            cumsum += pk
            k += 1
            if cumsum >= target:
                break

        if k < min_mem_len:
            continue

        P_ext = calculate_hitting_probabilities(k + 1, radius, distance, diff, Ts)
        P_main = P_ext[:k]
        P_extra = float(P_ext[k]) if k < len(P_ext) else 0.0

        P_scaled = P_main * N
        variances = N * P_main * (1.0 - P_main)

        return {
            "radius": radius, "distance": distance, "diffusion": diff,
            "Ts": Ts, "N": N, "mem_len": k, "P": P_main,
            "P_scaled": P_scaled, "variances": variances,
            "P_mem_len_extra": P_extra,
            "P_mem_len_extra_var": P_extra * (1.0 - P_extra),
        }

    raise RuntimeError(
        f"Could not generate a physical case with mem_len >= {min_mem_len} "
        f"after {max_tries} tries."
    )


# =========================
# Model (matches v3 training: global_dim=7, global_embed=96, cond_dim=128)
# =========================
class ConditionalFeatureModulation(nn.Module):
    def __init__(self, cond_dim, feat_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, 2 * feat_dim),
        )
        self.scale = nn.Parameter(torch.tensor(0.25, dtype=torch.float32))

    def forward(self, x, cond):
        gamma_beta = self.net(cond)
        gamma, beta = torch.chunk(gamma_beta, 2, dim=-1)
        if x.dim() == 3:
            gamma = gamma.unsqueeze(1)
            beta = beta.unsqueeze(1)
        return x * (1.0 + self.scale * gamma) + self.scale * beta


class SetSelfAttentionBlock(nn.Module):
    def __init__(self, d_model, num_heads, mlp_ratio=4.0, dropout=0.10):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=num_heads,
            dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        hidden = int(d_model * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, d_model), nn.Dropout(dropout))

    def forward(self, x, key_padding_mask=None):
        y = self.norm1(x)
        attn_out, _ = self.attn(y, y, y, need_weights=False, key_padding_mask=key_padding_mask)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x


class SetAttentionPooling(nn.Module):
    def __init__(self, d_model, hidden_dim=128):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(d_model, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, 1))

    def forward(self, x, key_padding_mask=None):
        logits = self.score(x)
        if key_padding_mask is not None:
            logits = logits.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
        weights = torch.softmax(logits, dim=1)
        weights = torch.nan_to_num(weights, nan=0.0)
        return (weights * x).sum(dim=1)


class BERFirstTokenSetTransformerBoundedLogHalfMultiTask(nn.Module):
    def __init__(self, max_past_seq_len, token_dim=4, first_token_dim=4,
                 threshold_dim=1, global_dim=7, d_model=128, num_set_layers=4,
                 num_heads=4, mlp_ratio=4.0, dropout=0.10,
                 num_ordinal_thresholds=NUM_ORDINAL_THRESHOLDS):
        super().__init__()
        self.max_past_seq_len = max_past_seq_len
        self.d_model = d_model
        self.num_ordinal_thresholds = num_ordinal_thresholds

        self.first_proj = nn.Sequential(
            nn.Linear(first_token_dim, d_model), nn.LayerNorm(d_model),
            nn.GELU(), nn.Dropout(dropout))
        self.token_proj = nn.Sequential(
            nn.Linear(token_dim, d_model), nn.LayerNorm(d_model),
            nn.GELU(), nn.Dropout(dropout))
        self.thr_embed = nn.Sequential(
            nn.Linear(threshold_dim, 32), nn.GELU(),
            nn.Linear(32, 32), nn.GELU())
        self.global_embed = nn.Sequential(
            nn.Linear(global_dim, 96), nn.GELU(),
            nn.Linear(96, 96), nn.GELU())

        cond_dim = 32 + 96
        self.first_cond_mod = ConditionalFeatureModulation(cond_dim, d_model, 256)
        self.set_cond_mod = ConditionalFeatureModulation(cond_dim, d_model, 256)

        self.set_blocks = nn.ModuleList([
            SetSelfAttentionBlock(d_model, num_heads, mlp_ratio, dropout)
            for _ in range(num_set_layers)])

        self.first_refine = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_model),
            nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model, d_model))
        self.final_set_norm = nn.LayerNorm(d_model)
        self.set_attn_pool = SetAttentionPooling(d_model, 128)

        set_summary_dim = 3 * d_model
        head_in = d_model + set_summary_dim + cond_dim

        self.reg_head = nn.Sequential(
            nn.Linear(head_in, 256), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.15),
            nn.Linear(128, 1))
        self.ord_head = nn.Sequential(
            nn.Linear(head_in, 256), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(0.15),
            nn.Linear(128, num_ordinal_thresholds))

    @staticmethod
    def raw_to_log10ber(pred_raw_unconstrained):
        log10_half = torch.log10(torch.tensor(
            0.5, device=pred_raw_unconstrained.device,
            dtype=pred_raw_unconstrained.dtype))
        return log10_half - F.softplus(pred_raw_unconstrained)

    @staticmethod
    def raw_to_ber(pred_raw_unconstrained):
        pred_log = BERFirstTokenSetTransformerBoundedLogHalfMultiTask.raw_to_log10ber(
            pred_raw_unconstrained)
        return torch.pow(10.0, pred_log).clamp(min=EPS, max=0.5)

    def forward(self, first_token, past_tokens, global_feats, threshold, key_padding_mask=None):
        thr = self.thr_embed(threshold)
        g = self.global_embed(global_feats)
        cond = torch.cat([thr, g], dim=-1)

        first_x = self.first_proj(first_token)
        first_x = self.first_cond_mod(first_x, cond)
        first_x = first_x + self.first_refine(first_x)

        set_x = self.token_proj(past_tokens)
        set_x = self.set_cond_mod(set_x, cond)
        for block in self.set_blocks:
            set_x = block(set_x, key_padding_mask=key_padding_mask)
        set_x = self.final_set_norm(set_x)

        if key_padding_mask is not None:
            valid_mask = (~key_padding_mask).unsqueeze(-1).float()
        else:
            valid_mask = torch.ones(
                set_x.shape[0], set_x.shape[1], 1,
                device=set_x.device, dtype=set_x.dtype)

        pooled_attn = self.set_attn_pool(set_x, key_padding_mask=key_padding_mask)
        valid_counts = valid_mask.sum(dim=1).clamp(min=1.0)
        pooled_mean = (set_x * valid_mask).sum(dim=1) / valid_counts

        if key_padding_mask is not None:
            x_for_max = set_x.masked_fill(key_padding_mask.unsqueeze(-1), float("-inf"))
        else:
            x_for_max = set_x
        pooled_max = x_for_max.amax(dim=1)
        pooled_max = torch.nan_to_num(pooled_max, nan=0.0, posinf=0.0, neginf=0.0)

        set_summary = torch.cat([pooled_attn, pooled_mean, pooled_max], dim=-1)
        fused = torch.cat([first_x, set_summary, cond], dim=-1)

        pred_raw_unconstrained = self.reg_head(fused)
        ord_logits = self.ord_head(fused)
        return pred_raw_unconstrained, ord_logits


# =========================
# Build inference inputs (7 global features, position-independent scaling)
# =========================
def prepare_inference_features(case, thresholds, scalers, device):
    """Build model inputs for a threshold sweep over a single physical case.

    Returns:
        first_t, past_t, global_t, thr_t, mask_t — all torch tensors on device
        global_t has shape [B, 7]: [z0, z1, hmg, nsid, log_hd, da, herf]
    """
    B = len(thresholds)
    mem_len = case["mem_len"]
    N = float(case["N"])

    P_raw = np.asarray(case["P"], dtype=np.float32)
    var_raw = np.asarray(case["variances"], dtype=np.float32)
    thr_raw = np.asarray(thresholds, dtype=np.float32).reshape(-1, 1)

    # Feature engineering
    taps_feat = (P_raw * N).astype(np.float32)
    vars_feat = var_raw.astype(np.float32)
    abs_feat = np.abs(taps_feat).astype(np.float32)
    snr_feat = np.log10((taps_feat ** 2) / (var_raw + EPS) + EPS).astype(np.float32)

    # Broadcast to [B, mem_len]
    taps_2d = np.broadcast_to(taps_feat[None, :], (B, mem_len)).copy()
    vars_2d = np.broadcast_to(vars_feat[None, :], (B, mem_len)).copy()
    abs_2d = np.broadcast_to(abs_feat[None, :], (B, mem_len)).copy()
    snr_2d = np.broadcast_to(snr_feat[None, :], (B, mem_len)).copy()

    L_past = mem_len - 1
    valid_past = np.ones((B, L_past), dtype=bool)

    # --- Threshold-dependent global features ---
    first_mean = taps_2d[:, 0:1]
    past_means = taps_2d[:, 1:]
    first_var = vars_2d[:, 0:1]
    past_vars = vars_2d[:, 1:]

    mu0 = (0.5 * past_means.sum(axis=1, keepdims=True)).astype(np.float32)
    mu1 = (first_mean + mu0).astype(np.float32)
    var0 = (0.5 * past_vars.sum(axis=1, keepdims=True)).astype(np.float32)
    var1 = (first_var + var0).astype(np.float32)
    std0 = np.sqrt(np.maximum(var0, EPS)).astype(np.float32)
    std1 = np.sqrt(np.maximum(var1, EPS)).astype(np.float32)

    z0 = ((thr_raw - mu0) / (std0 + EPS)).astype(np.float32)
    z1 = ((mu1 - thr_raw) / (std1 + EPS)).astype(np.float32)
    harmonic = safe_harmonic_z(z0, z1)
    abs_diff = np.abs(z0 - z1).astype(np.float32)
    hmg = (harmonic - 0.25 * abs_diff).astype(np.float32)
    hmg = np.nan_to_num(hmg, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

    # --- Scenario-level features (threshold-independent) ---
    signal = first_mean  # [B, 1]
    isi = past_means.sum(axis=1, keepdims=True)
    nsid = ((signal - isi) / (signal + isi + EPS)).astype(np.float32)

    gap = signal
    d0_inf = (gap / (std0 + EPS)).astype(np.float32)
    d1_inf = (gap / (std1 + EPS)).astype(np.float32)
    hd = (2.0 * d0_inf * d1_inf / (d0_inf + d1_inf + EPS)).astype(np.float32)
    log_hd = np.log10(hd + EPS).astype(np.float32)
    da = (np.abs(d0_inf - d1_inf) / (d0_inf + d1_inf + EPS)).astype(np.float32)

    past_taps_sum = past_means.sum(axis=1, keepdims=True)
    past_shares = past_means / (past_taps_sum + EPS)
    herf = (past_shares ** 2).sum(axis=1, keepdims=True).astype(np.float32)

    # --- Scale first token ---
    ft_tap = ((taps_2d[:, 0] - scalers["first_tap_mean"]) / scalers["first_tap_std"]).astype(np.float32)
    ft_var = ((vars_2d[:, 0] - scalers["first_var_mean"]) / scalers["first_var_std"]).astype(np.float32)
    ft_abs = ((abs_2d[:, 0] - scalers["first_abs_mean"]) / scalers["first_abs_std"]).astype(np.float32)
    ft_snr = ((snr_2d[:, 0] - scalers["first_snr_mean"]) / scalers["first_snr_std"]).astype(np.float32)
    first_token = np.stack([ft_tap, ft_var, ft_abs, ft_snr], axis=1).astype(np.float32)

    # --- Scale past tokens ---
    pt_tap = apply_shared_scale(taps_2d[:, 1:], scalers["past_tap_mean"], scalers["past_tap_std"], valid_past)
    pt_var = apply_shared_scale(vars_2d[:, 1:], scalers["past_var_mean"], scalers["past_var_std"], valid_past)
    pt_abs = apply_shared_scale(abs_2d[:, 1:], scalers["past_abs_mean"], scalers["past_abs_std"], valid_past)
    pt_snr = apply_shared_scale(snr_2d[:, 1:], scalers["past_snr_mean"], scalers["past_snr_std"], valid_past)
    past_tokens = np.stack([pt_tap, pt_var, pt_abs, pt_snr], axis=2).astype(np.float32)

    # --- Scale globals (7 features) ---
    z0_s = scalers["z0_scaler"].transform(z0).astype(np.float32)
    z1_s = scalers["z1_scaler"].transform(z1).astype(np.float32)
    hmg_s = scalers["harmonic_minus_gap_scaler"].transform(hmg).astype(np.float32)
    nsid_s = scalers["nsid_scaler"].transform(nsid).astype(np.float32)
    log_hd_s = scalers["log_hd_scaler"].transform(log_hd).astype(np.float32)
    da_s = scalers["da_scaler"].transform(da).astype(np.float32)
    herf_s = scalers["herf_scaler"].transform(herf).astype(np.float32)
    global_feats = np.concatenate(
        [z0_s, z1_s, hmg_s, nsid_s, log_hd_s, da_s, herf_s], axis=1
    ).astype(np.float32)

    # --- Scale threshold ---
    thr_log = np.log10(thr_raw + EPS).astype(np.float32)
    thr_s = scalers["thr_scaler"].transform(thr_log).astype(np.float32)

    # --- Padding mask (no padding needed here) ---
    pad_mask = np.zeros((B, L_past), dtype=bool)

    first_t = torch.from_numpy(first_token).to(device)
    past_t = torch.from_numpy(past_tokens).to(device)
    global_t = torch.from_numpy(global_feats).to(device)
    thr_t = torch.from_numpy(thr_s).to(device)
    mask_t = torch.from_numpy(pad_mask).to(device)

    return first_t, past_t, global_t, thr_t, mask_t


# =========================
# Prediction helpers
# =========================
def predict_ber_for_thresholds(model, case, thresholds, scalers, device):
    """Predict BER for a threshold vector.

    This recomputes threshold-dependent engineered features, so it is safe to call
    for shifted threshold vectors such as thr-eps and thr+eps.
    """
    first_t, past_t, global_t, thr_t, mask_t = prepare_inference_features(
        case, thresholds, scalers, device,
    )

    with torch.no_grad():
        pred_raw_out, ord_logits = model(
            first_t, past_t, global_t, thr_t, key_padding_mask=mask_t,
        )
        pred_bers = model.raw_to_ber(pred_raw_out).cpu().numpy().reshape(-1)
        pred_log = model.raw_to_log10ber(pred_raw_out).cpu().numpy().reshape(-1)
        pred_regions = ordinal_logits_to_region_labels_torch(ord_logits).cpu().numpy().reshape(-1)
        pred_region_probs = torch.sigmoid(ord_logits).cpu().numpy()

    pred_bers = np.clip(pred_bers, EPS, 0.5)
    pred_log = np.log10(np.clip(pred_bers, EPS, 0.5)).astype(np.float32)
    return pred_bers, pred_log, pred_regions, pred_region_probs


def predict_ber_smoothed_3point(
    model,
    case,
    thresholds,
    scalers,
    device,
    eps_frac=SMOOTHING_EPS_FRAC,
    smoothing_space=SMOOTHING_SPACE,
    weights=SMOOTHING_WEIGHTS,
):
    """Three-point test-time smoothing over threshold.

    Predicts at [thr-eps, thr, thr+eps] and averages either in log-BER space or
    raw-BER space. Crucially, this recomputes global features for each shifted
    threshold, rather than only perturbing the already-scaled threshold tensor.
    """
    thresholds = np.asarray(thresholds, dtype=np.float32)
    w_minus, w_mid, w_plus = weights
    weight_sum = float(w_minus + w_mid + w_plus)
    w_minus, w_mid, w_plus = w_minus / weight_sum, w_mid / weight_sum, w_plus / weight_sum

    thr_span = float(np.max(thresholds) - np.min(thresholds))
    eps_thr = float(eps_frac * max(thr_span, EPS))

    thr_minus = np.clip(thresholds - eps_thr, 0.0, None).astype(np.float32)
    thr_mid = thresholds.astype(np.float32)
    thr_plus = (thresholds + eps_thr).astype(np.float32)

    ber_minus, log_minus, _, _ = predict_ber_for_thresholds(
        model, case, thr_minus, scalers, device,
    )
    ber_mid, log_mid, regions_mid, probs_mid = predict_ber_for_thresholds(
        model, case, thr_mid, scalers, device,
    )
    ber_plus, log_plus, _, _ = predict_ber_for_thresholds(
        model, case, thr_plus, scalers, device,
    )

    if smoothing_space.lower() == "raw":
        ber_smooth = (
            w_minus * ber_minus +
            w_mid * ber_mid +
            w_plus * ber_plus
        )
        ber_smooth = np.clip(ber_smooth, EPS, 0.5).astype(np.float32)
        log_smooth = np.log10(ber_smooth).astype(np.float32)
    elif smoothing_space.lower() == "log":
        log_smooth = (
            w_minus * log_minus +
            w_mid * log_mid +
            w_plus * log_plus
        ).astype(np.float32)
        ber_smooth = np.clip(10.0 ** log_smooth, EPS, 0.5).astype(np.float32)
    else:
        raise ValueError("smoothing_space must be either 'log' or 'raw'.")

    return {
        "ber": ber_smooth,
        "log10_ber": log_smooth,
        "regions": regions_mid,
        "region_probs": probs_mid,
        "raw_mid_ber": ber_mid,
        "raw_mid_log10_ber": log_mid,
        "eps_threshold": eps_thr,
        "smoothing_space": smoothing_space,
    }


# =========================
# Generate scenario
# =========================
rng = np.random.default_rng(RANDOM_SEED)
case = generate_physical_case(
    rng,
    min_mem_len=PHYSICS_MIN_MEM_LEN,
    max_mem_len=PHYSICS_MAX_MEM_LEN,
    arrival_coverage=ARRIVAL_COVERAGE,
)

print("Generated physical scenario")
print("radius    =", case["radius"])
print("distance  =", case["distance"])
print("diffusion =", case["diffusion"])
print("Ts        =", case["Ts"])
print("N         =", case["N"])
print("mem_len   =", case["mem_len"])
print("P         =", case["P"])
print("P_scaled  =", case["P_scaled"])
print("variances =", case["variances"])

# =========================
# Threshold sweep — ground truth
# =========================
thr_min = 0.0
thr_max = float(np.sum(case["P_scaled"]))
thresholds = np.linspace(thr_min, thr_max, N_THRESHOLDS)

print("Threshold search interval:", thr_min, "to", thr_max)
print("sum(P_scaled) =", np.sum(case["P_scaled"]))

real_bers = np.array([
    calculate_ber_vectorized(
        mem_len=case["mem_len"],
        threshold=thr,
        P_scaled=case["P_scaled"],
        variances=case["variances"],
    )
    for thr in thresholds
])

real_bers = np.clip(real_bers, EPS, 0.5)
real_regions = raw_to_region_labels_np(real_bers)

# =========================
# Load model + scalers
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scalers = joblib.load(SCALER_PATH)

max_past_seq_len = scalers.get("train_max_past_seq_len", scalers.get("max_past_seq_len", case["mem_len"] - 1))
ordinal_thresholds = scalers.get("ordinal_thresholds", ORDINAL_THRESHOLDS)
num_ordinal = len(ordinal_thresholds)
global_dim = scalers.get("global_dim", 7)

model = BERFirstTokenSetTransformerBoundedLogHalfMultiTask(
    max_past_seq_len=max_past_seq_len,
    token_dim=scalers.get("past_token_dim", 4),
    first_token_dim=scalers.get("first_token_dim", 4),
    threshold_dim=1,
    global_dim=global_dim,
    d_model=128,
    num_set_layers=4,
    num_heads=4,
    mlp_ratio=4.0,
    dropout=0.10,
    num_ordinal_thresholds=num_ordinal,
).to(device)

state = torch.load(MODEL_PATH, map_location=device, weights_only=True)
model.load_state_dict(state)
model.eval()

print(f"\nModel loaded: max_past_seq_len={max_past_seq_len}, "
      f"global_dim={global_dim}, num_ordinal={num_ordinal}")
print(f"Scaling strategy: {scalers.get('scaling_strategy', 'unknown')}")

# =========================
# Predict across thresholds
# =========================
raw_bers, raw_log, raw_regions, raw_region_probs = predict_ber_for_thresholds(
    model, case, thresholds, scalers, device,
)

if USE_TEST_TIME_SMOOTHING:
    smooth = predict_ber_smoothed_3point(
        model,
        case,
        thresholds,
        scalers,
        device,
        eps_frac=SMOOTHING_EPS_FRAC,
        smoothing_space=SMOOTHING_SPACE,
        weights=SMOOTHING_WEIGHTS,
    )
    pred_bers = smooth["ber"]
    pred_log = smooth["log10_ber"]
    pred_regions = smooth["regions"]
    pred_region_probs = smooth["region_probs"]
    print(
        f"Test-time smoothing: ON | space={smooth['smoothing_space']} | "
        f"eps_threshold={smooth['eps_threshold']:.6g}"
    )
else:
    pred_bers = raw_bers
    pred_log = raw_log
    pred_regions = raw_regions
    pred_region_probs = raw_region_probs
    print("Test-time smoothing: OFF")

pred_bers = np.clip(pred_bers, EPS, 0.5)
raw_bers = np.clip(raw_bers, EPS, 0.5)

# =========================
# Comparison table
# =========================
results = pd.DataFrame({
    "threshold": thresholds,
    "real_BER": real_bers,
    "estimated_BER": pred_bers,
    "estimated_BER_raw_unsmoothed": raw_bers,
    "smoothing_delta": pred_bers - raw_bers,
    "real_region": real_regions,
    "predicted_region": pred_regions,
    "abs_error": np.abs(pred_bers - real_bers),
    "abs_log10_error": np.abs(
        np.log10(np.clip(pred_bers, EPS, 0.5)) -
        np.log10(np.clip(real_bers, EPS, 0.5))
    ),
    "region_abs_error": np.abs(pred_regions.astype(np.int64) - real_regions.astype(np.int64)),
})

print(results.head(15))

print("\nSummary")
print("Mean abs raw error        :", results["abs_error"].mean())
print("Mean abs log10 error      :", results["abs_log10_error"].mean())
print("Max  abs log10 error      :", results["abs_log10_error"].max())
print("Mean region abs error     :", results["region_abs_error"].mean())
print("Exact region accuracy     :", np.mean(results["real_region"] == results["predicted_region"]))
if USE_TEST_TIME_SMOOTHING:
    print("Mean abs smoothing delta  :", float(np.mean(np.abs(results["smoothing_delta"]))))
    print("Max  abs smoothing delta  :", float(np.max(np.abs(results["smoothing_delta"]))))

best_real_idx = np.argmin(real_bers)
best_est_idx = np.argmin(pred_bers)

print("\nBest threshold from real BER      :", thresholds[best_real_idx])
print("Minimum real BER                  :", real_bers[best_real_idx])
print("Real BER region there             :", REGION_LABELS.get(int(real_regions[best_real_idx]), "?"))

print("Best threshold from estimated BER :", thresholds[best_est_idx])
print("Estimated BER at that threshold   :", pred_bers[best_est_idx])
print("Predicted BER region there        :", REGION_LABELS.get(int(pred_regions[best_est_idx]), "?"))

# =========================
# Plot 1: log-scale BER
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["estimated_BER"], label="Estimated BER" + (" smoothed" if USE_TEST_TIME_SMOOTHING else ""))
if USE_TEST_TIME_SMOOTHING:
    plt.plot(results["threshold"], results["estimated_BER_raw_unsmoothed"], label="Estimated BER raw", alpha=0.45)
plt.plot(results["threshold"], results["real_BER"], label="Real BER")
plt.yscale("log")
plt.xlabel("Threshold")
plt.ylabel("BER")
plt.title(f"Threshold vs BER (log scale) — mem_len={case['mem_len']}")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "01_ber_log_scale.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/01_ber_log_scale.png")

# =========================
# Plot 2: linear-scale BER
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["estimated_BER"], label="Estimated BER" + (" smoothed" if USE_TEST_TIME_SMOOTHING else ""))
if USE_TEST_TIME_SMOOTHING:
    plt.plot(results["threshold"], results["estimated_BER_raw_unsmoothed"], label="Estimated BER raw", alpha=0.45)
plt.plot(results["threshold"], results["real_BER"], label="Real BER")
plt.yscale("linear")
plt.xlabel("Threshold")
plt.ylabel("BER")
plt.title(f"Threshold vs BER (linear scale) — mem_len={case['mem_len']}")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "02_ber_linear_scale.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/02_ber_linear_scale.png")

# =========================
# Plot 3: predicted vs true region
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["predicted_region"], label="Predicted region")
plt.plot(results["threshold"], results["real_region"], label="True region")
plt.xlabel("Threshold")
plt.ylabel("BER Region Class")
plt.title(f"Threshold vs BER Region — mem_len={case['mem_len']}")
plt.yticks(list(REGION_LABELS.keys()),
           [REGION_LABELS[k] for k in sorted(REGION_LABELS.keys())],
           fontsize=7)
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "03_region_comparison.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/03_region_comparison.png")

# =========================
# Plot 4: ordinal threshold probabilities
# =========================
plt.figure(figsize=(12, 7))
for i, thr_val in enumerate(ordinal_thresholds):
    plt.plot(thresholds, pred_region_probs[:, i], label=f"P(y >= {thr_val:g})")
plt.xlabel("Threshold")
plt.ylabel("Ordinal Probability")
plt.title(f"Ordinal Head Outputs Across Threshold Sweep — mem_len={case['mem_len']}")
plt.legend(fontsize=7, ncol=2)
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "04_ordinal_probabilities.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/04_ordinal_probabilities.png")

# =========================
# Plot 5: absolute error by threshold
# =========================
plt.figure(figsize=(10, 6))
plt.plot(results["threshold"], results["abs_error"], label="Abs raw error", alpha=0.8)
plt.plot(results["threshold"], results["abs_log10_error"], label="Abs log10 error", alpha=0.8)
plt.yscale("log")
plt.xlabel("Threshold")
plt.ylabel("Error")
plt.title(f"Prediction Error Across Threshold Sweep — mem_len={case['mem_len']}")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "05_error_by_threshold.png"), dpi=150)
plt.close()
print(f"Saved: {PLOT_DIR}/05_error_by_threshold.png")

# =========================
# Plot 6: smoothing delta
# =========================
if USE_TEST_TIME_SMOOTHING:
    plt.figure(figsize=(10, 6))
    plt.plot(results["threshold"], results["smoothing_delta"], label="Smoothed - raw estimate")
    plt.axhline(0.0, linewidth=1.0)
    plt.xlabel("Threshold")
    plt.ylabel("BER delta")
    plt.title(f"Test-Time Smoothing Delta — mem_len={case['mem_len']}")
    plt.legend()
    plt.grid(True, which="both", alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, "06_smoothing_delta.png"), dpi=150)
    plt.close()
    print(f"Saved: {PLOT_DIR}/06_smoothing_delta.png")


Generated physical scenario
radius    = 4.999903475776163
distance  = 13.063595613360427
diffusion = 63.15707394395149
Ts        = 0.5834416162676823
N         = 2026
mem_len   = 14
P         = [0.03545102 0.042582   0.02704795 0.01857409 0.01368091 0.01059455
 0.00850978 0.00702648 0.00592783 0.00508774 0.00442855 0.00390015
 0.00346895 0.00311165]
P_scaled  = [71.82377081 86.27113641 54.79914164 37.63111579 27.71752004 21.46456524
 17.24082032 14.23564842 12.00979138 10.30776824  8.9722469   7.90171201
  7.0280834   6.304198  ]
variances = [69.27754472 82.59753869 53.31693734 36.93215188 27.33831919 21.23715776
 17.09410468 14.13562193 11.93859933 10.25532496  8.93251283  7.87089412
  7.00370336  6.28458156]
Threshold search interval: 0.0 to 383.7075186062634
sum(P_scaled) = 383.7075186062634

Model loaded: max_past_seq_len=14, global_dim=7, num_ordinal=13
Scaling strategy: position_independent
Test-time smoothing: ON | space=log | eps_threshold=1.91854
    threshold  real_BER  estim